# T1 — traduzione con contesto | QLoRA napoletano, stadi A / A2 / B

Questo notebook e' la parte **T1** del notebook unico `fine-tuning-test7`.
Gli altri due sono `fine-tuning-T2.ipynb` e `fine-tuning-T3.ipynb`.
La cella successiva e' l'introduzione originale, lasciata intatta.

Un notebook = **un modello x un task**: `MODEL` si sceglie al Passo 17.

## Come sono divise le tre parti

| sezione | dove sta |
|---|---|
| ambiente, split, `common.py` e gli altri moduli | in tutti e tre (sono solo `%%writefile`, costo zero) |
| lessico, dati A2, stadio A, stadio A2 | in tutti e tre, ma si eseguono **una volta sola per modello** (vedi sotto) |
| SFT del task e la sua valutazione | solo nel notebook del task |
| `pipeline_due_stadi.py` (composizione via adapter T1) | T2 e T3 |
| best-of-n / reranking (`rerank_t3.py`) | T2 e T3 (per T1 la traduzione ha un modo: greedy) |
| igiene dello split | T2 e T3 |
| sweep dello scalare LoRA, DPO, arm conservativi | T3 |
| run multi-task (MT) | addestrato in T3, valutabile su T1 e T2 dai rispettivi notebook |

## L'ordine in cui conviene eseguirli

Gli stadi A e A2 sono **condivisi**: un run per modello, non uno per task.
Rieseguirli in tutti e tre i notebook costa 3x il tempo GPU senza cambiare
niente. E T2/T3 hanno bisogno dell'adapter di T1 per la pipeline a due stadi.
Quindi:

1. **`fine-tuning-T1.ipynb`** per intero. Produce l'adapter dello stadio A2 e
   l'adapter T1. *Save Version > Save & Run All* alla fine.
2. In **T2** e **T3**: allega l'output di T1 come Dataset e compila al Passo 17
   `ADAPTER_A2_ESTERNO` (e `ADAPTER_T1_ESTERNO`). I Passi 22 e 23 si saltano da
   soli e il notebook parte direttamente dallo stadio B.

Se preferisci l'indipendenza totale, lascia le due variabili vuote: ogni
notebook rifa' A e A2 da capo. E' corretto, costa ~2-3 h di GPU in piu' ciascuno.

## Un'avvertenza sulle tabelle riassuntive

`metriche_finali.py` raccoglie **tutti** i `.preds.jsonl` presenti in
`/kaggle/working/eval`. In un notebook per task quella cartella contiene solo
gli arm di questo task: le tabelle sono corrette ma parziali per costruzione.
Per il confronto fra task, allega gli output dei tre notebook come Dataset e
lancia `metriche_finali.py` una volta sola su una cartella che li contiene tutti.

# Fine-tuning QLoRA napoletano — stadi A / A2 / B su T1, T2, T3

Un notebook = **un modello**. Duplicalo tre volte e cambia solo `MODEL` nella
cella 12.

## Perche' questa versione

La versione precedente addestrava i tre task direttamente sul modello di base (o
sul solo stadio A). Il risultato misurato: **i modelli imparavano il compito, non
il dialetto**. La causa e' nei dati, ed e' quantificabile su `dataset_finale.csv`
(2.568 coppie):

| | |
|---|---|
| righe con `italiano` identico a `napoletano` | 588 (22,9%) |
| righe senza nemmeno una parola dialettale | 714 (27,8%) |
| ... di cui fonte `gemma4` | 60,6% delle sue 515 righe |
| ... di cui fonte `golden` | 19,6% delle sue 2.053 righe |
| tipi dialettali totali | 1.291 (48% dei token napoletani) |
| tipi che coprono il 57% della massa dialettale | **50** |

Per un quarto del corpus la loss e' minimizzata dalla strategia "ricopia
l'input". E il segnale dialettale, dove c'e', e' concentrato su poche decine di
forme (`'o`, `'e`, `'a`, `ca`, `nun`, `pecche'`, `pe'`, `accussi'`, `cchiu'`,
`quanno`) piu' una coda di 730 hapax.

Da qui le due aggiunte:

1. **Stadio A2 — iniezione lessicale.** Un anello fra il continued pretraining
   (che sposta la distribuzione ma non insegna nessuna corrispondenza) e l'SFT
   sui task (che la insegna a livello di frase, dove e' diluita su decine di
   token). Item corti: glossa, cloze in contesto, turni brevi ad alta densita'
   dialettale. Il cloze porta il contesto conversazionale, quindi allena anche
   il condizionamento pragmatico richiesto da T2/T3.
2. **Pesatura della loss** sui token delle forme dialettali durante l'SFT, con
   attribuzione via `offset_mapping` del tokenizer fast.

Piu' un lessico allineato italiano->napoletano estratto dai dati con IBM Model 1
(nessuna voce scritta a mano), che serve a costruire A2, a pesare la loss e a
misurare i risultati.

## Quanti addestramenti

**Per modello: 5.** Stadio A (una volta), stadio A2 (una volta), poi i tre task
che partono tutti dallo stesso adapter A2.

| stadio | run per modello | riusato da |
|---|---|---|
| A — continued pretraining | 1 | tutti e tre i task |
| A2 — iniezione lessicale | 1 | tutti e tre i task |
| B — SFT sui task | 3 | — |

**Su tre modelli: 15 run.** Non 27: A e A2 non si ripetono per task.

I run gia' fatti non si buttano e non si rifanno: Llama T1/T2/T3 in versione
"solo stadio B" diventano l'**arm di ablation sul curriculum** (senza CPT, senza
iniezione lessicale, senza pesatura), con i numeri gia' in cassa. Quindi il
disegno finale ha 15 run nuovi + 3 arm di confronto a costo zero.

Ordine di grandezza sul tempo GPU (P100, misurato sui run Llama precedenti:
8,2 h per i tre task in bf16 emulato, atteso meno in fp16 nativo):

| | per modello |
|---|---|
| A | ~10 min |
| A2 | ~1,5-2,5 h (3.646 item, 3 epoche) |
| B (tre task) | ~4-6 h |
| **totale** | **~6-9 h** |

Con la quota Kaggle di 30 h GPU/settimana ci sta un modello per sessione e i tre
modelli in una settimana, ma **non tutto in una sessione**: il tetto e' 12 h.
Salva la versione dopo A2 e riparti da li' per i tre task.

## Novita' di questa versione (v2): il fine turno di T3

Due patch a `evaluate_task.py` e un file nuovo, `rerank_t3.py`. Il punto di
partenza e' `controllo_solo_lunghezza = 0,776` su `accuracy_umano_vs_macchina =
0,80` nel run `minerva-7b-instruct-v1.0__T3__ft__nucleus`: il 97% del potere
discriminante del classificatore avversariale veniva dalla LUNGHEZZA, non dal
napoletano. Causa: `generate()` non aveva il token di fine turno del chat
template fra gli `eos_token_id`, quindi non si fermava mai e arrivava sempre a
`max_new_tokens`; e `repetition_penalty`/`no_repeat_ngram_size` non arrivavano
dal `TaskConfig` alla valutazione. Vedi il **Passo 30**.

Il **Passo 31** attacca invece il merito, sempre senza dati nuovi: sweep dello
scalare LoRA, igiene dei target, e DPO sui distrattori appaiati per lunghezza
(`distrattori_per_lunghezza()` usata per ADDESTRARE e non solo per misurare).

I numeri di T3 prodotti dalla versione precedente non sono confrontabili con
questi: rilancia il Passo 30c prima di scrivere qualsiasi tabella.

## Prima di eseguire, in Settings

| Impostazione | Valore |
|---|---|
| Accelerator | **GPU P100** (o T4 x2) |
| Internet | **On** — serve per scaricare i checkpoint da HuggingFace |
| Secrets | `HF_TOKEN` allegato (Llama-2 e Gemma-3 sono gated; Minerva no) |
| Data | **due** dataset: lo split rigenerato **e** `dataset_finale.csv` |

Il CSV e' una novita' rispetto a prima: `lessico.py` e `dati_lessicali.py`
lavorano sulle coppie allineate, che nei JSON dello split non ci sono in forma
diretta.

## Sequenza dei passi

```
 1  ambiente e GPU
 2  cartella di lavoro
 3  trova lo split e verifica che sia quello aggiornato
 4  trova il CSV e copia lo split in working        <- NUOVO (serve scrivibile)
 5  token HuggingFace
 6  scrive common.py                                 (con le patch A2 + pesatura)
 7  scrive pretrain_dialect.py
 8  scrive lessico.py                               <- NUOVO
 9  scrive dati_lessicali.py                        <- NUOVO
10  scrive pesi_lessicali.py                        <- NUOVO
11  scrive finetune_a2_lessico.py                   <- NUOVO
12  scrive finetune_t1_traduzione.py
13  scrive finetune_t2_completamento.py
14  scrive finetune_t3_replica.py
15  scrive evaluate_task.py
16  verifica i file e le config per task
17  scegli il MODELLO di questo notebook
18  smoke test su un modello piccolo
19  estrai il lessico dal SOLO train                <- NUOVO
20  costruisci i dati dello stadio A2               <- NUOVO
21  autotest delle metriche lessicali               <- NUOVO
22  STADIO A   — continued pretraining        (1 run)
23  STADIO A2  — iniezione lessicale          (1 run)  <- NUOVO
24  STADIO B   — T1                           (1 run)
25  STADIO B   — T2                           (1 run)
26  STADIO B   — T3                           (1 run)
27+ valutazione, riepilogo, salvataggio
28  metriche_finali.py: CSV delle generazioni + BERTScore
29  pipeline_due_stadi.py: T2 e T3 come composizione
30  T3/T2: fine turno corretto + best-of-n            <- NUOVO
31  migliorare l'addestramento senza dati nuovi       <- NUOVO
```

I titoli delle celle sono numerati per **passo logico**, nell'ordine in cui vanno
eseguiti dall'alto verso il basso. Ogni titolo ha una riga **Cosa fare** con
l'azione richiesta e, dove serve, cosa controllare nell'output prima di andare
avanti.

## Attenzione allo split

Serve ancora lo split prodotto dalla versione **aggiornata** di
`split_dataset.py` (T2 con contesto conversazionale, T3 con soglia separata sul
turno di contesto). La cella 3 lo verifica e si ferma se trova la vecchia.


## Correzioni applicate a questa versione

**1. `[generazione fallita: RuntimeError]` durante il training (stadio A2 e B).**
Il messaggio completo nei log era `Tensors must have same number of dimensions:
got 2 and 3`: `model.generate()` e' rotta su `Gemma3ForConditionalGeneration`
nella transformers dell'immagine Kaggle (in `_sample`, `next_tokens` esce con una
dimensione di troppo). `evaluate_task.py` aggirava gia' il bug con un loop di
decoding manuale, ma `common.py` no: i due callback di generazione chiamavano
`generate()` diretta. Effetto: `eval_gen_chrf` non veniva **mai** calcolata —
cioe' l'unica metrica che vede il fallimento sull'EOS — e la generazione di
controllo stampava un segnaposto. Ora `common.py` ha `genera_una()`, che su
Gemma-3 usa il loop manuale (batch 1, cache KV, ripiego senza cache) e sugli
altri modelli `generate()` con fallback automatico e traceback stampato una volta
sola. Una generazione che fallisce non sparisce piu' dietro un segnaposto.

**2. Id di fine turno mancanti nel monitoraggio.** Le due generazioni durante il
training non passavano `eos_token_id`, quindi arrivavano sempre a
`max_new_tokens` e l'avviso "non sta imparando a emettere EOS" era un artefatto
del decoding. Ora `run()` calcola `id_fine_turno()` sui target renderizzati di
train — stessa logica di `evaluate_task.py`, quindi i due chrF sono confrontabili.

**3. Gradient checkpointing nel callback degli esempi.** Restava attivo durante la
generazione: transformers forza `use_cache=False` e ogni token ricalcolava il
forward sull'intera sequenza. Era il motivo per cui il run sembrava bloccato dopo
ogni eval. Il callback della metrica lo disattivava gia', questo no.

**4. Smoke test: 403 su repo gated.** Girava su `minerva-small`
(`sapienzanlp/Minerva-3B-base-v1.0`), gated e non accessibile da questo account.
Ora `gemma-tiny`. In piu', `run()` intercetta il `GatedRepoError` e stampa cosa
fare, invece di tre schermate di traceback di `huggingface_hub`.

**5. Stesso crash negli script a valle.** `diagnosi_cecita.py` (ablazione del
contesto), `rerank_t3.py` (candidati best-of-n) e `prova_esempi.py` generavano
ancora con `generate()` batched: su Gemma sarebbero crollati allo stesso modo piu'
avanti in questo notebook. Ora passano dal decoder per-item. Nota metodologica:
il loop manuale non implementa *typical sampling*, quindi su Gemma `rerank_t3.py`
usa nucleus e lo dichiara a schermo — va riportato nel confronto fra modelli.

**6. Uscite di test tutte vuote, chrF++ 0.00 (la causa a monte).** La versione
precedente aveva messo una guardia nel decoder manuale di Gemma-3 - l'EOS resta
vietato finche' l'uscita e' vuota - ma non aveva toccato il punto in cui il
problema nasce: `id_fine_turno()` aggiunge agli eos *l'ultimo token del target
renderizzato*, e su Gemma-3 quel token e' il **107**, cioe' `'\n'`, perche' il
chat template scrive `<end_of_turn>` seguito da un a capo (verificato in
`tokenizer.json`: 106 = `<end_of_turn>`, 107 = `'\n'`). Con un a capo fra gli
eos, un modello che lo emette come primo token chiude subito il turno e
restituisce la stringa vuota: su T1 e' successo su **267/267** item, con chrF++
0.00, nessun errore e nessun avviso. Ora la chiusura del turno viene cercata a
monte della spaziatura (si ottiene il 106) e i token di soli spazi vengono
scartati dalla lista, in `common.py` e in `evaluate_task.py`. La correzione non
riguarda solo Gemma: il ramo `model.generate()` usato dagli altri due modelli
non ha la guardia del decoder manuale, quindi con un `'\n'` fra gli eos produce
le stesse uscite vuote - su Llama T1 la lunghezza mediana delle generazioni era
appunto 0.

**Cosa va rieseguito.** Non il training: gli adapter A, A2 e T1 sono validi (in
teacher forcing chrF 77.35, e la contrastiva riconosce la traduzione giusta fra
cinque candidati nel 100% dei casi, quindi il modello non e' compromesso). Basta
rilanciare la **valutazione su test** e `metriche_finali.py` - le celle della
sezione "Correzione 2" in fondo - dopo aver spostato la cartella `eval/` vecchia,
perche' `metriche_finali.py` globba tutti i `*.preds.jsonl` che vi trova.

**Cosa NON e' stato toccato:** il registry dei modelli, gli iperparametri, gli
split e la logica di selezione del checkpoint. Gli adapter A e A2 gia' addestrati
restano validi: la classe di caricamento del modello non cambia.


In [1]:
# --- 1. Ambiente -------------------------------------------------------------
!pip install -q peft bitsandbytes sacrebleu rouge_score scikit-learn scipy

import torch, importlib
for m in ("torch", "transformers", "peft", "bitsandbytes", "accelerate"):
    try:
        print(f"{m:14s}", importlib.import_module(m).__version__)
    except Exception as e:
        print(f"{m:14s} NON DISPONIBILE ({e})")

if not torch.cuda.is_available():
    raise SystemExit("Nessuna GPU. Settings > Accelerator > GPU P100 (o T4 x2).")
cc = torch.cuda.get_device_capability(0)
print("\nGPU:", torch.cuda.get_device_name(0), f"(cc {cc[0]}.{cc[1]})",
      f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("bf16 nativo (serve cc >= 8):", cc[0] >= 8,
      "-> precisione:", "bf16" if cc[0] >= 8 else "fp16")
print("(torch.cuda.is_bf16_supported() direbbe:", torch.cuda.is_bf16_supported(),
      "- include l'emulazione, per questo non lo usiamo)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 8.1 MB/s eta 0:00:00
torch          2.10.0+cu128
transformers   5.0.0
peft           0.19.1
bitsandbytes   0.50.2
accelerate     1.13.0

GPU: Tesla T4 (cc 7.5) | 15.6 GB
bf16 nativo (serve cc >= 8): False -> precisione: fp16
(torch.cuda.is_bf16_supported() direbbe: True - include l'emulazione, per questo non lo usiamo)


In [2]:
# --- 2. Cartella di lavoro ---------------------------------------------------
import os, sys, pathlib
pathlib.Path("/kaggle/working/training").mkdir(parents=True, exist_ok=True)
os.chdir("/kaggle/working/training")
sys.path.insert(0, "/kaggle/working/training")     # perché `from common import ...` funzioni
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # una GPU sola anche su T4 x2
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("cwd:", os.getcwd())

cwd: /kaggle/working/training


In [3]:
# --- 3. Trova lo split e VERIFICA che sia la versione aggiornata -------------
import glob, os, json

LAYOUTS = ("layout1_traduzione_con_contesto", "layout2_completamento_turno",
           "layout3_replica_conversazionale")

candidati = sorted({os.path.dirname(p) for p in
                    glob.glob("/kaggle/input/**/" + LAYOUTS[0], recursive=True)})
if not candidati:
    print("Contenuto di /kaggle/input:")
    for p in sorted(glob.glob("/kaggle/input/*/*"))[:40]:
        print("  ", p)
    raise SystemExit("Cartella split non trovata: allega il dataset dello split al notebook.")

SPLIT = candidati[-1]      # l'ultimo in ordine alfabetico: di solito il piu' recente
print("SPLIT =", SPLIT)
if len(candidati) > 1:
    print("! più candidati:", candidati, "\n  se non è quello giusto, imposta SPLIT a mano")

for d in LAYOUTS:
    for s in ("train", "dev", "test"):
        p = os.path.join(SPLIT, d, f"{s}.json")
        n = len(json.load(open(p, encoding="utf-8"))) if os.path.exists(p) else 0
        print(f"  {'OK   ' if n else 'MANCA'} {d}/{s}.json  ({n} istanze)")

# --- guardia: T2 DEVE avere il contesto conversazionale ---------------------
t2 = json.load(open(os.path.join(SPLIT, LAYOUTS[1], "train.json"), encoding="utf-8"))
con_ctx = sum(1 for r in t2 if "Conversazione finora" in r["prompt"])
print(f"\nT2: {con_ctx}/{len(t2)} prompt con contesto conversazionale")
if con_ctx < 0.5 * len(t2):
    raise SystemExit(
        "STOP: questo è il VECCHIO split, senza contesto in T2.\n"
        "Rigenera con la versione aggiornata di split_dataset.py, caricala come "
        "nuovo dataset Kaggle e allega quello. Proseguire produrrebbe run non "
        "confrontabili con gli altri modelli.")

t3 = json.load(open(os.path.join(SPLIT, LAYOUTS[2], "train.json"), encoding="utf-8"))
print(f"T3: {len(t3)} istanze di train (atteso ~748 col filtro sul contesto rilassato, "
      f"~450 col vecchio)")
print("\nEsempio di prompt T2:\n" + "-"*60 + f"\n{t2[5]['prompt']}\n"
      f"--- TARGET: {t2[5]['target']}")

SPLIT = /kaggle/input/datasets/lucagiuliano/split-dati/split/split
  OK    layout1_traduzione_con_contesto/train.json  (1134 istanze)
  OK    layout1_traduzione_con_contesto/dev.json  (270 istanze)
  OK    layout1_traduzione_con_contesto/test.json  (267 istanze)
  OK    layout2_completamento_turno/train.json  (683 istanze)
  OK    layout2_completamento_turno/dev.json  (142 istanze)
  OK    layout2_completamento_turno/test.json  (147 istanze)
  OK    layout3_replica_conversazionale/train.json  (748 istanze)
  OK    layout3_replica_conversazionale/dev.json  (187 istanze)
  OK    layout3_replica_conversazionale/test.json  (192 istanze)

T2: 682/683 prompt con contesto conversazionale
T3: 748 istanze di train (atteso ~748 col filtro sul contesto rilassato, ~450 col vecchio)

Esempio di prompt T2:
------------------------------------------------------------
Conversazione finora:
A: assaggiammo
B: ce sta
B: ca pure si nun 'o fernesc~ cioè i' me faccio nu bicchiere
---
Continua il turno in na

## Passo 4 — trova il CSV e rendi lo split scrivibile

**Cosa fare:** niente, se hai allegato entrambi i dataset. Esegui e leggi.

Due cose accadono qui:

1. Si localizza `dataset_finale.csv`. Se non lo trova, allegalo come Dataset
   (Add Data) e riesegui: `lessico.py` e `dati_lessicali.py` non possono
   funzionare senza le coppie allineate.
2. **Lo split viene copiato in `/kaggle/working/split`.** `/kaggle/input` e'
   read-only e i dati dello stadio A2 devono finire *dentro* la cartella di
   split, accanto ai tre layout, perche' `load_split` li trovi. Sono pochi file
   JSON, la copia costa un secondo.

Da qui in avanti `SPLIT` punta alla copia scrivibile: tutte le celle successive
la usano, comprese quelle dello stadio A che prima puntavano a `/kaggle/input`.

In [4]:
# --- Passo 4. CSV delle coppie allineate + split scrivibile -----------------------
import glob, shutil, os

cand_csv = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
CSV = next((p for p in cand_csv if "dataset_finale" in os.path.basename(p)),
           cand_csv[0] if cand_csv else None)
if CSV is None:
    print("CSV non trovato. Contenuto di /kaggle/input:")
    for p in sorted(glob.glob("/kaggle/input/*/*"))[:40]:
        print("  ", p)
    raise SystemExit("Allega dataset_finale.csv come Dataset e riesegui.")
print("CSV =", CSV)
if len(cand_csv) > 1:
    print("  ! piu' CSV presenti:", cand_csv, "\n    se non e' quello giusto, imposta CSV a mano")

SPLIT_RO = SPLIT                      # quello montato in read-only
SPLIT = "/kaggle/working/split"
if os.path.abspath(SPLIT_RO) != os.path.abspath(SPLIT):
    shutil.copytree(SPLIT_RO, SPLIT, dirs_exist_ok=True)
print("SPLIT (scrivibile) =", SPLIT)
for d in sorted(os.listdir(SPLIT)):
    n = len(glob.glob(os.path.join(SPLIT, d, "*.json")))
    print(f"  {d}  ({n} file json)")

import pandas as pd
df_check = pd.read_csv(CSV)
print(f"\nCSV: {len(df_check)} righe, colonne {list(df_check.columns)}")
mancanti = {"conversazione", "turn_index", "italiano", "napoletano", "speaker"} - set(df_check.columns)
if mancanti:
    raise SystemExit(f"Colonne mancanti nel CSV: {mancanti}")
ident = (df_check["italiano"].str.strip().str.lower()
         == df_check["napoletano"].str.strip().str.lower()).sum()
print(f"Righe con italiano == napoletano: {ident} ({ident/len(df_check):.1%}) "
      f"-- e' la quota che rende la baseline copia-italiano difficile da battere")

CSV = /kaggle/input/datasets/lucagiuliano/split-dati/dataset_finale.csv
  ! piu' CSV presenti: ['/kaggle/input/datasets/lucagiuliano/split-dati/dataset_finale.csv', '/kaggle/input/datasets/lucagiuliano/split-dati/split/split/layout1_traduzione_con_contesto/dev.csv', '/kaggle/input/datasets/lucagiuliano/split-dati/split/split/layout1_traduzione_con_contesto/test.csv', '/kaggle/input/datasets/lucagiuliano/split-dati/split/split/layout1_traduzione_con_contesto/train.csv', '/kaggle/input/datasets/lucagiuliano/split-dati/split/split/layout2_completamento_turno/dev.csv', '/kaggle/input/datasets/lucagiuliano/split-dati/split/split/layout2_completamento_turno/test.csv', '/kaggle/input/datasets/lucagiuliano/split-dati/split/split/layout2_completamento_turno/train.csv', '/kaggle/input/datasets/lucagiuliano/split-dati/split/split/layout3_replica_conversazionale/dev.csv', '/kaggle/input/datasets/lucagiuliano/split-dati/split/split/layout3_replica_conversazionale/test.csv', '/kaggle/input/datasets/

In [5]:
# --- Passo 5. Token HuggingFace ----------------------------------------------------
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("Token caricato:", os.environ["HF_TOKEN"][:6] + "...")
# common.py lo legge dai Secrets anche da solo, ma averlo nell'ambiente serve
# ai sottoprocessi lanciati con !python.

Token caricato: hf_nkE...


## Passo 6 — `common.py`

**Cosa fare:** esegui. Scrive il nucleo condiviso dei quattro entrypoint. In questa versione contiene le patch per lo stadio A2 (`LAYOUT_DIRS["A2"]`) e per la pesatura lessicale (`--lessico`, `--peso-dial`): senza `--lessico` il comportamento e' identico ai run precedenti.


In [6]:
%%writefile /kaggle/working/training/common.py
#!/usr/bin/env python3
"""
common.py — nucleo condiviso dei tre script di fine-tuning per task.

NON si esegue da solo. Viene importato da:
    finetune_t1_traduzione.py
    finetune_t2_completamento.py
    finetune_t3_replica.py

Ogni entrypoint definisce solo un TaskConfig (layout, lunghezze, epoche,
metrica di selezione, parametri di generazione qualitativa) e chiama run().
Tutto il resto - caricamento modello, QLoRA, masking della loss, metriche,
resume, summary - vive qui in una sola copia: se cambia, cambia per tutti e
tre i task contemporaneamente, e il confronto cross-task resta valido.

Differenze rispetto al finetune.py monolitico:
  * eval_steps DERIVATO dagli step per epoca (era fisso a 100: su T3, con 87
    step totali, non partiva nessuna eval e l'early stopping era inerte)
  * bf16/fp16 rilevati a runtime: le GPU Kaggle (T4, P100) sono pre-Ampere e
    NON supportano bf16. Il vecchio script forzava bnb_4bit_compute_dtype=
    bfloat16 e non impostava mai fp16/bf16 in TrainingArguments
  * attn_implementation="eager": obbligatorio per Gemma-2/3 (logit soft-capping,
    SDPA e FlashAttention lo implementano male o non lo implementano)
  * token HuggingFace dai Kaggle Secrets invece di .napoli/.api
  * percorsi split espliciti (/kaggle/input, read-only) separati dall'output
    (/kaggle/working, scrivibile)
  * resume da un dataset Kaggle montato in read-only (i checkpoint vengono
    copiati in working prima di riprendere)
"""

from __future__ import annotations

import argparse
import csv
import datetime
import json
import math
import os
import re
import shutil
import sys
import time
from collections import Counter
from dataclasses import dataclass, field

LORA_TARGET_CANDIDATES = {"q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"}

# Alias -> repo_id. Sostituisce model.txt: su Kaggle i modelli arrivano dall'hub
# (o da un dataset montato), non da un file locale della macchina di sviluppo.
# I tre checkpoint sono quelli del tuo model.txt, non altri: cambiare versione
# a metà matrice renderebbe i 9 run non confrontabili.
# NOTA sulle taglie: 7B / 7B / 4B. Gemma-3-4b è la più piccola dei tre, quindi
# un suo punteggio più basso non è automaticamente un limite del modello: va
# dichiarato come asimmetria di taglia tra gli arm del confronto.
MODEL_REGISTRY = {
    "llama":   "meta-llama/Llama-2-7b-chat-hf",
    "minerva": "sapienzanlp/Minerva-7B-instruct-v1.0",
    "gemma":   "google/gemma-3-4b-it",
    # fallback per smoke test o se la taglia grande non regge la GPU
    "minerva-small": "sapienzanlp/Minerva-3B-base-v1.0",
    "gemma-tiny":    "google/gemma-3-1b-it",
}

# Gemma-3-4b-it è multimodale: contiene un vision tower con moduli che si
# chiamano q_proj/k_proj/v_proj/o_proj esattamente come quelli del language
# model. Selezionare i target LoRA per SUFFISSO ci attaccherebbe adapter anche
# sull'encoder visivo - parametri allenabili buttati e un confronto sporco con
# gli altri due modelli, che di vision tower non ne hanno.
VISION_MARKERS = ("vision_tower", "vision_model", "multi_modal_projector",
                  "visual", "image_encoder", "patch_embed")

LAYOUT_DIRS = {
    "T1": "layout1_traduzione_con_contesto",
    "T2": "layout2_completamento_turno",
    "T3": "layout3_replica_conversazionale",
    # Stadio A2: iniezione lessicale. Vive nella stessa cartella di split degli
    # altri layout e ha lo stesso formato ({prompt, target, layout, ...}), cosi'
    # load_split e ChatDataset non cambiano.
    "A2": "stadio_a2_lessico",
}


# --------------------------------------------------------------------------- #
# Configurazione per task
# --------------------------------------------------------------------------- #

@dataclass
class TaskConfig:
    """Tutto (e solo) ciò che distingue un task dagli altri due."""
    layout: str                      # "T1" | "T2" | "T3"
    nome: str                        # nome leggibile, finisce nel summary
    descrizione: str
    max_seq_len: int
    epochs: float
    lr: float = 1e-4
    metric: str = "chrf"             # metrica di selezione del checkpoint
    greater_is_better: bool = True
    gen_max_new_tokens: int = 64
    repetition_penalty: float = 1.2
    no_repeat_ngram_size: int = 3
    patience: int = 4
    evals_per_epoch: int = 2         # da cui si deriva eval_steps
    # capacita' LoRA: su dataset da poche centinaia di esempi r=16 overfitta,
    # quindi ogni task puo' fissare la propria invece di ereditare un default
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    note: list[str] = field(default_factory=list)


# --------------------------------------------------------------------------- #
# Utility
# --------------------------------------------------------------------------- #

def _library_versions():
    """Registrate nel summary: l'immagine Kaggle si aggiorna, e se il run 1 gira
    su una versione di transformers e il run 7 su un'altra il confronto
    cross-modello ha un confondente che dopo non puoi piu' rimuovere."""
    import importlib
    out = {}
    for name in ("torch", "transformers", "peft", "bitsandbytes", "accelerate",
                 "sacrebleu", "trl"):
        try:
            out[name] = importlib.import_module(name).__version__
        except Exception:
            out[name] = None
    return out


def slug(repo_id: str) -> str:
    return repo_id.split("/")[-1].lower()


def approx_params_b(repo_id: str):
    m = re.search(r"(\d+(?:\.\d+)?)b", slug(repo_id))
    return float(m.group(1)) if m else None


def resolve_model(name: str) -> str:
    if name in MODEL_REGISTRY:
        return MODEL_REGISTRY[name]
    if "/" in name:                                   # repo_id esplicito
        return name
    sys.exit(f"ERRORE: modello {name!r} non riconosciuto. Alias disponibili: "
             f"{', '.join(MODEL_REGISTRY)} (oppure passa un repo_id org/nome).")


def load_hf_token(explicit=None):
    """Kaggle Secrets -> variabile d'ambiente -> None.
    Su Kaggle: Add-ons > Secrets, chiave HF_TOKEN, e spunta 'Attach to notebook'.
    Llama e Gemma sono repo gated: senza token il download fallisce (Minerva no)."""
    if explicit:
        return explicit
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")


def load_split(split_dir: str, layout: str, split_name: str, max_samples=None):
    path = os.path.join(split_dir, LAYOUT_DIRS[layout], f"{split_name}.json")
    if not os.path.exists(path):
        sys.exit(f"ERRORE: {path!r} non trovato.\n"
                 f"Su Kaggle lo split va caricato come Dataset e passato con --split-dir, "
                 f"es. --split-dir /kaggle/input/napoletano-split/split")
    rows = json.load(open(path, encoding="utf-8"))
    rows = [r for r in rows if r.get("layout") == layout]     # difesa: file misti
    return rows[:max_samples] if max_samples else rows


def find_lora_target_modules(model):
    """Ritorna i nomi COMPLETAMENTE QUALIFICATI dei moduli target, escludendo il
    vision tower.

    Selezione per nome e non per tipo: Linear4bit di bitsandbytes non è
    nn.Linear, ma la convenzione q/k/v/o/gate/up/down_proj è la stessa su Llama,
    Mistral (Minerva) e Gemma.

    Nomi completi e non suffissi perché su Gemma-3-4b-it il vision tower ha
    moduli omonimi: passando `["q_proj", ...]` PEFT li matcherebbe per endswith
    e attaccherebbe adapter anche all'encoder visivo. PEFT accetta nomi esatti
    (`key in target_modules`), quindi la lista completa è precisa.
    """
    names, skipped = [], 0
    for n, _ in model.named_modules():
        if n.rsplit(".", 1)[-1] not in LORA_TARGET_CANDIDATES:
            continue
        if any(m in n for m in VISION_MARKERS):
            skipped += 1
            continue
        names.append(n)
    if not names:
        sys.exit("ERRORE: nessun modulo target LoRA trovato (q/k/v/o/gate/up/down_proj). "
                 "Architettura non riconosciuta: adatta LORA_TARGET_CANDIDATES.")
    leaves = sorted({n.rsplit(".", 1)[-1] for n in names})
    print(f"Target LoRA: {len(names)} moduli, tipi {leaves}")
    if skipped:
        print(f"  esclusi {skipped} moduli omonimi nel vision tower "
              f"(modello multimodale: gli adapter vanno solo sul language model)")
    return names


def render_prompt(tokenizer, prompt, target=None):
    """Wrappa prompt (e opzionalmente target) nel chat template del tokenizer.

    Nessun ruolo 'system': l'istruzione è già dentro il testo del prompt scritto
    da split_dataset.py, identico byte per byte tra training e inferenza. Questo
    rende il formato compatibile anche con Gemma, che il ruolo system non lo
    ammette.

    Modelli BASE (senza chat template, es. Minerva-3B-base): fallback testuale
    semplice - stesso CONTENUTO, senza tag di ruolo mai visti in pretraining.

    Il template di Gemma-3 in alcune versioni itera su `content` come lista di
    blocchi tipizzati e va in errore su una stringa: da qui il secondo tentativo
    con il contenuto incapsulato.
    """
    if not getattr(tokenizer, "chat_template", None):
        if target is None:
            return prompt + "\n"
        return prompt + "\n" + target + (tokenizer.eos_token or "")

    def apply(user_content, target_content):
        msg = [{"role": "user", "content": user_content}]
        if target_content is None:
            return tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        return tokenizer.apply_chat_template(
            msg + [{"role": "assistant", "content": target_content}],
            tokenize=False, add_generation_prompt=False)

    try:
        return apply(prompt, target)
    except Exception:
        wrap = lambda s: [{"type": "text", "text": s}]
        return apply(wrap(prompt), None if target is None else wrap(target))


# --------------------------------------------------------------------------- #
# Generazione reale durante il training
# --------------------------------------------------------------------------- #

# Token che chiudono il turno assistant nei chat template dei tre modelli. Sono
# gli stessi di evaluate_task.py: se qui mancassero, la generazione di controllo
# durante il training arriverebbe sempre a max_new_tokens e sembrerebbe che il
# modello non impari a fermarsi, mentre il problema sarebbe solo nel decoding.
CANDIDATI_EOT = ("<end_of_turn>", "<|im_end|>", "<|eot_id|>", "<|end|>",
                 "<|endoftext|>", "</s>", "<end_of_text>")


def _solo_spazi(tok, i):
    """True se l'id decodifica in soli spazi (o in niente).

    Serve a non usare come fine turno un token che non chiude nulla. Su Gemma-3
    il chat template scrive '<end_of_turn>' seguito da un a capo, quindi
    l'ULTIMO token del target renderizzato e' il 107, cioe' '\\n' (verificato
    in tokenizer.json: 106 = '<end_of_turn>', 107 = '\\n'). Passarlo a
    generate() come eos significa fermare la generazione al primo a capo: se il
    modello lo emette subito, l'uscita e' la stringa vuota.
    """
    try:
        return not tok.decode([int(i)], skip_special_tokens=False).strip()
    except Exception:
        return False


def _chiusura_target(tok, ids_testo):
    """Ultimo token NON di spaziatura della sequenza renderizzata.

    E' il token che chiude davvero il turno (su Gemma-3 il 106,
    '<end_of_turn>'), non l'a capo che il template gli mette dopo.
    """
    for i in reversed(list(ids_testo)):
        if not _solo_spazi(tok, i):
            return int(i)
    return None


def id_fine_turno(tok, rows=(), verbose=True):
    """Insieme degli id che chiudono il turno assistant, da passare al decoding.

    La fonte di verita' non e' la generation_config del modello ma l'ultimo
    token del testo renderizzato in training: e' esattamente cio' che il modello
    ha imparato a produrre per chiudere. Stessa logica di evaluate_task.py, cosi'
    la generazione di monitoraggio e quella di valutazione si fermano allo stesso
    punto e i due chrF sono confrontabili.
    """
    ids = set()
    if tok.eos_token_id is not None:
        ids.add(int(tok.eos_token_id))
    for s in CANDIDATI_EOT:
        i = tok.convert_tokens_to_ids(s)
        if isinstance(i, int) and i >= 0 and i != tok.unk_token_id:
            ids.add(int(i))
    coda = []
    for r in list(rows)[:8]:
        t = tok(render_prompt(tok, r["prompt"], r["target"]),
                add_special_tokens=False)["input_ids"]
        if t:
            ultimo = _chiusura_target(tok, t)
            if ultimo is not None:
                coda.append(ultimo)
    if coda:
        ids.add(Counter(coda).most_common(1)[0][0])
    # BUG CORRETTO. Nessun token di sola spaziatura fra gli eos: fermarsi su un
    # a capo prima di aver prodotto contenuto restituisce la stringa vuota, e
    # una stringa vuota vale 0.00 in tutte le metriche senza sembrare un errore.
    tenuti = set(i for i in ids if not _solo_spazi(tok, i))
    ids = tenuti or ids
    out = sorted(ids)
    if verbose:
        print(f"  fine turno per la generazione di monitoraggio: {out} "
              f"({[tok.convert_ids_to_tokens([i])[0] for i in out]})")
    return out


# Stato del motore di generazione: si stampa una volta sola per run, altrimenti
# ogni eval ripete lo stesso traceback.
_GEN_FALLBACK = {"annunciato": False, "forza_manuale": False}


def _decode_manuale(model, tokenizer, enc, max_new_tokens, eos_ids=(),
                    repetition_penalty=1.0, no_repeat_ngram_size=0,
                    usa_cache=True):
    """Loop di decoding greedy scritto a mano, batch 1, senza model.generate().

    PERCHE' ESISTE. Su google/gemma-3-4b-it (Gemma3ForConditionalGeneration)
    model.generate() e' rotto nella transformers dell'immagine Kaggle: _sample
    riceve next_tokens con una dimensione di troppo e crolla con
    "Tensors must have same number of dimensions: got 2 and 3". Il sintomo, nei
    log dello stadio A2, era

        [generazione reale saltata: RuntimeError: Tensors must have same ...]
        generato ( 3 parole): '[generazione fallita: RuntimeError]'

    cioe' l'UNICA metrica che vede il fallimento sull'EOS (eval_gen_chrf) non
    veniva mai calcolata, e la generazione di controllo stampava un segnaposto.
    evaluate_task.py aggirava gia' il bug con lo stesso loop; qui mancava, quindi
    il training girava cieco. Affettando esplicitamente logits[:, -1, :] la forma
    e' sempre [1, vocab] e il problema non si pone.

    Con cache KV (usa_cache=True) il costo e' un forward sul prompt piu' uno per
    token; senza cache si ricalcola tutto ad ogni passo, quindi e' solo il
    ripiego se la cache non e' supportata.
    """
    import torch
    ids = enc["input_ids"]
    attn = enc.get("attention_mask")
    if attn is None:
        attn = torch.ones_like(ids)
    eos_set = {int(e) for e in (eos_ids or ())}
    prompt_len = ids.shape[1]
    generati = []
    past, cur = None, ids
    for _ in range(max_new_tokens):
        kw = dict(input_ids=cur, attention_mask=attn, use_cache=usa_cache)
        if usa_cache and past is not None:
            kw["past_key_values"] = past
        out = model(**kw)
        if usa_cache:
            past = getattr(out, "past_key_values", None)
            if past is None:                       # il modello ignora la cache
                usa_cache = False
        logits = out.logits[:, -1, :].float()      # [1, vocab] sempre
        if not torch.isfinite(logits).all():
            raise RuntimeError(
                "logit non finiti (NaN/inf) durante la generazione: il forward "
                "e' in overflow fp16, non e' un problema di decoding. "
                "Vedi stabilizza_fp16() in questo file.")
        seq = ids[0].tolist()
        if repetition_penalty and repetition_penalty != 1.0:
            for t in set(seq):
                v = logits[0, t]
                logits[0, t] = v / repetition_penalty if v > 0 else v * repetition_penalty
        if no_repeat_ngram_size and len(seq) >= no_repeat_ngram_size:
            n = no_repeat_ngram_size
            prefisso = tuple(seq[-(n - 1):]) if n > 1 else ()
            for i in range(len(seq) - n + 1):
                if tuple(seq[i:i + n - 1]) == prefisso:
                    logits[0, seq[i + n - 1]] = float("-inf")
        # Niente EOS finche' l'uscita sarebbe vuota. Fra gli id di fine turno
        # c'e' il 107, che e' un semplice '\n' e non un token speciale: se il
        # modello lo emette al primo passo, senza questa guardia il loop esce
        # subito e restituisce stringa vuota. In valutazione e' costato un
        # chrF++ 0.00 su tutte e 267 le uscite di test, senza nessun errore.
        if tokenizer is not None and not tokenizer.decode(
                generati, skip_special_tokens=True).strip():
            for e in eos_set:
                logits[0, e] = float("-inf")
        nxt = logits.argmax(dim=-1, keepdim=True)  # [1, 1]
        ids = torch.cat([ids, nxt], dim=-1)
        generati.append(int(nxt.item()))
        attn = torch.cat([attn, torch.ones_like(nxt)], dim=-1)
        cur = nxt if usa_cache else ids
        if int(nxt.item()) in eos_set:
            break
    return ids[0][prompt_len:]


def genera_una(model, tokenizer, testo, max_new_tokens, eos_ids=(),
               repetition_penalty=1.0, no_repeat_ngram_size=0):
    """Genera la continuazione di UN prompt gia' renderizzato. Ritorna testo.

    Un item alla volta, nessun padding: la generazione batched con left-padding
    su Gemma-3 crasha in transformers (_sample: shape [prompt_len] vs [batch]).
    Con i 12-32 item della metrica di generazione il costo e' accettabile.

    Su Gemma-3 si usa direttamente il loop manuale; sugli altri modelli si prova
    model.generate() e si ripiega sul loop se solleva un'eccezione, stampando il
    traceback UNA volta. Una generazione che fallisce non deve piu' sparire
    dietro un segnaposto: o si genera, o si vede perche' no.
    """
    import torch
    enc = tokenizer(testo, return_tensors="pt",
                    add_special_tokens=False).to(model.device)
    mt = (getattr(getattr(model, "config", None), "model_type", "") or "").lower()
    manuale = _GEN_FALLBACK["forza_manuale"] or "gemma3" in mt
    if not manuale:
        gen_kw = dict(max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
                      pad_token_id=tokenizer.pad_token_id)
        if eos_ids:
            gen_kw["eos_token_id"] = list(eos_ids)
        if repetition_penalty and repetition_penalty != 1.0:
            gen_kw["repetition_penalty"] = repetition_penalty
        if no_repeat_ngram_size:
            gen_kw["no_repeat_ngram_size"] = no_repeat_ngram_size
        try:
            with torch.no_grad():
                g = model.generate(**enc, **gen_kw)
            return tokenizer.decode(g[0][enc["input_ids"].shape[1]:],
                                    skip_special_tokens=True)
        except Exception:
            import traceback
            _GEN_FALLBACK["forza_manuale"] = True
            if not _GEN_FALLBACK["annunciato"]:
                _GEN_FALLBACK["annunciato"] = True
                print("  [model.generate() ha fallito: passo al loop di decoding "
                      "manuale per tutto il run. Traceback una volta sola:]")
                traceback.print_exc()
    try:
        with torch.no_grad():
            out_ids = _decode_manuale(model, tokenizer, enc, max_new_tokens,
                                      eos_ids, repetition_penalty,
                                      no_repeat_ngram_size, usa_cache=True)
    except Exception:                              # cache KV non supportata qui
        with torch.no_grad():
            out_ids = _decode_manuale(model, tokenizer, enc, max_new_tokens,
                                      eos_ids, repetition_penalty,
                                      no_repeat_ngram_size, usa_cache=False)
    return tokenizer.decode(out_ids, skip_special_tokens=True)


def _forza_token_type_ids(model):
    """Gemma-3 (Gemma3ForConditionalGeneration) pretende token_type_ids in input
    durante il training: la maschera causale multimodale li usa per distinguere i
    token immagine da quelli testo. Qui addestriamo SOLO su testo, quindi sono
    tutti 0 (nessun token immagine). Li iniettiamo di default nel forward, cosi'
    nessuno degli script a valle (A, A2, SFT) deve saperlo. Patch applicata solo a
    Gemma-3: gli altri modelli (Llama, Minerva) non accettano token_type_ids.
    """
    mt = getattr(getattr(model, "config", None), "model_type", "") or ""
    if "gemma3" not in mt.lower() and "gemma3" not in type(model).__name__.lower():
        return model
    import torch
    _orig_forward = model.forward
    def forward(*args, **kw):
        ii = kw.get("input_ids")
        # solo in TRAINING Gemma-3 pretende token_type_ids; in eval/generate no,
        # e iniettarli li' rompe la generazione con la cache (shape mismatch).
        if model.training and kw.get("token_type_ids") is None and ii is not None:
            kw["token_type_ids"] = torch.zeros_like(ii)
        return _orig_forward(*args, **kw)
    model.forward = forward
    return model


def scegli_dtype(repo_id, scelta="auto"):
    """(dtype, bf16_ok) per QUESTA GPU e QUESTO modello. Regola del notebook T3.

    Non basta guardare la compute capability. Su una GPU senza bf16 nativo
    (T4, cc 7.5) il ripiego naturale e' fp16, ma Gemma in fp16 va in overflow:
    le attivazioni escono dal range del formato, i logit diventano NaN e la
    generazione restituisce stringhe vuote (in campionamento crolla con
    'probability tensor contains inf/nan'; in greedy no, perche' argmax su NaN
    restituisce l'id 0, che su Gemma e' '<pad>' - stesso guasto, silenzioso).

    La loss in teacher forcing puo' restare sana mentre la generazione e' gia'
    morta, quindi il sintomo NON e' la loss. In questo progetto il training
    sopravviveva in fp16 solo perche' prepare_model_for_kbit_training riporta a
    fp32 i parametri non quantizzati; gli script di inferenza, che caricano il
    modello nudo, no. Da qui le 267 uscite vuote su 267 in valutazione.

    Per Gemma senza bf16 si usa fp32: circa 2x piu' lento, ma finito. Il
    notebook T3, stesso modello e stessa T4, gira cosi' e riporta "vuote": 0 in
    tutti gli arm di valutazione.
    """
    import torch
    bf16_ok = torch.cuda.get_device_capability(0)[0] >= 8
    if scelta == "bf16":
        return torch.bfloat16, True
    if scelta == "fp16":
        return torch.float16, False
    if scelta == "fp32":
        return torch.float32, False
    if bf16_ok:
        return torch.bfloat16, True
    if "gemma" in str(repo_id).lower():
        return torch.float32, False
    return torch.float16, False


def controlla_finito(model, tokenizer, testo, dove="", rendi=True):
    """Un forward solo, per fallire in cinque secondi invece che dopo un'ora.

    Se i logit non sono finiti nessun numero a valle e' interpretabile: loss che
    non scende, grad_norm=nan, chrF 0 e uscite vuote sono conseguenze, non
    misure. Va chiamato dopo aver costruito il PeftModel e prima del Trainer.
    """
    import torch
    t = render_prompt(tokenizer, testo) if rendi else testo
    enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=512,
                    add_special_tokens=False).to(model.device)
    era = model.training
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits
    model.train(era)
    ok = bool(torch.isfinite(logits).all())
    print(f"  logit finiti: {ok} (dtype {logits.dtype})" + (f" [{dove}]" if dove else ""))
    if not ok:
        sys.exit("LOGIT NON FINITI: overflow numerico con questa precisione. "
                 "Rilancia con --dtype fp32, oppure usa una GPU con bf16 nativo "
                 "(L4, A100), che e' piu' veloce e piu' stabile.")


def stabilizza_fp16(model):
    """Porta a fp32 i parametri NON quantizzati quando si lavora in float16.

    PERCHE' ESISTE (uscite di valutazione tutte vuote, chrF++ 0.00).
    Su GPU pre-Ampere (Kaggle T4, cc 7.5) il compute dtype e' fp16. Gemma-3 ha
    attivazioni molto grandi: nel flusso residuale in fp16 si supera 65504, l'inf
    entra nelle RMSNorm e i logit finali diventano NaN. `argmax` su un tensore di
    NaN restituisce l'indice 0, che su Gemma e' `<pad>`: il decoder emette 64 pad
    di fila, `decode(skip_special_tokens=True)` restituisce la stringa vuota e
    OGNI metrica vale 0.00 senza che nessuno sollevi un'eccezione.

    Le due prove che era questo e non il decoding:
      - `logprob_riferimento_norm: NaN` in tutti e quattro i .metrics.json, e la
        contrastiva non genera niente, fa solo forward: il forward era gia' rotto;
      - la generazione di monitoraggio DURANTE il training stampava napoletano
        corretto sugli stessi prompt e con lo stesso adapter, perche' li' il
        modello era passato per prepare_model_for_kbit_training, che fa
        esattamente questo upcast. evaluate_task.py carica il modello "nudo".

    Resta come seconda linea di difesa per i modelli che in fp16 ci girano
    davvero (Llama, Minerva): su Gemma la prima linea e' scegli_dtype(), che in
    assenza di bf16 sceglie direttamente fp32.

    I pesi 4-bit non vengono toccati (Params4bit e' uint8, non fp16). Con gli
    embedding in fp32 le hidden state sono fp32: bitsandbytes casta l'ingresso a
    compute_dtype per il matmul e riporta l'uscita al dtype d'ingresso, quindi il
    residuo si accumula in fp32. Costo ~1,3 GB su un 4B; nessun cambiamento nei
    pesi, solo nella precisione con cui vengono sommati.
    """
    import torch
    n = 0
    for _, p in model.named_parameters():
        if p.dtype == torch.float16:
            p.data = p.data.to(torch.float32)
            n += 1
    if n:
        print(f"  stabilizzazione numerica: {n} tensori non quantizzati da fp16 "
              f"a fp32 (residuo in fp32, matmul 4-bit in fp16)")
    return model


def load_backbone(repo_id, dtype, **kw):
    """AutoModelForCausalLM, con fallback per i checkpoint multimodali.

    google/gemma-3-4b-it è registrato come Gemma3ForConditionalGeneration:
    AutoModelForCausalLM può rifiutarlo. AutoModelForImageTextToText lo carica
    (vision tower incluso, che noi congeliamo escludendolo dai target LoRA).
    Su Gemma-3 il forward viene patchato per fornire token_type_ids=0 (testo).
    """
    import torch
    from transformers import AutoModelForCausalLM

    def _prepara(m):
        # solo sui caricamenti quantizzati: in fp16 pieno l'upcast raddoppierebbe
        # la memoria dei pesi e non ci starebbe in VRAM.
        if dtype == torch.float16 and kw.get("quantization_config") is not None:
            m = stabilizza_fp16(m)
        return _forza_token_type_ids(m)

    try:
        model = from_pretrained_compat(AutoModelForCausalLM, repo_id, dtype, **kw)
        return _prepara(model), "CausalLM"
    except Exception as e_causal:
        try:
            from transformers import AutoModelForImageTextToText
        except ImportError:
            raise e_causal
        print(f"  AutoModelForCausalLM non applicabile ({type(e_causal).__name__}): "
              f"ripiego su AutoModelForImageTextToText (checkpoint multimodale).")
        model = from_pretrained_compat(AutoModelForImageTextToText, repo_id, dtype, **kw)
        return _prepara(model), "ImageTextToText"


class ChatDataset:
    """Un esempio = turno user (prompt già pronto) + risposta assistant.
    Loss mascherata a -100 sui token del prompt: si allena solo sull'assistant."""

    def __init__(self, rows, tokenizer, max_len):
        self.rows, self.tok, self.max_len = rows, tokenizer, max_len
        self.n_troncati = 0
        self.n_prefix_mismatch = 0
        for k, r in enumerate(rows):                  # diagnostica, non silenziosa
            p = self.tok(render_prompt(self.tok, r["prompt"]),
                         add_special_tokens=False)["input_ids"]
            f = self.tok(render_prompt(self.tok, r["prompt"], r["target"]),
                         add_special_tokens=False)["input_ids"]
            if len(f) > max_len:
                self.n_troncati += 1
            # Il masking presuppone che il prompt renderizzato sia un PREFISSO
            # esatto del testo completo. Se il chat template o la tokenizzazione
            # al confine prompt/target rompono questa proprietà, la loss finisce
            # calcolata su posizioni sbagliate - in silenzio. Meglio saperlo.
            if k < 200 and f[:len(p)] != p:
                self.n_prefix_mismatch += 1

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        prompt_ids = self.tok(render_prompt(self.tok, r["prompt"]),
                              add_special_tokens=False)["input_ids"]
        full_ids = self.tok(render_prompt(self.tok, r["prompt"], r["target"]),
                            add_special_tokens=False)["input_ids"][:self.max_len]
        labels = list(full_ids)
        for j in range(min(len(prompt_ids), len(labels))):
            labels[j] = -100
        return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}


def make_collate_fn(pad_id):
    import torch

    def collate(batch):
        n = max(len(b["input_ids"]) for b in batch)
        ids, attn, labels = [], [], []
        for b in batch:                               # padding a DESTRA in training
            k = n - len(b["input_ids"])
            ids.append(b["input_ids"] + [pad_id] * k)
            attn.append(b["attention_mask"] + [0] * k)
            labels.append(b["labels"] + [-100] * k)
        return {"input_ids": torch.tensor(ids, dtype=torch.long),
                "attention_mask": torch.tensor(attn, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)}
    return collate


def from_pretrained_compat(cls, repo_id, dtype, **kw):
    """transformers 5.x usa `dtype=`, le 4.x `torch_dtype=`. Prova il nuovo,
    ripiega sul vecchio: così lo stesso script gira sia sul venv locale sia
    sull'immagine Kaggle, che possono avere versioni diverse."""
    try:
        return cls.from_pretrained(repo_id, dtype=dtype, **kw)
    except TypeError:
        return cls.from_pretrained(repo_id, torch_dtype=dtype, **kw)


# --------------------------------------------------------------------------- #
# Metriche
# --------------------------------------------------------------------------- #

def build_metrics(tokenizer, task: TaskConfig):
    """compute_metrics + preprocess_logits_for_metrics.

    ATTENZIONE metodologica: tutto qui è calcolato in TEACHER FORCING (argmax
    dei logit posizione per posizione), non con generate() autoregressivo. È un
    proxy economico per il monitoraggio e per l'early stopping, NON il numero da
    riportare nel paper: un modello può sembrare buono qui e degenerare in
    ripetizioni a generazione libera. I numeri finali vanno da uno script di
    valutazione separato su test.json con generate() vero.
    """
    import sacrebleu
    try:
        from rouge_score import rouge_scorer
        rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)
    except ImportError:
        rouge = None
        print("  ! rouge_score non installato: rouge1/rougeL non calcolati "
              "(pip install rouge_score).")

    def preprocess_logits_for_metrics(logits, labels):
        # riduzione immediata ad argmax: tenere (batch x seq x vocab) per tutto il
        # dev satura la memoria (Gemma ha un vocabolario da 256k token)
        return logits.argmax(dim=-1)

    def word_prf1(hyp, ref):
        """Overlap bag-of-words (come l'F1 di SQuAD). Proxy LESSICALE su un task
        generativo, NON F1 di classificazione: etichettarlo come tale."""
        h, r = hyp.split(), ref.split()
        if not h or not r:
            return 0.0, 0.0, 0.0
        common = sum((Counter(h) & Counter(r)).values())
        if common == 0:
            return 0.0, 0.0, 0.0
        p, rc = common / len(h), common / len(r)
        return p, rc, 2 * p * rc / (p + rc)

    def score_set(hyps, refs):
        out = {"chrf": sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score,
               "bleu": sacrebleu.corpus_bleu(hyps, [refs]).score}
        prf1 = [word_prf1(h, r) for h, r in zip(hyps, refs)]
        out["word_precision"] = sum(p for p, _, _ in prf1) / len(prf1)
        out["word_recall"] = sum(r for _, r, _ in prf1) / len(prf1)
        out["word_f1"] = sum(f for _, _, f in prf1) / len(prf1)
        if rouge is not None:
            s = [rouge.score(r, h) for h, r in zip(hyps, refs)]   # (target, prediction)
            out["rouge1"] = sum(x["rouge1"].fmeasure for x in s) / len(s)
            out["rougeL"] = sum(x["rougeL"].fmeasure for x in s) / len(s)
        return out

    def compute_metrics(eval_pred):
        pred_ids, label_ids = eval_pred
        pred_ids = pred_ids[:, :-1]          # shift causal LM: pred[i] prevede label[i+1]
        label_ids = label_ids[:, 1:]
        hyps, refs, correct, total = [], [], 0, 0
        for p_row, l_row in zip(pred_ids, label_ids):
            mask = l_row != -100
            if mask.sum() == 0:
                continue
            hyps.append(tokenizer.decode(p_row[mask], skip_special_tokens=True))
            refs.append(tokenizer.decode(l_row[mask], skip_special_tokens=True))
            correct += int((p_row[mask] == l_row[mask]).sum())
            total += int(mask.sum())
        if not hyps:
            return {task.metric: 0.0}
        res = score_set(hyps, refs)
        res["token_accuracy"] = correct / total if total else 0.0
        return res

    return compute_metrics, preprocess_logits_for_metrics


def build_generation_metric_callback(TrainerCallback, torch, tokenizer, rows, task: TaskConfig,
                                     n=32, batch=8, eos_ids=()):
    """Aggiunge eval_gen_chrf e eval_gen_len_ratio, calcolate su GENERAZIONE REALE.

    Perche' serve: eval_chrf di compute_metrics e' in teacher forcing, cioe' si
    decodifica l'argmax su tante posizioni quante ne ha il target. L'ipotesi ha
    percio' SEMPRE la stessa lunghezza del riferimento, e la metrica e'
    strutturalmente cieca al fallimento sull'EOS. Misurato su Minerva T3:
    eval_chrf in teacher forcing 22.1, chrF su generazione reale 10.5, con
    output 2.7 volte piu' lunghi dell'umano. Nessuna metrica in teacher forcing
    (ne' chrF ne' loss) puo' selezionare un checkpoint su quella base.

    Decoding greedy anche su T3: qui serve un segnale STABILE fra una eval e
    l'altra per la selezione: il nucleus sampling introdurrebbe varianza fra
    valutazioni. Il nucleus va usato nella valutazione finale.

    Mutare il dict `metrics` dentro on_evaluate propaga alla selezione del
    checkpoint, quindi metric_for_best_model="gen_chrf" funziona.
    """
    import sacrebleu

    sub = rows[:n]

    class GenerationMetricCallback(TrainerCallback):
        def on_evaluate(self, args, state, control, model=None, metrics=None, **kwargs):
            if model is None or metrics is None or not sub:
                return
            was_training, was_cache = model.training, model.config.use_cache
            prev_side = tokenizer.padding_side
            model.eval()
            # Con gradient_checkpointing attivo transformers forza use_cache=False,
            # quindi generate() gira SENZA cache KV e ricalcola il forward su tutta
            # la sequenza per ogni token nuovo: con 32 item, 64 token e prompt da
            # ~200 sono ~64x il lavoro previsto, e il run sembra bloccato. Va
            # disattivato per la durata della generazione e riattivato dopo.
            ckpt_era_attivo = bool(getattr(model, "is_gradient_checkpointing", False))
            if ckpt_era_attivo:
                model.gradient_checkpointing_disable()
            model.config.use_cache = True
            tokenizer.padding_side = "left"          # obbligatorio in generazione
            hyps = []
            try:
                # Generazione UN ITEM ALLA VOLTA (batch=1, nessun padding), via
                # genera_una: su Gemma-3 usa il loop di decoding manuale, sugli
                # altri model.generate() con ripiego automatico. Prima qui c'era
                # una chiamata diretta a model.generate(), che su Gemma-3 solleva
                # RuntimeError e faceva saltare eval_gen_chrf ad ogni eval.
                for r in sub:
                    txt = genera_una(model, tokenizer,
                                     render_prompt(tokenizer, r["prompt"]),
                                     task.gen_max_new_tokens, eos_ids)
                    hyps.append(txt.strip().split("\n")[0].strip())
                refs = [r["target"] for r in sub]
                metrics["eval_gen_chrf"] = sacrebleu.corpus_chrf(
                    hyps, [refs], word_order=2).score
                lh = sum(len(x.split()) for x in hyps) / len(hyps)
                lr = sum(len(x.split()) for x in refs) / len(refs)
                metrics["eval_gen_len_ratio"] = lh / lr if lr else 0.0
                print(f"  [generazione reale su {len(sub)} item] chrF "
                      f"{metrics['eval_gen_chrf']:.2f} | lunghezza "
                      f"{metrics['eval_gen_len_ratio']:.2f}x l'umano")
            except Exception as e:
                import traceback
                print(f"  [generazione reale saltata: {type(e).__name__}: {e}]")
                print("   eval_gen_chrf non calcolato; la selezione del checkpoint "
                      "usa eval_chrf (teacher forcing) e non ne dipende.")
                print("   ATTENZIONE: senza eval_gen_chrf nessuna metrica vede il "
                      "fallimento sull'EOS. Traceback:")
                traceback.print_exc()
            finally:
                tokenizer.padding_side = prev_side
                model.config.use_cache = was_cache
                if ckpt_era_attivo:
                    model.gradient_checkpointing_enable(
                        gradient_checkpointing_kwargs={"use_reentrant": False})
                if was_training:
                    model.train()

    return GenerationMetricCallback()


def build_sample_callback(TrainerCallback, torch, tokenizer, samples, task: TaskConfig,
                          eos_ids=()):
    """Ad ogni eval genera per davvero (no teacher forcing) su 3 esempi FISSI di
    dev. È l'unico modo per accorgersi in tempo reale che il chrF proxy sale
    mentre il testo generato degenera in loop.

    Tre correzioni rispetto alla versione precedente:
      1. la generazione passa da genera_una (loop manuale su Gemma-3), quindi
         non stampa piu' '[generazione fallita: RuntimeError]';
      2. si passano gli id di fine turno: senza, il decoding arriva sempre a
         max_new_tokens e il messaggio "non sta imparando a emettere EOS" e'
         un artefatto del decoding, non una misura;
      3. il gradient checkpointing viene disattivato per la durata della
         generazione. Con il checkpointing attivo transformers forza
         use_cache=False, quindi ogni token nuovo ricalcolava il forward
         sull'intera sequenza: era il motivo per cui questa cella sembrava
         bloccata dopo ogni eval. Il callback della metrica lo faceva gia',
         questo no.
    """

    class SampleGenerationCallback(TrainerCallback):
        def on_evaluate(self, args, state, control, model=None, **kwargs):
            if model is None or not samples:
                return
            was_training, was_cache = model.training, model.config.use_cache
            model.eval()
            ckpt_era_attivo = bool(getattr(model, "is_gradient_checkpointing", False))
            if ckpt_era_attivo:
                model.gradient_checkpointing_disable()
            model.config.use_cache = True
            print(f"\n--- Generazione reale ({task.layout}) allo step {state.global_step} ---")
            try:
                for prompt, target in samples:
                    try:
                        gen = genera_una(model, tokenizer,
                                         render_prompt(tokenizer, prompt),
                                         task.gen_max_new_tokens, eos_ids,
                                         task.repetition_penalty,
                                         task.no_repeat_ngram_size)
                    except Exception as e:
                        import traceback
                        gen = f"[generazione fallita: {type(e).__name__}: {e}]"
                        traceback.print_exc()
                    # prompt COMPLETO: troncarlo in stampa fa sembrare che il
                    # modello non veda il contesto, quando invece lo riceve tutto
                    print("  --- prompt (completo) ---")
                    for riga in prompt.split("\n"):
                        print(f"  | {riga}")
                    nt, ng = len(target.split()), len(gen.split())
                    print(f"  target   ({nt:2d} parole): {target!r}")
                    print(f"  generato ({ng:2d} parole): {gen!r}")
                    if ng > 3 * max(nt, 1):
                        print(f"  ! generato {ng/max(nt,1):.1f}x piu' lungo del target: "
                              f"il modello non sta imparando a emettere EOS")
                    print()
                print("---\n")
            finally:
                model.config.use_cache = was_cache
                if ckpt_era_attivo:
                    model.gradient_checkpointing_enable(
                        gradient_checkpointing_kwargs={"use_reentrant": False})
                if was_training:
                    model.train()

    return SampleGenerationCallback()


# --------------------------------------------------------------------------- #
# CLI condivisa
# --------------------------------------------------------------------------- #

def build_parser(task: TaskConfig) -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(
        description=f"Fine-tuning QLoRA — {task.layout} ({task.nome}). {task.descrizione}",
        formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True,
                    help=f"alias ({'/'.join(list(MODEL_REGISTRY)[:3])}...) o repo_id org/nome")
    ap.add_argument("--split-dir", default="/kaggle/working/split",
                    help="cartella che contiene layout1_*/layout2_*/layout3_* (read-only su Kaggle)")
    ap.add_argument("--out-dir", default="/kaggle/working/runs")
    ap.add_argument("--resume-dir", default=None,
                    help="cartella (es. output di una sessione Kaggle precedente montata in "
                         "/kaggle/input) da cui copiare i checkpoint per riprendere il training")
    ap.add_argument("--no-resume", action="store_true")
    ap.add_argument("--precision", choices=["qlora4bit", "bf16", "fp16"], default="qlora4bit")
    ap.add_argument("--dtype", choices=["auto", "bf16", "fp16", "fp32"], default="auto",
                    help="precisione di calcolo. auto = bf16 se la GPU ce l'ha, "
                         "fp32 per Gemma senza bf16 (in fp16 il GradScaler scarta "
                         "gli step: grad_norm=nan e learning_rate=0), fp16 per gli altri")
    ap.add_argument("--seed", type=int, default=42)
    # iperparametri: default dal TaskConfig, sovrascrivibili
    ap.add_argument("--max-seq-len", type=int, default=task.max_seq_len)
    ap.add_argument("--epochs", type=float, default=task.epochs)
    ap.add_argument("--lr", type=float, default=task.lr)
    ap.add_argument("--batch-size", type=int, default=2)
    ap.add_argument("--eval-batch-size", type=int, default=2)
    ap.add_argument("--grad-accum", type=int, default=8,
                    help="batch efficace = batch-size * grad-accum. Tenerlo COSTANTE (16) su "
                         "tutti e 9 i run: è una variabile di confronto, non un parametro libero")
    ap.add_argument("--warmup-ratio", type=float, default=0.03)
    ap.add_argument("--weight-decay", type=float, default=0.01)
    ap.add_argument("--neftune-alpha", type=float, default=5.0)
    ap.add_argument("--lora-r", type=int, default=task.lora_r)
    ap.add_argument("--lora-alpha", type=int, default=task.lora_alpha)
    ap.add_argument("--lora-dropout", type=float, default=task.lora_dropout)
    ap.add_argument("--train-embeddings", action="store_true",
                    help="allena embed_tokens/lm_head per intero (non a basso rango). "
                         "Costoso in VRAM, soprattutto su Gemma (vocabolario 256k)")
    ap.add_argument("--evals-per-epoch", type=int, default=task.evals_per_epoch,
                    help="da cui si DERIVA eval_steps. Con dataset di poche centinaia di "
                         "esempi un eval_steps fisso rischia di non far partire nessuna eval")
    ap.add_argument("--eval-subset", type=int, default=200,
                    help="esempi di dev per l'eval periodica (0 = tutto il dev)")
    ap.add_argument("--patience", type=int, default=task.patience)
    ap.add_argument("--metric", default=task.metric,
                    help="metrica di selezione del checkpoint (chrf, bleu, loss, word_f1...)")
    ap.add_argument("--no-sample-generation", action="store_true")
    ap.add_argument("--gen-metric-n", type=int, default=12,
                    help="item di dev su cui calcolare chrF e rapporto di lunghezza con "
                         "GENERAZIONE REALE ad ogni eval (0 disattiva). Sono le uniche "
                         "metriche che vedono il fallimento sull'EOS: quelle in teacher "
                         "forcing hanno per costruzione la stessa lunghezza del riferimento")
    ap.add_argument("--no-gradient-checkpointing", action="store_true")
    ap.add_argument("--max-train-samples", type=int, default=None, help="smoke test")
    ap.add_argument("--max-dev-samples", type=int, default=None, help="smoke test")
    ap.add_argument("--init-adapter", default=None,
                    help="adapter di partenza, tipicamente l'output dello stadio A "
                         "(pretrain_dialect.py). Lo stesso adapter viene CONTINUATO, "
                         "non impilato: r/alpha/dropout li eredita da quello e i "
                         "corrispondenti --lora-* vengono ignorati")
    ap.add_argument("--ctx-metric-n", type=int, default=64,
                    help="item di dev su cui misurare la sensibilita' al contesto "
                         "(0 disattiva). Aggiunge eval_ctx_delta e eval_ctx_acc, "
                         "usabili come --metric ctx_acc. Costa 2 forward per item, "
                         "nessuna generazione")
    ap.add_argument("--lessico", default=None,
                    help="output di lessico.py. Se presente, la loss viene PESATA "
                         "sui token delle forme dialettali (vedi --peso-dial). "
                         "Senza questo flag il comportamento e' identico ai run "
                         "precedenti, quindi i due arm sono confrontabili")
    ap.add_argument("--peso-dial", type=float, default=3.0,
                    help="peso dei token dialettali contro 1.0 degli altri. La media "
                         "e' ponderata, non sommata: la scala della loss resta "
                         "confrontabile con i run non pesati e il lr non va ritoccato")
    ap.add_argument("--hf-token", default=None)
    return ap


# --------------------------------------------------------------------------- #
# Run
# --------------------------------------------------------------------------- #

def run(task: TaskConfig):
    args = build_parser(task).parse_args()

    import torch
    from transformers import (AutoTokenizer, BitsAndBytesConfig,
                              EarlyStoppingCallback, Trainer, TrainerCallback,
                              TrainingArguments, set_seed)
    from transformers.trainer_utils import get_last_checkpoint
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    set_seed(args.seed)
    repo_id = resolve_model(args.model)

    if not torch.cuda.is_available():
        sys.exit("ERRORE: nessuna GPU CUDA rilevata. Su Kaggle: Settings > Accelerator > "
                 "GPU P100 (preferibile) oppure T4 x2. Questo script non è pensato per la CPU.")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    # NON usare torch.cuda.is_bf16_supported(): nelle versioni recenti di PyTorch
    # ha including_emulation=True per default e risponde True anche sulla T4
    # (Turing, cc 7.5), che il bf16 in hardware non ce l'ha. Il risultato e' bf16
    # EMULATO, molto piu' lento dei tensor core fp16 nativi. Il bf16 vero parte da
    # Ampere, cc 8.0.
    _cc = torch.cuda.get_device_capability(0)
    bf16_ok = _cc[0] >= 8
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    compute_dtype, _ = scegli_dtype(repo_id, args.dtype)
    nome_dt = str(compute_dtype).replace("torch.", "")
    print(f"=== {task.layout} ({task.nome}) | {repo_id} ===")
    print(f"GPU: {gpu_name} (cc {_cc[0]}.{_cc[1]}), {vram_gb:.1f} GB VRAM | "
          f"bf16 nativo: {bf16_ok} -> precisione: {nome_dt}")
    if compute_dtype == torch.float32:
        print("  Gemma su GPU senza bf16: fp32, quindi niente autocast fp16 e niente "
              "GradScaler.\n"
              "  In fp16 il GradScaler scarta gli step con gradienti inf/nan e nei log "
              "compaiono\n"
              "  grad_norm=nan con learning_rate=0: sono step che NON hanno aggiornato "
              "l'adapter.\n"
              "  Costo: circa 2x di tempo per step.")
    elif not bf16_ok:
        print("  GPU pre-Ampere: fp16 (tensor core nativi). Se nei log compaiono "
              "grad_norm=nan e learning_rate=0 gli step vengono scartati: rilancia "
              "con --dtype fp32.")
    pb = approx_params_b(repo_id)
    if pb:
        peso = pb * (0.5 if args.precision == "qlora4bit" else 2)
        print(f"  ~{pb:.0f}B parametri -> ~{peso:.1f} GB di soli pesi in {args.precision} "
              f"(più adapter, ottimizzatore, attivazioni)")
        if peso > vram_gb * 0.6:
            print("  ATTENZIONE: frazione alta della VRAM. Rischio OOM.")

    token = load_hf_token(args.hf_token)
    if token is None:
        print("! Nessun HF_TOKEN: Llama e Gemma sono gated e il download fallirà. "
              "Su Kaggle: Add-ons > Secrets > HF_TOKEN.", file=sys.stderr)

    # --- tokenizer ---------------------------------------------------------
    try:
        tokenizer = AutoTokenizer.from_pretrained(repo_id, token=token, use_fast=True)
    except Exception as e:
        # 403 GatedRepoError: il repo esiste ma l'accesso non e' stato concesso a
        # QUESTO account. Il traceback di huggingface_hub e' lungo tre schermate
        # e nasconde la sola cosa da fare, quindi la si dice qui.
        if "gated" in str(e).lower() or "403" in str(e):
            sys.exit(
                f"ERRORE: {repo_id} e' un repo gated e questo account non ha "
                f"l'accesso.\n"
                f"  1. apri https://huggingface.co/{repo_id} e accetta la licenza "
                f"(per Minerva l'accesso va richiesto e approvato a mano);\n"
                f"  2. verifica che HF_TOKEN nei Kaggle Secrets sia dello STESSO "
                f"account che ha accettato;\n"
                f"  3. oppure usa un alias accessibile: "
                f"{', '.join(MODEL_REGISTRY)}.")
        raise
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    if not getattr(tokenizer, "chat_template", None):
        print(f"  ! {repo_id} non ha chat template (modello BASE): fallback testuale.")

    # --- modello -----------------------------------------------------------
    print(f"Carico il modello ({args.precision})...")
    common_kw = dict(device_map={"": 0}, token=token, low_cpu_mem_usage=True,
                     attn_implementation="eager")   # obbligatorio per Gemma-2 (soft-capping)
    if args.precision == "qlora4bit":
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_use_double_quant=True,
                                   bnb_4bit_compute_dtype=compute_dtype)
        try:
            model, arch_kind = load_backbone(repo_id, compute_dtype,
                                             quantization_config=quant, **common_kw)
        except Exception as e:
            sys.exit(f"ERRORE: caricamento 4-bit fallito: {e}\n"
                     f"Verifica che bitsandbytes sia compatibile con la CUDA dell'immagine "
                     f"Kaggle, oppure ripiega su --precision fp16 (per un 7B+ probabilmente "
                     f"non ci sta in {vram_gb:.0f} GB).")
    else:
        dt = torch.bfloat16 if args.precision == "bf16" else torch.float16
        model, arch_kind = load_backbone(repo_id, dt, **common_kw)

    use_ckpt = not args.no_gradient_checkpointing
    model.config.use_cache = False
    if args.precision == "qlora4bit":
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=use_ckpt,
            gradient_checkpointing_kwargs={"use_reentrant": False} if use_ckpt else None)
    elif use_ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

    target_modules = find_lora_target_modules(model)
    if args.init_adapter:
        # Stadio B: si CONTINUA ad addestrare l'adapter dello stadio A (continued
        # pretraining sul napoletano), non se ne impila un secondo sopra.
        # is_trainable=True e' indispensabile: senza, l'adapter viene caricato
        # congelato e il training non aggiorna niente.
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, args.init_adapter, is_trainable=True)
        print(f"Adapter iniziale caricato da {args.init_adapter} (stadio A -> stadio B)")
    else:
        model = get_peft_model(model, LoraConfig(
            r=args.lora_r, lora_alpha=args.lora_alpha, lora_dropout=args.lora_dropout,
            target_modules=target_modules, bias="none", task_type="CAUSAL_LM",
            modules_to_save=["embed_tokens", "lm_head"] if args.train_embeddings else None))
    model.print_trainable_parameters()

    # --- dati (solo questo layout) -----------------------------------------
    train_rows = load_split(args.split_dir, task.layout, "train", args.max_train_samples)
    dev_full = load_split(args.split_dir, task.layout, "dev", args.max_dev_samples)
    if not train_rows:
        sys.exit(f"ERRORE: nessuna istanza {task.layout} nel train. Controlla --split-dir.")
    print(f"Dati {task.layout}: train {len(train_rows)} | dev {len(dev_full)}")
    controlla_finito(model, tokenizer, train_rows[0]["prompt"], task.layout)

    fixed_samples = []
    if not args.no_sample_generation:
        fixed_samples = [(r["prompt"], r["target"]) for r in dev_full[:3]]

    dev_rows = dev_full
    if args.eval_subset and len(dev_rows) > args.eval_subset:
        import random
        dev_rows = list(dev_rows)
        random.Random(args.seed).shuffle(dev_rows)
        dev_rows = dev_rows[:args.eval_subset]
        print(f"  dev sottocampionato a {len(dev_rows)} per l'eval periodica")

    # --- pesatura lessicale (opzionale) ------------------------------------
    # Senza --lessico si usano ChatDataset/Trainer come prima: e' l'arm di
    # confronto "senza pesatura", non un ripiego.
    lex = None
    if args.lessico:
        from pesi_lessicali import (ChatDatasetPesato, carica_lessico,
                                    make_collate_pesato)
        lex = carica_lessico(args.lessico)
        print(f"Pesatura lessicale attiva: {len(lex['dialettali'])} tipi dialettali, "
              f"peso {args.peso_dial}")

    def costruisci_ds(rows):
        if lex:
            return ChatDatasetPesato(rows, tokenizer, args.max_seq_len, lex,
                                     peso_dial=args.peso_dial)
        return ChatDataset(rows, tokenizer, args.max_seq_len)

    train_ds = costruisci_ds(train_rows)
    dev_ds = costruisci_ds(dev_rows)
    if lex:
        # Passata di controllo su un campione: se la quota di token pesati e'
        # vicina a zero il lessico non sta agganciando i target (tokenizer slow,
        # o forme normalizzate in modo diverso) e la pesatura e' inerte.
        for i in range(min(300, len(train_ds))):
            train_ds[i]
        print("  " + train_ds.riepilogo_pesi())
    if train_ds.n_prefix_mismatch or dev_ds.n_prefix_mismatch:
        print(f"  !! PREFIX MISMATCH: {train_ds.n_prefix_mismatch} train, "
              f"{dev_ds.n_prefix_mismatch} dev (sui primi 200 controllati). Il prompt "
              f"renderizzato non e' un prefisso esatto del testo completo: il masking "
              f"della loss e' disallineato. NON usare questo run, va risolto il "
              f"rendering del chat template per questo modello.")
    if train_ds.n_troncati or dev_ds.n_troncati:
        print(f"  ! troncati a --max-seq-len {args.max_seq_len}: "
              f"{train_ds.n_troncati} train, {dev_ds.n_troncati} dev. "
              f"Un target troncato è un target sbagliato: alza max-seq-len.")

    # --- cadenza di eval DERIVATA (il bug del monolite) --------------------
    eff_batch = args.batch_size * args.grad_accum
    steps_per_epoch = max(1, math.ceil(len(train_rows) / eff_batch))
    max_steps = max(1, math.ceil(args.epochs * steps_per_epoch))
    eval_steps = max(5, steps_per_epoch // max(1, args.evals_per_epoch))
    warmup_steps = max(1, round(args.warmup_ratio * max_steps))
    n_evals = max_steps // eval_steps
    print(f"Batch efficace {eff_batch} | {steps_per_epoch} step/epoca | {max_steps} step totali")
    print(f"eval ogni {eval_steps} step -> ~{n_evals} valutazioni | warmup {warmup_steps} step")
    if n_evals < args.patience + 1:
        print(f"  ! solo ~{n_evals} eval contro patience={args.patience}: l'early stopping non "
              f"potrà mai scattare. Alza --evals-per-epoch o --epochs.")

    metric_name = args.metric
    greater = task.greater_is_better if metric_name == task.metric else metric_name != "loss"
    compute_metrics, preprocess_logits = build_metrics(tokenizer, task)

    run_slug = f"{slug(repo_id)}__{task.layout}"
    out_dir = os.path.join(args.out_dir, run_slug)
    os.makedirs(out_dir, exist_ok=True)

    # --- resume ------------------------------------------------------------
    # /kaggle/working viene azzerato tra le sessioni: per riprendere davvero,
    # l'output della sessione precedente va montato come dataset di input e
    # passato con --resume-dir. Qui lo copiamo in working (Trainer deve scrivere).
    if args.resume_dir and not args.no_resume:
        src = os.path.join(args.resume_dir, run_slug)
        src = src if os.path.isdir(src) else args.resume_dir
        if get_last_checkpoint(src):
            for name in os.listdir(src):
                if name.startswith("checkpoint-"):
                    dst = os.path.join(out_dir, name)
                    if not os.path.exists(dst):
                        shutil.copytree(os.path.join(src, name), dst)
            print(f"Checkpoint copiati da {src} in {out_dir}")
    resume_from = None if args.no_resume else get_last_checkpoint(out_dir)
    if resume_from:
        print(f"Riprendo da {resume_from}")

    training_args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.eval_batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        weight_decay=args.weight_decay,
        max_grad_norm=0.3,
        neftune_noise_alpha=args.neftune_alpha if args.neftune_alpha > 0 else None,
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_strategy="steps",
        save_steps=eval_steps,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model=metric_name,
        greater_is_better=greater,
        logging_steps=max(1, eval_steps // 4),
        report_to="none",
        seed=args.seed,
        data_seed=args.seed,
        # La precisione dell'ottimizzazione segue quella del modello (regola T3):
        # con compute_dtype fp32 non c'e' autocast e non c'e' GradScaler, quindi
        # nessuno step viene scartato per gradienti inf/nan.
        fp16=(compute_dtype == torch.float16),
        bf16=(compute_dtype == torch.bfloat16),
        optim="paged_adamw_8bit",
        remove_unused_columns=False,
        group_by_length=True,
        dataloader_num_workers=2,
    )

    # ORDINE IMPORTANTE: i callback girano nell'ordine della lista, e
    # EarlyStoppingCallback.on_evaluate legge metrics[metric_for_best_model]. Se
    # sta prima di chi aggiunge eval_gen_chrf al dict, non lo trova e si
    # disattiva con "early stopping required metric_for_best_model, but did not
    # find eval_gen_chrf". La selezione del best checkpoint invece funziona in
    # ogni caso, perche' avviene dopo che tutti i callback hanno finito.
    # Id di fine turno, calcolati sui target renderizzati di train: servono a
    # entrambe le generazioni di monitoraggio. Senza, il decoding non si ferma
    # mai prima di max_new_tokens e ogni diagnosi sull'EOS e' falsata.
    EOT = id_fine_turno(tokenizer, train_rows)

    callbacks = []
    if args.gen_metric_n > 0:
        callbacks.append(build_generation_metric_callback(
            TrainerCallback, torch, tokenizer, dev_rows, task,
            n=args.gen_metric_n, eos_ids=EOT))
    # PRIMA di EarlyStoppingCallback, che legge metrics[metric_for_best_model]:
    # se il callback che scrive eval_ctx_acc girasse dopo, non lo troverebbe e
    # l'early stopping si disattiverebbe in silenzio.
    if args.ctx_metric_n > 0:
        from contesto_metrica import build_context_callback
        callbacks.append(build_context_callback(
            TrainerCallback, torch, tokenizer, dev_rows, task,
            n=args.ctx_metric_n, seed=args.seed))
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=args.patience))
    if fixed_samples:
        callbacks.append(build_sample_callback(TrainerCallback, torch, tokenizer,
                                               fixed_samples, task, eos_ids=EOT))

    ClasseTrainer = Trainer
    collate_fn = make_collate_fn(tokenizer.pad_token_id)
    if lex:
        from pesi_lessicali import TrainerPesato
        ClasseTrainer = TrainerPesato
        collate_fn = make_collate_pesato(tokenizer.pad_token_id)

    trainer = ClasseTrainer(model=model, args=training_args,
                      train_dataset=train_ds, eval_dataset=dev_ds,
                      data_collator=collate_fn,
                      compute_metrics=compute_metrics,
                      preprocess_logits_for_metrics=preprocess_logits,
                      callbacks=callbacks)

    print(f"\nAvvio training. Output: {out_dir}\n")
    t0 = time.time()
    interrotto = False
    try:
        trainer.train(resume_from_checkpoint=resume_from)
    except KeyboardInterrupt:
        interrotto = True
        print("\nInterrotto (Ctrl+C / timeout sessione).")
    elapsed = time.time() - t0

    final_dir = os.path.join(out_dir, "adapter_final")
    if not interrotto:
        trainer.save_model(final_dir)
        tokenizer.save_pretrained(final_dir)
    elif trainer.state.best_model_checkpoint:
        final_dir = trainer.state.best_model_checkpoint
        print(f"(interrotto: uso il best checkpoint {final_dir})")

    log = trainer.state.log_history
    for row in log:
        if "eval_loss" in row:
            row["eval_perplexity"] = math.exp(min(row["eval_loss"], 20))
    csv_path = os.path.join(out_dir, "metrics_history.csv")
    if log:
        with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.DictWriter(f, fieldnames=sorted({k for r in log for k in r}),
                               extrasaction="ignore")
            w.writeheader()
            w.writerows(log)

    summary = {
        "task": {"layout": task.layout, "nome": task.nome,
                 "descrizione": task.descrizione, "note": task.note},
        "repo_id": repo_id, "completed": not interrotto,
        "precision": args.precision, "seed": args.seed,
        "hardware": {"gpu": gpu_name, "vram_gb": round(vram_gb, 1), "bf16": bf16_ok,
                     "dtype": nome_dt},
        "architettura_caricata": arch_kind,
        "versioni": _library_versions(),
        "init_adapter": args.init_adapter,
        "due_stadi": bool(args.init_adapter),
        "lora": {"r": args.lora_r, "alpha": args.lora_alpha, "dropout": args.lora_dropout,
                 "target_modules": target_modules,
                 "embeddings_allenati": args.train_embeddings},
        "hyperparams": {"lr": args.lr, "epochs": args.epochs,
                        "effective_batch_size": eff_batch,
                        "steps_per_epoch": steps_per_epoch, "max_steps": max_steps,
                        "eval_steps": eval_steps, "n_evals_previste": n_evals,
                        "warmup_steps": warmup_steps, "max_seq_len": args.max_seq_len,
                        "weight_decay": args.weight_decay,
                        "neftune_alpha": args.neftune_alpha or None,
                        "patience": args.patience},
        "dataset": {"train": len(train_rows), "dev_full": len(dev_full),
                    "dev_in_eval": len(dev_rows),
                    "troncati_train": train_ds.n_troncati, "troncati_dev": dev_ds.n_troncati,
                    "prefix_mismatch_train": train_ds.n_prefix_mismatch,
                    "prefix_mismatch_dev": dev_ds.n_prefix_mismatch,
                    # Provenienza: quale strategia di generazione ha vinto l'arbitraggio in
                    # build_final_dataset.py per ciascun turno. NON usato per filtrare né per
                    # pesare: il training vede tutto il dataset. Registrato qui solo perché la
                    # sezione "costruzione del dataset" del paper deve poter dichiarare la
                    # composizione effettiva di ciò su cui si è addestrato.
                    "provenienza_train": dict(Counter(r.get("fonte", "n/d") for r in train_rows)),
                    "provenienza_dev": dict(Counter(r.get("fonte", "n/d") for r in dev_full))},
        "selezione_checkpoint": {"metrica": metric_name, "greater_is_better": greater,
                                 "best_value": trainer.state.best_metric,
                                 "best_step": trainer.state.best_global_step,
                                 "best_checkpoint": trainer.state.best_model_checkpoint},
        "training_seconds": round(elapsed, 1),
        "final_adapter_dir": final_dir,
        "metrics_history_csv": csv_path if log else None,
        "avvertenza_metriche": "Tutte le metriche di questo run sono in TEACHER FORCING "
                               "(argmax dei logit), proxy per il monitoraggio e l'early "
                               "stopping. I numeri da riportare vanno da uno script di "
                               "valutazione separato su test.json con generate() reale.",
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    }
    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"\n{'Interrotto' if interrotto else 'Fatto'} in {elapsed/60:.1f} min")
    print(f"Adapter: {final_dir}")
    print(f"Miglior {metric_name} (dev): {trainer.state.best_metric}")
    print(f"Riepilogo: {os.path.join(out_dir, 'summary.json')}")
    if interrotto:
        print("Per riprendere: salva la versione del notebook, montane l'output come dataset "
              "e rilancia con --resume-dir /kaggle/input/<nome-dataset>")


Writing /kaggle/working/training/common.py


## Passo 7 — `pretrain_dialect.py`

**Cosa fare:** esegui. Scrive lo script dello stadio A. Non addestra niente adesso.


In [7]:
%%writefile /kaggle/working/training/pretrain_dialect.py
#!/usr/bin/env python3
"""
STADIO A — continued pretraining sul napoletano, prima dell'SFT sui task.

Perche' serve
-------------
L'SFT sui task addestra coppie prompt->target: l'adapter impara a condizionare i
PRIMI token della risposta, ma la distribuzione linguistica sottostante resta
quella del modello di base, cioe' italiana. Effetto osservato nei log: la
generazione parte in napoletano e scivola in italiano man mano che si allontana
dal prompt ("pero' poi ce sta sempre l'incognita che puo' essere un veleno per
la"). Il continued pretraining sposta la DISTRIBUZIONE, non solo il
condizionamento iniziale.

Inoltre usa tutto il testo napoletano disponibile invece delle sole parole di
target: sul pool di train sono ~11.000 parole contro le 6.005 che T3 usa come
supervisione.

Si esegue UNA VOLTA per modello e serve tutti e tre i task:

    python pretrain_dialect.py --model minerva --split-dir /kaggle/working/split
    python finetune_t1_traduzione.py --model minerva --split-dir ... \\
        --init-adapter /kaggle/working/cpt/minerva-7b-instruct-v1.0/adapter_final
    (idem per T2 e T3)

Anti-leakage
------------
Il testo viene ricostruito ESCLUSIVAMENTE dai file train.json dei tre layout.
I pool di dev e test non vengono mai letti per il training: i turni di test
comparirebbero come testo di pretraining, che e' leakage puro e invaliderebbe
tutta la valutazione. Il dev viene usato solo per la perplexity di monitoraggio.

Formato
-------
Righe "PARLANTE: turno napoletano", identiche al formato dei contesti di
T1/T2/T3, cosi' fra stadio A e stadio B non c'e' disallineamento di formato.
Nessun chat template, nessun masking: language modeling puro, loss su tutto.
I turni vengono concatenati e impacchettati in blocchi di --block-size token.
"""

import argparse
import json
import math
import os
import sys
from pathlib import Path

from common import (LAYOUT_DIRS, MODEL_REGISTRY, controlla_finito,
                   find_lora_target_modules, load_backbone, load_hf_token,
                   load_split, render_prompt, resolve_model, scegli_dtype, slug)


def ricostruisci_turni(split_dir, split_name):
    """Ricostruisce i turni napoletani dai file di UN solo split.

    Ogni layout espone il turno in modo diverso, quindi si prende l'unione
    tenendo la ricostruzione piu' lunga per ciascun (conversazione, turn_index):
      T1: target = turno intero
      T2: prefisso (dentro il prompt) + target = turno intero
      T3: target = turno intero
    """
    turni = {}
    for layout in ("T1", "T2", "T3"):
        try:
            rows = load_split(split_dir, layout, split_name)
        except SystemExit:
            continue
        for r in rows:
            chiave = (r["conversazione"], int(r["turn_index"]))
            if layout == "T2":
                marker = "Continua il turno in napoletano: "
                pref = r["prompt"].split(marker)[-1].strip() if marker in r["prompt"] else ""
                testo = (pref + " " + r["target"]).strip()
            else:
                testo = r["target"].strip()
            spk = r.get("speaker", "?")
            if len(testo) > len(turni.get(chiave, ("", ""))[1]):
                turni[chiave] = (spk, testo)
    ordinati = [turni[k] for k in sorted(turni)]
    return ordinati


def costruisci_testo(turni):
    """Una riga per turno, nello stesso formato dei contesti dei tre layout."""
    return "\n".join(f"{spk}: {testo}" for spk, testo in turni)


class BlocchiDataset:
    """Language modeling puro: input_ids == labels, nessun masking."""

    def __init__(self, testo, tok, block_size):
        ids = tok(testo, add_special_tokens=False)["input_ids"]
        n = (len(ids) // block_size) * block_size
        if n == 0:                                  # testo piu' corto di un blocco
            n, block_size = len(ids), len(ids)
        self.blocchi = [ids[i:i + block_size] for i in range(0, n, block_size)]
        self.n_token = len(ids)

    def __len__(self):
        return len(self.blocchi)

    def __getitem__(self, i):
        b = self.blocchi[i]
        return {"input_ids": b, "attention_mask": [1] * len(b), "labels": list(b)}


def collate(batch):
    import torch
    return {k: torch.tensor([b[k] for b in batch], dtype=torch.long) for k in batch[0]}


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1],
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True, help=f"alias ({'/'.join(MODEL_REGISTRY)}) o repo_id")
    ap.add_argument("--split-dir", default="/kaggle/working/split")
    ap.add_argument("--out-dir", default="/kaggle/working/cpt")
    ap.add_argument("--block-size", type=int, default=512)
    ap.add_argument("--epochs", type=float, default=3)
    ap.add_argument("--lr", type=float, default=5e-5,
                    help="piu' basso dell'SFT: qui si sposta la distribuzione, non si "
                         "impara un mapping, e con 11k parole e' facile degradare il modello")
    ap.add_argument("--batch-size", type=int, default=1)
    ap.add_argument("--grad-accum", type=int, default=8)
    ap.add_argument("--lora-r", type=int, default=16)
    ap.add_argument("--lora-alpha", type=int, default=32)
    ap.add_argument("--lora-dropout", type=float, default=0.05)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--hf-token", default=None)
    ap.add_argument("--dtype", choices=["auto", "bf16", "fp16", "fp32"], default="auto",
                    help="auto = fp32 per Gemma senza bf16 nativo: in fp16 il "
                         "GradScaler scarta gli step (grad_norm=nan, lr=0)")
    ap.add_argument("--dump-text", action="store_true",
                    help="salva il testo di pretraining su file, per ispezionarlo")
    a = ap.parse_args()

    import torch
    from transformers import (AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback,
                              Trainer, TrainingArguments, set_seed)
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    set_seed(a.seed)
    repo_id = resolve_model(a.model)
    if not torch.cuda.is_available():
        sys.exit("Nessuna GPU CUDA.")
    cc = torch.cuda.get_device_capability(0)
    bf16_ok = cc[0] >= 8
    dtype, _ = scegli_dtype(repo_id, a.dtype)
    print(f"=== STADIO A — continued pretraining | {repo_id} ===")
    print(f"GPU {torch.cuda.get_device_name(0)} (cc {cc[0]}.{cc[1]}) -> "
          f"{str(dtype).replace('torch.', '')}")
    if dtype == torch.float32:
        print("  (Gemma senza bf16 nativo: fp32. In fp16 il GradScaler scarta gli "
              "step con gradienti inf/nan\n"
              "   e nei log compaiono grad_norm=nan e learning_rate=0, cioe' step "
              "che non aggiornano l'adapter.)")

    token = load_hf_token(a.hf_token)
    tok = AutoTokenizer.from_pretrained(repo_id, token=token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # --- testo: SOLO dal pool di train -------------------------------------
    turni_tr = ricostruisci_turni(a.split_dir, "train")
    turni_dv = ricostruisci_turni(a.split_dir, "dev")
    if not turni_tr:
        sys.exit("Nessun turno ricostruito dal train: controlla --split-dir.")
    testo_tr, testo_dv = costruisci_testo(turni_tr), costruisci_testo(turni_dv)
    parole_tr = len(testo_tr.split())
    print(f"Turni ricostruiti: train {len(turni_tr)} ({parole_tr} parole) | "
          f"dev {len(turni_dv)} ({len(testo_dv.split())} parole)")
    print("  (dev e test non entrano mai nel training: sarebbe leakage)")
    # trasparenza: alcune formule brevissime ricorrono naturalmente in piu' punti
    # del corpus, quindi compaiono sia in train sia in test. Non e' leakage - e'
    # lo stesso fenomeno per cui "sì sì sì" appare in mezza conversazione - ma il
    # numero va riportato invece di essere taciuto.
    turni_te = ricostruisci_turni(a.split_dir, "test")
    comuni = {t for _, t in turni_tr} & {t for _, t in turni_te}
    if comuni:
        print(f"  {len(comuni)} turni identici presenti sia in train sia in test "
              f"(formule ricorrenti, mediana {sorted(len(c.split()) for c in comuni)[len(comuni)//2]} "
              f"parole): non e' leakage di split, e' ricorrenza lessicale nel parlato")

    outdir = Path(a.out_dir) / slug(repo_id)
    outdir.mkdir(parents=True, exist_ok=True)
    if a.dump_text:
        (outdir / "testo_pretraining.txt").write_text(testo_tr, encoding="utf-8")
        print(f"  testo salvato in {outdir / 'testo_pretraining.txt'}")

    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    model, arch = load_backbone(repo_id, dtype, quantization_config=quant,
                               device_map={"": 0}, token=token, low_cpu_mem_usage=True,
                               attn_implementation="eager")
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False})
    target_modules = find_lora_target_modules(model)
    model = get_peft_model(model, LoraConfig(
        r=a.lora_r, lora_alpha=a.lora_alpha, lora_dropout=a.lora_dropout,
        target_modules=target_modules, bias="none", task_type="CAUSAL_LM"))
    model.print_trainable_parameters()
    controlla_finito(model, tok, testo_tr[:2000], "stadio A", rendi=False)

    ds_tr = BlocchiDataset(testo_tr, tok, a.block_size)
    ds_dv = BlocchiDataset(testo_dv, tok, a.block_size) if turni_dv else None
    print(f"Blocchi da {a.block_size} token: train {len(ds_tr)} "
          f"({ds_tr.n_token} token totali)" + (f" | dev {len(ds_dv)}" if ds_dv else ""))
    if len(ds_tr) < 4:
        print("  ! pochissimi blocchi: abbassa --block-size per averne di piu'")

    eff = a.batch_size * a.grad_accum
    spe = max(1, math.ceil(len(ds_tr) / eff))
    max_steps = max(1, math.ceil(a.epochs * spe))
    print(f"Batch efficace {eff} | {spe} step/epoca | {max_steps} step totali")

    args = TrainingArguments(
        output_dir=str(outdir),
        num_train_epochs=a.epochs,
        per_device_train_batch_size=a.batch_size,
        per_device_eval_batch_size=a.batch_size,
        gradient_accumulation_steps=a.grad_accum,
        learning_rate=a.lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        max_grad_norm=0.3,
        # NEFTune e' pensato per l'instruction tuning: qui si modella testo, non
        # si segue un'istruzione, quindi resta disattivato
        eval_strategy="epoch" if ds_dv else "no",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=bool(ds_dv),
        metric_for_best_model="loss" if ds_dv else None,
        greater_is_better=False if ds_dv else None,
        logging_steps=max(1, spe // 4),
        report_to="none",
        seed=a.seed,
        # precisione dell'ottimizzazione = precisione del modello: in fp32 non
        # c'e' GradScaler, quindi nessuno step scartato per gradienti inf/nan.
        fp16=(dtype == torch.float16), bf16=(dtype == torch.bfloat16),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        remove_unused_columns=False,
    )
    trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_dv,
                      data_collator=collate,
                      callbacks=[EarlyStoppingCallback(2)] if ds_dv else None)
    trainer.train()

    final = outdir / "adapter_final"
    trainer.save_model(str(final))
    tok.save_pretrained(str(final))

    log = trainer.state.log_history
    for r in log:
        if "eval_loss" in r:
            r["eval_perplexity"] = math.exp(min(r["eval_loss"], 20))
    ppl = [r["eval_perplexity"] for r in log if "eval_perplexity" in r]
    riepilogo = {
        "stadio": "A — continued pretraining",
        "repo_id": repo_id, "architettura": arch,
        "gpu": torch.cuda.get_device_name(0), "bf16": bf16_ok,
        "dtype": str(dtype).replace("torch.", ""),
        "lora": {"r": a.lora_r, "alpha": a.lora_alpha, "dropout": a.lora_dropout,
                 "n_moduli_target": len(target_modules)},
        "hyperparams": {"lr": a.lr, "epochs": a.epochs, "block_size": a.block_size,
                        "effective_batch_size": eff, "max_steps": max_steps},
        "dati": {"turni_train": len(turni_tr), "parole_train": parole_tr,
                 "token_train": ds_tr.n_token, "blocchi_train": len(ds_tr),
                 "turni_dev": len(turni_dv),
                 "fonte": "solo i file train.json dei tre layout — dev e test esclusi"},
        "perplexity_dev": {"iniziale": round(ppl[0], 2) if ppl else None,
                           "finale": round(ppl[-1], 2) if ppl else None,
                           "migliore": round(min(ppl), 2) if ppl else None},
        "best_metric": trainer.state.best_metric,
        "adapter_final": str(final),
        "_uso": "passalo ai tre script di SFT con --init-adapter",
    }
    (outdir / "summary_cpt.json").write_text(
        json.dumps(riepilogo, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(riepilogo["perplexity_dev"], indent=2))
    print(f"\nAdapter stadio A: {final}")
    print(f"Usalo con:  --init-adapter {final}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/pretrain_dialect.py


## Passo 8 — `lessico.py`

**Cosa fare:** esegui la cella (scrive il file, non addestra niente).

Estrae il lessico allineato italiano->napoletano con IBM Model 1 (EM su
`P(nap|ita)` con token NULL), filtrando su `p >= 0.15` e almeno 3 co-occorrenze.
Dalle 2.568 coppie escono ~467 voci; dal solo train ~383, di cui ~317 con forma
dialettale. Le prime sono pulite: `non->nun` p=1.00, `che->ca` p=0.97,
`perche'->pecche'`, `quindi->quinni`, `cosi'->accussi'`, `vabbe'->vabbuo'`.

Produce anche l'inventario dei **tipi dialettali** (forme napoletane assenti dal
lato italiano del corpus), che serve sia a pesare la loss sia a misurare.

**Anti-leakage:** il lessico e' un artefatto addestrato sui dati. Estratto da
tutto il CSV conterrebbe le corrispondenze osservate in dev e test, e ogni
metrica che lo usa sarebbe contaminata. Con `--split-dir` le coppie vengono
filtrate sulle chiavi `(conversazione, turn_index)` dei `train.json`. La
modalita' `--solo-csv` esiste per l'esplorazione, si dichiara a schermo, e
`carica_lessico` **rifiuta** di aprire un file prodotto cosi'.

In [8]:
%%writefile /kaggle/working/training/lessico.py
#!/usr/bin/env python3
"""
lessico.py — estrae un lessico allineato italiano -> napoletano dai SOLI turni
di train, piu' l'inventario dei tipi lessicali dialettali.

Perche' serve
-------------
"Insegnare il dialetto termine per termine" richiede di sapere QUALI sono i
termini. Non esiste un glossario allegato al dataset, ma il dataset e' un corpus
parallelo allineato a livello di turno: da 2.568 coppie si ricava un lessico per
allineamento statistico (IBM Model 1, EM su P(nap|ita) con token NULL) senza
scrivere a mano nemmeno una voce.

L'output serve a tre cose diverse, tutte a valle:
  1. costruire il dataset dello stadio A2 (dati_lessicali.py);
  2. pesare la loss sui token dialettali (pesi_lessicali.py);
  3. misurare se il dialetto e' stato imparato (recall lessicale, italianismi).

Anti-leakage
------------
Il lessico e' un artefatto ADDESTRATO sui dati: se lo estrai da tutto il CSV,
dentro ci finiscono le corrispondenze osservate in dev e test, e ogni metrica
che lo usa e' contaminata. Con --split-dir le coppie vengono filtrate sulle
chiavi (conversazione, turn_index) presenti nei train.json dei tre layout.
La modalita' --solo-csv esiste per l'ispezione esplorativa e lo dichiara a
schermo: non usarla per produrre il lessico dei run finali.

Uso
---
    python lessico.py --csv dataset_finale.csv --split-dir /kaggle/working/split \\
        --out lessico_train.json
"""

import argparse
import json
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

LAYOUT_DIRS = ("layout1_traduzione_con_contesto", "layout2_completamento_turno",
               "layout3_replica_conversazionale")

# Le apostrofate sono la spina dorsale del napoletano scritto ('o, 'e, pe',
# 'nfatti): un \w+ le spezzerebbe e il lessico verrebbe fuori a pezzi.
TOK = re.compile(r"[a-zàèéìòóùâêîôû'\u2019\-]+")


def tokenizza(s):
    return TOK.findall(str(s).lower().replace("\u2019", "'"))


def chiavi_train(split_dir):
    """(conversazione, turn_index) dei turni che compaiono nel train di almeno
    un layout. Si leggono i JSON dello split invece di ri-splittare il CSV:
    l'unica definizione di 'train' resta quella di split_dataset.py."""
    chiavi = set()
    for d in LAYOUT_DIRS:
        p = Path(split_dir) / d / "train.json"
        if not p.exists():
            print(f"  ! manca {p}", file=sys.stderr)
            continue
        for r in json.loads(p.read_text(encoding="utf-8")):
            chiavi.add((r["conversazione"], int(r["turn_index"])))
    return chiavi


def ibm1(coppie, iterazioni=12):
    """EM su P(nap | ita). Ritorna il dizionario di traduzione lessicale.

    Model 1 ignora l'ordine delle parole: e' un limite che qui non morde, perche'
    italiano e napoletano hanno sintassi vicinissima e le frasi sono corte
    (mediana 4 parole). Model 2 o HMM aggiungerebbero un modello di distorsione
    per un guadagno trascurabile su 2.500 coppie.
    """
    vocab_nap = {w for _, n in coppie for w in n}
    t = defaultdict(lambda: 1.0 / len(vocab_nap))
    for _ in range(iterazioni):
        conta, totale = defaultdict(float), defaultdict(float)
        for it, nap in coppie:
            sorgente = ["<NULL>"] + it
            for nw in nap:
                z = sum(t[(nw, sw)] for sw in sorgente)
                if z == 0:
                    continue
                for sw in sorgente:
                    d = t[(nw, sw)] / z
                    conta[(nw, sw)] += d
                    totale[sw] += d
        t = defaultdict(float, {k: v / totale[k[1]] for k, v in conta.items()
                                if totale[k[1]] > 0})
    return t


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1],
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--csv", required=True)
    ap.add_argument("--split-dir", default=None,
                    help="cartella con layout1_*/layout2_*/layout3_*: filtra sul train")
    ap.add_argument("--solo-csv", action="store_true",
                    help="usa TUTTO il csv (dev e test compresi): solo esplorazione")
    ap.add_argument("--out", default="lessico_train.json")
    ap.add_argument("--p-min", type=float, default=0.15,
                    help="probabilita' minima di traduzione")
    ap.add_argument("--cooc-min", type=int, default=3,
                    help="co-occorrenze minime: sotto le 3 il lessico si riempie di rumore")
    ap.add_argument("--escludi-fonte", nargs="*", default=[],
                    help="fonti da escludere, es. --escludi-fonte gemma4")
    a = ap.parse_args()

    import pandas as pd

    if not a.split_dir and not a.solo_csv:
        sys.exit("Serve --split-dir (oppure --solo-csv se stai solo esplorando).")

    df = pd.read_csv(a.csv)
    n0 = len(df)
    if a.escludi_fonte:
        df = df[~df["fonte"].isin(a.escludi_fonte)]
        print(f"Fonti escluse {a.escludi_fonte}: {n0} -> {len(df)} righe")

    if a.split_dir:
        chiavi = chiavi_train(a.split_dir)
        prima = len(df)
        df = df[[(c, int(i)) in chiavi
                 for c, i in zip(df["conversazione"], df["turn_index"])]]
        print(f"Filtro sul train dello split: {prima} -> {len(df)} righe "
              f"({len(chiavi)} chiavi di train)")
        if len(df) < 0.2 * prima:
            print("  ! pochissime righe sopravvissute: le chiavi del CSV e dello "
                  "split non combaciano, controlla conversazione/turn_index")
    else:
        print("!!! --solo-csv: il lessico include dev e test. Artefatto NON "
              "utilizzabile per training o valutazione.")

    coppie = [(tokenizza(i), tokenizza(n))
              for i, n in zip(df["italiano"], df["napoletano"])]
    coppie = [(i, n) for i, n in coppie if i and n]
    print(f"Coppie allineate: {len(coppie)}")

    freq_it, freq_nap, cooc = Counter(), Counter(), Counter()
    for it, nap in coppie:
        for sw in set(it):
            freq_it[sw] += 1
            for nw in set(nap):
                cooc[(sw, nw)] += 1
        for nw in set(nap):
            freq_nap[nw] += 1

    # Tipi dialettali: forme napoletane che non esistono sul lato italiano del
    # corpus. E' un'approssimazione (una forma napoletana omografa dell'italiano
    # non viene contata) ma e' quella che si puo' calcolare senza un dizionario
    # esterno, e cattura il segnale che ci interessa: 'o, nun, ca, pecche',
    # accussi', cchiu', quanno.
    tipi_dialettali = sorted({w for w in freq_nap if w not in freq_it},
                            key=lambda w: -freq_nap[w])
    massa = sum(freq_nap[w] for w in tipi_dialettali)
    print(f"Tipi dialettali: {len(tipi_dialettali)} "
          f"({massa} occorrenze, {massa/max(1,sum(freq_nap.values())):.1%} dei token napoletani)")

    t = ibm1(coppie)
    voci = []
    for (nw, sw), p in t.items():
        if sw == "<NULL>" or nw == sw or p < a.p_min or cooc[(sw, nw)] < a.cooc_min:
            continue
        voci.append({"italiano": sw, "napoletano": nw, "p": round(p, 4),
                     "cooc": cooc[(sw, nw)], "freq_it": freq_it[sw],
                     "freq_nap": freq_nap[nw],
                     "dialettale": nw in set(tipi_dialettali)})
    voci.sort(key=lambda v: (-v["cooc"], -v["p"]))
    print(f"Voci di lessico: {len(voci)} "
          f"({sum(v['dialettale'] for v in voci)} con forma dialettale)")

    # Una entrata italiana puo' avere piu' rese (di -> 'e / 'o): si tengono
    # tutte, e chi consuma il lessico decide se prendere l'argmax o l'insieme.
    fuori = {
        "_meta": {"fonte_csv": a.csv, "split_dir": a.split_dir,
                  "solo_csv": a.solo_csv, "fonti_escluse": a.escludi_fonte,
                  "coppie": len(coppie), "p_min": a.p_min, "cooc_min": a.cooc_min},
        "tipi_dialettali": tipi_dialettali,
        "freq_nap": {w: freq_nap[w] for w in tipi_dialettali},
        # Vocabolario napoletano COMPLETO del train (non solo i tipi dialettali):
        # serve a misurare le forme inedite, cioe' le parole che il modello
        # produce e che non compaiono da nessuna parte nei dati di training.
        # Senza questo, forme inventate per analogia morfologica come
        # "puparuolo" o "cusinajo" passano per napoletano valido in ogni metrica.
        "vocabolario_nap": dict(freq_nap),
        "lessico": voci,
    }
    Path(a.out).write_text(json.dumps(fuori, ensure_ascii=False, indent=1),
                           encoding="utf-8")
    print(f"\nScritto {a.out}")
    print("Prime 25 voci:")
    for v in voci[:25]:
        print(f"  {v['italiano']:>10s} -> {v['napoletano']:<12s} "
              f"p={v['p']:.2f} cooc={v['cooc']}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/lessico.py


## Passo 9 — `dati_lessicali.py`

**Cosa fare:** esegui la cella (scrive il file).

Costruisce gli item dello stadio A2 dai turni di train. Tre forme:

| tipo | quanti (indicativo) | cosa insegna |
|---|---|---|
| `glossa` | ~250 | la corrispondenza nuda `italiano -> napoletano` |
| `cloze` | ~2.700 | il termine **dentro** la frase e dentro la conversazione |
| `turno_breve` | ~1.000 | turni <= 8 parole ad alta densita' dialettale, interi |

Il `cloze` e' l'item centrale: contesto napoletano dei 3 turni precedenti +
frase italiana + resa napoletana con un buco -> la parola mancante. Il modello
deve produrre la forma dialettale *e* capire quando e' appropriata, che e'
esattamente il condizionamento pragmatico che serve a T2 e T3. La `glossa` resta
minoritaria: da sola produrrebbe un glossario parlante.

**Anti-leakage:** solo turni di train, lessico di train, e contesto costruito
solo con turni di train. Se un turno precedente non e' di train la finestra di
contesto si **tronca** invece di saltarlo: un contesto con un buco in mezzo e'
una conversazione che non e' mai esistita.

In [9]:
%%writefile /kaggle/working/training/dati_lessicali.py
#!/usr/bin/env python3
"""
dati_lessicali.py — costruisce il dataset dello STADIO A2 (iniezione lessicale),
il pezzo che manca fra il continued pretraining e l'SFT sui tre task.

Il problema che risolve
-----------------------
Lo stadio A (pretrain_dialect.py) sposta la distribuzione con language modeling
puro, ma non insegna nessuna CORRISPONDENZA: il modello vede napoletano, non
vede che "pecche'" sta al posto di "perche'". L'SFT sui task insegna la
corrispondenza ma a livello di frase intera, dove il segnale lessicale e'
diluito su decine di token e la strada piu' facile per abbassare la loss e'
copiare l'italiano. Nel mezzo manca la supervisione a livello di TERMINE.

Tre forme di item, dalla piu' isolata alla piu' contestuale:

  glossa       "In napoletano «perche'» si dice ___"  -> pecche'
               Ancora la corrispondenza. Da sola produrrebbe un glossario
               parlante, per questo e' la quota minoritaria.

  cloze        contesto napoletano + frase italiana + resa napoletana con un
               buco -> la parola dialettale mancante.
               E' l'item centrale: il termine va prodotto DENTRO la frase e
               dentro la conversazione, quindi il modello impara anche quando
               quella forma e' appropriata, che e' esattamente la richiesta del
               prompt con contesto pragmatico.

  turno_breve  turni corti (<= --max-parole) con almeno una parola dialettale,
               tradotti per intero. Fa da ponte verso T1/T2/T3: stessa forma di
               compito, ma su una finestra dove il termine dialettale pesa molto
               nella loss.

Gli item escono nello stesso formato dei tre layout ({prompt, target, layout,
conversazione, turn_index}), quindi ChatDataset e il resto di common.py li
digeriscono senza modifiche. Serve solo aggiungere la voce in LAYOUT_DIRS:

    LAYOUT_DIRS["A2"] = "stadio_a2_lessico"

Anti-leakage
------------
Solo turni di train (chiavi lette dai train.json dello split), lessico
estratto dal train, e contesto conversazionale costruito solo con turni di
train: se il contesto pescasse dai turni vicini senza filtro, i target di
dev/test entrerebbero nel training come testo di contesto.
Il dev di A2 e' un campione tenuto fuori dagli item di train (--quota-dev),
serve solo per l'early stopping dello stadio A2.

Uso
---
    python lessico.py --csv dataset_finale.csv --split-dir split --out lessico_train.json
    python dati_lessicali.py --csv dataset_finale.csv --split-dir split \\
        --lessico lessico_train.json --out-dir split/stadio_a2_lessico
"""

import argparse
import json
import random
import re
import sys
from collections import Counter
from pathlib import Path

LAYOUT_DIRS = ("layout1_traduzione_con_contesto", "layout2_completamento_turno",
               "layout3_replica_conversazionale")
TOK = re.compile(r"[a-zàèéìòóùâêîôû'\u2019\-]+")

ISTR_GLOSSA = ("Il napoletano usa una forma diversa dall'italiano per questa "
               "parola. Scrivi soltanto la forma napoletana.")
ISTR_CLOZE = ("Completa la frase napoletana inserendo la parola che manca al "
              "posto di ___. Scrivi soltanto quella parola.")
ISTR_TURNO = "Traduci in napoletano."


def norm(s):
    return str(s).replace("\u2019", "'")


def tokenizza(s):
    return TOK.findall(norm(s).lower())


def chiavi_train(split_dir):
    chiavi = set()
    for d in LAYOUT_DIRS:
        p = Path(split_dir) / d / "train.json"
        if not p.exists():
            continue
        for r in json.loads(p.read_text(encoding="utf-8")):
            chiavi.add((r["conversazione"], int(r["turn_index"])))
    if not chiavi:
        sys.exit(f"Nessun train.json trovato sotto {split_dir}")
    return chiavi


def costruisci_contesto(righe_conv, i, chiavi_ok, n=3):
    """Fino a n turni napoletani precedenti, ma solo quelli di train.

    Se un turno precedente non e' di train si TRONCA la finestra invece di
    saltarlo: un contesto con un buco in mezzo e' una conversazione che non e'
    mai esistita, e il modello impara a raccordare turni non adiacenti.
    """
    fuori = []
    for j in range(i - 1, max(-1, i - 1 - n), -1):
        r = righe_conv[j]
        if (r["conversazione"], int(r["turn_index"])) not in chiavi_ok:
            break
        fuori.append(f"{r['speaker']}: {norm(r['napoletano'])}")
    return list(reversed(fuori))


def blocco_contesto(ctx):
    if not ctx:
        return ""
    return "Conversazione finora (in napoletano):\n" + "\n".join(ctx) + "\n\n"


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1],
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--csv", required=True)
    ap.add_argument("--split-dir", required=True)
    ap.add_argument("--lessico", required=True, help="output di lessico.py")
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--max-parole", type=int, default=8,
                    help="soglia per gli item turno_breve")
    ap.add_argument("--cloze-per-turno", type=int, default=2,
                    help="quanti buchi diversi generare dallo stesso turno")
    ap.add_argument("--min-freq-glossa", type=int, default=3,
                    help="frequenza minima nel train perche' un termine diventi glossa")
    ap.add_argument("--richiedi-contesto", action="store_true",
                    help="scarta gli item di cloze e turno_breve senza blocco di "
                         "contesto. Con un train sparso la finestra si tronca spesso "
                         "e la maggioranza degli item finisce senza contesto: quegli "
                         "item insegnano a produrre napoletano senza guardare indietro, "
                         "che e' il fallimento osservato su T2 e T3")
    ap.add_argument("--quota-glossa", type=float, default=None,
                    help="con --richiedi-contesto, tetto alla quota di item glossa "
                         "(che per natura non hanno contesto), es. 0.10")
    ap.add_argument("--quota-dev", type=float, default=0.08)
    ap.add_argument("--escludi-fonte", nargs="*", default=[])
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()

    import pandas as pd

    rnd = random.Random(a.seed)
    lex = json.loads(Path(a.lessico).read_text(encoding="utf-8"))
    if lex["_meta"].get("solo_csv"):
        sys.exit("Il lessico e' stato estratto con --solo-csv (include dev/test). "
                 "Rigeneralo con --split-dir prima di costruire i dati.")
    dialettali = set(lex["tipi_dialettali"])
    freq_nap = lex["freq_nap"]

    df = pd.read_csv(a.csv)
    if a.escludi_fonte:
        df = df[~df["fonte"].isin(a.escludi_fonte)]
    df = df.sort_values(["conversazione", "turn_index"]).reset_index(drop=True)
    chiavi = chiavi_train(a.split_dir)

    per_conv = {c: g.to_dict("records") for c, g in df.groupby("conversazione")}
    items, conta = [], Counter()

    # ---- glossa ------------------------------------------------------------
    # Una voce per forma napoletana: si prende l'italiano con la co-occorrenza
    # piu' alta, cosi' "'o" non genera tre item quasi identici (il/lo/nel).
    migliore = {}
    for v in lex["lessico"]:
        if not v["dialettale"] or freq_nap.get(v["napoletano"], 0) < a.min_freq_glossa:
            continue
        pre = migliore.get(v["napoletano"])
        if pre is None or v["cooc"] > pre["cooc"]:
            migliore[v["napoletano"]] = v
    for nw, v in migliore.items():
        items.append({"layout": "A2", "tipo": "glossa",
                      "conversazione": "_lessico", "turn_index": -1,
                      "prompt": f"{ISTR_GLOSSA}\n\nItaliano: {v['italiano']}\nNapoletano:",
                      "target": nw})
        conta["glossa"] += 1

    # ---- cloze e turno_breve ----------------------------------------------
    for conv, righe in per_conv.items():
        for i, r in enumerate(righe):
            chiave = (r["conversazione"], int(r["turn_index"]))
            if chiave not in chiavi:
                continue
            ita, nap = norm(r["italiano"]).strip(), norm(r["napoletano"]).strip()
            if not ita or not nap:
                continue
            parole = nap.split()
            idx_dial = [k for k, p in enumerate(parole)
                        if TOK.findall(p.lower()) and TOK.findall(p.lower())[0] in dialettali]
            if not idx_dial:
                continue                      # nessun segnale dialettale: si salta
            ctx = costruisci_contesto(righe, i, chiavi)

            for k in rnd.sample(idx_dial, min(a.cloze_per_turno, len(idx_dial))):
                bucato = list(parole)
                bucato[k] = "___"
                items.append({
                    "layout": "A2", "tipo": "cloze",
                    "conversazione": conv, "turn_index": int(r["turn_index"]),
                    "prompt": (f"{ISTR_CLOZE}\n\n{blocco_contesto(ctx)}"
                               f"Italiano: {ita}\nNapoletano: {' '.join(bucato)}\n"
                               f"Parola mancante:"),
                    "target": parole[k]})
                conta["cloze"] += 1

            if len(parole) <= a.max_parole:
                items.append({
                    "layout": "A2", "tipo": "turno_breve",
                    "conversazione": conv, "turn_index": int(r["turn_index"]),
                    "prompt": (f"{ISTR_TURNO}\n\n{blocco_contesto(ctx)}"
                               f"Italiano: {ita}\nNapoletano:"),
                    "target": nap})
                conta["turno_breve"] += 1

    # --- filtro sul contesto -----------------------------------------------
    if a.richiedi_contesto:
        prima = len(items)
        items = [i for i in items
                 if i["tipo"] == "glossa" or "Conversazione finora" in i["prompt"]]
        conta_f = Counter(i["tipo"] for i in items)
        print(f"\n--richiedi-contesto: {prima} -> {len(items)} item "
              f"({dict(conta_f)})")
        if a.quota_glossa is not None:
            gl = [i for i in items if i["tipo"] == "glossa"]
            altri = [i for i in items if i["tipo"] != "glossa"]
            tetto = int(a.quota_glossa * len(altri) / max(1e-9, 1 - a.quota_glossa))
            if len(gl) > tetto:
                gl = rnd.sample(gl, tetto)
                print(f"  glossa ridotta a {len(gl)} item "
                      f"(tetto {a.quota_glossa:.0%} del totale)")
            items = gl + altri
        senza = sum(1 for i in items if "Conversazione finora" not in i["prompt"])
        print(f"  item senza contesto residui: {senza}/{len(items)} "
              f"({senza/max(1,len(items)):.1%}) — sono le glossa")

    rnd.shuffle(items)
    n_dev = int(len(items) * a.quota_dev)
    dev, train = items[:n_dev], items[n_dev:]

    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    for nome, dati in (("train", train), ("dev", dev)):
        (out / f"{nome}.json").write_text(
            json.dumps(dati, ensure_ascii=False, indent=1), encoding="utf-8")

    print(f"Item per tipo: {dict(conta)}")
    print(f"Totale {len(items)} -> train {len(train)}, dev {len(dev)}")
    print(f"Scritto in {out}")
    print("\nEsempio cloze:\n" + "-" * 60)
    ex = next(i for i in items if i["tipo"] == "cloze")
    print(ex["prompt"] + f"\n--- TARGET: {ex['target']}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/dati_lessicali.py


## Passo 10 — `pesi_lessicali.py`

**Cosa fare:** esegui la cella (scrive il file).

Due cose, entrambe agganciate a `common.py` dalle patch che sono gia' nella
cella di `common.py` di questo notebook:

**1. Pesatura della loss.** In un turno di 12 parole di cui 3 dialettali, la
cross-entropy media premia chi indovina le 9 parole condivise con l'italiano.
Pesare i token dialettali (3.0 contro 1.0) sposta il gradiente dove il dialetto
vive. L'attribuzione usa `offset_mapping` del tokenizer fast: e' l'unico modo
esatto di sapere quali sottoparole appartengono a una forma dialettale. Con un
tokenizer slow la pesatura si disattiva e lo dichiara, invece di indovinare i
confini e pesare i token sbagliati. La media e' **ponderata, non sommata**: la
scala della loss resta confrontabile con i run senza pesatura e il lr non va
ritoccato.

**2. Metriche lessicali.** chrF sale anche solo copiando l'italiano (baseline
copia-italiano: 35,92) perche' le due varieta' condividono gran parte dei
caratteri. Queste no:

| metrica | copia-italiano | riferimento umano |
|---|---|---|
| recall dialettale | 0,001 | 1,000 |
| tasso italianismi | 1,000 | 0,043 |
| tasso copia | 1,000 | 0,229 |

Il **tasso di italianismi** e' la piu' diagnostica: fra le parole italiane per
cui il lessico conosce una resa dialettale, quante restano in forma italiana
nell'output. Misura direttamente lo scivolamento verso l'italiano, e ha fondo
scala e tetto noti.

In [10]:
%%writefile /kaggle/working/training/pesi_lessicali.py
#!/usr/bin/env python3
"""
pesi_lessicali.py — due cose che vanno dentro common.py:

  1. PESATURA DELLA LOSS sui token dialettali.
     In un turno di 12 parole di cui 3 dialettali, la cross-entropy media
     premia chi indovina le 9 parole condivise con l'italiano: copiare porta
     gia' molto lontano. Pesare i token dialettali (peso --peso-dial contro 1.0)
     cambia il gradiente dove il dialetto vive davvero. E' la versione
     "termine per termine" applicata dentro l'SFT, senza cambiare i task.

  2. METRICHE LESSICALI.
     chrF sale anche solo copiando l'italiano (baseline copia-italiano: 35,92)
     perche' le due varieta' condividono gran parte dei caratteri. Serve una
     misura che risponda a "ha imparato le parole?":

       recall_dialettale   quanti dei tipi dialettali del riferimento compaiono
                           nell'ipotesi (multiset, non solo presenza)
       precisione_dial.    quanti dei tipi dialettali prodotti sono attestati
                           nel riferimento: sanziona chi spara 'o e nun a caso
       tasso_italianismi   fra le parole italiane per cui il lessico conosce una
                           resa dialettale, quante restano in forma italiana
                           nell'ipotesi. E' la metrica piu' diagnostica: misura
                           esattamente lo scivolamento verso l'italiano
       tasso_copia         ipotesi identiche alla frase italiana di partenza

Uso come libreria, dentro common.py
-----------------------------------
    from pesi_lessicali import (ChatDatasetPesato, make_collate_pesato,
                                TrainerPesato, carica_lessico)

    lex = carica_lessico(args.lessico)                      # None se non passato
    Dataset = ChatDatasetPesato if lex else ChatDataset
    ds_tr = Dataset(rows_tr, tok, cfg.max_seq_len, lessico=lex, peso_dial=args.peso_dial)
    collate = make_collate_pesato(pad_id) if lex else make_collate_fn(pad_id)
    Tr = TrainerPesato if lex else Trainer

I pesi viaggiano come colonna del batch, quindi TrainingArguments deve avere
remove_unused_columns=False (nel tuo common.py c'e' gia').

Autotest
--------
    python pesi_lessicali.py --csv dataset_finale.csv --lessico lessico_train.json
Valuta la baseline copia-italiano e il riferimento umano con le stesse metriche:
serve a vedere il fondo scala prima di leggere i numeri di un modello.
"""

from __future__ import annotations

import argparse
import json
import re
from collections import Counter
from pathlib import Path

TOK = re.compile(r"[a-zàèéìòóùâêîôû'\u2019\-]+")


def norm(s):
    return str(s).replace("\u2019", "'")


def tokenizza(s):
    return TOK.findall(norm(s).lower())


def carica_lessico(path):
    """Ritorna {'dialettali': set, 'it2nap': {ita: set(nap dialettali)}}."""
    if not path:
        return None
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    if d["_meta"].get("solo_csv"):
        raise SystemExit("Lessico estratto con --solo-csv: contiene dev/test, "
                         "non usarlo ne' per pesare la loss ne' per valutare.")
    it2nap = {}
    for v in d["lessico"]:
        if v["dialettale"]:
            it2nap.setdefault(v["italiano"], set()).add(v["napoletano"])
    return {"dialettali": set(d["tipi_dialettali"]), "it2nap": it2nap}


# --------------------------------------------------------------------------- #
# 1. Pesatura della loss
# --------------------------------------------------------------------------- #

def _span_dialettali(target, dialettali):
    """Span di caratteri (dentro `target`) occupati da parole dialettali."""
    return [(m.start(), m.end()) for m in TOK.finditer(norm(target).lower())
            if m.group() in dialettali]


class ChatDatasetPesato:
    """Come ChatDataset, piu' una colonna `pesi` per-token.

    Attribuzione via offset_mapping del tokenizer fast: e' l'unico modo esatto
    di sapere quali token coprono quali caratteri, e quindi quali sottoparole
    appartengono a una forma dialettale. Con un tokenizer slow gli offset non
    esistono: si ricade su pesi uniformi e lo si dichiara, invece di indovinare
    i confini e pesare i token sbagliati.
    """

    def __init__(self, rows, tokenizer, max_len, lessico, peso_dial=3.0,
                 render=None):
        if render is None:                      # import locale: evita cicli
            from common import render_prompt
            render = render_prompt
        self.rows, self.tok, self.max_len = rows, tokenizer, max_len
        self.dial = lessico["dialettali"]
        self.peso_dial = peso_dial
        self.render = render
        self.fast = bool(getattr(tokenizer, "is_fast", False))
        self.n_troncati = 0
        self.n_prefix_mismatch = 0
        self.n_token_pesati = 0
        self.n_token_target = 0
        if not self.fast:
            print("! tokenizer non-fast: nessun offset_mapping, pesi uniformi "
                  "(la pesatura lessicale e' DISATTIVATA per questo run)")
        for k, r in enumerate(rows[:200]):
            p = self.tok(self.render(self.tok, r["prompt"]),
                         add_special_tokens=False)["input_ids"]
            f = self.tok(self.render(self.tok, r["prompt"], r["target"]),
                         add_special_tokens=False)["input_ids"]
            if len(f) > max_len:
                self.n_troncati += 1
            if f[:len(p)] != p:
                self.n_prefix_mismatch += 1

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        testo_p = self.render(self.tok, r["prompt"])
        testo_f = self.render(self.tok, r["prompt"], r["target"])
        n_prompt = len(self.tok(testo_p, add_special_tokens=False)["input_ids"])

        enc = self.tok(testo_f, add_special_tokens=False,
                       return_offsets_mapping=self.fast)
        ids = enc["input_ids"][:self.max_len]
        labels = list(ids)
        for j in range(min(n_prompt, len(labels))):
            labels[j] = -100

        pesi = [1.0] * len(ids)
        if self.fast:
            off = enc["offset_mapping"][:len(ids)]
            base = testo_f.rfind(norm(r["target"]))
            if base >= 0:
                span = [(base + s, base + e)
                        for s, e in _span_dialettali(r["target"], self.dial)]
                for j, (a, b) in enumerate(off):
                    if labels[j] == -100 or a == b:
                        continue
                    self.n_token_target += 1
                    if any(a < fe and b > fs for fs, fe in span):
                        pesi[j] = self.peso_dial
                        self.n_token_pesati += 1
        return {"input_ids": ids, "attention_mask": [1] * len(ids),
                "labels": labels, "pesi": pesi}

    def riepilogo_pesi(self):
        if not self.n_token_target:
            return "pesatura: nessun token osservato (dataset non ancora iterato)"
        q = self.n_token_pesati / self.n_token_target
        return (f"pesatura: {self.n_token_pesati}/{self.n_token_target} token di "
                f"target dialettali ({q:.1%}), peso {self.peso_dial}")


def make_collate_pesato(pad_id):
    import torch

    def collate(batch):
        n = max(len(b["input_ids"]) for b in batch)
        ids, attn, lab, pesi = [], [], [], []
        for b in batch:
            k = n - len(b["input_ids"])
            ids.append(b["input_ids"] + [pad_id] * k)
            attn.append(b["attention_mask"] + [0] * k)
            lab.append(b["labels"] + [-100] * k)
            pesi.append(b["pesi"] + [0.0] * k)
        return {"input_ids": torch.tensor(ids, dtype=torch.long),
                "attention_mask": torch.tensor(attn, dtype=torch.long),
                "labels": torch.tensor(lab, dtype=torch.long),
                "pesi": torch.tensor(pesi, dtype=torch.float)}
    return collate


def _trainer_pesato():
    """Costruito a runtime: importare Trainer al top del modulo costerebbe il
    caricamento di transformers anche a chi usa solo le metriche."""
    import torch
    from transformers import Trainer

    class TrainerPesato(Trainer):
        """Loss pesata sui token dialettali, con la normalizzazione CORRETTA
        rispetto all'accumulo di gradiente.

        Il punto delicato: transformers guarda se il forward del modello accetta
        **kwargs (un PeftModel lo accetta sempre) e in quel caso NON divide la
        loss per gli step di accumulo, perche' si aspetta una loss gia'
        normalizzata su `num_items_in_batch` (il numero di token supervisionati
        nell'intera finestra di accumulo). Restituire una media per micro-batch
        e ignorare num_items_in_batch gonfia i gradienti di un fattore
        grad_accum: si vede in grad_norm a tre cifre e in max_grad_norm che
        clippa a ogni step, quindi il lr effettivo non e' quello impostato.

        Qui i pesi vengono RINORMALIZZATI a media 1 sui token supervisionati
        del micro-batch, poi si somma e si divide per num_items_in_batch. Cosi':
          * la scala resta identica a quella di un run non pesato (media 1),
            quindi il lr non va ritoccato e i log sono confrontabili;
          * la semantica dell'accumulo e' quella attesa da transformers;
          * il RAPPORTO fra token dialettali e non dialettali resta peso_dial.
        """

        def compute_loss(self, model, inputs, return_outputs=False,
                         num_items_in_batch=None, **kw):
            pesi = inputs.pop("pesi", None)
            labels = inputs.get("labels")
            out = model(**inputs)
            # shift standard del causal LM: la posizione t predice t+1
            lg = out.logits[:, :-1, :].contiguous()
            lb = labels[:, 1:].contiguous()
            perdite = torch.nn.functional.cross_entropy(
                lg.view(-1, lg.size(-1)).float(), lb.view(-1),
                ignore_index=-100, reduction="none").view(lb.shape)

            maschera = (lb != -100).float()
            n_sup = maschera.sum().clamp(min=1.0)
            if pesi is None:
                w = maschera
            else:
                w = pesi[:, 1:].contiguous() * maschera
                w = w * (n_sup / w.sum().clamp(min=1.0))   # media 1 sui supervisionati

            if num_items_in_batch is None:
                # nessun accumulo dichiarato: media ponderata, come prima
                loss = (perdite * w).sum() / n_sup
            else:
                # transformers sommera' i micro-batch senza dividere
                loss = (perdite * w).sum() / num_items_in_batch
            return (loss, out) if return_outputs else loss

    return TrainerPesato


class _Lazy:
    def __call__(self, *a, **k):
        return _trainer_pesato()(*a, **k)

    def __mro_entries__(self, bases):
        return (_trainer_pesato(),)


TrainerPesato = _Lazy()


# --------------------------------------------------------------------------- #
# 2. Metriche lessicali
# --------------------------------------------------------------------------- #

def valuta(fonti_ita, riferimenti, ipotesi, lessico):
    """Metriche lessicali su una lista di triple (italiano, riferimento, ipotesi)."""
    dial, it2nap = lessico["dialettali"], lessico["it2nap"]
    rec_num = rec_den = pre_num = pre_den = 0
    ital_num = ital_den = 0
    copie = 0
    for src, rif, ipo in zip(fonti_ita, riferimenti, ipotesi):
        t_src, t_rif, t_ipo = tokenizza(src), tokenizza(rif), tokenizza(ipo)
        c_rif = Counter(w for w in t_rif if w in dial)
        c_ipo = Counter(w for w in t_ipo if w in dial)
        rec_num += sum((c_rif & c_ipo).values()); rec_den += sum(c_rif.values())
        pre_num += sum((c_rif & c_ipo).values()); pre_den += sum(c_ipo.values())
        set_ipo = set(t_ipo)
        for w in t_src:
            if w not in it2nap:
                continue                      # nessuna resa dialettale attestata
            ital_den += 1
            # italianismo: la forma italiana e' rimasta e nessuna resa dialettale
            # nota compare al suo posto
            if w in set_ipo and not (it2nap[w] & set_ipo):
                ital_num += 1
        if norm(ipo).strip().lower() == norm(src).strip().lower():
            copie += 1
    n = max(1, len(ipotesi))
    r = rec_num / rec_den if rec_den else 0.0
    p = pre_num / pre_den if pre_den else 0.0
    # La precisione e' illeggibile quando il denominatore e' minuscolo: la
    # baseline copia-italiano produce 13 forme dialettali per caso (omografie) e
    # ne "azzecca" 12, quindi precisione 0.92 accanto a un recall di 0.002. Il
    # numero non e' sbagliato, e' privo di significato: va letto solo insieme al
    # conteggio, e sotto 30 forme prodotte si dichiara non interpretabile.
    avvertenza = None
    if pre_den < 30:
        avvertenza = (f"precisione_dialettale calcolata su sole {pre_den} forme "
                      f"dialettali prodotte: non interpretabile")
    return {
        "recall_dialettale": round(r, 4),
        "precisione_dialettale": round(p, 4),
        "f1_dialettale": round(2 * p * r / (p + r), 4) if p + r else 0.0,
        "tasso_italianismi": round(ital_num / ital_den, 4) if ital_den else None,
        "tasso_copia": round(copie / n, 4),
        "_conteggi": {"tipi_dial_riferimento": rec_den, "tipi_dial_ipotesi": pre_den,
                      "occasioni_lessicali": ital_den, "n": len(ipotesi)},
        "_avvertenza": avvertenza,
    }


def main():
    ap = argparse.ArgumentParser(description="autotest delle metriche lessicali")
    ap.add_argument("--csv", required=True)
    ap.add_argument("--lessico", required=True)
    ap.add_argument("--escludi-fonte", nargs="*", default=[])
    a = ap.parse_args()

    import pandas as pd
    lex = carica_lessico(a.lessico)
    df = pd.read_csv(a.csv)
    if a.escludi_fonte:
        df = df[~df["fonte"].isin(a.escludi_fonte)]
    ita, nap = df["italiano"].tolist(), df["napoletano"].tolist()

    print("Baseline COPIA-ITALIANO (l'ipotesi e' la frase italiana):")
    print(json.dumps(valuta(ita, nap, ita, lex), ensure_ascii=False, indent=1))
    print("\nTetto: RIFERIMENTO UMANO come ipotesi (controllo di sanita'):")
    print(json.dumps(valuta(ita, nap, nap, lex), ensure_ascii=False, indent=1))
    for f, g in df.groupby("fonte"):
        print(f"\nSolo fonte={f} ({len(g)} righe), baseline copia:")
        print(json.dumps(valuta(g["italiano"].tolist(), g["napoletano"].tolist(),
                                g["italiano"].tolist(), lex),
                         ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/pesi_lessicali.py


## Passo 10b — `contesto_metrica.py`

**Cosa fare:** esegui (scrive il file).

Misura se il modello **usa** il contesto conversazionale, invece di produrre
napoletano plausibile in astratto. Per ogni item di dev calcola la
log-verosimiglianza del target due volte, con il contesto reale e con il contesto
di un altro punto della conversazione:

    delta = logP(target | contesto vero) - logP(target | contesto sbagliato)
    acc   = quota di item con delta > 0

`acc` a 0,50 significa contesto ignorato. Due forward per item, nessuna
generazione: si puo' tenere accesa a ogni valutazione e usarla per selezionare il
checkpoint con `--metric ctx_acc`.

Serve perche' nel run T2 precedente le generazioni erano napoletano fluente e
morfologicamente corretto, ma semanticamente slegate dal contesto — e `chrF` non
lo vedeva (20,88 in teacher forcing su continuazioni che parlavano d'altro).

In [11]:
%%writefile /kaggle/working/training/contesto_metrica.py
#!/usr/bin/env python3
"""
contesto_metrica.py — misura se il modello USA il contesto conversazionale,
durante il training e come metrica di selezione del checkpoint.

Premessa onesta
---------------
Una metrica non fa fare niente al modello: non c'e' nessuna metrica che "lo
costringa" a stare sul senso. Cio' che il modello fa lo decidono l'obiettivo e i
dati. Quello che una metrica decide e' **quale checkpoint tieni**, e su questo
task e' una leva vera: selezionare su chrF premia il checkpoint che indovina le
parole del riferimento, e su T2/T3, dove il riferimento e' uno fra molti validi,
quel checkpoint e' spesso quello generico o quello che ricopia il prefisso.
Selezionare su una metrica di coerenza tiene un checkpoint diverso.

La misura: delta di verosimiglianza contesto vero vs contesto sbagliato
----------------------------------------------------------------------
Per ogni item di dev si calcola la log-verosimiglianza del target per token, due
volte:

    L_vero  = log P(target | contesto REALE)
    L_falso = log P(target | contesto di un ALTRO punto della conversazione)

    delta = L_vero - L_falso        (per token, quindi non premia i target corti)
    acc   = quota di item con L_vero > L_falso

Se il modello ignora il contesto, delta ~ 0 e acc ~ 0.50: sta producendo un
napoletano plausibile in astratto, slegato da cio' che e' stato detto prima. E'
esattamente il "spara termini a caso" nella sua forma misurabile. Se delta > 0 e
acc sale, il contesto sta entrando nella predizione.

E' l'immagine speculare della contrastiva che hai gia' in evaluate_task.py: la'
i distrattori stanno sul lato del TARGET (quale replica e' quella vera), qui
stanno sul lato del CONTESTO (quale conversazione ha prodotto questa replica).
Le due cose non sono ridondanti: un modello puo' riconoscere la replica giusta
perche' e' l'unica in napoletano fluente, senza usare il contesto.

Costo: due forward per item, nessuna generazione. Su 64 item e' l'ordine di
grandezza di una eval in teacher forcing, quindi si puo' tenere accesa a ogni
valutazione.

Cosa NON risolve
----------------
Con ~450-750 istanze di training la coerenza pragmatica non si insegna: viene dal
modello di base. Questa metrica serve a **non distruggerla** — a scegliere il
checkpoint e a fermarsi al punto giusto — non a crearla.

Integrazione in common.py
-------------------------
Nella lista dei callback, PRIMA di EarlyStoppingCallback (che legge
metrics[metric_for_best_model] e se non lo trova si disattiva):

    if args.ctx_metric_n > 0:
        from contesto_metrica import build_context_callback
        callbacks.append(build_context_callback(
            TrainerCallback, torch, tokenizer, dev_rows, task,
            n=args.ctx_metric_n, seed=args.seed))
    callbacks.append(EarlyStoppingCallback(...))

piu' l'argomento:

    ap.add_argument("--ctx-metric-n", type=int, default=64,
                    help="item di dev su cui misurare la sensibilita' al contesto "
                         "(0 disattiva). Aggiunge eval_ctx_delta e eval_ctx_acc")

Da quel momento `--metric ctx_acc` seleziona il checkpoint sulla coerenza.

Uso consigliato
---------------
    # T3: seleziona sulla coerenza invece che su chrF
    python finetune_t3_replica.py --model minerva --split-dir SPLIT --metric ctx_acc

    # T1: lascia chrF come metrica di selezione (la traduzione HA un modo) ma
    # leggi ctx_acc nei log: se resta a 0.50, il blocco di contesto e' decorativo
    python finetune_t1_traduzione.py --model minerva --split-dir SPLIT
"""

from __future__ import annotations

import random

INTESTAZIONI_CONTESTO = ("Conversazione finora", "Conversazione", "Contesto")


def _e_separatore(riga):
    """Fine del blocco di contesto.

    I layout non usano tutti la stessa convenzione: T1 chiude il contesto con una
    riga vuota, T2 e T3 con una riga di trattini. Cercare solo la riga vuota fa
    inghiottire l'intero prompt, istruzione compresa, e la misura diventa priva
    di senso senza dare errore. Si accetta quindi entrambe le forme.
    """
    t = riga.strip()
    return (not t) or (len(t) >= 3 and set(t) <= set("-=*_"))


def estrai_blocco_contesto(prompt):
    """Ritorna (indice_intestazione, indice_fine, righe_del_blocco).

    Il blocco e' delimitato dall'intestazione e dalla prima riga vuota. Se non
    c'e' intestazione l'item non ha contesto e va escluso dalla misura: senza
    contesto il delta non e' definito, e includerlo come 0 diluirebbe la metrica
    verso 0.50 facendola sembrare cieca quando invece manca l'input.
    """
    righe = prompt.split("\n")
    i = next((k for k, r in enumerate(righe)
              if any(h in r for h in INTESTAZIONI_CONTESTO)), None)
    if i is None:
        return None
    j = next((k for k in range(i + 1, len(righe)) if _e_separatore(righe[k])),
             len(righe))
    if j <= i + 1:
        return None                              # intestazione senza turni
    return i, j, righe[i + 1:j]


def sostituisci_contesto(prompt, nuovi_turni):
    """Rimpiazza i turni di contesto tenendo intestazione, istruzione e suffissi."""
    trovato = estrai_blocco_contesto(prompt)
    if trovato is None:
        return None
    i, j, _ = trovato
    righe = prompt.split("\n")
    return "\n".join(righe[:i + 1] + list(nuovi_turni) + righe[j:])


def prepara_coppie(rows, seed=0):
    """Per ogni item con contesto: (prompt_vero, prompt_falso, target).

    Il contesto falso viene da un ALTRO item, non da turni sintetici: cosi' il
    confronto e' fra due contesti entrambi plausibili e ben formati, e il delta
    misura la pertinenza, non la stranezza del testo.
    """
    con_ctx = []
    for r in rows:
        t = estrai_blocco_contesto(r["prompt"])
        if t is not None:
            con_ctx.append((r, t[2]))
    if len(con_ctx) < 2:
        return []
    rnd = random.Random(seed)
    coppie = []
    for k, (r, _) in enumerate(con_ctx):
        # si pesca un donatore diverso da se stesso; con 2+ item il ciclo termina
        while True:
            d = rnd.randrange(len(con_ctx))
            if d != k:
                break
        falso = sostituisci_contesto(r["prompt"], con_ctx[d][1])
        if falso and falso != r["prompt"]:
            coppie.append((r["prompt"], falso, r["target"]))
    return coppie


def _logp_per_token(model, tokenizer, torch, prompt_testi, target_testi,
                    render, max_len, batch=8):
    """log P(target | prompt) diviso per il numero di token di target."""
    fuori = []
    prev_side = tokenizer.padding_side
    tokenizer.padding_side = "right"             # scoring, non generazione
    try:
        for i in range(0, len(prompt_testi), batch):
            pp = prompt_testi[i:i + batch]
            tt = target_testi[i:i + batch]
            interi = [render(tokenizer, p, t) for p, t in zip(pp, tt)]
            n_prompt = [len(tokenizer(render(tokenizer, p), add_special_tokens=False)
                             ["input_ids"]) for p in pp]
            enc = tokenizer(interi, return_tensors="pt", padding=True, truncation=True,
                            max_length=max_len, add_special_tokens=False).to(model.device)
            with torch.no_grad():
                logits = model(**enc).logits.float()
            lp = torch.log_softmax(logits[:, :-1, :], dim=-1)
            ids = enc["input_ids"][:, 1:]
            att = enc["attention_mask"][:, 1:]
            scelti = lp.gather(-1, ids.unsqueeze(-1)).squeeze(-1)
            for b in range(ids.size(0)):
                maschera = att[b].clone().bool()
                maschera[:max(0, n_prompt[b] - 1)] = False   # solo i token di target
                n = int(maschera.sum())
                fuori.append(float(scelti[b][maschera].sum() / n) if n else 0.0)
    finally:
        tokenizer.padding_side = prev_side
    return fuori


def valuta_sensibilita(model, tokenizer, torch, rows, render, max_len=512,
                       n=64, batch=8, seed=0):
    """Ritorna {'ctx_delta', 'ctx_acc', 'n'} oppure None se manca il contesto."""
    coppie = prepara_coppie(rows, seed=seed)[:n]
    if not coppie:
        return None
    veri = [c[0] for c in coppie]
    falsi = [c[1] for c in coppie]
    targ = [c[2] for c in coppie]
    lv = _logp_per_token(model, tokenizer, torch, veri, targ, render, max_len, batch)
    lf = _logp_per_token(model, tokenizer, torch, falsi, targ, render, max_len, batch)
    delta = [a - b for a, b in zip(lv, lf)]
    return {"ctx_delta": sum(delta) / len(delta),
            "ctx_acc": sum(1 for d in delta if d > 0) / len(delta),
            "logp_vero": sum(lv) / len(lv),
            "logp_falso": sum(lf) / len(lf),
            "n": len(coppie)}


def build_context_callback(TrainerCallback, torch, tokenizer, rows, task,
                           n=64, batch=8, seed=0, render=None):
    """Aggiunge eval_ctx_delta e eval_ctx_acc al dict delle metriche.

    Mutare `metrics` dentro on_evaluate propaga alla selezione del checkpoint,
    quindi --metric ctx_acc funziona. Va inserito PRIMA di EarlyStoppingCallback.
    """
    if render is None:
        from common import render_prompt
        render = render_prompt

    max_len = getattr(task, "max_seq_len", 512)
    coppie_disponibili = len(prepara_coppie(rows, seed=seed))
    if coppie_disponibili == 0:
        print("! sensibilita' al contesto non misurabile: nessun item di dev ha un "
              "blocco di contesto. Metrica disattivata (non e' un errore per un "
              "layout senza contesto).")

    class ContextSensitivityCallback(TrainerCallback):
        def on_evaluate(self, args, state, control, model=None, metrics=None, **kwargs):
            if model is None or metrics is None or coppie_disponibili == 0:
                return
            era_training = model.training
            model.eval()
            try:
                res = valuta_sensibilita(model, tokenizer, torch, rows, render,
                                         max_len=max_len, n=n, batch=batch, seed=seed)
                if res is None:
                    return
                metrics["eval_ctx_delta"] = res["ctx_delta"]
                metrics["eval_ctx_acc"] = res["ctx_acc"]
                print(f"  [contesto su {res['n']} item] delta {res['ctx_delta']:+.4f} "
                      f"nat/token | acc {res['ctx_acc']:.3f} "
                      f"(0.50 = contesto ignorato)")
            finally:
                if era_training:
                    model.train()

    return ContextSensitivityCallback()


Writing /kaggle/working/training/contesto_metrica.py


## Passo 10c — `decodifica_contestuale.py`

**Cosa fare:** esegui (scrive il file). Si usa DOPO il training, sugli adapter
che hai gia': non richiede di riaddestrare niente.

A ogni passo di generazione calcola i logit due volte, col prompt completo e col
blocco di contesto rimosso, e li combina:

    logit = (1 + gamma) * logit(y | contesto) - gamma * logit(y | senza contesto)

La differenza fra i due rami e' il contributo del contesto alla predizione:
moltiplicarla per gamma amplifica cio' che il contesto suggerisce e sopprime cio'
che il modello direbbe comunque. Costa 2x il forward in generazione, zero in
training.

`--sweep 0 0.5 1.0 1.5` sul dev sceglie gamma. Vanno guardati **due numeri
insieme**: chrF e rep-3. Un gamma che alza il chrF facendo esplodere le
ripetizioni non e' un miglioramento, e' un artefatto del decoding.

Se il chrF resta piatto per ogni gamma, lo script te lo dice: significa che non
c'e' segnale di contesto da amplificare, e il problema e' a monte.

In [12]:
%%writefile /kaggle/working/training/decodifica_contestuale.py
#!/usr/bin/env python3
"""
decodifica_contestuale.py — costringe il modello a usare il contesto in
INFERENZA, senza riaddestrare niente.

L'idea
------
A ogni passo si calcolano i logit due volte, con il prompt completo e con il
blocco di contesto rimosso, e si combinano:

    logit = (1 + gamma) * logit(y | contesto)  -  gamma * logit(y | senza contesto)

E' il classifier-free guidance applicato al testo (context-aware decoding).
La differenza fra i due termini e' esattamente il contributo del contesto alla
predizione: moltiplicarla per gamma amplifica cio' che il contesto suggerisce e
sopprime cio' che il modello direbbe comunque. Sul fallimento tipico di T2/T3 —
napoletano fluente e morfologicamente corretto ma semanticamente slegato da cio'
che precede — e' il rimedio piu' diretto, e costa solo 2x il forward in
generazione.

gamma
-----
gamma=0 e' la generazione normale. Valori utili stanno fra 0,3 e 1,5. Troppo
alto degrada la fluenza: il modello inizia a preferire token rari solo perche'
il ramo senza contesto li sconsiglia. Non esiste un valore giusto a priori, si
sceglie sul dev con --sweep guardando DUE numeri insieme (chrF e ripetizioni):
un gamma che alza la pertinenza distruggendo la lingua non e' un miglioramento.

Cosa NON e'
-----------
Non insegna niente al modello: e' un intervento sul decoding. Se il modello non
ha proprio codificato il contesto nei suoi stati, non c'e' segnale da
amplificare e il gamma non produrra' effetto. In quel caso il problema e' a
monte (dati, curriculum) e questo script te lo dice: se il chrF resta piatto per
ogni gamma, il contesto non e' nella rappresentazione.

Uso
---
    # scegli gamma sul dev
    python decodifica_contestuale.py --model minerva --task T3 \\
        --adapter /kaggle/working/runs/<slug>__T3/adapter_final \\
        --split-dir /kaggle/working/split --split dev --sweep 0 0.5 1.0 1.5 --n 60

    # genera con il gamma scelto
    python decodifica_contestuale.py --model minerva --task T3 --adapter ... \\
        --split test --gamma 1.0 --out predizioni_cfg.json

Come libreria (per evaluate_task.py o prova_esempi.py):
    from decodifica_contestuale import genera_cfg, rimuovi_contesto
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

from contesto_metrica import _e_separatore, estrai_blocco_contesto


def rimuovi_contesto(prompt):
    """Toglie intestazione, turni di contesto e la riga vuota che li chiude.

    Ritorna None se il prompt non ha contesto: in quel caso i due rami sarebbero
    identici, la differenza sarebbe zero e il CFG un puro raddoppio di costo.
    Quegli item vanno generati normalmente.
    """
    trovato = estrai_blocco_contesto(prompt)
    if trovato is None:
        return None
    i, j, _ = trovato
    righe = prompt.split("\n")
    k = j
    while k < len(righe) and _e_separatore(righe[k]):
        k += 1                     # via anche il separatore (riga vuota o ---)
    return "\n".join(righe[:i] + righe[k:])


def genera_cfg(model, tokenizer, torch, testo_con, testo_senza, gamma=1.0,
               max_new_tokens=64, do_sample=False, top_p=0.9, temperature=0.8,
               eos_id=None):
    """Generazione con guidance sul contesto. Ritorna la stringa generata.

    Si tengono due cache KV separate e si avanzano in parallelo. Non si usa
    model.generate perche' servirebbe un LogitsProcessor con accesso a un
    secondo stato, che con la cache diventa piu' fragile di un ciclo esplicito.
    """
    eos_id = eos_id if eos_id is not None else tokenizer.eos_token_id
    dev = model.device
    a = tokenizer(testo_con, return_tensors="pt", add_special_tokens=False).to(dev)
    b = tokenizer(testo_senza, return_tensors="pt", add_special_tokens=False).to(dev)

    with torch.no_grad():
        oa = model(**a, use_cache=True)
        ob = model(**b, use_cache=True)
    ca, cb = oa.past_key_values, ob.past_key_values
    la, lb = oa.logits[:, -1, :].float(), ob.logits[:, -1, :].float()

    prodotti = []
    for _ in range(max_new_tokens):
        # la differenza fra i due rami e' il contributo del contesto
        logit = (1.0 + gamma) * la - gamma * lb if gamma else la

        if do_sample:
            logit = logit / max(temperature, 1e-5)
            ordinati, indici = torch.sort(logit, descending=True, dim=-1)
            cum = torch.softmax(ordinati, dim=-1).cumsum(dim=-1)
            taglia = cum - torch.softmax(ordinati, dim=-1) > top_p
            ordinati = ordinati.masked_fill(taglia, float("-inf"))
            scelto = indici.gather(-1, torch.multinomial(
                torch.softmax(ordinati, dim=-1), 1))
        else:
            scelto = logit.argmax(dim=-1, keepdim=True)

        tid = int(scelto[0, 0])
        if tid == eos_id:
            break
        prodotti.append(tid)

        with torch.no_grad():
            oa = model(input_ids=scelto, past_key_values=ca, use_cache=True)
            ob = model(input_ids=scelto, past_key_values=cb, use_cache=True)
        ca, cb = oa.past_key_values, ob.past_key_values
        la, lb = oa.logits[:, -1, :].float(), ob.logits[:, -1, :].float()

    return tokenizer.decode(prodotti, skip_special_tokens=True)


def rerank_contestuale(model, tokenizer, torch, testo_con, testo_senza,
                       candidati, lam=1.0, render=None):
    """Alternativa piu' semplice al CFG: si generano k candidati e si ordinano
    per logP(y|contesto) - lam * logP(y|senza contesto).

    Piu' debole del CFG (sceglie fra cio' che il campionamento ha prodotto,
    invece di guidare la produzione) ma non richiede un ciclo di decoding
    custom: si puo' innestare in evaluate_task.py cambiando poche righe.
    """
    from contesto_metrica import _logp_per_token
    if render is None:
        from common import render_prompt
        render = render_prompt
    n = len(candidati)
    lc = _logp_per_token(model, tokenizer, torch, [testo_con] * n, candidati,
                         render, 512)
    ls = _logp_per_token(model, tokenizer, torch, [testo_senza] * n, candidati,
                         render, 512)
    punteggi = [c - lam * s for c, s in zip(lc, ls)]
    ordine = sorted(range(n), key=lambda k: -punteggi[k])
    return candidati[ordine[0]], [punteggi[k] for k in ordine]


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1],
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True)
    ap.add_argument("--task", required=True, choices=["T1", "T2", "T3"])
    ap.add_argument("--adapter", required=True)
    ap.add_argument("--split-dir", default="/kaggle/working/split")
    ap.add_argument("--split", default="dev")
    ap.add_argument("--n", type=int, default=60, help="item da valutare")
    ap.add_argument("--gamma", type=float, default=1.0)
    ap.add_argument("--sweep", type=float, nargs="*", default=None,
                    help="prova piu' gamma e confronta, es. --sweep 0 0.5 1.0 1.5")
    ap.add_argument("--max-new", type=int, default=64)
    ap.add_argument("--sample", action="store_true")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--out", default=None)
    ap.add_argument("--hf-token", default=None)
    a = ap.parse_args()

    import sacrebleu
    import torch
    from peft import PeftModel
    from transformers import AutoTokenizer, BitsAndBytesConfig

    from common import scegli_dtype
    from common import (load_backbone, load_hf_token, load_split, render_prompt,
                        resolve_model)

    righe = load_split(a.split_dir, a.task, a.split)[:a.n]
    if not righe:
        sys.exit(f"Nessuna riga {a.task}/{a.split} in {a.split_dir}")
    coppie = []
    senza_ctx = 0
    for r in righe:
        nudo = rimuovi_contesto(r["prompt"])
        if nudo is None:
            senza_ctx += 1
            continue
        coppie.append((r["prompt"], nudo, r["target"]))
    print(f"{a.task}/{a.split}: {len(coppie)} item con contesto "
          f"({senza_ctx} senza, esclusi: per loro il CFG non ha effetto)")
    if not coppie:
        sys.exit("Nessun item con contesto: il CFG non e' applicabile a questo layout.")

    repo_id = resolve_model(a.model)
    token = load_hf_token(a.hf_token)
    tok = AutoTokenizer.from_pretrained(repo_id, token=token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype, _ = scegli_dtype(repo_id)      # fp32 su Gemma senza bf16
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_use_double_quant=True,
                               bnb_4bit_compute_dtype=dtype)
    model, _ = load_backbone(repo_id, dtype, quantization_config=quant,
                             device_map={"": 0}, token=token,
                             low_cpu_mem_usage=True, attn_implementation="eager")
    model = PeftModel.from_pretrained(model, a.adapter)
    model.eval()
    model.config.use_cache = True

    def ripetizioni(testi):
        """quota di 3-grammi ripetuti: sale quando il gamma degrada la lingua"""
        tot = rip = 0
        for t in testi:
            w = t.split()
            g = [tuple(w[i:i + 3]) for i in range(max(0, len(w) - 2))]
            tot += len(g)
            rip += len(g) - len(set(g))
        return round(rip / tot, 4) if tot else 0.0

    gammi = a.sweep if a.sweep is not None else [a.gamma]
    rifer = [c[2] for c in coppie]
    risultati = {}
    for g in gammi:
        torch.manual_seed(a.seed)
        ipo = []
        for con, nudo, _ in coppie:
            t = genera_cfg(model, tok, torch, render_prompt(tok, con),
                           render_prompt(tok, nudo), gamma=g,
                           max_new_tokens=a.max_new, do_sample=a.sample)
            ipo.append(t.strip().split("\n")[0].strip())
        chrf = round(sacrebleu.corpus_chrf(ipo, [rifer], word_order=2).score, 2)
        lung = sum(len(h.split()) for h in ipo) / max(1, sum(len(r.split()) for r in rifer))
        risultati[g] = {"chrf": chrf, "rapporto_lunghezza": round(lung, 3),
                        "rep3": ripetizioni(ipo), "ipotesi": ipo}
        print(f"  gamma={g:<4} chrF {chrf:6.2f} | lunghezza {lung:.2f}x | "
              f"rep-3 {risultati[g]['rep3']:.4f}")

    if len(gammi) > 1:
        best = max(risultati, key=lambda g: risultati[g]["chrf"])
        print(f"\nMiglior chrF a gamma={best}. Ma controlla rep-3 e lunghezza: "
              f"un gamma che alza il chrF\nfacendo esplodere le ripetizioni non e' "
              f"un miglioramento, e' un artefatto del decoding.")
        piatto = max(r["chrf"] for r in risultati.values()) - \
            min(r["chrf"] for r in risultati.values())
        if piatto < 1.0:
            print("! chrF piatto su tutti i gamma: non c'e' segnale di contesto da "
                  "amplificare.\n  Il problema e' a monte (dati o curriculum), non "
                  "nel decoding.")

    if a.out:
        Path(a.out).write_text(json.dumps(
            {str(g): v for g, v in risultati.items()}, ensure_ascii=False, indent=1),
            encoding="utf-8")
        print(f"\nScritto {a.out}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/decodifica_contestuale.py


## Nota — bug corretto in `contesto_metrica.py`

La prima versione cercava la fine del blocco di contesto come "prima riga vuota".
I tuoi layout non usano tutti la stessa convenzione: **T1 chiude con una riga
vuota, T2 e T3 con una riga di trattini** (`---`, come si vede nei prompt
stampati durante il training). Con la vecchia regola il blocco inghiottiva
l'intero prompt, istruzione compresa, e `ctx_acc` avrebbe misurato una cosa
priva di senso su T2 e T3 senza dare errore.

Le celle di `contesto_metrica.py` e `prova_esempi.py` in questo notebook sono
gia' quelle corrette: la fine del blocco e' ora una riga vuota **oppure** una
riga di soli `-=*_`.

## Passo 11 — `finetune_a2_lessico.py`

**Cosa fare:** esegui la cella (scrive il file).

L'entrypoint dello stadio A2. Stessa forma dei tre entrypoint dei task: definisce
un `TaskConfig` con `layout="A2"` e chiama `run()`. Tutto il resto (QLoRA,
masking, metriche, resume, summary) e' quello di `common.py`, in una sola copia.

Parametri scelti: `lr=5e-5` e 3 epoche. Gli item sono ~3.600 su ~300 tipi, quindi
ogni tipo viene visto molte volte; spingere di piu' fa memorizzare la coppia
come lookup e degrada la fluenza a valle. `gen_max_new_tokens=24` e
`repetition_penalty=1.0` perche' i target sono corti, spesso di una sola parola,
e su un target di una parola la penalita' di ripetizione e' solo dannosa.

**I numeri di questo run non vanno nel confronto cross-modello.** Servono a
verificare che l'iniezione abbia preso.

In [13]:
%%writefile /kaggle/working/training/finetune_a2_lessico.py
#!/usr/bin/env python3
"""
STADIO A2 — iniezione lessicale, fra il continued pretraining (stadio A) e
l'SFT sui tre task (stadio B).

Cosa addestra
-------------
Gli item prodotti da dati_lessicali.py: glossa (corrispondenza it->nap nuda),
cloze in contesto (il termine dialettale va prodotto dentro la frase e dentro
la conversazione), turno_breve (turni corti ad alta densita' dialettale).
Target corti - da 1 parola a poche - quindi il segnale lessicale non e' diluito
su decine di token come nell'SFT sui turni interi.

Posizione nella catena
----------------------
    python lessico.py          --csv dataset_finale.csv --split-dir SPLIT --out lessico_train.json
    python dati_lessicali.py   --csv dataset_finale.csv --split-dir SPLIT \\
                               --lessico lessico_train.json --out-dir SPLIT/stadio_a2_lessico
    python pretrain_dialect.py --model minerva --split-dir SPLIT                       # A
    python finetune_a2_lessico.py --model minerva --split-dir SPLIT \\
        --init-adapter /kaggle/working/cpt/<slug>/adapter_final                         # A2
    python finetune_t1_traduzione.py --model minerva --split-dir SPLIT \\
        --init-adapter /kaggle/working/runs/<slug>-A2/adapter_final                     # B
    (idem T2 e T3, tutti e tre dallo STESSO adapter A2)

Un solo run di A2 per modello, condiviso dai tre task: 3 run in piu' su 9, non 9.

Perche' lr basso e una sola epoca in piu' non serve
---------------------------------------------------
lr=5e-5 e 3 epoche: gli item sono ~4.000 su un lessico di ~300 tipi, quindi ogni
tipo viene visto molte volte. Spingere di piu' fa memorizzare la coppia
(italiano, napoletano) come lookup e degrada la fluenza a valle - si vede nel
dev di A2 che continua a scendere mentre il chrF di T1 peggiora. Se succede,
taglia a 2 epoche invece di alzare la patience.

metric="exact" non e' disponibile in common.py: si seleziona su chrF, che su
target di una parola coincide di fatto con l'accuratezza a livello di carattere.
"""

from common import TaskConfig, run

TASK = TaskConfig(
    layout="A2",
    nome="iniezione lessicale",
    descrizione="glossa + cloze in contesto + turni brevi ad alta densita' dialettale",
    max_seq_len=512,          # i cloze portano 3 turni di contesto
    epochs=3,
    lr=5e-5,
    metric="chrf",
    greater_is_better=True,
    gen_max_new_tokens=24,    # target corti: 64 token sono solo occasioni di divagare
    repetition_penalty=1.0,   # su un target di una parola la penalita' e' dannosa
    no_repeat_ngram_size=0,
    patience=3,
    evals_per_epoch=3,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    note=[
        "Stadio intermedio: i numeri di questo run NON vanno nel confronto "
        "cross-modello, servono solo a verificare che l'iniezione abbia preso.",
        "Richiede LAYOUT_DIRS['A2'] = 'stadio_a2_lessico' in common.py.",
        "Controllo di sanita': il recall dialettale su T1/T2/T3 deve salire "
        "rispetto agli stessi task partiti dal solo stadio A. Se non sale, il "
        "problema non e' l'iniezione ma il formato dei prompt.",
    ],
)

if __name__ == "__main__":
    run(TASK)


Writing /kaggle/working/training/finetune_a2_lessico.py


## Passo 12 — `finetune_t1_traduzione.py`

**Cosa fare:** esegui. Scrive l'entrypoint di T1.


In [14]:
%%writefile /kaggle/working/training/finetune_t1_traduzione.py
#!/usr/bin/env python3
"""
FASE 3 — T1: traduzione italiano -> napoletano con contesto conversazionale.

Layout 1 (layout1_traduzione_con_contesto): il prompt contiene i 3 turni
precedenti già in napoletano più la frase italiana da tradurre; il target è la
resa napoletana del turno corrente.

Perché una configurazione propria e non gli stessi valori degli altri due task:
  * è il layout con più istanze (~1134 in train) -> più step per epoca, quindi
    meno epoche sono sufficienti e l'eval può essere meno frequente
  * il prompt è il più lungo dei tre (3 turni di contesto + istruzione + frase):
    serve max_seq_len ampio, altrimenti il TARGET viene troncato e la loss si
    calcola su un riferimento mutilato
  * esiste un unico riferimento corretto -> chrF è una metrica di selezione
    legittima, non un proxy debole

Uso su Kaggle:
    !python finetune_t1_traduzione.py --model minerva \
        --split-dir /kaggle/input/napoletano-split/split
"""

from common import TaskConfig, run

TASK = TaskConfig(
    layout="T1",
    nome="traduzione con contesto",
    descrizione="italiano + 3 turni di contesto napoletano -> napoletano",
    max_seq_len=512,          # contesto lungo: sotto i 512 token si troncano i target
    epochs=4,
    lr=1e-4,
    metric="chrf",            # riferimento unico: chrF è appropriato
    greater_is_better=True,
    gen_max_new_tokens=64,
    repetition_penalty=1.15,
    no_repeat_ngram_size=0,   # su una traduzione breve un trigramma ripetuto può essere corretto
    patience=4,
    evals_per_epoch=2,        # ~71 step/epoca -> eval ogni ~35 step, ~8 eval nel run
    note=[
        "Metrica primaria del paper: chrF++ su test con generate() reale, non questo proxy.",
        "Baseline obbligatoria: copia dell'italiano invariato (le coppie identiche nel corpus "
        "la rendono tutt'altro che banale).",
    ],
)

if __name__ == "__main__":
    run(TASK)


Writing /kaggle/working/training/finetune_t1_traduzione.py


## Passo 15 — `evaluate_task.py`

**Cosa fare:** esegui. Scrive lo script di valutazione con `generate()` reale, usato dopo il training.


In [15]:
%%writefile /kaggle/working/training/evaluate_task.py
#!/usr/bin/env python3
"""
evaluate_task.py — valutazione di UN run (modello x layout) sul test set.

Importa da common.py: stesso registry di modelli, stesso rendering dei prompt.
Il prompt in valutazione e' byte-identico a quello visto in training.

Uso:
  # sistema fine-tuned
  python evaluate_task.py --model llama --task T3 --split-dir /kaggle/working/split \\
      --adapter /kaggle/working/runs/llama-2-7b-chat-hf__T3/adapter_final
  # baseline del modello base
  python evaluate_task.py --model llama --task T3 --split-dir ... --zero-shot
  python evaluate_task.py --model llama --task T3 --split-dir ... --few-shot 5

--------------------------------------------------------------------------
LE METRICHE, E PERCHE' QUESTE
--------------------------------------------------------------------------
T2 e T3 sono task UNO-A-MOLTI: date le stesse 3 battute di contesto, molte
repliche diverse sono corrette. Nessuna metrica calcolata contro l'unico turno
realmente pronunciato puo' misurare l'adeguatezza. Quindi:

A. DISCRIMINAZIONE AVVERSARIALE (reference-free, e' la metrica principale)
   Un classificatore prova a distinguere le repliche umane da quelle generate.
   Accuracy 5-fold CV: ~0.50 = indistinguibile, 1.00 = riconoscibile subito.
   Vengono riportati SEMPRE due controlli, senza i quali il numero non si legge:
     - umano-vs-umano: due meta' casuali del solo insieme umano. Deve dare
       ~0.50 (misurato su questo corpus: 0.520 +- 0.044). Se da' di piu', il
       classificatore sta leggendo artefatti e il risultato non vale.
     - solo-lunghezza: stessa classificazione usando come unica feature il
       numero di parole. Esclude che si stia misurando "la macchina e' piu'
       corta dell'umano".

B. CONTRASTIVA (trasforma uno-a-molti in risposta-unica)
   Il modello deve assegnare la log-likelihood piu' alta al target vero fra
   k+1 candidati. Distrattori = target reali di altri item, appaiati per
   lunghezza (negativi difficili). Accuracy, baseline casuale 1/(k+1).
   Nessuna generazione: deterministica e non contaminata dal decoding.

C. DISTRIBUZIONALI (corpus vs corpus, non item vs item)
   distinct-1/2/3, rep-2/3, distribuzione delle lunghezze con test di
   Kolmogorov-Smirnov contro i riferimenti umani. Sono bersagli BILATERALI:
   distinct troppo basso = collasso su formule frequenti, troppo alto =
   insalata di parole. I valori umani vengono ricalcolati sul test corrente,
   non hardcoded.

D. DIALETTALITA' (reference-free)
   Classificatore italiano-vs-napoletano addestrato sulle coppie parallele del
   TRAIN di T1 (mai sul test). Su questo corpus: accuracy 0.994, AUC 0.998,
   P(nap) 0.914 sui riferimenti umani contro 0.094 sulle frasi italiane.
   Restituisce P(napoletano) medio sugli output generati.

E. LIKELIHOOD DEL RIFERIMENTO, normalizzata per lunghezza
   Reference-based ma tollerante: non chiede di produrre quella stringa, solo
   di considerarla plausibile. Le altre continuazioni valide non competono.

F. chrF++ / BLEU
   Riportate, ma su T2 e T3 sono un LIMITE INFERIORE, non una misura di
   adeguatezza. Insieme vengono stampate le baseline non-banali misurate sul
   test corrente (copia dell'italiano per T1, ripeti-il-prefisso per T2,
   replica-da-un-altro-punto per T3): sono quelle il riferimento, non lo zero.

Nessuna di queste metriche, da sola, misura la qualita': ciascuna e' aggirabile
in isolamento (dialettalita' massima con dialetto senza senso, distinct massimo
con parole casuali). Si leggono come vettore.
"""

import argparse
import json
import math
import os
import random
import re
import statistics
import sys
from collections import Counter
from pathlib import Path

from common import (LAYOUT_DIRS, MODEL_REGISTRY, load_backbone, load_hf_token,
                   load_split, render_prompt, resolve_model, scegli_dtype, slug)

MAX_NEW = {"T1": 64, "T2": 48, "T3": 32}   # T3 a 32: la mediana dei turni
# umani e' 4 parole (~8 token) e il 68% sta sotto le 6. A 64 token il modello
# continuava a generare fino a 40 parole contro un target di 10, e la deriva
# semantica si accumulava proprio nella coda. Con solo la lunghezza come
# feature un classificatore distingueva generato da umano al 57,9%: il
# troncamento va fatto prima di dare peso alla discriminazione avversariale.
ISTRUZIONE = {"T1": "Traduci in napoletano: ", "T2": "Continua il turno in napoletano: "}


# --------------------------------------------------------------------------- #
# utility testuali
# --------------------------------------------------------------------------- #

def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^\w' ]+", " ", (s or "").lower())).strip()


CANDIDATI_EOT = ("<|im_end|>", "<end_of_turn>", "<|eot_id|>", "<|end|>",
                 "<|endoftext|>", "</s>", "<end_of_text>")


def _solo_spazi(tok, i):
    """True se l'id decodifica in soli spazi (o in niente).

    Serve a non usare come fine turno un token che non chiude nulla. Su Gemma-3
    il chat template scrive '<end_of_turn>' seguito da un a capo, quindi
    l'ULTIMO token del target renderizzato e' il 107, cioe' '\\n' (verificato
    in tokenizer.json: 106 = '<end_of_turn>', 107 = '\\n'). Passarlo a
    generate() come eos significa fermare la generazione al primo a capo: se il
    modello lo emette subito, l'uscita e' la stringa vuota.
    """
    try:
        return not tok.decode([int(i)], skip_special_tokens=False).strip()
    except Exception:
        return False


def _chiusura_target(tok, ids_testo):
    """Ultimo token NON di spaziatura della sequenza renderizzata.

    E' il token che chiude davvero il turno (su Gemma-3 il 106,
    '<end_of_turn>'), non l'a capo che il template gli mette dopo.
    """
    for i in reversed(list(ids_testo)):
        if not _solo_spazi(tok, i):
            return int(i)
    return None


def id_fine_turno(tok, rows, verbose=True):
    """Insieme degli id che chiudono il turno assistant, da passare a generate().

    IL PUNTO. Con un chat template tipo ChatML il turno assistant chiude con
    <|im_end|>, che NON e' tok.eos_token (</s>). In training la label c'e':
    apply_chat_template con add_generation_prompt=False lo include e il masking
    di ChatDataset lo lascia dentro, quindi il modello IMPARA a emetterlo. Ma se
    generate() non lo ha fra gli eos_token_id non si ferma mai e arriva sempre a
    max_new_tokens. Il sintomo misurato su minerva T3 nucleus: lunghezza media
    17.57 parole contro 6.99 umano, ks 0.589 con p=0, e soprattutto
    controllo_solo_lunghezza 0.776 su accuracy_umano_vs_macchina 0.80 - cioe' il
    97% del potere discriminante del classificatore avversariale veniva dalla
    lunghezza. Una frase tagliata al tetto si legge come "sconclusionata" anche
    quando la deriva semantica non c'e'.

    La fonte di verita' non e' la config del modello: e' l'ultimo token del
    testo renderizzato in training, che e' esattamente cio' che il modello ha
    imparato a produrre per chiudere.
    """
    ids = set()
    if tok.eos_token_id is not None:
        ids.add(int(tok.eos_token_id))
    for s in CANDIDATI_EOT:
        i = tok.convert_tokens_to_ids(s)
        if isinstance(i, int) and i >= 0 and i != tok.unk_token_id:
            ids.add(int(i))
    coda = []
    for r in rows[:8]:
        t = tok(render_prompt(tok, r["prompt"], r["target"]),
                add_special_tokens=False)["input_ids"]
        if t:
            ultimo = _chiusura_target(tok, t)
            if ultimo is not None:
                coda.append(ultimo)
    if coda:
        ultimo = Counter(coda).most_common(1)[0][0]
        ids.add(ultimo)
        if verbose:
            print(f"  chiusura del target renderizzato: id={ultimo} "
                  f"repr={tok.convert_ids_to_tokens([ultimo])[0]!r}")
    # BUG CORRETTO (uscite tutte vuote, chrF++ 0.00). L'ultimo token del target
    # renderizzato su Gemma-3 e' il 107, cioe' '\n', perche' il chat template
    # scrive '<end_of_turn>' + a capo: usato come eos fermava la generazione al
    # primo a capo. Ora la chiusura si cerca a monte della spaziatura e nessun
    # token di soli spazi resta nella lista. Vale per tutti e tre i modelli:
    # il ramo model.generate() non ha la guardia del decoder manuale, quindi
    # con un '\n' fra gli eos produce le stesse uscite vuote (su Llama T1 la
    # lunghezza mediana delle generazioni era 0).
    scartati = sorted(i for i in ids if _solo_spazi(tok, i))
    tenuti = set(i for i in ids if not _solo_spazi(tok, i))
    ids = tenuti or ids
    if scartati and verbose:
        print(f"  scartati come fine turno (solo spaziatura): {scartati} "
              f"({[tok.convert_ids_to_tokens([i])[0] for i in scartati]})")
    out = sorted(ids)
    if verbose:
        print(f"  tok.eos_token = {tok.eos_token!r} (id {tok.eos_token_id})")
        print(f"  fine turno da usare = {out} "
              f"({[tok.convert_ids_to_tokens([i])[0] for i in out]})")
    return out


def frazione_al_tetto(tok, hyps, max_new):
    """Frazione di output che raggiunge max_new, cioe' che non ha mai emesso
    fine di turno. Sopra ~0.05 la qualita' apparente e' dominata dal
    troncamento e non dalla pragmatica: leggere le altre metriche prima di
    aver abbassato questa non ha senso."""
    if not hyps:
        return 0.0
    n = sum(1 for h in hyps
            if len(tok(h, add_special_tokens=False)["input_ids"]) >= max_new - 1)
    return n / len(hyps)


def fonte_italiana(prompt):
    """Estrae la frase italiana dal prompt di T1 (l'istruzione la contiene)."""
    marker = ISTRUZIONE["T1"]
    return prompt.split(marker)[-1].strip() if marker in prompt else None


def prefisso_nap(prompt):
    """Estrae il prefisso napoletano dal prompt di T2."""
    marker = ISTRUZIONE["T2"]
    return prompt.split(marker)[-1].strip() if marker in prompt else None


MARCATORI_RUOLO = ("<|im_start|>", "<|im_end|>", "<start_of_turn>",
                   "<end_of_turn>", "<|eot_id|>", "<|start_header_id|>",
                   "<|end_header_id|>")


def pulisci_generato(txt):
    """Primo turno utile della generazione.

    Come prima si taglia alla prima riga, ma PRIMA si togliono i marcatori di
    ruolo: con un template custom skip_special_tokens non sempre li copre, e
    tagliare alla prima riga quando la prima riga E' il marcatore restituisce
    stringa vuota, cioe' un item perso in silenzio.
    """
    t = txt
    for m in MARCATORI_RUOLO:
        t = t.replace(m, "\n")
    for riga in t.split("\n"):
        r = riga.strip()
        if r.lower() in ("assistant", "user", "model", "system"):
            continue
        if r:
            return r
    return ""


def distinct(texts, n):
    grams, tot = set(), 0
    for t in texts:
        w = t.split()
        for i in range(len(w) - n + 1):
            grams.add(tuple(w[i:i + n]))
            tot += 1
    return len(grams) / tot if tot else 0.0


def rep_n(texts, n):
    c = 0
    for t in texts:
        w = t.split()
        g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
        if g and len(set(g)) < len(g):
            c += 1
    return c / len(texts) if texts else 0.0


def loop_massimo(t):
    """Lunghezza della piu' lunga sequenza di token identici consecutivi."""
    w = t.split()
    best = cur = 1 if w else 0
    for i in range(1, len(w)):
        cur = cur + 1 if w[i].lower() == w[i - 1].lower() else 1
        best = max(best, cur)
    return best


def degenerazione(hyps, refs):
    """Due patologie che distinct-n e rep-n non separano: i loop del decoder e il
    collasso sull'apertura piu' frequente."""
    def incipit(ts):
        c = Counter(t.split()[0].lower() for t in ts if t.split())
        top, n = (c.most_common(1)[0] if c else ("", 0))
        return {"token": top, "frazione": round(n / max(len(ts), 1), 3)}
    lm = [loop_massimo(t) for t in hyps]
    lr = [loop_massimo(t) for t in refs]
    return {
        "frazione_con_loop_3plus": round(sum(1 for x in lm if x >= 3) / max(len(lm), 1), 3),
        "loop_massimo": max(lm) if lm else 0,
        "incipit_dominante": incipit(hyps),
        "_umano": {"frazione_con_loop_3plus": round(sum(1 for x in lr if x >= 3) / max(len(lr), 1), 3),
                  "loop_massimo": max(lr) if lr else 0,
                  "incipit_dominante": incipit(refs)},
        "_lettura": "loop = degenerazione del decoder (il greedy su task aperti la produce "
                    "per costruzione). incipit_dominante alto = collasso modale: se il 76% "
                    "delle repliche inizia con la stessa parola e negli umani il massimo e' "
                    "il 6%, il modello ha imparato una formula, non una distribuzione.",
    }


def distribuzionali(hyps, refs):
    from scipy import stats
    Lh = [len(x.split()) for x in hyps]
    Lr = [len(x.split()) for x in refs]
    ks = stats.ks_2samp(Lh, Lr)
    return {
        "distinct_1": round(distinct(hyps, 1), 4),
        "distinct_2": round(distinct(hyps, 2), 4),
        "distinct_3": round(distinct(hyps, 3), 4),
        "rep_2": round(rep_n(hyps, 2), 4),
        "rep_3": round(rep_n(hyps, 3), 4),
        "lunghezza_media": round(statistics.mean(Lh), 2) if Lh else 0,
        "lunghezza_mediana": statistics.median(Lh) if Lh else 0,
        "ks_lunghezza_stat": round(float(ks.statistic), 4),
        "ks_lunghezza_pvalue": round(float(ks.pvalue), 4),
        "_umano": {
            "distinct_1": round(distinct(refs, 1), 4),
            "distinct_2": round(distinct(refs, 2), 4),
            "distinct_3": round(distinct(refs, 3), 4),
            "rep_2": round(rep_n(refs, 2), 4),
            "rep_3": round(rep_n(refs, 3), 4),
            "lunghezza_media": round(statistics.mean(Lr), 2) if Lr else 0,
            "lunghezza_mediana": statistics.median(Lr) if Lr else 0,
        },
        "_lettura": "bersagli bilaterali: confronta con _umano, non massimizzare. "
                    "ks_pvalue alto = distribuzione delle lunghezze compatibile con l'umano.",
    }


# --------------------------------------------------------------------------- #
# A. discriminazione avversariale
# --------------------------------------------------------------------------- #

def _clf():
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    return make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
        LogisticRegression(max_iter=2000, C=5))


def discriminazione_avversariale(hyps, refs, seed=0):
    """Accuracy 5-fold nel distinguere umano da macchina, piu' i due controlli."""
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score

    X = [norm(x) for x in refs] + [norm(x) for x in hyps]
    y = [0] * len(refs) + [1] * len(hyps)
    if min(len(refs), len(hyps)) < 20:
        return {"errore": "troppi pochi esempi per la cross-validation"}
    sc = cross_val_score(_clf(), X, y, cv=5, scoring="accuracy")

    # controllo 1: umano-vs-umano. Deve stare a ~0.50.
    rng = random.Random(seed)
    hh = [norm(x) for x in refs]
    rng.shuffle(hh)
    yhh = [0] * (len(hh) // 2) + [1] * (len(hh) - len(hh) // 2)
    sc_hh = cross_val_score(_clf(), hh, yhh, cv=5, scoring="accuracy")

    # controllo 2: solo la lunghezza come feature
    L = np.array([[len(x.split())] for x in X])
    sc_len = cross_val_score(LogisticRegression(max_iter=1000), L, y, cv=5, scoring="accuracy")

    return {
        "accuracy_umano_vs_macchina": round(float(sc.mean()), 3),
        "std": round(float(sc.std()), 3),
        "controllo_umano_vs_umano": round(float(sc_hh.mean()), 3),
        "controllo_solo_lunghezza": round(float(sc_len.mean()), 3),
        "_lettura": "~0.50 = indistinguibile dall'umano (ottimo). 1.00 = riconoscibile "
                    "subito. Valido SOLO se controllo_umano_vs_umano resta vicino a 0.50: "
                    "altrimenti il classificatore legge artefatti. Se "
                    "controllo_solo_lunghezza e' alto, sta misurando la lunghezza.",
    }


# --------------------------------------------------------------------------- #
# D. dialettalita'
# --------------------------------------------------------------------------- #

def costruisci_discriminatore_dialetto(split_dir):
    """Addestrato sulle coppie parallele del TRAIN di T1: la frase italiana sta
    dentro il prompt, la resa napoletana e' il target. Mai addestrato sul test."""
    from sklearn.metrics import accuracy_score, roc_auc_score
    from sklearn.model_selection import train_test_split

    rows = load_split(split_dir, "T1", "train")
    X, y = [], []
    for r in rows:
        ita, nap = fonte_italiana(r["prompt"]), r["target"]
        if not ita:
            continue
        a, b = norm(ita), norm(nap)
        if a == b or len(a.split()) < 3:      # coppie identiche: non separabili per definizione
            continue
        X += [a, b]
        y += [0, 1]
    if len(X) < 100:
        return None, {"errore": "coppie parallele insufficienti"}
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.15, random_state=0, stratify=y)
    clf = _clf().fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]
    info = {
        "n_frasi_train": len(Xtr),
        "accuracy": round(float(accuracy_score(yte, p > 0.5)), 3),
        "auc": round(float(roc_auc_score(yte, p)), 3),
    }
    return clf, info


# --------------------------------------------------------------------------- #
# B/E. likelihood e contrastiva
# --------------------------------------------------------------------------- #

def logprob_target(model, tok, prompt, target, max_len, device):
    """log P(target | prompt) sommata e normalizzata per numero di token."""
    import torch
    p_ids = tok(render_prompt(tok, prompt), add_special_tokens=False)["input_ids"]
    f_ids = tok(render_prompt(tok, prompt, target), add_special_tokens=False)["input_ids"][:max_len]
    n_t = len(f_ids) - len(p_ids)
    if n_t <= 0:
        return None
    ids = torch.tensor([f_ids], device=device)
    with torch.no_grad():
        logits = model(input_ids=ids).logits[0].float()
    lp = torch.log_softmax(logits[:-1], dim=-1)
    tgt = ids[0][1:]
    tot = sum(float(lp[i, tgt[i]]) for i in range(len(p_ids) - 1, len(tgt)))
    return tot / n_t


def distrattori_per_lunghezza(rows, i, k, rng):
    """Negativi difficili: target reali di altri item, con lunghezza simile."""
    n_target = len(rows[i]["target"].split())
    cand = [j for j in range(len(rows))
            if j != i and abs(len(rows[j]["target"].split()) - n_target) <= 2]
    if len(cand) < k:
        cand = [j for j in range(len(rows)) if j != i]
    return [rows[j]["target"] for j in rng.sample(cand, min(k, len(cand)))]


def contrastiva(model, tok, rows, k, max_len, device, seed=0, limite=None):
    rng = random.Random(seed)
    sub = rows if limite is None else rows[:limite]
    ok, n, lp_ref = 0, 0, []
    for i, r in enumerate(sub):
        cand = [r["target"]] + distrattori_per_lunghezza(rows, i, k, rng)
        scores = [logprob_target(model, tok, r["prompt"], c, max_len, device) for c in cand]
        if scores[0] is None or any(s is None for s in scores):
            continue
        lp_ref.append(scores[0])
        ok += int(max(range(len(scores)), key=lambda j: scores[j]) == 0)
        n += 1
        if n % 25 == 0:
            print(f"  contrastiva {n}/{len(sub)}", end="\r")
    if not n:
        return {}
    return {
        "accuracy": round(ok / n, 3),
        "baseline_casuale": round(1 / (k + 1), 3),
        "k_distrattori": k,
        "n": n,
        "logprob_riferimento_norm": round(statistics.mean(lp_ref), 4),
        "_lettura": "distrattori = target reali di altri item appaiati per lunghezza. "
                    "logprob normalizzata per token: piu' alta (meno negativa) = il "
                    "modello considera plausibile una continuazione valida.",
    }


# --------------------------------------------------------------------------- #
# F. metriche di riferimento e baseline
# --------------------------------------------------------------------------- #

def riferimento(hyps, refs):
    import sacrebleu
    return {
        "chrf++": round(sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score, 2),
        "bleu": round(sacrebleu.corpus_bleu(hyps, [refs]).score, 2),
    }


def bootstrap_chrf(hyps, refs, n=1000, seed=0):
    import sacrebleu
    rng = random.Random(seed)
    N = len(refs)
    vals = []
    for _ in range(n):
        s = [rng.randrange(N) for _ in range(N)]
        vals.append(sacrebleu.corpus_chrf([hyps[i] for i in s], [[refs[i] for i in s]],
                                          word_order=2).score)
    vals.sort()
    return {"ci95_basso": round(vals[int(0.025 * n)], 2),
            "ci95_alto": round(vals[int(0.975 * n)], 2)}


def baseline_non_banali(task, rows, refs):
    """Il riferimento non e' lo zero: e' quello che ottiene una strategia stupida."""
    import sacrebleu
    out = {}
    def chrf(h, r):
        return round(sacrebleu.corpus_chrf(h, [r], word_order=2).score, 2)
    if task == "T1":
        ita = [fonte_italiana(r["prompt"]) or "" for r in rows]
        out["copia_italiano"] = chrf(ita, refs)
    if task == "T2":
        out["ripeti_prefisso"] = chrf([prefisso_nap(r["prompt"]) or "" for r in rows], refs)
    sh = refs[:]
    random.Random(0).shuffle(sh)
    out["riferimento_di_un_altro_item"] = chrf(sh, refs)
    out["_lettura"] = "un sistema che non batte queste in modo netto non ha imparato nulla."
    return out


def stratifica_per_lunghezza(hyps, refs, soglia=2):
    """Riscontri (<= soglia parole) contro turni contenutistici.

    Nel corpus il 33.7% dei turni napoletani sta a <= 2 parole e i piu'
    frequenti sono segnali di ascolto (mh 95, si' 71, mhmh 58, no 32). Sono un
    sotto-task diverso: chrF++ contro un riferimento di una parola non misura
    niente, e aggregare i due annega il segnale sui turni contenutistici, che
    sono quelli di cui si parla quando si dice "generazione libera".
    """
    out = {}
    for nome, tieni in (("riscontri", lambda n: n <= soglia),
                        ("contenutistici", lambda n: n > soglia)):
        idx = [i for i, r in enumerate(refs) if tieni(len(r.split()))]
        if len(idx) < 20:
            out[nome] = {"n": len(idx), "nota": "troppi pochi item"}
            continue
        h = [hyps[i] for i in idx]
        f = [refs[i] for i in idx]
        out[nome] = {
            "n": len(idx),
            "chrf++": riferimento(h, f)["chrf++"],
            "lunghezza_media_generata": round(
                statistics.mean(len(x.split()) for x in h), 2),
            "lunghezza_media_riferimento": round(
                statistics.mean(len(x.split()) for x in f), 2),
        }
    out["_lettura"] = ("il numero aggregato e' una media fra due distribuzioni "
                       "diverse: guarda 'contenutistici' per la generazione libera.")
    return out


# --------------------------------------------------------------------------- #
# generazione
# --------------------------------------------------------------------------- #

def prompt_few_shot(tok, r, shots):
    msgs = []
    for s in shots:
        msgs.append({"role": "user", "content": s["prompt"]})
        msgs.append({"role": "assistant", "content": s["target"]})
    msgs.append({"role": "user", "content": r["prompt"]})
    if getattr(tok, "chat_template", None):
        try:
            return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return "".join(m["content"] + "\n" for m in msgs)


def _decode_gemma3_manuale(model, enc, decoding, max_new, min_new, top_p,
                           temperature, eos_ids, repetition_penalty,
                           no_repeat_ngram_size, tok=None):
    """model.generate() sul checkpoint multimodale di Gemma-3 e' rotto in questa
    transformers (_sample restituisce next_tokens con una dimensione di troppo:
    'Tensors must have same number of dimensions'). Qui facciamo il loop noi,
    affettando esplicitamente logits[:, -1, :] cosi' la forma e' sempre [1, vocab].
    Senza cache KV: piu' lento ma robusto. Rispetta greedy/nucleus, min_new_tokens,
    repetition_penalty, no_repeat_ngram_size ed eos multipli, come model.generate.

    BUG CORRETTO (uscite tutte vuote). Fra gli id di fine turno c'e' il 107, che
    NON e' un token speciale ma un semplice '\\n' (il chat template mette un a
    capo dopo <end_of_turn>, quindi e' l'ultimo token del target renderizzato).
    Con min_new=1 la guardia `step < min_new - 1` non scattava al primo passo:
    se il modello emetteva '\\n' come primo token il loop usciva subito e
    restituiva stringa vuota. Su T1 e' successo su tutte e 267 le uscite di test,
    con chrF++ 0.00 e nessun errore da nessuna parte. Ora l'EOS resta vietato
    finche' non e' stato prodotto almeno un carattere non-spazio: fermarsi su
    '\\n' dopo del contenuto e' giusto, fermarsi prima no.
    """
    import torch
    ids = enc["input_ids"]
    attn = enc.get("attention_mask")
    eos_set = set(eos_ids) if eos_ids else set()
    # Token che non devono MAI essere emessi: decodificano nel nulla, quindi
    # riempiono l'uscita senza produrre testo. Su Gemma <pad> e' l'id 0, cioe'
    # esattamente cio' che argmax restituisce su logit NaN: senza questo divieto
    # un forward in overflow produce 64 pad e una stringa vuota che sembra una
    # traduzione sbagliata. Mai vietare un id che serve a fermarsi.
    mai = set()
    if tok is not None:
        for t in (getattr(tok, "pad_token_id", None), getattr(tok, "bos_token_id", None)):
            if t is not None and int(t) not in eos_set:
                mai.add(int(t))
    generati = []
    for step in range(max_new):
        out = model(input_ids=ids, attention_mask=attn, use_cache=False)
        logits = out.logits[:, -1, :].float()               # [1, vocab]
        if not torch.isfinite(logits).all():
            raise RuntimeError(
                f"logit non finiti (NaN/inf) al passo {step} della generazione. "
                "Il forward e' in overflow fp16: argmax su NaN restituisce l'id 0 "
                "('<pad>') e l'uscita decodificata e' la stringa vuota. Non e' un "
                "problema di eos o di decoding. Rimedio: common.stabilizza_fp16(), "
                "chiamata da load_backbone() sui caricamenti 4-bit in fp16.")
        for t in mai:
            logits[0, t] = float("-inf")
        seq = ids[0].tolist()
        # repetition penalty (semantica HF)
        if repetition_penalty and repetition_penalty != 1.0:
            for t in set(seq):
                v = logits[0, t]
                logits[0, t] = v / repetition_penalty if v > 0 else v * repetition_penalty
        # no_repeat_ngram_size: blocca i token che chiuderebbero un n-gram ripetuto
        if no_repeat_ngram_size and len(seq) >= no_repeat_ngram_size:
            n = no_repeat_ngram_size
            prefisso = tuple(seq[-(n - 1):]) if n > 1 else ()
            vietati = set()
            for i in range(len(seq) - n + 1):
                if tuple(seq[i:i + n - 1]) == prefisso:
                    vietati.add(seq[i + n - 1])
            for t in vietati:
                logits[0, t] = float("-inf")
        # niente EOS prima di min_new_tokens, e niente EOS finche' l'uscita
        # sarebbe vuota: senza questa seconda condizione un '\n' al primo passo
        # produce una stringa vuota e un chrF++ 0.00 silenzioso
        if tok is not None:
            vuoto = not tok.decode(generati, skip_special_tokens=True).strip()
        else:
            vuoto = not generati
        if step < max(0, min_new - 1) or vuoto:
            for e in eos_set:
                logits[0, e] = float("-inf")
        if decoding == "nucleus":
            probs = torch.softmax(logits / max(temperature, 1e-6), dim=-1)
            sp, si = torch.sort(probs, descending=True)
            cum = torch.cumsum(sp, dim=-1)
            keep = cum <= top_p
            keep[..., 0] = True
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))    # [1,1]
        else:
            nxt = logits.argmax(dim=-1, keepdim=True)         # [1,1]
        ids = torch.cat([ids, nxt], dim=-1)
        generati.append(int(nxt.item()))
        if attn is not None:
            attn = torch.cat([attn, torch.ones_like(nxt)], dim=-1)
        if int(nxt.item()) in eos_set:
            break
    return ids


def controllo_numerico(model, tok, rows):
    """Un forward su un prompt vero, prima di generare 267 volte.

    Costa un secondo e sostituisce l'unico modo in cui questo guasto si e'
    manifestato finora: 267 uscite vuote, chrF++ 0.00, logprob NaN, nessuna
    eccezione e quattro valutazioni buttate.
    """
    import torch
    if not rows:
        return
    enc = tok(render_prompt(tok, rows[0]["prompt"]), return_tensors="pt",
              add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model(**enc, use_cache=False)
    log = out.logits[:, -1, :].float()
    if not torch.isfinite(log).all():
        raise SystemExit(
            "ERRORE: il forward produce logit non finiti (NaN/inf).\n"
            "  Causa: fp16 su GPU pre-Ampere, il residuale di Gemma-3 va in "
            "overflow. Ogni metrica che segue sarebbe 0.00 o NaN.\n"
            "  Rimedio: riesegui la cella di common.py (contiene "
            "stabilizza_fp16, chiamata da load_backbone) e rilancia.")
    top = int(log.argmax(-1))
    nome = tok.convert_ids_to_tokens([top])[0]
    print(f"  controllo numerico: logit finiti, primo token = {top} {nome!r}")
    if top == getattr(tok, "pad_token_id", None):
        print("  ! il primo token e' <pad>: il modello non produrra' testo. "
              "Controlla che l'adapter sia quello di QUESTO modello.")


def genera(model, tok, rows, shots, max_new, device, batch=8, decoding="greedy",
           top_p=0.9, temperature=0.8, seed=42, eos_token_id=None,
           repetition_penalty=1.0, no_repeat_ngram_size=0, typical_p=None):
    """decoding='greedy' per T1/T2, 'nucleus' per T3.

    Il greedy e' mode-seeking: su un task APERTO cerca sempre la continuazione
    piu' probabile e il risultato e' collasso sull'apertura piu' frequente piu'
    loop di ripetizione (Holtzman et al. 2020). Misurato su Minerva T3: il 76%
    delle repliche iniziava con 'ma' contro il 5.7% del token piu' frequente
    negli umani, e il 14.6% conteneva un token ripetuto >=3 volte di fila, con
    un massimo di 39 ripetizioni consecutive.

    Le repliche umane sono campioni da una distribuzione, non modi. Per T3 serve
    nucleus sampling; il seed fisso preserva la riproducibilita'.

    BUG CORRETTO. `repetition_penalty` e `no_repeat_ngram_size` erano nel
    TaskConfig di T3 (1.2 e 3) e agivano sulla generazione di monitoraggio
    durante il training, ma qui non arrivavano: il ramo nucleus impostava solo
    do_sample/top_p/temperature/top_k. La valutazione di test girava quindi
    senza NESSUN controllo di ripetizione, da cui rep_2 0.151 contro 0.0365
    umano e loop_massimo 63. Ora sono parametri espliciti e finiscono nel
    metrics.json, cosi' un run non e' confrontabile con un altro per sbaglio.

    `eos_token_id` va passato: vedi id_fine_turno(). Senza, generate() usa
    generation_config, che su piu' di un modello del registry non contiene il
    token di fine turno del chat template.

    `typical_p`: locally typical sampling (Meister et al. 2023). Su target con
    mediana 4 parole il nucleus a top_p 0.9 e T 0.8 e' troppo caldo - un
    campionamento sbagliato sui primi due token non ha spazio per essere
    recuperato. Alternativa a top_p, non aggiuntiva.
    """
    import torch
    if decoding == "nucleus":
        torch.manual_seed(seed)
    # Gemma-3: la generazione batched con left-padding crasha in transformers
    # (_sample: shape [prompt_len] vs [batch]). Si genera un item alla volta
    # (batch=1, nessun padding): piu' lento ma robusto. Gli altri modelli no.
    mt = getattr(getattr(model, "config", None), "model_type", "") or ""
    if "gemma3" in mt.lower():
        batch = 1
    hyps = []
    for i in range(0, len(rows), batch):
        b = rows[i:i + batch]
        testi = [prompt_few_shot(tok, r, shots) if shots else render_prompt(tok, r["prompt"])
                 for r in b]
        enc = tok(testi, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
        gen_kw = dict(max_new_tokens=max_new, min_new_tokens=1,
                      pad_token_id=tok.pad_token_id)
        if eos_token_id:
            gen_kw["eos_token_id"] = eos_token_id
        if repetition_penalty and repetition_penalty != 1.0:
            gen_kw["repetition_penalty"] = repetition_penalty
        if no_repeat_ngram_size:
            gen_kw["no_repeat_ngram_size"] = no_repeat_ngram_size
        if decoding == "nucleus":
            gen_kw.update(do_sample=True, temperature=temperature, top_k=0)
            if typical_p:
                gen_kw["typical_p"] = typical_p
            else:
                gen_kw["top_p"] = top_p
        else:
            gen_kw.update(do_sample=False, num_beams=1)
        with torch.no_grad():
            if "gemma3" in mt.lower():
                g = _decode_gemma3_manuale(
                    model, enc, decoding, max_new, 1, top_p, temperature,
                    eos_token_id, repetition_penalty, no_repeat_ngram_size,
                    tok=tok)
            else:
                g = model.generate(**enc, **gen_kw)
        for j in range(len(b)):
            txt = tok.decode(g[j][enc["input_ids"].shape[1]:], skip_special_tokens=True)
            hyps.append(pulisci_generato(txt))
        print(f"  generazione {min(i + batch, len(rows))}/{len(rows)}", end="\r")
    print()
    # GUARDIA. Un'uscita vuota non e' un punteggio basso: e' assenza di output,
    # e il chrF++ la conta come 0 senza distinguerla da una traduzione sbagliata.
    # Se non lo si dice qui, arriva a valle come una barra a zero in un grafico.
    vuoti = sum(1 for h in hyps if not h.strip())
    if vuoti:
        print(f"  ! {vuoti}/{len(hyps)} uscite VUOTE ({100*vuoti/len(hyps):.1f}%). "
              f"Il chrF++ che segue e' abbassato da stringhe assenti, non da "
              f"traduzioni sbagliate: guarda il decoding (eos di fine turno) "
              f"prima di interpretare le metriche.")
    return hyps


# --------------------------------------------------------------------------- #

def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1],
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True, help=f"alias ({'/'.join(MODEL_REGISTRY)}) o repo_id")
    ap.add_argument("--task", required=True, choices=["T1", "T2", "T3"])
    ap.add_argument("--split-dir", default="/kaggle/working/split")
    ap.add_argument("--split", default="test")
    ap.add_argument("--adapter", default=None)
    ap.add_argument("--zero-shot", action="store_true")
    ap.add_argument("--few-shot", type=int, default=0)
    ap.add_argument("--out", default="/kaggle/working/eval")
    ap.add_argument("--max-len", type=int, default=512)
    ap.add_argument("--max-new", type=int, default=None)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--decoding", choices=["greedy", "nucleus", "auto"], default="auto",
                    help="auto = greedy su T1/T2 (hanno un modo), nucleus su T3 (aperto). "
                         "Riporta ENTRAMBI su T3: la differenza e' essa stessa un risultato")
    ap.add_argument("--top-p", type=float, default=0.9)
    ap.add_argument("--typical-p", type=float, default=None,
                    help="locally typical sampling invece di top_p (alternativa, "
                         "non aggiuntiva). Su T3 provare 0.9 con --temperature 0.7")
    ap.add_argument("--temperature", type=float, default=0.8)
    ap.add_argument("--repetition-penalty", type=float, default=None,
                    help="default: 1.15 su T3 nucleus, 1.0 altrove. Prima non "
                         "arrivava affatto a generate()")
    ap.add_argument("--no-repeat-ngram", type=int, default=None,
                    help="default: 3 su T3 nucleus, 0 altrove")
    ap.add_argument("--gen-seed", type=int, default=42)
    ap.add_argument("--contrastive-k", type=int, default=4)
    ap.add_argument("--contrastive-limit", type=int, default=100,
                    help="la contrastiva costa (k+1) forward per item: limitala")
    ap.add_argument("--skip-contrastive", action="store_true")
    ap.add_argument("--hf-token", default=None)
    ap.add_argument("--dtype", choices=["auto", "bf16", "fp16", "fp32"], default="auto",
                    help="auto = bf16 se la GPU lo ha, fp32 per Gemma senza bf16 "
                         "(in fp16 i logit vanno in NaN e le uscite escono vuote), "
                         "fp16 per gli altri")
    a = ap.parse_args()

    import torch
    from transformers import AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    repo_id = resolve_model(a.model)
    max_new = a.max_new or MAX_NEW[a.task]
    token = load_hf_token(a.hf_token)
    device = 0

    tok = AutoTokenizer.from_pretrained(repo_id, token=token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"                          # obbligatorio in generazione

    cc = torch.cuda.get_device_capability(0)
    dtype, _ = scegli_dtype(repo_id, a.dtype)
    print(f"GPU cc {cc[0]}.{cc[1]} -> {str(dtype).replace('torch.', '')}"
          + ("  (Gemma senza bf16 nativo: in fp16 il forward va in NaN e le "
             "uscite escono vuote, quindi fp32)" if dtype == torch.float32 else ""))
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    model, arch = load_backbone(repo_id, dtype, quantization_config=quant,
                               device_map={"": device}, token=token,
                               low_cpu_mem_usage=True, attn_implementation="eager")
    if a.adapter:
        model = PeftModel.from_pretrained(model, a.adapter)
    model.eval()
    model.config.use_cache = True

    rows = load_split(a.split_dir, a.task, a.split)
    controllo_numerico(model, tok, rows)
    refs = [r["target"] for r in rows]
    shots = []
    if a.few_shot:
        tr = load_split(a.split_dir, a.task, "train")
        shots = tr[:a.few_shot]
    tag = "ft" if a.adapter else (f"fs{a.few_shot}" if a.few_shot else "zs")
    print(f"=== {slug(repo_id)} {a.task} {tag} | {len(rows)} esempi di {a.split} ===")

    dec = a.decoding if a.decoding != "auto" else ("nucleus" if a.task == "T3" else "greedy")

    # Fine turno: senza questo generate() non si ferma e arriva sempre al tetto.
    print("Fine turno:")
    eot = id_fine_turno(tok, rows)
    gc_eos = getattr(model.generation_config, "eos_token_id", None)
    gc_set = set(gc_eos if isinstance(gc_eos, (list, tuple))
                 else ([gc_eos] if gc_eos is not None else []))
    mancanti = [i for i in eot if i not in gc_set]
    if mancanti:
        print(f"  ! {mancanti} NON erano in generation_config: senza il fix "
              f"generate() arrivava sempre a max_new={max_new}")

    rep = a.repetition_penalty if a.repetition_penalty is not None else (
        1.15 if dec == "nucleus" else 1.0)
    nrn = a.no_repeat_ngram if a.no_repeat_ngram is not None else (
        3 if dec == "nucleus" else 0)
    print(f"Decoding: {dec}" + (
        f" ({'typical_p=%s' % a.typical_p if a.typical_p else 'top_p=%s' % a.top_p}, "
        f"T={a.temperature}, rep={rep}, no_repeat={nrn}, seed={a.gen_seed})"
        if dec == "nucleus" else ""))
    hyps = genera(model, tok, rows, shots, max_new, device, a.batch,
                  decoding=dec, top_p=a.top_p, temperature=a.temperature,
                  seed=a.gen_seed, eos_token_id=eot, repetition_penalty=rep,
                  no_repeat_ngram_size=nrn, typical_p=a.typical_p)
    tetto = frazione_al_tetto(tok, hyps, max_new)
    print(f"Frazione al tetto: {tetto:.3f}" +
          ("   <- ALTO: stai misurando un troncamento, non la pragmatica"
           if tetto > 0.05 else "   ok"))

    etichetta_dec = "" if dec == "greedy" else (
        "__typical" if a.typical_p else f"__{dec}")
    run = f"{slug(repo_id)}__{a.task}__{tag}" + etichetta_dec
    res = {"run": run, "decoding": dec, "repo_id": repo_id, "task": a.task, "tag": tag,
           "architettura": arch, "n_test": len(rows),
           "F_riferimento": riferimento(hyps, refs),
           "F_bootstrap_chrf": bootstrap_chrf(hyps, refs),
           "F_baseline": baseline_non_banali(a.task, rows, refs),
           "C_distribuzionali": distribuzionali(hyps, refs),
           "C_degenerazione": degenerazione(hyps, refs),
           "A_avversariale": discriminazione_avversariale(hyps, refs),
           "G_troncamento": {
               "frazione_al_tetto": round(tetto, 3),
               "max_new_tokens": max_new,
               "_lettura": "frazione di output che raggiunge max_new senza "
                           "emettere fine di turno. Sopra ~0.05 le altre "
                           "metriche misurano il troncamento.",
           },
           "H_stratificato": stratifica_per_lunghezza(hyps, refs),
           "gen_kw": {"decoding": dec, "temperature": a.temperature,
                      "top_p": None if a.typical_p else a.top_p,
                      "typical_p": a.typical_p, "repetition_penalty": rep,
                      "no_repeat_ngram_size": nrn, "eos_token_id": eot,
                      "max_new_tokens": max_new, "seed": a.gen_seed}}

    clf_dial, info = costruisci_discriminatore_dialetto(a.split_dir)
    if clf_dial is not None:
        pm = clf_dial.predict_proba([norm(x) for x in hyps])[:, 1]
        pr = clf_dial.predict_proba([norm(x) for x in refs])[:, 1]
        res["D_dialettalita"] = {
            "P_nap_generato": round(float(pm.mean()), 3),
            "P_nap_riferimenti_umani": round(float(pr.mean()), 3),
            "discriminatore": info,
            "_lettura": "confronta P_nap_generato con P_nap_riferimenti_umani: "
                        "il bersaglio e' quello, non 1.0.",
        }
    else:
        res["D_dialettalita"] = info

    if not a.skip_contrastive:
        tok.padding_side = "right"                     # scoring, non generazione
        print("Contrastiva (nessuna generazione, solo forward)...")
        res["B_contrastiva"] = contrastiva(model, tok, rows, a.contrastive_k, a.max_len,
                                           device, limite=a.contrastive_limit)

    outdir = Path(a.out)
    outdir.mkdir(parents=True, exist_ok=True)
    with (outdir / f"{run}.preds.jsonl").open("w", encoding="utf-8") as f:
        for r, h in zip(rows, hyps):
            f.write(json.dumps({"id": r.get("id"), "prompt": r["prompt"],
                                "target": r["target"], "hyp": h}, ensure_ascii=False) + "\n")
    (outdir / f"{run}.metrics.json").write_text(
        json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(res, ensure_ascii=False, indent=2))
    print(f"\nSalvato in {outdir}/{run}.metrics.json")


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/evaluate_task.py


## Passo 15b — `prova_esempi.py`

**Cosa fare:** esegui (scrive il file). Serve dopo il training, per provare gli
adapter su frasi tue con prompt **byte-identici** a quelli visti in addestramento.

I prompt vengono presi da una riga vera di `test.json` e si sostituisce solo la
parte variabile: un prompt ricostruito a mano da' output peggiori del modello
reale, e non sapresti se stai vedendo un limite del modello o un errore di
formato. Con `--lessico` stampa, prima di caricare il modello, quali parole della
tua frase hanno una resa dialettale attestata nel corpus e quali no.

In [16]:
%%writefile /kaggle/working/training/prova_esempi.py
#!/usr/bin/env python3
"""
prova_esempi.py — prova un adapter addestrato su frasi tue, con prompt
BYTE-IDENTICI a quelli del training.

Perche' non basta scrivere il prompt a mano
-------------------------------------------
I tre layout hanno formati precisi (blocco di contesto, istruzione, separatori) e
il fine-tuning ha legato il comportamento a quel formato. Un prompt ricostruito
"a occhio" produce output peggiori del modello reale, e non sai se stai vedendo
un limite del modello o un errore di formattazione. Qui il prompt viene preso da
una riga vera di test.json e si sostituisce SOLO la parte variabile dopo
l'istruzione, che e' l'unico punto in cui il tuo input entra.

Cosa aspettarsi per ciascun task
--------------------------------
T1  Traduci in napoletano: <frase italiana>
    Funziona sul formato. Ma il corpus e' parlato conversazionale con mediana 4
    parole: frasi lunghe e di dominio assente (meteo, oggetti mai citati) daranno
    resa parziale. Non e' un bug, e' copertura lessicale.

T2  Continua il turno in napoletano: <prima meta' del turno, IN NAPOLETANO>
    Intra-lingua: il prefisso deve essere GIA' in napoletano. Passare un prefisso
    italiano e' un task mai addestrato. Se non hai un prefisso napoletano, usa
    --da-test per vedere il task come e' stato addestrato.

T3  <3 turni di contesto in napoletano> + istruzione a rispondere
    Non e' instruction-following: non risponde a "parlami di qualcosa". Serve
    contesto conversazionale in napoletano. Con --contesto lo passi tu, altrimenti
    si usa quello di una riga di test.

Uso
---
    # traduzione di frasi tue
    python prova_esempi.py --model minerva --task T1 \\
        --adapter /kaggle/working/runs/Minerva-7B-instruct-v1.0__T1/adapter_final \\
        --frase "Oggi e' proprio una giornata nuvolosa, dovevo portare l'ombrello" \\
        --frase "Non lo so, forse domani"

    # completamento (prefisso napoletano)
    python prova_esempi.py --model minerva --task T2 --adapter ... \\
        --frase "Nun 'o saccio, pecche'"

    # replica con contesto tuo
    python prova_esempi.py --model minerva --task T3 --adapter ... \\
        --contesto "A: Comme staje?" --contesto "B: Nun c'e' male, e tu?"

    # il task come e' stato addestrato, su righe vere di test
    python prova_esempi.py --model minerva --task T3 --adapter ... --da-test 5

Diagnostica lessicale
---------------------
Con --lessico stampa, per ogni frase italiana di T1, quali parole hanno una resa
dialettale attestata nel corpus e quali no. Le seconde non possono essere
tradotte: il modello non le ha mai viste in napoletano. E' la risposta a "perche'
mi ha lasciato 'ombrello' in italiano".
"""

import argparse
import json
import re
import sys

from common import (MODEL_REGISTRY, load_backbone, load_hf_token, load_split,
                    render_prompt, resolve_model, scegli_dtype, slug)

ISTRUZIONE = {"T1": "Traduci in napoletano: ",
              "T2": "Continua il turno in napoletano: "}
MAX_NEW = {"T1": 64, "T2": 48, "T3": 64}


def splicing(prompt_modello, task, testo):
    """Sostituisce la parte variabile di un prompt reale, tenendo il resto.

    Per T1 e T2 l'istruzione delimita esattamente il punto di innesto. Se il
    marker non c'e', il layout e' cambiato: meglio fermarsi che tirare a indovinare.
    """
    marker = ISTRUZIONE[task]
    if marker not in prompt_modello:
        sys.exit(f"ERRORE: '{marker.strip()}' non compare nel prompt di test di "
                 f"{task}. Il layout e' cambiato: rigenera lo split o aggiorna "
                 f"ISTRUZIONE in questo script.")
    testa, _, coda = prompt_modello.rpartition(marker)
    # la coda oltre la frase (eventuale suffisso del layout) va conservata
    resto = coda.split("\n", 1)
    suffisso = "\n" + resto[1] if len(resto) > 1 else ""
    return testa + marker + testo + suffisso


def prompt_t3_con_contesto(prompt_modello, turni):
    """Riscrive il blocco di contesto di un prompt T3 con i turni forniti."""
    righe = prompt_modello.split("\n")
    # il blocco di contesto e' delimitato dall'intestazione e dalla riga vuota
    try:
        i = next(k for k, r in enumerate(righe) if "Conversazione" in r)
    except StopIteration:
        sys.exit("ERRORE: nessun blocco di contesto nel prompt T3 di test.")
    from contesto_metrica import _e_separatore
    j = next((k for k in range(i + 1, len(righe)) if _e_separatore(righe[k])),
             len(righe))
    return "\n".join(righe[:i + 1] + list(turni) + righe[j:])


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1],
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True, help=f"alias ({'/'.join(MODEL_REGISTRY)}) o repo_id")
    ap.add_argument("--task", required=True, choices=["T1", "T2", "T3"])
    ap.add_argument("--adapter", default=None, help="senza adapter = modello di base")
    ap.add_argument("--split-dir", default="/kaggle/working/split")
    ap.add_argument("--frase", action="append", default=[],
                    help="ripetibile. T1: frase italiana. T2: prefisso NAPOLETANO")
    ap.add_argument("--contesto", action="append", default=[],
                    help="ripetibile, per T3: turni di contesto in napoletano, es. \"A: Comme staje?\"")
    ap.add_argument("--da-test", type=int, default=0,
                    help="quante righe vere di test provare, per vedere il task come e' addestrato")
    ap.add_argument("--lessico", default=None, help="output di lessico.py, per la diagnostica")
    ap.add_argument("--decoding", choices=["greedy", "nucleus"], default=None,
                    help="default: greedy su T1/T2, nucleus su T3 (come in evaluate_task.py)")
    ap.add_argument("--top-p", type=float, default=0.9)
    ap.add_argument("--temperature", type=float, default=0.8)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--max-new", type=int, default=None)
    ap.add_argument("--hf-token", default=None)
    a = ap.parse_args()

    if not a.frase and not a.contesto and not a.da_test:
        sys.exit("Passa almeno --frase, --contesto o --da-test.")

    righe_test = load_split(a.split_dir, a.task, "test")
    if not righe_test:
        sys.exit(f"Nessuna riga di test per {a.task} in {a.split_dir}.")
    modello_prompt = righe_test[0]["prompt"]

    # --- costruzione dei prompt --------------------------------------------
    casi = []                                     # (etichetta, prompt, riferimento)
    for i, r in enumerate(righe_test[:a.da_test]):
        casi.append((f"test[{i}]", r["prompt"], r["target"]))
    if a.task == "T3":
        if a.contesto:
            casi.append(("contesto tuo",
                         prompt_t3_con_contesto(modello_prompt, a.contesto), None))
        for f in a.frase:
            print(f"! T3 ignora --frase ({f!r}): serve --contesto. T3 continua una "
                  f"conversazione, non risponde a un'istruzione.")
    else:
        for f in a.frase:
            casi.append((f"tua: {f[:40]}", splicing(modello_prompt, a.task, f), None))

    # --- diagnostica lessicale (prima di caricare il modello) ---------------
    if a.lessico and a.task == "T1" and a.frase:
        from pesi_lessicali import carica_lessico, tokenizza
        lex = carica_lessico(a.lessico)
        print("\n=== copertura lessicale delle tue frasi ===")
        print("Una parola senza resa attestata NON puo' essere tradotta: quella "
              "forma napoletana non compare nel corpus di train.\n")
        for f in a.frase:
            noti, ignoti = [], []
            for w in tokenizza(f):
                (noti if w in lex["it2nap"] else ignoti).append(w)
            print(f"  {f}")
            print(f"    resa attestata ({len(noti)}): {' '.join(noti) or '-'}")
            print(f"    nessuna resa  ({len(ignoti)}): {' '.join(ignoti) or '-'}")
            tot = len(noti) + len(ignoti)
            print(f"    copertura: {len(noti)/tot:.0%}\n" if tot else "")

    # --- modello ------------------------------------------------------------
    import torch
    from transformers import AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    repo_id = resolve_model(a.model)
    token = load_hf_token(a.hf_token)
    tok = AutoTokenizer.from_pretrained(repo_id, token=token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    dtype, _ = scegli_dtype(repo_id)      # fp32 su Gemma senza bf16
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_use_double_quant=True,
                               bnb_4bit_compute_dtype=dtype)
    model, arch = load_backbone(repo_id, dtype, quantization_config=quant,
                                device_map={"": 0}, token=token,
                                low_cpu_mem_usage=True, attn_implementation="eager")
    if a.adapter:
        model = PeftModel.from_pretrained(model, a.adapter)
        print(f"\nAdapter: {a.adapter}")
    else:
        print("\n! nessun adapter: stai provando il modello di BASE (baseline zero-shot)")
    model.eval()
    model.config.use_cache = True

    dec = a.decoding or ("nucleus" if a.task == "T3" else "greedy")
    max_new = a.max_new or MAX_NEW[a.task]
    # Id di fine turno: senza, la generazione arriva sempre a max_new_tokens e
    # le frasi di prova sembrano piu' lunghe e sconclusionate di quanto siano.
    from common import id_fine_turno
    eot = id_fine_turno(tok, [], verbose=False)
    torch.manual_seed(a.seed)
    print(f"{slug(repo_id)} | {a.task} | arch={arch} | decoding={dec} | max_new={max_new}")

    for etichetta, prompt, riferimento in casi:
        testo = render_prompt(tok, prompt)
        enc = tok(testo, return_tensors="pt", add_special_tokens=False).to(0)
        kw = dict(max_new_tokens=max_new, pad_token_id=tok.pad_token_id,
                  eos_token_id=eot)
        if dec == "nucleus":
            kw.update(do_sample=True, top_p=a.top_p, temperature=a.temperature)
        else:
            kw.update(do_sample=False)
        # Gemma-3: generate() e' rotta su Gemma3ForConditionalGeneration in
        # questa transformers ("Tensors must have same number of dimensions").
        # Stesso loop manuale di evaluate_task.py, cosi' le frasi di prova si
        # generano con lo stesso decoder della valutazione.
        mt = (getattr(getattr(model, "config", None), "model_type", "") or "").lower()
        with torch.no_grad():
            if "gemma3" in mt:
                from evaluate_task import _decode_gemma3_manuale
                out = _decode_gemma3_manuale(model, enc, dec, max_new, 1,
                                             a.top_p, a.temperature, eot, 1.0, 0,
                                             tok=tok)
            else:
                out = model.generate(**enc, **kw)
        gen = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        gen = gen.strip().split("\n")[0].strip()      # come in evaluate_task.py

        print("\n" + "=" * 70)
        print(f"[{etichetta}]")
        print("-" * 70)
        print(prompt)
        print("-" * 70)
        print(f"GENERATO:    {gen}")
        if riferimento:
            print(f"RIFERIMENTO: {riferimento}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/prova_esempi.py


In [17]:
# --- Passo 16. Verifica dei file e delle config -----------------------------------
!ls -la /kaggle/working/training/*.py
import importlib, common
importlib.reload(common)

# solo i due entrypoint che servono qui: A2 (condiviso) e T1
for m in ("finetune_a2_lessico", "finetune_t1_traduzione"):
    t = importlib.reload(importlib.import_module(m)).TASK
    print(f"{t.layout}  seq={t.max_seq_len}  ep={t.epochs}  lr={t.lr:g}  "
          f"r={t.lora_r}/{t.lora_alpha} drop={t.lora_dropout}  "
          f"metrica={t.metric}  patience={t.patience}")

print("\nAlias modello:")
for k, v in common.MODEL_REGISTRY.items():
    print(f"  {k:16s} {v}")

-rw-r--r-- 1 root root 68385 Sep  3 23:24 /kaggle/working/training/common.py
-rw-r--r-- 1 root root 10528 Sep  3 23:24 /kaggle/working/training/contesto_metrica.py
-rw-r--r-- 1 root root 11428 Sep  3 23:24 /kaggle/working/training/dati_lessicali.py
-rw-r--r-- 1 root root 11236 Sep  3 23:24 /kaggle/working/training/decodifica_contestuale.py
-rw-r--r-- 1 root root 43620 Sep  3 23:24 /kaggle/working/training/evaluate_task.py
-rw-r--r-- 1 root root  3172 Sep  3 23:24 /kaggle/working/training/finetune_a2_lessico.py
-rw-r--r-- 1 root root  1928 Sep  3 23:24 /kaggle/working/training/finetune_t1_traduzione.py
-rw-r--r-- 1 root root  8726 Sep  3 23:24 /kaggle/working/training/lessico.py
-rw-r--r-- 1 root root 14610 Sep  3 23:24 /kaggle/working/training/pesi_lessicali.py
-rw-r--r-- 1 root root 13514 Sep  3 23:24 /kaggle/working/training/pretrain_dialect.py
-rw-r--r-- 1 root root 11395 Sep  3 23:24 /kaggle/working/training/prova_esempi.py
A2  seq=512  ep=3  lr=5e-05  r=16/32 drop=0.05  metrica=ch

In [18]:
# --- Passo 17. Configurazione di questo notebook ----------------------------------
MODEL = "gemma"     # llama | minerva | gemma   (uno per notebook)
SEED  = 42
TASK  = "T1"           # fisso: questo notebook addestra solo T1

# --- artefatti condivisi con gli altri due notebook -------------------------------
# Stadi A e A2: UN run per modello, non uno per task. Se li hai gia' fatti
# (tipicamente nel notebook T1) e ne hai allegato l'output come Dataset, incolla
# qui il percorso dell'adapter A2: i Passi 22 e 23 si saltano da soli.
# Vuoto = A e A2 vengono eseguiti in questo notebook (~2-3 h di GPU).
ADAPTER_A2_ESTERNO = ""   # es. "/kaggle/input/nap-t1/runs/gemma-3-4b-it__A2/adapter_final"

print("Modello:", common.MODEL_REGISTRY[MODEL], "| task:", TASK, "| split:", SPLIT)
print("A2 esterno:", ADAPTER_A2_ESTERNO or "(no: A e A2 girano qui)")

Modello: google/gemma-3-4b-it | task: T1 | split: /kaggle/working/split
A2 esterno: (no: A e A2 girano qui)


## Passo 18 — smoke test

Un minuto su `gemma-tiny` (google/gemma-3-1b-it), per verificare percorsi, token,
chat template e masking della loss **prima** di scaricare il 4b.

La versione precedente girava su `minerva-small`: repo gated senza accesso
(403) e, soprattutto, chat template di un altro modello — uno smoke test che
non testava il percorso che poi si usa davvero.

`--gen-metric-n 4` invece di `--no-sample-generation`: qui serve proprio che
la generazione reale giri, perche' e' il pezzo che era rotto.

Se compare `PREFIX MISMATCH`, fermati: il prompt renderizzato non è un prefisso
esatto del testo completo e la loss sta guardando le posizioni sbagliate.
Se compare un avviso sui troncamenti, alza `--max-seq-len`.


In [19]:
# --- Passo 18. Smoke test ---------------------------------------------------
# CORRETTO: era --model minerva-small, cioe' sapienzanlp/Minerva-3B-base-v1.0,
# un repo GATED a cui questo account non ha accesso -> 403 GatedRepoError e lo
# smoke test non partiva. Su un notebook Gemma va comunque usato un checkpoint
# Gemma: minerva-small avrebbe verificato il chat template sbagliato.
# gemma-tiny = google/gemma-3-1b-it (stessa licenza gia' accettata per il 4b).
!python finetune_t1_traduzione.py --model gemma-tiny --split-dir {SPLIT} \
    --max-train-samples 48 --max-dev-samples 24 --epochs 1 --eval-subset 24 \
    --gen-metric-n 4


=== T1 (traduzione con contesto) | google/gemma-3-1b-it ===
GPU: Tesla T4 (cc 7.5), 15.6 GB VRAM | bf16 nativo: False -> precisione: float32
  Gemma su GPU senza bf16: fp32, quindi niente autocast fp16 e niente GradScaler.
  In fp16 il GradScaler scarta gli step con gradienti inf/nan e nei log compaiono
  grad_norm=nan con learning_rate=0: sono step che NON hanno aggiornato l'adapter.
  Costo: circa 2x di tempo per step.
  ~1B parametri -> ~0.5 GB di soli pesi in qlora4bit (più adapter, ottimizzatore, attivazioni)
config.json: 100%|█████████████████████████████| 899/899 [00:00<00:00, 3.45MB/s]
tokenizer_config.json: 1.16MB [00:00, 33.8MB/s]
tokenizer.json: 100%|██████████████████████| 33.4M/33.4M [00:00<00:00, 40.6MB/s]
added_tokens.json: 100%|██████████████████████| 35.0/35.0 [00:00<00:00, 227kB/s]
special_tokens_map.json: 100%|█████████████████| 662/662 [00:00<00:00, 3.71MB/s]
Carico il modello (qlora4bit)...
model.safetensors: 100%|████████████████████| 2.00G/2.00G [00:08<00:00, 232

## Passo 19 — estrai il lessico dal solo train

**Cosa fare:** esegui e **guarda le prime voci stampate**. Se le prime 10 non
sono corrispondenze evidenti (`non->nun`, `che->ca`, `la->'a`), l'allineamento
non ha convergito e non ha senso proseguire: controlla che le colonne
`italiano`/`napoletano` del CSV siano nell'ordine giusto.

Controlla anche la riga "Filtro sul train dello split": se sopravvivono
pochissime righe, le chiavi `(conversazione, turn_index)` del CSV e dello split
non combaciano.

`--escludi-fonte gemma4` e' commentato di proposito. Le 515 righe `gemma4` hanno
0,87 parole dialettali per turno contro 3,39 delle `golden`, quindi diluiscono il
segnale; ma escluderle cambia il pool di training e va dichiarato come scelta,
non fatto di default. Se le escludi qui, escludile anche nel passo 21.

In [20]:
# --- Passo 19. Lessico allineato italiano -> napoletano (solo train) --------------
!python lessico.py --csv {CSV} --split-dir {SPLIT} --out /kaggle/working/lessico_train.json
# variante: aggiungi  --escludi-fonte gemma4   (poi anche nel passo 21)

LEX_JSON = "/kaggle/working/lessico_train.json"
LEX = f"--lessico {LEX_JSON}"      # entra nelle celle T1/T2/T3
import json
_l = json.load(open(LEX_JSON))
print(f"\nTipi dialettali: {len(_l['tipi_dialettali'])} | voci: {len(_l['lessico'])}")

Filtro sul train dello split: 2568 -> 1147 righe (1147 chiavi di train)
Coppie allineate: 1147
Tipi dialettali: 988 (4711 occorrenze, 50.5% dei token napoletani)
Voci di lessico: 348 (290 con forma dialettale)

Scritto /kaggle/working/lessico_train.json
Prime 25 voci:
         che -> ca           p=1.00 cooc=227
         non -> nun          p=1.00 cooc=204
          la -> 'a           p=1.00 cooc=141
          di -> 'e           p=1.00 cooc=129
          un -> nu           p=1.00 cooc=104
          il -> 'o           p=1.00 cooc=93
         poi -> po'          p=1.00 cooc=92
       anche -> pure         p=1.00 cooc=88
      perché -> pecché       p=1.00 cooc=88
          lo -> 'o           p=1.00 cooc=77
          se -> si           p=1.00 cooc=66
         per -> pe'          p=1.00 cooc=65
         una -> na           p=1.00 cooc=65
          io -> i'           p=1.00 cooc=64
          ci -> ce           p=1.00 cooc=64
      quindi -> quinni       p=1.00 cooc=63
          lì -> llà   

## Passo 20 — costruisci i dati dello stadio A2

**Cosa fare:** esegui e leggi l'esempio di `cloze` stampato in fondo. Deve avere
il contesto conversazionale, la frase italiana, la resa napoletana con `___` e un
target di una sola parola dialettale. Se il target e' una parola italiana comune,
il set dei tipi dialettali e' sbagliato.

Gli item finiscono in `{SPLIT}/stadio_a2_lessico/`, accanto ai tre layout, e
`common.py` li vede grazie a `LAYOUT_DIRS["A2"]`.

In [21]:
# --- Passo 20. Dataset dello stadio A2 --------------------------------------
# --richiedi-contesto: senza il filtro, con un train di 1147 turni su 2568 la
# finestra di contesto si tronca quasi sempre e il 58% degli item finisce senza
# contesto. Quegli item insegnano a produrre napoletano senza guardare indietro,
# e A2 e' l'anello immediatamente prima di T2 e T3.
# Effetto misurato: 2216 -> 1032 item, di cui il 90% con contesto (il residuo
# sono le glossa, che per natura non ce l'hanno, tenute al 10%).
# Meno item significa anche meta' del tempo GPU su A2.
!python dati_lessicali.py --csv {CSV} --split-dir {SPLIT} \
    --lessico {LEX_JSON} --out-dir {SPLIT}/stadio_a2_lessico \
    --richiedi-contesto --quota-glossa 0.10


--richiedi-contesto: 2910 -> 1930 item ({'glossa': 226, 'cloze': 1286, 'turno_breve': 418})
  glossa ridotta a 189 item (tetto 10% del totale)
  item senza contesto residui: 189/1893 (10.0%) — sono le glossa
Item per tipo: {'glossa': 226, 'cloze': 2026, 'turno_breve': 658}
Totale 1893 -> train 1742, dev 151
Scritto in /kaggle/working/split/stadio_a2_lessico

Esempio cloze:
------------------------------------------------------------
Completa la frase napoletana inserendo la parola che manca al posto di ___. Scrivi soltanto quella parola.

Conversazione finora (in napoletano):
PKP007: cioè si tu dice "faccio smart working, vaco a faticà 'a machu picchu", tipo okay, o "faccio cunzulenze"
PKP135: assolutamente, cioè tu
PKP135: no no, ma 'nfatti i' 'ncopp'a chesto, cioè pur'io 'a penzo ô stesso modo, però 'a nu lato penzo "saje che ce sta"

Italiano: vaffanculo per come si sta adesso anche
Napoletano: vaffanculo, pe' comme se sta ___ pure
Parola mancante:
--- TARGET: mo,


## Passo 21 — autotest delle metriche lessicali

**Cosa fare:** esegui e guarda i due blocchi. Servono come fondo scala e tetto,
da leggere **prima** dei numeri di qualsiasi modello:

- baseline copia-italiano: recall dialettale ~0,00 e tasso italianismi 1,00
- riferimento umano: recall 1,00 e tasso italianismi ~0,04 (non zero, perche' a
  volte la forma italiana e' legittimamente quella giusta)

Un modello che finisce vicino al primo blocco ha imparato il compito e non il
dialetto, qualunque cosa dica il chrF. Il terzo blocco spacca per fonte e mostra
quanto le righe `gemma4` siano povere di segnale.

In [22]:
# --- Passo 21. Fondo scala e tetto delle metriche lessicali -----------------------
!python pesi_lessicali.py --csv {CSV} --lessico {LEX_JSON}

Baseline COPIA-ITALIANO (l'ipotesi e' la frase italiana):
{
 "recall_dialettale": 0.0017,
 "precisione_dialettale": 0.9231,
 "f1_dialettale": 0.0034,
 "tasso_italianismi": 1.0,
 "tasso_copia": 1.0,
 "_conteggi": {
  "tipi_dial_riferimento": 7035,
  "tipi_dial_ipotesi": 13,
  "occasioni_lessicali": 5960,
  "n": 2568
 },
 "_avvertenza": "precisione_dialettale calcolata su sole 13 forme dialettali prodotte: non interpretabile"
}

Tetto: RIFERIMENTO UMANO come ipotesi (controllo di sanita'):
{
 "recall_dialettale": 1.0,
 "precisione_dialettale": 1.0,
 "f1_dialettale": 1.0,
 "tasso_italianismi": 0.0466,
 "tasso_copia": 0.229,
 "_conteggi": {
  "tipi_dial_riferimento": 7035,
  "tipi_dial_ipotesi": 7035,
  "occasioni_lessicali": 5960,
  "n": 2568
 },
 "_avvertenza": null
}

Solo fonte=gemma4 (515 righe), baseline copia:
{
 "recall_dialettale": 0.0,
 "precisione_dialettale": 0.0,
 "f1_dialettale": 0.0,
 "tasso_italianismi": 1.0,
 "tasso_copia": 1.0,
 "_conteggi": {
  "tipi_dial_riferimento": 4

## Passo 22 — STADIO A: continued pretraining sul napoletano

Si esegue **una volta per modello** e serve tutti e tre i task.

Perche': l'SFT sui task addestra coppie prompt->target, quindi l'adapter impara a
condizionare i *primi* token della risposta, ma la distribuzione linguistica
sottostante resta quella del modello di base. Nei log si vede: la generazione
parte in napoletano e scivola in italiano man mano che si allontana dal prompt.
Il continued pretraining sposta la distribuzione, non solo il condizionamento
iniziale.

E usa tutto il testo disponibile invece delle sole parole di target: **11.229
parole** sul pool di train contro le 6.005 che T3 usa come supervisione, cioe'
1,9 volte tanto.

Il testo viene ricostruito **solo** dai `train.json` dei tre layout: dev e test
non entrano mai nel training, altrimenti i turni di test comparirebbero come
testo di pretraining e la valutazione sarebbe invalida.

Costa una decina di minuti. Con `DUE_STADI = False` si salta e si torna all'SFT
diretto, che e' l'altro arm dell'ablation.

**Cosa fare:** esegui e controlla la perplexity di dev nella cella successiva: da iniziale a finale deve **scendere**. Se sale, il continued pretraining sta degradando il modello invece di spostarne la distribuzione: abbassa `--lr` a 2e-5 e rifai.

**Un solo run per modello.**


## Correzione 4 — `grad_norm: nan` e `learning_rate: 0` nello stadio A

Nel run precedente le prime righe di log erano:

```
{'loss': '6.337', 'grad_norm': 'nan', 'learning_rate': '0',   'epoch': '0.1951'}
{'loss': '6.196', 'grad_norm': 'nan', 'learning_rate': '0',   'epoch': '0.3902'}
{'loss': '6.11',  'grad_norm': 'nan', 'learning_rate': '0',   'epoch': '0.5854'}
```

`grad_norm: nan` con `learning_rate: 0` non e' il warmup: e' il **GradScaler di
fp16** che trova gradienti inf/nan, scarta l'ottimizzazione e non fa avanzare
neanche lo scheduler. Quegli step non hanno aggiornato l'adapter. Su uno stadio
A da **18 step totali**, quattro scartati sono un quarto del training.

E' lo stesso overflow che in valutazione dava le uscite vuote, visto dal lato del
gradiente. L'upcast dei tensori non quantizzati non basta qui, perche'
`TrainingArguments(fp16=True)` attiva l'autocast: i matmul tornano in fp16 a
prescindere dal dtype dei parametri.

Quindi la regola di `scegli_dtype()` vale ora anche per il training: su Gemma
senza bf16 nativo il modello e' in fp32 e `TrainingArguments` ha
`fp16=False, bf16=False`. Niente autocast, niente GradScaler, nessuno step
scartato. Prima di addestrare, `controlla_finito()` fa un forward e stampa
`logit finiti: True`.

**Costo.** Circa 2x per step. Sui tempi del run precedente: stadio A ~3 min ->
~6, stadio A2 ~51 min -> ~1h40, stadio B (T1) ~46 min -> ~1h30. In tutto circa
3h15 contro 1h40: sta dentro le 12 ore di sessione, ma non lasciarlo per ultimo.

**Cosa rifare.** Gli stadi A, A2 e B di Gemma vanno rifatti: gli adapter attuali
vengono da un training con step scartati. Se vuoi tenere il confronto, salva
prima `/kaggle/working/runs` e `/kaggle/working/cpt` con un nome diverso. Per
Llama e Minerva non cambia niente: in fp16 girano davvero, e `--dtype fp16`
resta disponibile se ti serve riprodurre un run vecchio.


In [23]:
DUE_STADI = True     # False = SFT diretto (arm di confronto)

if ADAPTER_A2_ESTERNO:
    print("Stadi A e A2 saltati: adapter A2 preso dall'esterno.")
    print("  ", ADAPTER_A2_ESTERNO)
elif DUE_STADI:
    !python pretrain_dialect.py --model {MODEL} --split-dir {SPLIT} --dump-text
else:
    print("stadio A saltato: SFT diretto")

=== STADIO A — continued pretraining | google/gemma-3-4b-it ===
GPU Tesla T4 (cc 7.5) -> float32
  (Gemma senza bf16 nativo: fp32. In fp16 il GradScaler scarta gli step con gradienti inf/nan
   e nei log compaiono grad_norm=nan e learning_rate=0, cioe' step che non aggiornano l'adapter.)
config.json: 100%|█████████████████████████████| 855/855 [00:00<00:00, 3.49MB/s]
tokenizer_config.json: 1.16MB [00:00, 43.5MB/s]
tokenizer.json: 100%|██████████████████████| 33.4M/33.4M [00:00<00:00, 53.2MB/s]
added_tokens.json: 100%|██████████████████████| 35.0/35.0 [00:00<00:00, 176kB/s]
special_tokens_map.json: 100%|█████████████████| 662/662 [00:00<00:00, 3.50MB/s]
Turni ricostruiti: train 1147 (11229 parole) | dev 271 (2184 parole)
  (dev e test non entrano mai nel training: sarebbe leakage)
  4 turni identici presenti sia in train sia in test (formule ricorrenti, mediana 3 parole): non e' leakage di split, e' ricorrenza lessicale nel parlato
  testo salvato in /kaggle/working/cpt/gemma-3-4b-it/te

In [24]:
# --- Passo 22b. Adapter dello stadio A --------------------------------------------
import glob, json, os
INIT = ""
if ADAPTER_A2_ESTERNO:
    print("(saltato: lo stadio A2 arriva dall'esterno)")
elif DUE_STADI:
    c = glob.glob("/kaggle/working/cpt/*/adapter_final")
    assert c, "adapter dello stadio A non trovato: la cella sopra e' andata a buon fine?"
    INIT = f"--init-adapter {c[0]}"
    print("Lo stadio A2 partira' da:", c[0])
    print(json.dumps(json.load(open(os.path.join(os.path.dirname(c[0]),
          "summary_cpt.json")))["perplexity_dev"], indent=2))
else:
    print("Lo stadio B partira' dal modello di base")

Lo stadio A2 partira' da: /kaggle/working/cpt/gemma-3-4b-it/adapter_final
{
  "iniziale": 89.05,
  "finale": 54.17,
  "migliore": 54.17
}


## Passo 23 — STADIO A2: iniezione lessicale

**Cosa fare:** esegui, poi controlla tre cose nell'output:

1. `Adapter iniziale caricato da .../cpt/...` — A2 **continua** l'adapter dello
   stadio A, non ne impila un secondo sopra. Se questa riga manca, A2 sta
   ripartendo dal modello di base e la catena e' rotta.
2. La riga `Dati A2: train ... | dev ...` deve dire ~3.600 / ~300.
3. Il `chrf` di dev deve salire nelle prime valutazioni. Su target di una parola
   coincide di fatto con l'accuratezza a livello di carattere, quindi valori alti
   (>80) sono normali e non indicano overfitting.

Se il dev continua a scendere mentre poi il chrF di T1 peggiora, taglia a
`--epochs 2` invece di alzare la patience: il rischio qui e' la memorizzazione
del lessico come lookup.

**Un solo run per modello.** Lo stesso adapter A2 alimenta T1, T2 e T3.

In [25]:
# --- Passo 23. Stadio A2 — iniezione lessicale (un run per modello) ---------------
if ADAPTER_A2_ESTERNO:
    print("(saltato: adapter A2 esterno)")
elif DUE_STADI:
    !python finetune_a2_lessico.py --model {MODEL} --split-dir {SPLIT} --seed {SEED} {INIT}
else:
    print("stadio A2 saltato: arm 'solo stadio B' dell'ablation")

=== A2 (iniezione lessicale) | google/gemma-3-4b-it ===
GPU: Tesla T4 (cc 7.5), 15.6 GB VRAM | bf16 nativo: False -> precisione: float32
  Gemma su GPU senza bf16: fp32, quindi niente autocast fp16 e niente GradScaler.
  In fp16 il GradScaler scarta gli step con gradienti inf/nan e nei log compaiono
  grad_norm=nan con learning_rate=0: sono step che NON hanno aggiornato l'adapter.
  Costo: circa 2x di tempo per step.
  ~4B parametri -> ~2.0 GB di soli pesi in qlora4bit (più adapter, ottimizzatore, attivazioni)
Carico il modello (qlora4bit)...
Loading weights: 100%|█| 883/883 [00:07<00:00, 112.06it/s, Materializing param=m
Target LoRA: 238 moduli, tipi ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
  esclusi 81 moduli omonimi nel vision tower (modello multimodale: gli adapter vanno solo sul language model)
Adapter iniziale caricato da /kaggle/working/cpt/gemma-3-4b-it/adapter_final (stadio A -> stadio B)
trainable params: 29,802,496 || all params: 4,329,88

In [26]:
# --- Passo 23b. Da dove parte lo stadio B ----------------------------------------
# Catena: base -> A (distribuzione) -> A2 (lessico) -> B (il task di questo
# notebook). Ogni anello CONTINUA l'adapter precedente (--init-adapter con
# is_trainable), quindi l'ultimo adapter porta dentro tutti gli stadi che lo
# precedono.
import glob, json, os
INIT2 = INIT
if ADAPTER_A2_ESTERNO:
    assert os.path.isdir(ADAPTER_A2_ESTERNO), \
        f"percorso inesistente: {ADAPTER_A2_ESTERNO} — hai allegato il Dataset?"
    INIT2 = f"--init-adapter {ADAPTER_A2_ESTERNO}"
    print("Stadio B partira' dall'A2 esterno:", ADAPTER_A2_ESTERNO)
elif DUE_STADI:
    c2 = glob.glob("/kaggle/working/runs/*__A2/adapter_final")
    if not c2:
        print("! adapter A2 non trovato: lo stadio B partira' dal solo stadio A.")
        print("  Controlla che la cella sopra sia arrivata in fondo.")
    else:
        INIT2 = f"--init-adapter {c2[0]}"
        s = json.load(open(os.path.join(os.path.dirname(c2[0]), "summary.json")))
        print("Stadio B partira' da:", c2[0])
        print("  A2 best:", s["selezione_checkpoint"]["metrica"],
              "=", s["selezione_checkpoint"]["best_value"])
print("INIT2 =", INIT2 or "(modello di base)")

Stadio B partira' da: /kaggle/working/runs/gemma-3-4b-it__A2/adapter_final
  A2 best: chrf = 66.44650995279467
INIT2 = --init-adapter /kaggle/working/runs/gemma-3-4b-it__A2/adapter_final


## Passo 24 — STADIO B: T1, traduzione con contesto

**Cosa fare:** esegui la cella qui sotto e misura il `s/it` nel primo minuto.
Parte da `{INIT2}` (l'adapter A2) e porta `{LEX}` (la pesatura della loss).

All'avvio controlla due righe:

- `Pesatura lessicale attiva: N tipi dialettali, peso 3.0`
- `pesatura: X/Y token di target dialettali (Z%)` — Z deve stare fra il 20% e il
  60%. Se e' ~0% il lessico non aggancia i target e la pesatura e' inerte:
  verifica che il tokenizer sia fast.

T1 e' l'unico dei tre task in cui `{LEX}` ha senso, ed e' il motivo per cui la
pesatura esiste: e' l'unico task cross-lingua, quindi l'unico dove il modello
puo' abbassare la loss ricopiando l'italiano. In T2 e T3 il prompt e' gia'
napoletano e non c'e' niente da ricopiare.

Il batch efficace resta 16 su tutti i run della matrice: e' la variabile che
tiene comparabili i confronti fra modelli, non un parametro da regolare per
cella. Se sfori il tempo, le leve sono `--batch-size 4 --grad-accum 4` (batch
efficace invariato), `--no-gradient-checkpointing` e `--eval-subset 100`.

## Cosa rifare, e cosa no

Nel `TrainerPesato` della versione precedente c'era un bug di normalizzazione:
`transformers` non divide la loss per gli step di accumulo quando il forward del
modello accetta `**kwargs` (un `PeftModel` lo accetta sempre), perche' si aspetta
una loss normalizzata su `num_items_in_batch`. La mia restituiva una media per
micro-batch, quindi i gradienti accumulati erano ~8x troppo grandi. Si vede nel
log di T2: train loss 27 contro `eval_loss` 3,04 (rapporto 8,9 = il tuo
`grad_accum`) e `grad_norm` a 270/163/100, cioe' `max_grad_norm=1.0` che clippa a
ogni step.

| run | stato | perche' |
|---|---|---|
| **Stadio A** (CPT) | **si tiene** | non passa `--lessico`, usa il `Trainer` standard. Perplexity dev 19,72 -> 17,37 |
| **Stadio A2** | **si tiene** | la cella 23 non passa `{LEX}`: nessun `TrainerPesato`, nessun bug. Sono 2h15m che non rifai |
| **T1** | da rifare | girato con `{LEX}` |
| **T2** | da rifare | girato con `{LEX}`, piu' i nuovi parametri |
| **T3** | da rifare se girato | idem |

Quindi si riparte dalla cella 23b (`INIT2`), non da capo.

### Nota sul budget

Il run gira su **Tesla T4**, non sulla P100 dichiarata nei Settings: fp16 invece
di bf16, e A2 viaggiava a 16,3 s/it (504 step = ~2h15m). Se devi rifare A2 per
un altro modello, `--epochs 2` lo porta a ~1h30m, oppure
`dati_lessicali.py --cloze-per-turno 1` dimezza gli item di cloze.
---

## Aggiornamento — cosa cambia in questa versione

Correzione a quanto detto prima: **il bug dell'accumulo era cosmetico, non
sostanziale.** Il clipping agisce prima dello step dell'ottimizzatore, e
`grad_norm` stava sempre sopra `max_grad_norm=1.0` (51-270 prima, 7,1 dopo): un
fattore scalare uniforme su un gradiente che viene poi rinormalizzato a norma 1
non cambia ne' direzione ne' magnitudine dell'aggiornamento. I log erano
illeggibili, la traiettoria no. **I numeri dei run vecchi restano usabili.**

| modifica | dove | perche' |
|---|---|---|
| generazione T3 a 32 token | `finetune_t3_replica.py`, `evaluate_task.py` | i turni umani hanno mediana 4 parole; a 64 token il modello arrivava a 40 parole contro un target di 10, e la deriva semantica si accumulava nella coda |
| `--richiedi-contesto` su A2 | cella del passo 20 | il 58% degli item A2 era senza contesto: 2216 -> 1032 item, 90% contestuali, e meta' del tempo GPU |
| via `--metric ctx_acc` da T3 | cella di T3 | `ctx_acc` e' piatta durante il training (0,609 / 0,609 / 0,578): non discrimina fra checkpoint |
| `vocabolario_nap` nel lessico | `lessico.py` | serve alle forme inedite |
| forme inedite | `metriche_finali.py` | `puparuolo`, `cusinajo`, `pruove ccusarelle`: forme inventate per analogia, fluenti, che superano il discriminatore ita/nap e che nessun'altra metrica vede |

### Cosa va rieseguito

1. **Passo 19** (`lessico.py`): pochi secondi, serve per `vocabolario_nap`.
2. **Passo 20** (`dati_lessicali.py`) con il nuovo filtro.
3. **Stadio A2**: sui nuovi item. ~1032 invece di 2039, quindi sotto l'ora.
4. **T1, T2, T3** dal nuovo adapter A2.

Lo **stadio A non si rifa'** (nessuna modifica lo riguarda).

### Tarature da riportare, non da massimizzare

Tre metriche hanno bersagli bilaterali e vanno lette contro il valore umano, non
spinte verso l'alto:

| metrica | valore umano misurato | come si legge |
|---|---|---|
| `P(nap)` | **0,877** su T3 | sopra 0,88 il modello sta *spargendo* clitici piu' di un parlante |
| forme inedite | ~0,08 sui riferimenti di test | conta solo l'eccesso su quel valore |
| discriminazione avversariale | controllo umano-vs-umano **0,425** | banda di rumore ~±0,08: fra 0,35 e 0,58 e' indistinguibile |

E una nota metodologica da mettere nel testo: il discriminatore ita/nap ha
accuracy 0,979 / AUC 1,00, ma **esclude per costruzione** le coppie con italiano
identico al napoletano (22,9% del corpus) e i turni sotto le 3 parole. Va scritto,
altrimenti sembra che si rivendichi un identificatore di varieta' quasi perfetto
quando si ha un rilevatore di clitici apostrofati.


In [27]:
# T1 — traduzione con contesto (~1134 istanze, selezione su chrF)
# Baseline da battere: copia dell'italiano = 35,92 chrF++
# {LEX} SI: T1 e' l'unico task cross-lingua, quindi l'unico dove il modello puo'
# abbassare la loss ricopiando l'italiano. E' il caso per cui la pesatura esiste.
!python finetune_t1_traduzione.py --model {MODEL} --split-dir {SPLIT} --seed {SEED} \
    {INIT2} {LEX}

=== T1 (traduzione con contesto) | google/gemma-3-4b-it ===
GPU: Tesla T4 (cc 7.5), 15.6 GB VRAM | bf16 nativo: False -> precisione: float32
  Gemma su GPU senza bf16: fp32, quindi niente autocast fp16 e niente GradScaler.
  In fp16 il GradScaler scarta gli step con gradienti inf/nan e nei log compaiono
  grad_norm=nan con learning_rate=0: sono step che NON hanno aggiornato l'adapter.
  Costo: circa 2x di tempo per step.
  ~4B parametri -> ~2.0 GB di soli pesi in qlora4bit (più adapter, ottimizzatore, attivazioni)
Carico il modello (qlora4bit)...
Loading weights: 100%|█| 883/883 [00:07<00:00, 112.21it/s, Materializing param=m
Target LoRA: 238 moduli, tipi ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
  esclusi 81 moduli omonimi nel vision tower (modello multimodale: gli adapter vanno solo sul language model)
Adapter iniziale caricato da /kaggle/working/runs/gemma-3-4b-it__A2/adapter_final (stadio A -> stadio B)
trainable params: 29,802,496 || all params:

## Valutazione immediata (opzionale ma consigliata)

Gli adapter appena addestrati sono in `/kaggle/working/runs/`, quindi si possono
valutare qui senza salvare la versione e rimontarla come dataset.

**Attenzione al tetto di 12 ore della sessione.** I tre run di training di Llama
hanno preso 8,2 ore; queste tre celle aggiungono circa un'ora. Se il training e'
andato lungo, salta questa sezione e usa `kaggle_valutazione_napoletano.ipynb`
in una sessione nuova, montando l'output di questa come dataset.

Le baseline zero-shot e few-shot NON sono qui: non dipendono dal fine-tuning,
quindi conviene farle una volta sola nel notebook di valutazione.

Prima cella: taratura del discriminatore. Se il controllo umano-vs-umano si
allontana da 0,50 il classificatore sta leggendo artefatti e le metriche
avversariali non valgono — guardala prima dei risultati.

In [28]:
# --- Taratura del discriminatore ita/nap ------------------------------------------
import importlib, evaluate_task as E
importlib.reload(E)
clf, info = E.costruisci_discriminatore_dialetto(SPLIT)
print("discriminatore ita-vs-nap:", info)
refs = [r["target"] for r in E.load_split(SPLIT, TASK, "test")]
print(f"P(nap) sui riferimenti umani {TASK}: %.3f" % clf.predict_proba(
    [E.norm(x) for x in refs])[:, 1].mean())
ctrl = E.discriminazione_avversariale(refs[:len(refs)//2], refs[len(refs)//2:])
print("controllo umano-vs-umano:", ctrl["controllo_umano_vs_umano"],
      "| solo-lunghezza:", ctrl["controllo_solo_lunghezza"])

discriminatore ita-vs-nap: {'n_frasi_train': 1859, 'accuracy': 0.979, 'auc': 1.0}
P(nap) sui riferimenti umani T1: 0.880
controllo umano-vs-umano: 0.485 | solo-lunghezza: 0.566


In [29]:
# --- Adapter di questo notebook ---------------------------------------------------
import glob, os, common

# L'adapter va scelto per MODELLO oltre che per task: un adapter LoRA e' legato
# alla taglia esatta del base su cui e' stato addestrato. Filtrare solo per task
# (__T1) puo' pescare l'adapter dello smoke test (Passo 18, --model gemma-tiny,
# hidden 1152) e passarlo a un base diverso -> size mismatch in evaluate_task.py.
_slug = common.slug(common.resolve_model(MODEL))          # es. 'llama-2-7b-chat-hf'
RUNS = {os.path.basename(os.path.dirname(p)).split("__")[-1]: p
        for p in glob.glob(f"/kaggle/working/runs/{_slug}__*/adapter_final")}
print(f"Modello: {_slug} | adapter di questo modello in working:", RUNS)
assert TASK in RUNS, (
    f"adapter {_slug}__{TASK} non trovato. Hai eseguito la cella di training "
    f"(Passo 24) con MODEL={MODEL!r}? Lo smoke test (Passo 18) NON conta: gira "
    "su gemma-tiny e produce un adapter di un altro modello. Dopo un riavvio "
    "della sessione /kaggle/working e' vuoto: riallega l'output come Dataset.")
ADAPTER = RUNS[TASK]
print("Valutero':", ADAPTER)

Modello: gemma-3-4b-it | adapter di questo modello in working: {'A2': '/kaggle/working/runs/gemma-3-4b-it__A2/adapter_final', 'T1': '/kaggle/working/runs/gemma-3-4b-it__T1/adapter_final'}
Valutero': /kaggle/working/runs/gemma-3-4b-it__T1/adapter_final


In [30]:
!python evaluate_task.py --model {MODEL} --task {TASK} --split-dir {SPLIT} \
    --adapter {ADAPTER} --contrastive-limit 100

GPU cc 7.5 -> float32  (Gemma senza bf16 nativo: in fp16 il forward va in NaN e le uscite escono vuote, quindi fp32)
Loading weights: 100%|█| 883/883 [00:07<00:00, 117.41it/s, Materializing param=m
  controllo numerico: logit finiti, primo token = 236745 't'
=== gemma-3-4b-it T1 ft | 267 esempi di test ===
Fine turno:
  chiusura del target renderizzato: id=106 repr='<end_of_turn>'
  tok.eos_token = '<eos>' (id 1)
  fine turno da usare = [1, 106, 212] (['<eos>', '<end_of_turn>', '</s>'])
  ! [212] NON erano in generation_config: senza il fix generate() arrivava sempre a max_new=64
Decoding: greedy

Frazione al tetto: 0.000   ok
Contrastiva (nessuna generazione, solo forward)...
{
  "run": "gemma-3-4b-it__T1__ft",
  "decoding": "greedy",
  "repo_id": "google/gemma-3-4b-it",
  "task": "T1",
  "tag": "ft",
  "architettura": "CausalLM",
  "n_test": 267,
  "F_riferimento": {
    "chrf++": 74.78,
    "bleu": 54.16
  },
  "F_bootstrap_chrf": {
    "ci95_basso": 72.52,
    "ci95_alto": 77.03
  

## Passo 27 — riepilogo dei run

**Cosa fare:** esegui a training finito e leggi la colonna `best`/`step`.


In [31]:
import json, glob
righe = []
for p in sorted(glob.glob("/kaggle/working/runs/*/summary.json")):
    s = json.load(open(p))
    sel, ds, hp = s["selezione_checkpoint"], s["dataset"], s["hyperparams"]
    righe.append(dict(task=s["task"]["layout"], modello=s["repo_id"].split("/")[-1],
                      arch=s.get("architettura_caricata"),
                      train=ds["train"], metrica=sel["metrica"], best=sel["best_value"],
                      step=f'{sel["best_step"]}/{hp["max_steps"]}',
                      minuti=round(s["training_seconds"]/60, 1),
                      troncati=ds["troncati_train"],
                      mismatch=ds.get("prefix_mismatch_train"),
                      ok=s["completed"]))
try:
    import pandas as pd
    display(pd.DataFrame(righe))
except ImportError:
    for r in righe:
        print(r)

# best_step vicino a max_steps => il modello stava ancora migliorando: alza --epochs.
# best_step molto presto => overfitting precoce: abbassa --lora-r o alza il dropout.

,task,modello,arch,train,metrica,best,step,minuti,troncati,mismatch,ok
0,T1,gemma-3-1b-it,CausalLM,48,chrf,NaN,None/3,0.3,0,0,True
1,A2,gemma-3-4b-it,CausalLM,1742,chrf,66.446510,324/327,140.3,0,0,True
2,T1,gemma-3-4b-it,CausalLM,1134,chrf,77.777747,245/284,105.9,0,0,True


## Passo 28 — `metriche_finali.py`: CSV delle generazioni + BERTScore

**Cosa fare:** esegui la cella che scrive il file, poi le due sotto.

Non rigenera niente: `evaluate_task.py` salva gia' le generazioni in
`{run}.preds.jsonl`. Questo script le raccoglie tutte (tre task, e tutti i
modelli se i file sono nella stessa cartella) e produce **un CSV solo** piu' il
riepilogo delle metriche.

### Perche' BERTScore va tarato

Nessun encoder pubblico conosce il napoletano: mBERT, XLM-R e i modelli italiani
lo trattano come italiano rumoroso. Quindi BERTScore misura similarita'
semantica *nello spazio dell'italiano*, ed e' quasi cieco alla correttezza
dialettale — anzi, **premia la copia dell'italiano**, che e' proprio il
fallimento che stai misurando. Da solo non e' riportabile.

Lo script calcola sempre due tarature accanto al sistema:

| serie | cos'e' | come si legge |
|---|---|---|
| `pavimento` | riferimento di un ALTRO item | BERTScore fra due frasi napoletane senza relazione: il vero zero della scala |
| `copia_italiano` | solo T1, la frase italiana come ipotesi | se il sistema non stacca nettamente questo valore, BERTScore non sta premiando la traduzione |

L'intervallo utile e' **F1 meno pavimento**, non F1 in assoluto. Nel collaudo su
dati sintetici il pavimento stava a 0,617 e la copia dell'italiano a 0,749:
significa che un sistema a 0,78 sarebbe appena sopra la copia, non "buono al 78%".

In [32]:
%%writefile /kaggle/working/training/metriche_finali.py
#!/usr/bin/env python3
"""
metriche_finali.py — dai .preds.jsonl di evaluate_task.py produce
  1. UN CSV con tutte le generazioni di test (T1, T2, T3, tutti i modelli)
  2. BERTScore + chrF++ + BLEU + metriche lessicali, con le baseline di taratura

Non rigenera niente: `evaluate_task.py` scrive gia' {run}.preds.jsonl con
{id, prompt, target, hyp}. La generazione e' la parte cara e non va rifatta se
una metrica fallisce.

BERTScore sul napoletano: leggere con attenzione
------------------------------------------------
Nessun encoder pubblico conosce il napoletano. mBERT, XLM-R e i modelli italiani
lo trattano come italiano rumoroso, quindi BERTScore qui misura la similarita'
semantica *vista attraverso lo spazio dell'italiano*. Due conseguenze concrete:

  * e' quasi cieco alla correttezza dialettale: un output in italiano e un
    riferimento in napoletano finiscono vicini;
  * **premia la copia dell'italiano**, che e' esattamente il fallimento che stai
    cercando di misurare.

Per questo non va mai riportato da solo. Lo script calcola sempre due tarature:

  pavimento    riferimento di UN ALTRO item (napoletano non correlato). E' il
               valore che BERTScore da' a due frasi della stessa varieta' che
               non c'entrano niente: sotto quel numero non si scende, e la
               distanza fra pavimento e sistema e' l'intervallo utile reale.
  copia-it     solo T1: la frase italiana di partenza come ipotesi. Se il tuo
               modello non stacca nettamente questo valore, BERTScore non sta
               premiando la traduzione.

Il tetto (riferimento contro se stesso) e' 1.0 per costruzione e non si stampa.

Uso
---
    pip install bert-score --quiet
    python metriche_finali.py --preds-dir /kaggle/working/eval \\
        --lessico /kaggle/working/lessico_train.json \\
        --csv /kaggle/working/predizioni_test.csv \\
        --out /kaggle/working/metriche_finali.json

    # senza BERTScore (piu' veloce, nessun download)
    python metriche_finali.py --preds-dir ... --no-bertscore
"""

import argparse
import json
import random
import re
import sys
from pathlib import Path

TOK = re.compile(r"[a-zàèéìòóùâêîôû'\u2019\-]+")
ISTRUZIONE = {"T1": "Traduci in napoletano: ",
              "T2": "Continua il turno in napoletano: "}


def norm(s):
    return str(s).replace("\u2019", "'")


def tokenizza(s):
    return TOK.findall(norm(s).lower())


def parse_run(nome):
    """'minerva-7b-instruct-v1.0__T1__ft__nucleus' -> componenti."""
    p = nome.split("__")
    return {"modello": p[0],
            "task": p[1] if len(p) > 1 else "?",
            "tag": p[2] if len(p) > 2 else "?",
            "decoding": p[3] if len(p) > 3 else "greedy"}


def parte_variabile(prompt, task):
    """La frase italiana (T1) o il prefisso napoletano (T2) dentro il prompt."""
    m = ISTRUZIONE.get(task)
    if not m or m not in prompt:
        return ""
    return prompt.rpartition(m)[2].split("\n")[0].strip()


def blocco_contesto(prompt):
    """I turni di contesto, se il prompt ne ha. Utile da avere in colonna."""
    try:
        from contesto_metrica import estrai_blocco_contesto
    except ImportError:
        return ""
    t = estrai_blocco_contesto(prompt)
    return " | ".join(t[2]) if t else ""


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1],
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--preds-dir", default="/kaggle/working/eval")
    ap.add_argument("--csv", default="/kaggle/working/predizioni_test.csv")
    ap.add_argument("--out", default="/kaggle/working/metriche_finali.json")
    ap.add_argument("--lessico", default=None,
                    help="output di lessico.py: abilita le metriche lessicali")
    ap.add_argument("--bert-model", default="bert-base-multilingual-cased",
                    help="encoder per BERTScore. Alternative: xlm-roberta-large, "
                         "dbmdz/bert-base-italian-xxl-cased (tokenizza meglio le "
                         "forme apostrofate, ma nessuno dei tre conosce il napoletano)")
    ap.add_argument("--no-bertscore", action="store_true")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()

    import pandas as pd
    import sacrebleu

    file_preds = sorted(Path(a.preds_dir).glob("*.preds.jsonl"))
    if not file_preds:
        sys.exit(f"Nessun *.preds.jsonl in {a.preds_dir}. Esegui prima "
                 f"evaluate_task.py per i tre task.")
    print(f"Trovati {len(file_preds)} file di predizioni:")

    righe = []
    for f in file_preds:
        info = parse_run(f.name.replace(".preds.jsonl", ""))
        n = 0
        for i, linea in enumerate(f.read_text(encoding="utf-8").splitlines()):
            if not linea.strip():
                continue
            d = json.loads(linea)
            righe.append({**info, "idx": i, "id": d.get("id"),
                          "prompt": d["prompt"],
                          "contesto": blocco_contesto(d["prompt"]),
                          "ingresso": parte_variabile(d["prompt"], info["task"]),
                          "riferimento": norm(d["target"]),
                          "generato": norm(d["hyp"])})
            n += 1
        print(f"  {f.name:52s} {n:5d} item")

    df = pd.DataFrame(righe)

    # --- metriche per item ---------------------------------------------------
    df["chrf_item"] = [
        round(sacrebleu.sentence_chrf(h, [r], word_order=2).score, 2)
        for h, r in zip(df["generato"], df["riferimento"])]
    df["parole_rif"] = df["riferimento"].map(lambda s: len(s.split()))
    df["parole_gen"] = df["generato"].map(lambda s: len(s.split()))

    lex = None
    vocab_nap = None
    if a.lessico:
        from pesi_lessicali import carica_lessico
        lex = carica_lessico(a.lessico)
        dial = lex["dialettali"]
        df["dial_rif"] = df["riferimento"].map(lambda s: sum(w in dial for w in tokenizza(s)))
        df["dial_gen"] = df["generato"].map(lambda s: sum(w in dial for w in tokenizza(s)))

        # --- forme inedite ---------------------------------------------------
        # Parole prodotte che non compaiono da nessuna parte nel napoletano di
        # train. Cattura le forme inventate per analogia morfologica
        # ("puparuolo", "cusinajo", "pruove ccusarelle"): sono fluenti, superano
        # il discriminatore ita/nap, e nessuna delle altre metriche le vede.
        # Va TARATA: il vocabolario di train ha ~1.500 tipi, quindi anche i
        # riferimenti umani di test contengono forme fuori vocabolario. Il loro
        # tasso e' il pavimento naturale, e si calcola sulla stessa colonna.
        grezzo = json.loads(Path(a.lessico).read_text(encoding="utf-8"))
        vocab_nap = set(grezzo.get("vocabolario_nap", {}))
        if not vocab_nap:
            print("! il lessico non contiene 'vocabolario_nap': rigenera con la "
                  "versione aggiornata di lessico.py per avere le forme inedite")
        else:
            def inedite(testo):
                t = tokenizza(testo)
                return sum(1 for w in t if w not in vocab_nap), len(t)
            for col, dest in (("generato", "inedite_gen"), ("riferimento", "inedite_rif")):
                coppie = df[col].map(inedite)
                df[dest] = [c[0] for c in coppie]
                df[dest + "_su"] = [c[1] for c in coppie]

    # --- BERTScore -----------------------------------------------------------
    # Un solo passaggio su TUTTE le ipotesi e su tutte le serie di taratura:
    # caricare l'encoder una volta sola e' l'unica parte costosa.
    if not a.no_bertscore:
        try:
            from bert_score import score as bert_score
        except ImportError:
            sys.exit("bert-score non installato: pip install bert-score --quiet\n"
                     "(oppure usa --no-bertscore)")
        rnd = random.Random(a.seed)

        cand, rif, etichette = [], [], []
        for (mod, task, tag, dec), g in df.groupby(["modello", "task", "tag", "decoding"]):
            r = g["riferimento"].tolist()
            chiave = (mod, task, tag, dec)
            # sistema
            cand += g["generato"].tolist(); rif += r
            etichette += [(chiave, "sistema")] * len(r)
            # pavimento: riferimento di un altro item, stessa varieta', zero relazione
            perm = list(range(len(r)))
            rnd.shuffle(perm)
            perm = [p if p != k else (p + 1) % len(r) for k, p in enumerate(perm)]
            cand += [r[p] for p in perm]; rif += r
            etichette += [(chiave, "pavimento")] * len(r)
            # copia-italiano: solo T1, dove l'ingresso e' la frase italiana
            if task == "T1" and g["ingresso"].str.len().gt(0).all():
                cand += g["ingresso"].tolist(); rif += r
                etichette += [(chiave, "copia_italiano")] * len(r)

            print(f"\nBERTScore con {a.bert_model} su {len(cand)} coppie "f"(sistema + tarature)...")
            # generazioni/riferimenti vuoti -> placeholder: su stringa vuota bert_score
            # chiama build_inputs_with_special_tokens, non piu' esposto dal BertTokenizer
            # nelle transformers recenti (AttributeError). Same len/ordine = allineamento ok.
            cand = [c if c and c.strip() else "." for c in cand]
            rif  = [r if r and r.strip() else "." for r in rif]
            P, R, F = bert_score(cand, rif, model_type=a.bert_model,
                                 batch_size=a.batch, verbose=False)
        bs = {}
        for (chiave, serie), p, r_, f_ in zip(etichette, P.tolist(), R.tolist(), F.tolist()):
            bs.setdefault((chiave, serie), []).append((p, r_, f_))
        # il valore per item del solo sistema va anche nel CSV
        per_item = {}
        for (chiave, serie), vals in bs.items():
            if serie == "sistema":
                per_item[chiave] = [v[2] for v in vals]
        colonna = []
        for (mod, task, tag, dec), g in df.groupby(["modello", "task", "tag", "decoding"]):
            colonna += list(zip(g.index, per_item[(mod, task, tag, dec)]))
        for i, v in colonna:
            df.loc[i, "bertscore_f1"] = round(v, 4)

    # --- CSV -----------------------------------------------------------------
    Path(a.csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(a.csv, index=False)
    print(f"\nScritto {a.csv}: {len(df)} righe, colonne {list(df.columns)}")

    # --- riepilogo per run ---------------------------------------------------
    from pesi_lessicali import valuta
    riepilogo = {}
    print(f"\n{'run':40s} {'n':>5s} {'chrF++':>7s} {'BS-F1':>7s} {'BS-pav':>7s} "
          f"{'rec_d':>6s} {'ined':>6s} {'(um)':>6s} {'len':>5s}")
    for (mod, task, tag, dec), g in df.groupby(["modello", "task", "tag", "decoding"]):
        chiave = (mod, task, tag, dec)
        h, r = g["generato"].tolist(), g["riferimento"].tolist()
        voce = {
            "modello": mod, "task": task, "tag": tag, "decoding": dec, "n": len(g),
            "chrf++": round(sacrebleu.corpus_chrf(h, [r], word_order=2).score, 2),
            "bleu": round(sacrebleu.corpus_bleu(h, [r]).score, 2),
            "rapporto_lunghezza": round(g["parole_gen"].sum() / max(1, g["parole_rif"].sum()), 3),
        }
        if not a.no_bertscore:
            for serie in ("sistema", "pavimento", "copia_italiano"):
                v = bs.get((chiave, serie))
                if not v:
                    continue
                n = len(v)
                voce[f"bertscore_{serie}"] = {
                    "P": round(sum(x[0] for x in v) / n, 4),
                    "R": round(sum(x[1] for x in v) / n, 4),
                    "F1": round(sum(x[2] for x in v) / n, 4)}
            voce["_lettura_bertscore"] = (
                "confronta F1 con bertscore_pavimento (napoletano non correlato): "
                "quello e' il vero zero della scala. Su T1 confronta anche con "
                "copia_italiano: l'encoder non conosce il napoletano e premia la copia.")
        if lex:
            src = g["ingresso"].tolist() if task == "T1" else r
            voce["lessicali"] = {k: v for k, v in valuta(src, r, h, lex).items()
                                 if not k.startswith("_")}
        if vocab_nap:
            tg, sg = g["inedite_gen"].sum(), g["inedite_gen_su"].sum()
            tr_, sr = g["inedite_rif"].sum(), g["inedite_rif_su"].sum()
            voce["forme_inedite"] = {
                "tasso_generato": round(tg / max(1, sg), 4),
                "tasso_riferimenti_umani": round(tr_ / max(1, sr), 4),
                "_lettura": "il tasso dei riferimenti umani e' il pavimento "
                            "naturale (il vocabolario di train non copre tutto il "
                            "napoletano). Conta solo l'ECCESSO del generato su "
                            "quel valore: e' la quota di forme inventate."}
        riepilogo[f"{mod}__{task}__{tag}__{dec}"] = voce

        bsf = voce.get("bertscore_sistema", {}).get("F1", float("nan"))
        bsp = voce.get("bertscore_pavimento", {}).get("F1", float("nan"))
        recd = voce.get("lessicali", {}).get("recall_dialettale", float("nan"))
        ing = voce.get("forme_inedite", {})
        print(f"{mod[:14]+'__'+task+'__'+tag:40s} {len(g):5d} {voce['chrf++']:7.2f} "
              f"{bsf:7.4f} {bsp:7.4f} {recd:6.3f} "
              f"{ing.get('tasso_generato', float('nan')):6.3f} "
              f"{ing.get('tasso_riferimenti_umani', float('nan')):6.3f} "
              f"{voce['rapporto_lunghezza']:5.2f}")

    Path(a.out).write_text(json.dumps(riepilogo, ensure_ascii=False, indent=1),
                           encoding="utf-8")
    print(f"\nScritto {a.out}")
    if not a.no_bertscore:
        print("\nBS-pav e' BERTScore fra riferimenti NON correlati: e' il pavimento "
              "della scala.\nLa distanza BS-F1 meno BS-pav e' l'intervallo utile, "
              "non BS-F1 in assoluto.")
    if vocab_nap:
        print("ined = quota di parole generate fuori dal vocabolario napoletano di "
              "train; (um) e' lo\nstesso tasso sui riferimenti umani di test. Conta "
              "l'eccesso di ined su (um), non ined.")


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/metriche_finali.py


### Passo 28a — genera su test (se non l'hai gia' fatto)

Le tre celle di valutazione piu' sopra scrivono gia' i `.preds.jsonl`. Riesegui
queste solo se le hai saltate o se hai rifatto gli adapter.

In [33]:
!pip install bert-score --quiet
!python evaluate_task.py --model {MODEL} --task {TASK} --split-dir {SPLIT} \
    --adapter {ADAPTER} --split test --out /kaggle/working/eval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.8 MB/s eta 0:00:00
GPU cc 7.5 -> float32  (Gemma senza bf16 nativo: in fp16 il forward va in NaN e le uscite escono vuote, quindi fp32)
Loading weights: 100%|█| 883/883 [00:07<00:00, 110.44it/s, Materializing param=m
  controllo numerico: logit finiti, primo token = 236745 't'
=== gemma-3-4b-it T1 ft | 267 esempi di test ===
Fine turno:
  chiusura del target renderizzato: id=106 repr='<end_of_turn>'
  tok.eos_token = '<eos>' (id 1)
  fine turno da usare = [1, 106, 212] (['<eos>', '<end_of_turn>', '</s>'])
  ! [212] NON erano in generation_config: senza il fix generate() arrivava sempre a max_new=64
Decoding: greedy

Frazione al tetto: 0.000   ok
Contrastiva (nessuna generazione, solo forward)...
{
  "run": "gemma-3-4b-it__T1__ft",
  "decoding": "greedy",
  "repo_id": "google/gemma-3-4b-it",
  "task": "T1",
  "tag": "ft",
  "architettura": "CausalLM",
  "n_test": 267,
  "F_riferimento": {
    "chrf++": 74.78,
    "bleu": 54.16
 

### Passo 28b — CSV unico + metriche

Il CSV ha una riga per generazione, con `modello`, `task`, `tag`, `decoding`,
`prompt`, `contesto`, `ingresso`, `riferimento`, `generato`, il chrF di quella
riga, il BERTScore di quella riga e il conteggio di parole dialettali in
riferimento e generato. Concatenando i CSV dei tre notebook hai la tabella
completa dell'esperimento.

In [34]:
!python metriche_finali.py --preds-dir /kaggle/working/eval \
    --lessico {LEX_JSON} \
    --csv /kaggle/working/predizioni_test.csv \
    --out /kaggle/working/metriche_finali.json

import pandas as pd
pred = pd.read_csv("/kaggle/working/predizioni_test.csv")
print(f"\n{len(pred)} generazioni | task: {sorted(pred['task'].unique())}")
display(pred.groupby("task")[["chrf_item", "bertscore_f1", "dial_gen", "dial_rif"]]
        .mean().round(3))
display(pred[["task", "ingresso", "riferimento", "generato", "chrf_item"]].head(12))

Trovati 1 file di predizioni:
  gemma-3-4b-it__T1__ft.preds.jsonl                      267 item

BERTScore con bert-base-multilingual-cased su 801 coppie (sistema + tarature)...
config.json: 100%|█████████████████████████████| 625/625 [00:00<00:00, 3.53MB/s]
tokenizer_config.json: 100%|██████████████████| 49.0/49.0 [00:00<00:00, 365kB/s]
vocab.txt: 996kB [00:00, 21.2MB/s]
tokenizer.json: 1.96MB [00:00, 30.9MB/s]
model.safetensors: 100%|██████████████████████| 714M/714M [00:03<00:00, 209MB/s]
Loading weights: 100%|█| 199/199 [00:00<00:00, 1390.78it/s, Materializing param=
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED

,chrf_item,bertscore_f1,dial_gen,dial_rif
task,,,,
T1,76.907,0.922,3.052,3.03


,task,ingresso,riferimento,generato,chrf_item
0,T1,tutto a posto grazie sì sì,"tutto a posto, grazie, sì sì","tutto a posto, grazie, sì sì",100.00
1,T1,tutto bene grazie sì sì tutto buonissimo,"tutto bbuono, grazie, sì sì, tutto bbuonissimo","tutto bbuono, grazie, sì sì, tutto bbuonissimo",100.00
2,T1,l'unica cosa il vino non siamo riuscite a fini...,"ll'unica cosa, 'o vino nun simmo riuscite a 'o...","ll'unica cosa, 'o vino nun simmo riuscite a 'o...",83.84
3,T1,così al massimo la portiamo a casa,accussì a 'o massimo 'a purtammo â casa,"accussì, 'o massimo 'a purtammo â casa",86.79
4,T1,ce lo riportiamo,c' 'o repurtammo,ce ll'apportammo,26.62
5,T1,perfetto non è un problema,"perfetto, nun è nu problema","perfetto, nun è nu problema",100.00
6,T1,comunque era buonissimo questo era buonissimo,"comunque era bbonissimo, chisto era bbonissimo","comunque era bbonissimo, chisto era bbonissimo",100.00
7,T1,tutto il pollo fantastico,"tutto 'o puollo, fantastico",assaje 'o puollo fantastico,58.58
8,T1,per me sì che che mangio mh cibo indiano sì,"pe' me sì, ca, ca mangio mh cibo indiano, sì","pe' me sì, ca ca mangio, mh, cchiù indiano sì",61.31
9,T1,no io no,"no, i' no","no, i' no",100.00


## Passo 30 — il fine turno, anche qui

`generate()` non aveva il token di fine turno del chat template fra gli
`eos_token_id`: arrivava sempre a `max_new_tokens`. Su T3 questo ha falsato la
discriminazione avversariale (il 97% del potere discriminante veniva dalla
lunghezza), ma la patch e' in `evaluate_task.py` e riguarda **tutti** i task,
quindi la diagnostica va fatta anche su T1.

Il best-of-n invece **non** si fa su T1: la traduzione ha un modo, il
campionamento peggiora e greedy resta la scelta giusta. Di `rerank_t3.py` qui si
usa solo `--solo-diagnostica`, che non genera niente.

Tre righe da leggere nell'output:
1. `ultimo token del target renderizzato: id=... repr=...` — e' cio' che il
   modello deve imparare a produrre per fermarsi;
2. gli `eos_token_id` effettivamente passati a `generate()`;
3. la frazione di generazioni che arrivano al tetto di `max_new_tokens`.

### Passo 30a — `rerank_t3.py`: best-of-n con selezione a posteriori

**Cosa fare:** esegui (scrive il file, non genera niente).

Campiona n candidati per item e ne scegli uno con criteri **reference-free**
(nessuno guarda il target). Non riaddestra niente e non aggiunge conoscenza al
modello: **riduce la varianza del campionamento**, che è la parte
dell'incoerenza percepita che si può togliere senza più dati.

Tre criteri:

| modo | cosa massimizza | iperparametri |
|---|---|---|
| `mbr` | chrF++ medio del candidato contro gli altri candidati dello stesso item | nessuno |
| `composito` | `w_ctx·d_ctx + w_nap·d_nap + w_len·d_len` | 3, da tarare su dev |
| `ibrido` | somma dei due, standardizzati per item | 3 |

- `d_ctx` = `logP(cand | contesto reale) − logP(cand | senza contesto)`
  normalizzata per token. È lo stesso delta di `contesto_metrica.py`, riusato
  come **punteggio** invece che come metrica, con `rimuovi_contesto` di
  `decodifica_contestuale.py`. La contrastiva a 0,33 contro 0,20 casuale dice che
  il condizionamento sul contesto esiste ma è debole: è il regime in cui
  selezionare su questo delta ha effetto.
- `d_nap` = `−|P_nap(cand) − P_nap(riferimenti umani)|`. Il bersaglio è **0,877,
  non 1,0**: il run misurato sta a 0,944, cioè è *iper*-dialettale. Massimizzare
  la dialettalità peggiora.
- `d_len` = log-densità della lunghezza sotto la distribuzione empirica dei
  target di **train** (mai del test: usare il test come prior sarebbe tararsi
  sul set di valutazione).

Prima dei punteggi, **filtri hard** su patologie e non su preferenze stilistiche:
candidato vuoto, un token ripetuto ≥3 volte, copia normalizzata di un turno di
contesto, lunghezza oltre il doppio del 99° percentile umano.

MBR è citabile (Eikema & Aziz 2020; Freitag et al. 2022) e non ha iperparametri,
ma **ha un bias verso il candidato più lungo**: chrF premia il superstring, che
ha recall alto contro tutti i substring. Su un task dove il difetto è già
l'eccesso di lunghezza questo lavora contro di te, ed è la ragione per cui
`ibrido` esiste. Se in tabella `mbr` ha `lung_media` più alta di `1campione`, è
questo, e va scritto nei risultati.


In [35]:
%%writefile /kaggle/working/training/rerank_t3.py
#!/usr/bin/env python3
"""
rerank_t3.py — best-of-n con selezione a posteriori su T3 (e T2), senza
riaddestrare niente.

Perche' esiste
--------------
Le metriche del run `minerva-7b-instruct-v1.0__T3__ft__nucleus` dicono tre cose:

  lunghezza_media 17.57 contro 6.99 umano, ks 0.589 p=0
  controllo_solo_lunghezza 0.776 su accuracy_umano_vs_macchina 0.80
  rep_2 0.151 contro 0.0365 umano, loop_massimo 63

La seconda riga e' la piu' informativa: il 97% del potere discriminante del
classificatore avversariale viene dalla LUNGHEZZA. Il sistema non e'
riconoscibile perche' sbaglia il napoletano, e' riconoscibile perche' parla
troppo e viene troncato al tetto di max_new_tokens senza mai emettere fine di
turno. Una frase tagliata a meta' si legge come "sconclusionata" anche quando la
deriva semantica non c'e'.

Cosa fa questo script
---------------------
1. DIAGNOSTICA (sempre, prima di tutto): quali id di fine turno esistono, quali
   arrivano a generate(), e la frazione di generazioni che raggiunge il tetto.
   Se quella frazione e' alta, tutto il resto viene dopo: stai misurando un
   troncamento, non la pragmatica.

2. Campiona n candidati per item con un gen_kw CORRETTO (eos di fine turno,
   repetition_penalty, no_repeat_ngram, typical/nucleus) e ne sceglie uno.

3. Tre criteri di selezione, tutti reference-free (nessuno guarda il target):

   mbr        utilita' = chrF++ medio del candidato contro gli altri candidati
              dello stesso item. Nessun iperparametro. Elimina la coda di
              campionamenti anomali per costruzione: un candidato sconclusionato
              non ha consenso. (Eikema & Aziz 2020; Freitag et al. 2022)

   composito  combinazione lineare di:
                d_ctx  = logP(cand | contesto reale) - logP(cand | senza contesto)
                         normalizzata per token, con `rimuovi_contesto` di
                         decodifica_contestuale.py: e' lo stesso delta di
                         contesto_metrica.py, usato qui come punteggio invece che
                         come metrica.
                d_nap  = -|P_nap(cand) - P_nap(riferimenti umani)|
                         il bersaglio e' 0.877, NON 1.0: il run misurato sta a
                         0.944, cioe' IPER-dialettale. Massimizzare P_nap
                         peggiora.
                d_len  = log-densita' della lunghezza sotto la distribuzione
                         empirica dei target di TRAIN (mai del test).
              I pesi si tarano su dev con --taratura.

   ibrido     mbr + composito, standardizzati per item.

4. FILTRI HARD, applicati prima dei punteggi: candidato vuoto, loop di un token
   ripetuto >= 3 volte, copia normalizzata di un turno di contesto, lunghezza
   oltre il 99mo percentile umano. Un candidato che li viola viene scartato; se
   li violano tutti si tiene il meno peggio e lo si conta.

5. Salva {run}.preds.jsonl + {run}.metrics.json nello STESSO formato di
   evaluate_task.py, cosi' metriche_finali.py li raccoglie affiancati e la
   tabella comparativa esce da sola. Salva anche il candidato 0 come arm
   "1 campione" per un confronto appaiato a parita' di seme e di gen_kw.

6. Metriche STRATIFICATE per lunghezza del riferimento. Nel corpus il 33.7% dei
   turni sta a <= 2 parole e i piu' frequenti sono riscontri (mh 95, si' 71,
   mhmh 58): meta' del test e' un sotto-task diverso, e aggregare i due annega
   il segnale sui turni contenutistici.

Uso su Kaggle
-------------
  # 1. diagnostica, 30 secondi, prima di tutto il resto
  !python rerank_t3.py --model minerva --task T3 --split-dir {SPLIT} \
      --adapter {RUNS['T3']} --solo-diagnostica

  # 2. taratura dei pesi del composito sul DEV
  !python rerank_t3.py --model minerva --task T3 --split-dir {SPLIT} \
      --adapter {RUNS['T3']} --split dev --taratura --n 8

  # 3. run finale su test, tre arm in un colpo
  !python rerank_t3.py --model minerva --task T3 --split-dir {SPLIT} \
      --adapter {RUNS['T3']} --split test --n 12 --modo tutti \
      --pesi 1.0,2.0,0.5 --out /kaggle/working/eval

Costo su T4, 192 item, n=12
---------------------------
  generazione   192 * 12 sequenze da <= 32 token          ~6-10 min
  d_ctx         2 forward per candidato, batch 8          ~4-6 min
  P_nap         CPU, sklearn                              secondi
Con --modo mbr il d_ctx non serve e resta solo la generazione.
"""

import argparse
import json
import statistics
from collections import Counter
from pathlib import Path

from common import (load_backbone, load_hf_token, load_split, render_prompt,
                    resolve_model, scegli_dtype, slug)
from evaluate_task import (MAX_NEW, baseline_non_banali, bootstrap_chrf,
                           costruisci_discriminatore_dialetto, degenerazione,
                           discriminazione_avversariale, distribuzionali,
                           frazione_al_tetto, id_fine_turno, loop_massimo, norm,
                           pulisci_generato, riferimento,
                           stratifica_per_lunghezza)
from contesto_metrica import _logp_per_token, estrai_blocco_contesto
from decodifica_contestuale import rimuovi_contesto


# --------------------------------------------------------------------------- #
# 1. diagnostica: il modello emette fine di turno, e generate() la ascolta?
# --------------------------------------------------------------------------- #

def diagnostica(model, tok, rows, max_new, verbose=True):
    """id_fine_turno() stampa gia' quali id servono; qui si aggiunge il confronto
    con quello che generate() userebbe da sola."""
    eot = id_fine_turno(tok, rows, verbose=verbose)
    gc_eos = getattr(model.generation_config, "eos_token_id", None)
    gc_set = set(gc_eos if isinstance(gc_eos, (list, tuple)) else
                 ([gc_eos] if gc_eos is not None else []))
    if verbose:
        print(f"  generation_config.eos_token_id = {gc_eos}")
        mancanti = [i for i in eot if i not in gc_set]
        if mancanti:
            print(f"  ! {mancanti} NON sono in generation_config: senza il fix "
                  f"generate() non si ferma e arriva sempre a max_new={max_new}")
        else:
            print("  ok: generation_config copre tutti gli id di fine turno")
    return eot


# --------------------------------------------------------------------------- #
# 2. generazione di n candidati
# --------------------------------------------------------------------------- #

def genera_candidati(model, tok, torch, rows, n, max_new, eot_ids,
                     decoding="typical", top_p=0.9, typical_p=0.9,
                     temperature=0.7, repetition_penalty=1.15,
                     no_repeat_ngram=3, batch_prompt=2, seed=42):
    """Ritorna una lista di liste: cand[i] = n candidati per rows[i].

    batch_prompt e' basso di proposito: con num_return_sequences=n il batch
    effettivo e' batch_prompt * n. Su T4 con un 7B in 4 bit, batch_prompt=2 e
    n=12 danno 24 sequenze in volo, che ci stanno; alzare batch_prompt fa OOM
    prima di far guadagnare tempo.
    """
    torch.manual_seed(seed)
    prev_side = tok.padding_side
    tok.padding_side = "left"                     # obbligatorio in generazione
    fuori = []
    # Gemma-3: generate() e' rotta su Gemma3ForConditionalGeneration in questa
    # transformers ("Tensors must have same number of dimensions"), e con
    # num_return_sequences il crash arriva prima ancora del padding. Si usa lo
    # stesso loop manuale di evaluate_task.py, una sequenza alla volta: n volte
    # piu' lento, ma e' l'unico modo di avere i candidati su questo modello.
    mt = (getattr(getattr(model, "config", None), "model_type", "") or "").lower()
    if "gemma3" in mt:
        from evaluate_task import _decode_gemma3_manuale
        if decoding == "typical":
            print("  ! il loop manuale per Gemma-3 non implementa typical "
                  f"sampling: si usa nucleus con top_p={top_p}. Dichiaralo nel "
                  "confronto, non e' lo stesso decoder degli altri modelli.")
        try:
            for i, r in enumerate(rows, 1):
                enc = tok(render_prompt(tok, r["prompt"]), return_tensors="pt",
                          add_special_tokens=False).to(model.device)
                cand = []
                for _ in range(n):
                    with torch.no_grad():
                        g = _decode_gemma3_manuale(
                            model, enc, "nucleus", max_new, 1, top_p,
                            temperature, eot_ids, repetition_penalty,
                            no_repeat_ngram, tok=tok)
                    cand.append(pulisci_generato(tok.decode(
                        g[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)))
                fuori.append(cand)
                print(f"  generazione {i}/{len(rows)}", end="\r")
        finally:
            tok.padding_side = prev_side
        print()
        return fuori
    gen_kw = dict(max_new_tokens=max_new, min_new_tokens=1,
                  pad_token_id=tok.pad_token_id, eos_token_id=eot_ids,
                  num_return_sequences=n, do_sample=True, top_k=0,
                  temperature=temperature,
                  repetition_penalty=repetition_penalty,
                  no_repeat_ngram_size=no_repeat_ngram)
    if decoding == "typical":
        gen_kw["typical_p"] = typical_p           # locally typical sampling
    else:
        gen_kw["top_p"] = top_p                   # nucleus, per confronto
    try:
        for i in range(0, len(rows), batch_prompt):
            b = rows[i:i + batch_prompt]
            testi = [render_prompt(tok, r["prompt"]) for r in b]
            enc = tok(testi, return_tensors="pt", padding=True,
                      add_special_tokens=False).to(model.device)
            with torch.no_grad():
                g = model.generate(**enc, **gen_kw)
            larghezza = enc["input_ids"].shape[1]
            for j in range(len(b)):
                blocco = g[j * n:(j + 1) * n]
                fuori.append([pulisci_generato(tok.decode(s[larghezza:],
                                                 skip_special_tokens=True))
                              for s in blocco])
            print(f"  generazione {min(i + batch_prompt, len(rows))}/{len(rows)}",
                  end="\r")
    finally:
        tok.padding_side = prev_side
    print()
    return fuori


# --------------------------------------------------------------------------- #
# 3. filtri hard
# --------------------------------------------------------------------------- #

def turni_di_contesto(prompt):
    trovato = estrai_blocco_contesto(prompt)
    if trovato is None:
        return []
    return [norm(r) for r in trovato[2] if norm(r)]


def ammissibile(cand, ctx_norm, len_max):
    """False = da scartare. Sono patologie, non preferenze stilistiche."""
    if not cand.strip():
        return False, "vuoto"
    if loop_massimo(cand) >= 3:
        return False, "loop"
    c = norm(cand)
    if not c:
        return False, "vuoto"
    # Copia del contesto: il pattern degenere e' il turno di contesto ripetuto
    # verbatim. La sottostringa si accetta solo da 4 parole in su, altrimenti si
    # scartano echi legittimi ("nun saccio" dopo "B: nun saccio" e' una replica
    # plausibile, non degenerazione). Controlla `scarti_per_motivo` nel summary:
    # se copia_contesto e' sopra ~15% dei candidati il filtro sta stringendo
    # troppo per questo corpus.
    n_par = len(c.split())
    if any(c == t or (n_par >= 4 and c in t) for t in ctx_norm):
        return False, "copia_contesto"
    if len(cand.split()) > len_max:
        return False, "troppo_lungo"
    return True, None


# --------------------------------------------------------------------------- #
# 4. punteggi
# --------------------------------------------------------------------------- #

def utilita_mbr(cands, chrf_cache=None):
    """Per ogni candidato: chrF++ medio contro TUTTI gli altri candidati.

    E' MBR con utilita' chrF e distribuzione approssimata dai campioni. Non
    guarda il riferimento: e' la selezione del consenso, non della verita'.
    """
    import sacrebleu
    m = len(cands)
    if m == 1:
        return [0.0]
    fuori = []
    for a in range(m):
        s = 0.0
        for b in range(m):
            if a == b:
                continue
            s += sacrebleu.sentence_chrf(cands[a], [cands[b]], word_order=2).score
        fuori.append(s / (m - 1))
    return fuori


def costruisci_densita_lunghezza(split_dir, task, tetto=40):
    """log-densita' empirica della lunghezza in parole, dai target di TRAIN.

    Da train e non da test: usare la distribuzione del test come prior
    equivarrebbe a tarare il sistema sul set di valutazione.
    """
    tr = load_split(split_dir, task, "train")
    L = [min(len(r["target"].split()), tetto) for r in tr]
    c = Counter(L)
    tot = sum(c.values()) + 0.5 * (tetto + 1)
    import math
    logp = {k: math.log((c.get(k, 0) + 0.5) / tot) for k in range(tetto + 1)}
    minimo = min(logp.values())
    p99 = sorted(L)[int(0.99 * (len(L) - 1))]
    return logp, minimo, p99, {
        "media": round(statistics.mean(L), 2),
        "mediana": statistics.median(L),
        "p99": p99,
        "frazione_le2": round(sum(1 for x in L if x <= 2) / len(L), 3),
    }


def punteggio_lunghezza(cand, logp, minimo, tetto=40):
    return logp.get(min(len(cand.split()), tetto), minimo)


def punteggio_contesto(model, tok, torch, prompt, cands, max_len, batch=8):
    """d_ctx = logP(cand | prompt) - logP(cand | prompt senza contesto),
    normalizzata per token. Riusa _logp_per_token di contesto_metrica.py."""
    nudo = rimuovi_contesto(prompt)
    if nudo is None:
        return [0.0] * len(cands), False
    con = _logp_per_token(model, tok, torch, [prompt] * len(cands), cands,
                          render_prompt, max_len, batch=batch)
    senza = _logp_per_token(model, tok, torch, [nudo] * len(cands), cands,
                            render_prompt, max_len, batch=batch)
    return [a - b for a, b in zip(con, senza)], True


def standardizza(v):
    if len(v) < 2:
        return [0.0] * len(v)
    mu = statistics.mean(v)
    sd = statistics.pstdev(v)
    return [0.0] * len(v) if sd < 1e-9 else [(x - mu) / sd for x in v]


# --------------------------------------------------------------------------- #
# 5. selezione
# --------------------------------------------------------------------------- #

def seleziona(cands_per_item, rows, model, tok, torch, clf_dial, p_nap_bersaglio,
              logp_len, minimo_len, p99_len, modo, pesi, max_len, verbose=True):
    """Ritorna dict modo -> lista di ipotesi scelte, piu' la diagnostica."""
    w_ctx, w_nap, w_len = pesi
    scelte = {m: [] for m in ("1campione", "mbr", "composito", "ibrido")}
    conta_scarti = Counter()
    n_tutti_scartati = 0
    serve_ctx = modo in ("composito", "ibrido", "tutti")
    ctx_disponibile = 0

    for i, (r, cands) in enumerate(zip(rows, cands_per_item)):
        scelte["1campione"].append(cands[0])

        ctx_norm = turni_di_contesto(r["prompt"])
        ok, motivi = [], []
        for c in cands:
            a, m = ammissibile(c, ctx_norm, p99_len * 2)
            ok.append(a)
            if not a:
                conta_scarti[m] += 1
        vivi = [k for k in range(len(cands)) if ok[k]]
        if not vivi:                                  # nessuno passa: tieni tutti
            n_tutti_scartati += 1
            vivi = list(range(len(cands)))
        sub = [cands[k] for k in vivi]

        u_mbr = utilita_mbr(sub)
        scelte["mbr"].append(sub[max(range(len(sub)), key=lambda k: u_mbr[k])])

        if serve_ctx:
            d_ctx, ha_ctx = punteggio_contesto(model, tok, torch, r["prompt"],
                                               sub, max_len)
            ctx_disponibile += int(ha_ctx)
            p = clf_dial.predict_proba([norm(x) for x in sub])[:, 1] \
                if clf_dial is not None else [p_nap_bersaglio] * len(sub)
            d_nap = [-abs(float(x) - p_nap_bersaglio) for x in p]
            d_len = [punteggio_lunghezza(x, logp_len, minimo_len) for x in sub]
            comp = [w_ctx * a + w_nap * b + w_len * c
                    for a, b, c in zip(d_ctx, d_nap, d_len)]
            scelte["composito"].append(
                sub[max(range(len(sub)), key=lambda k: comp[k])])
            z = [a + b for a, b in zip(standardizza(u_mbr), standardizza(comp))]
            scelte["ibrido"].append(sub[max(range(len(z)), key=lambda k: z[k])])
        else:
            scelte["composito"].append(scelte["mbr"][-1])
            scelte["ibrido"].append(scelte["mbr"][-1])

        if verbose and (i + 1) % 20 == 0:
            print(f"  selezione {i + 1}/{len(rows)}", end="\r")
    if verbose:
        print()

    diag = {"scarti_per_motivo": dict(conta_scarti),
            "item_con_tutti_i_candidati_scartati": n_tutti_scartati,
            "item_con_contesto_estraibile": ctx_disponibile}
    return scelte, diag


# --------------------------------------------------------------------------- #
# 6. metriche, stratificate
# --------------------------------------------------------------------------- #

def blocco_metriche(task, rows, hyps, refs, clf_dial, info_dial, tok, max_new):
    res = {
        "F_riferimento": riferimento(hyps, refs),
        "F_bootstrap_chrf": bootstrap_chrf(hyps, refs),
        "F_baseline": baseline_non_banali(task, rows, refs),
        "C_distribuzionali": distribuzionali(hyps, refs),
        "C_degenerazione": degenerazione(hyps, refs),
        "A_avversariale": discriminazione_avversariale(hyps, refs),
        "G_troncamento": {
            "frazione_al_tetto": round(frazione_al_tetto(tok, hyps, max_new), 3),
            "_lettura": "frazione di output che raggiunge max_new senza emettere "
                        "fine di turno. Sopra ~0.05 la qualita' apparente e' "
                        "dominata dal troncamento, non dalla pragmatica.",
        },
    }
    if clf_dial is not None:
        pm = clf_dial.predict_proba([norm(x) for x in hyps])[:, 1]
        pr = clf_dial.predict_proba([norm(x) for x in refs])[:, 1]
        res["D_dialettalita"] = {
            "P_nap_generato": round(float(pm.mean()), 3),
            "P_nap_riferimenti_umani": round(float(pr.mean()), 3),
            "discriminatore": info_dial,
            "_lettura": "il bersaglio e' P_nap_riferimenti_umani, non 1.0: "
                        "sopra quel valore il sistema e' IPER-dialettale.",
        }
    return res


# --------------------------------------------------------------------------- #
# 7. taratura dei pesi su dev
# --------------------------------------------------------------------------- #

def obiettivo(hyps, refs, clf_dial):
    """Reference-free, da minimizzare. NON usa chrF: su un task uno-a-molti con
    riferimento singolo chrF++ 12.6 contro un floor di 9.79 non discrimina.

      |accuracy_avversariale - 0.50|   il sistema deve essere indistinguibile
    + |P_nap - P_nap_umano|            ne' sotto- ne' iper-dialettale
    + 0.5 * ks_lunghezza_stat          distribuzione delle lunghezze compatibile
    """
    a = discriminazione_avversariale(hyps, refs)
    if "errore" in a:
        return None, a
    d = distribuzionali(hyps, refs)
    ks = d.get("ks_lunghezza_stat", 1.0)
    if clf_dial is not None:
        pm = float(clf_dial.predict_proba([norm(x) for x in hyps])[:, 1].mean())
        pr = float(clf_dial.predict_proba([norm(x) for x in refs])[:, 1].mean())
        dn = abs(pm - pr)
    else:
        pm = pr = dn = 0.0
    val = abs(a["accuracy_umano_vs_macchina"] - 0.5) + dn + 0.5 * ks
    return val, {"adv": a["accuracy_umano_vs_macchina"],
                 "adv_solo_lunghezza": a["controllo_solo_lunghezza"],
                 "ks": ks, "p_nap": round(pm, 3), "p_nap_umano": round(pr, 3),
                 "obiettivo": round(val, 4)}


def taratura(cands_per_item, rows, refs, model, tok, torch, clf_dial,
             p_nap_bersaglio, logp_len, minimo_len, p99_len, max_len):
    griglia = [(wc, wn, wl)
               for wc in (0.0, 0.5, 1.0, 2.0)
               for wn in (0.0, 1.0, 2.0, 4.0)
               for wl in (0.0, 0.5, 1.0)]
    print(f"Taratura: {len(griglia)} combinazioni sul dev "
          f"(il d_ctx si calcola UNA volta e si riusa)")

    # d_ctx, d_nap, d_len non dipendono dai pesi: calcolali una volta sola
    cache = []
    for i, (r, cands) in enumerate(zip(rows, cands_per_item)):
        ctx_norm = turni_di_contesto(r["prompt"])
        vivi = [k for k, c in enumerate(cands)
                if ammissibile(c, ctx_norm, p99_len * 2)[0]] or list(range(len(cands)))
        sub = [cands[k] for k in vivi]
        d_ctx, _ = punteggio_contesto(model, tok, torch, r["prompt"], sub, max_len)
        p = clf_dial.predict_proba([norm(x) for x in sub])[:, 1] \
            if clf_dial is not None else [p_nap_bersaglio] * len(sub)
        cache.append((sub, d_ctx,
                      [-abs(float(x) - p_nap_bersaglio) for x in p],
                      [punteggio_lunghezza(x, logp_len, minimo_len) for x in sub]))
        print(f"  punteggi {i + 1}/{len(rows)}", end="\r")
    print()

    righe = []
    for (wc, wn, wl) in griglia:
        hyps = []
        for sub, dc, dn, dl in cache:
            s = [wc * a + wn * b + wl * c for a, b, c in zip(dc, dn, dl)]
            hyps.append(sub[max(range(len(s)), key=lambda k: s[k])])
        val, det = obiettivo(hyps, refs, clf_dial)
        if val is None:
            continue
        righe.append({"pesi": [wc, wn, wl], **det})
        print(f"  ({wc},{wn},{wl}) -> obiettivo {val:.4f}  adv {det['adv']:.3f} "
              f"ks {det['ks']:.3f} p_nap {det['p_nap']:.3f}")
    righe.sort(key=lambda x: x["obiettivo"])
    print("\nMigliori 5:")
    for r in righe[:5]:
        print("  ", json.dumps(r, ensure_ascii=False))
    print(f"\nUsa: --pesi {','.join(str(x) for x in righe[0]['pesi'])}")
    return righe


# --------------------------------------------------------------------------- #
# main
# --------------------------------------------------------------------------- #

def main():
    ap = argparse.ArgumentParser(
        description=__doc__.split("\n")[1],
        formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True)
    ap.add_argument("--task", default="T3", choices=["T2", "T3"],
                    help="T1 no: e' quasi deterministico, il best-of-n non serve")
    ap.add_argument("--split-dir", default="/kaggle/working/split")
    ap.add_argument("--split", default="test")
    ap.add_argument("--adapter", default=None)
    ap.add_argument("--out", default="/kaggle/working/eval")
    ap.add_argument("--n", type=int, default=12, help="candidati per item")
    ap.add_argument("--modo", default="tutti",
                    choices=["mbr", "composito", "ibrido", "tutti"])
    ap.add_argument("--pesi", default="1.0,2.0,0.5",
                    help="w_ctx,w_nap,w_len del composito (vedi --taratura)")
    ap.add_argument("--taratura", action="store_true",
                    help="griglia sui pesi; usalo su --split dev, non su test")
    ap.add_argument("--solo-diagnostica", action="store_true")
    ap.add_argument("--decoding", default="typical", choices=["typical", "nucleus"])
    ap.add_argument("--top-p", type=float, default=0.9)
    ap.add_argument("--typical-p", type=float, default=0.9)
    ap.add_argument("--temperature", type=float, default=0.7)
    ap.add_argument("--repetition-penalty", type=float, default=1.15)
    ap.add_argument("--no-repeat-ngram", type=int, default=3)
    ap.add_argument("--max-len", type=int, default=512)
    ap.add_argument("--max-new", type=int, default=None)
    ap.add_argument("--batch-prompt", type=int, default=2)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--limite", type=int, default=None,
                    help="usa solo i primi N item: per provare la pipeline")
    ap.add_argument("--hf-token", default=None)
    a = ap.parse_args()

    import torch
    from transformers import AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    repo_id = resolve_model(a.model)
    max_new = a.max_new or MAX_NEW[a.task]
    token = load_hf_token(a.hf_token)

    tok = AutoTokenizer.from_pretrained(repo_id, token=token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    dtype, _ = scegli_dtype(repo_id)      # fp32 su Gemma senza bf16
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_use_double_quant=True,
                               bnb_4bit_compute_dtype=dtype)
    model, arch = load_backbone(repo_id, dtype, quantization_config=quant,
                                device_map={"": 0}, token=token,
                                low_cpu_mem_usage=True,
                                attn_implementation="eager")
    if a.adapter:
        model = PeftModel.from_pretrained(model, a.adapter)
    model.eval()
    model.config.use_cache = True

    rows = load_split(a.split_dir, a.task, a.split)
    if a.limite:
        rows = rows[:a.limite]
    refs = [r["target"] for r in rows]
    tag = "ft" if a.adapter else "zs"
    base = f"{slug(repo_id)}__{a.task}__{tag}"
    print(f"=== {base} | {len(rows)} item di {a.split} | n={a.n} ===\n")

    print("[1] Diagnostica fine turno")
    eot = diagnostica(model, tok, rows, max_new)

    if a.solo_diagnostica:
        print("\n  campione di 16 item con il gen_kw corretto, per misurare il tetto")
        c = genera_candidati(model, tok, torch, rows[:16], 1, max_new, eot,
                             decoding=a.decoding, top_p=a.top_p,
                             typical_p=a.typical_p, temperature=a.temperature,
                             repetition_penalty=a.repetition_penalty,
                             no_repeat_ngram=a.no_repeat_ngram,
                             batch_prompt=a.batch_prompt, seed=a.seed)

        h = [x[0] for x in c]
        print(f"  frazione al tetto: {frazione_al_tetto(tok, h, max_new):.3f} "
              f"(deve stare sotto 0.05)")
        print(f"  lunghezza media: "
              f"{statistics.mean(len(x.split()) for x in h):.2f} parole "
              f"| riferimenti: "
              f"{statistics.mean(len(x.split()) for x in refs[:16]):.2f}")
        for r, x in list(zip(rows, h))[:8]:
            print(f"    RIF {r['target']!r}\n    GEN {x!r}")
        return

    print("\n[2] Prior di lunghezza dai target di TRAIN")
    logp_len, minimo_len, p99_len, stat_len = costruisci_densita_lunghezza(
        a.split_dir, a.task)
    print("  ", json.dumps(stat_len, ensure_ascii=False))

    print("\n[3] Discriminatore italiano-vs-napoletano (dal train di T1)")
    clf_dial, info_dial = costruisci_discriminatore_dialetto(a.split_dir)
    print("  ", json.dumps(info_dial, ensure_ascii=False))
    p_nap_bersaglio = float(
        clf_dial.predict_proba([norm(x) for x in refs])[:, 1].mean()
    ) if clf_dial is not None else 0.877
    print(f"  bersaglio P_nap (riferimenti umani di questo split): "
          f"{p_nap_bersaglio:.3f}  <- NON 1.0")

    print(f"\n[4] Generazione di {a.n} candidati per item "
          f"({a.decoding}, T={a.temperature}, rep={a.repetition_penalty})")
    cands = genera_candidati(model, tok, torch, rows, a.n, max_new, eot,
                             decoding=a.decoding, top_p=a.top_p,
                             typical_p=a.typical_p, temperature=a.temperature,
                             repetition_penalty=a.repetition_penalty,
                             no_repeat_ngram=a.no_repeat_ngram,
                             batch_prompt=a.batch_prompt, seed=a.seed)
    unici = statistics.mean(len(set(c)) for c in cands)
    print(f"  candidati distinti per item: {unici:.2f}/{a.n} "
          f"(se e' vicino a 1 la temperatura e' troppo bassa: non c'e' "
          f"niente da selezionare)")

    if a.taratura:
        print("\n[5] Taratura dei pesi")
        righe = taratura(cands, rows, refs, model, tok, torch, clf_dial,
                         p_nap_bersaglio, logp_len, minimo_len, p99_len,
                         a.max_len)
        Path(a.out).mkdir(parents=True, exist_ok=True)
        (Path(a.out) / f"{base}__taratura.json").write_text(
            json.dumps(righe, ensure_ascii=False, indent=2), encoding="utf-8")
        return

    pesi = tuple(float(x) for x in a.pesi.split(","))
    assert len(pesi) == 3, "--pesi vuole tre valori: w_ctx,w_nap,w_len"
    print(f"\n[5] Selezione (pesi composito: ctx={pesi[0]} nap={pesi[1]} "
          f"len={pesi[2]})")
    scelte, diag_sel = seleziona(cands, rows, model, tok, torch, clf_dial,
                                 p_nap_bersaglio, logp_len, minimo_len,
                                 p99_len, a.modo, pesi, a.max_len)
    print("  ", json.dumps(diag_sel, ensure_ascii=False))

    da_salvare = ["1campione"] + (
        ["mbr", "composito", "ibrido"] if a.modo == "tutti" else [a.modo])
    outdir = Path(a.out)
    outdir.mkdir(parents=True, exist_ok=True)

    print("\n[6] Metriche")
    riassunto = {}
    for m in da_salvare:
        hyps = scelte[m]
        run = f"{base}__{a.decoding}" if m == "1campione" else \
              f"{base}__{a.decoding}_bon{a.n}_{m}"
        res = {"run": run, "decoding": a.decoding, "repo_id": repo_id,
               "task": a.task, "tag": tag, "architettura": arch,
               "n_test": len(rows),
               "selezione": {"modo": m, "n_candidati": a.n,
                             "pesi_composito": list(pesi) if m in
                             ("composito", "ibrido") else None,
                             "gen_kw": {"decoding": a.decoding,
                                        "temperature": a.temperature,
                                        "typical_p": a.typical_p,
                                        "top_p": a.top_p,
                                        "repetition_penalty": a.repetition_penalty,
                                        "no_repeat_ngram_size": a.no_repeat_ngram,
                                        "eos_token_id": eot,
                                        "max_new_tokens": max_new},
                             "diagnostica": diag_sel},
               **blocco_metriche(a.task, rows, hyps, refs, clf_dial, info_dial,
                                 tok, max_new),
               "H_stratificato": stratifica_per_lunghezza(hyps, refs),
               }
        with (outdir / f"{run}.preds.jsonl").open("w", encoding="utf-8") as f:
            for r, h, cc_ in zip(rows, hyps, cands):
                f.write(json.dumps({"id": r.get("id"), "prompt": r["prompt"],
                                    "target": r["target"], "hyp": h,
                                    "candidati": cc_}, ensure_ascii=False) + "\n")
        (outdir / f"{run}.metrics.json").write_text(
            json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
        riassunto[m] = {
            "chrf++": res["F_riferimento"]["chrf++"],
            "adv": res["A_avversariale"].get("accuracy_umano_vs_macchina"),
            "adv_len": res["A_avversariale"].get("controllo_solo_lunghezza"),
            "adv_uu": res["A_avversariale"].get("controllo_umano_vs_umano"),
            "lung_media": res["C_distribuzionali"]["lunghezza_media"],
            "ks": res["C_distribuzionali"]["ks_lunghezza_stat"],
            "rep_2": res["C_distribuzionali"]["rep_2"],
            "loop3+": res["C_degenerazione"]["frazione_con_loop_3plus"],
            "p_nap": res.get("D_dialettalita", {}).get("P_nap_generato"),
            "al_tetto": res["G_troncamento"]["frazione_al_tetto"],
        }

    umano = {"lung_media": round(statistics.mean(len(x.split()) for x in refs), 2),
             "p_nap": round(p_nap_bersaglio, 3), "adv": 0.5}
    print("\n" + "=" * 78)
    print(f"{'arm':<12}{'chrf':>7}{'adv':>7}{'adv_len':>9}{'lung':>7}"
          f"{'ks':>7}{'rep_2':>8}{'loop3+':>8}{'p_nap':>7}{'tetto':>7}")
    print("-" * 78)
    for m, v in riassunto.items():
        print(f"{m:<12}{v['chrf++']:>7}{v['adv']:>7}{v['adv_len']:>9}"
              f"{v['lung_media']:>7}{v['ks']:>7}{v['rep_2']:>8}"
              f"{v['loop3+']:>8}{v['p_nap']:>7}{v['al_tetto']:>7}")
    print("-" * 78)
    print(f"{'UMANO':<12}{'-':>7}{'0.5':>7}{'-':>9}{umano['lung_media']:>7}"
          f"{'0':>7}{'0.037':>8}{'0.01':>8}{umano['p_nap']:>7}{'-':>7}")
    print("=" * 78)
    print("\nCome leggere: adv deve SCENDERE verso 0.5 e adv_len deve scendere\n"
          "con lei. Se adv scende ma adv_len resta alto, hai solo cambiato\n"
          "lunghezza senza migliorare il napoletano. chrF++ puo' restare piatto\n"
          "o calare: con riferimento singolo su task aperto non e' la metrica\n"
          "di merito (floor 9.79, il run misurato sta a 12.6).")
    print(f"\nSalvato in {outdir}/  -> metriche_finali.py li raccoglie affiancati")


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/rerank_t3.py


In [36]:
# --- Passo 30b. Diagnostica del fine turno ----------------------------------------
!python rerank_t3.py --model {MODEL} --task {TASK} --split-dir {SPLIT} \
    --adapter {ADAPTER} --solo-diagnostica

usage: rerank_t3.py [-h] --model MODEL [--task {T2,T3}]
                    [--split-dir SPLIT_DIR] [--split SPLIT]
                    [--adapter ADAPTER] [--out OUT] [--n N]
                    [--modo {mbr,composito,ibrido,tutti}] [--pesi PESI]
                    [--taratura] [--solo-diagnostica]
                    [--decoding {typical,nucleus}] [--top-p TOP_P]
                    [--typical-p TYPICAL_P] [--temperature TEMPERATURE]
                    [--repetition-penalty REPETITION_PENALTY]
                    [--no-repeat-ngram NO_REPEAT_NGRAM] [--max-len MAX_LEN]
                    [--max-new MAX_NEW] [--batch-prompt BATCH_PROMPT]
                    [--seed SEED] [--limite LIMITE] [--hf-token HF_TOKEN]
rerank_t3.py: error: argument --task: invalid choice: 'T1' (choose from T2, T3)


### Passo 30c — rilancia la valutazione con `generate()` corretto

**Cosa fare:** esegui. Sono i tre task con la patch applicata: questi diventano
la baseline corretta, e i numeri vecchi vanno tenuti solo come "prima del fix".

Su T3 girano **due** decodifiche: `nucleus` (top_p 0,9, T 0,8 — gli stessi
parametri di prima, così l'unica variabile che cambia è il fine turno) e
`typical` (locally typical sampling, `typical_p` 0,9, T 0,7). Con target di
mediana 4 parole il nucleus a 0,9/0,8 è troppo caldo: un campionamento sbagliato
sui primi due token non ha spazio per essere recuperato. La differenza fra i due
è un risultato, non una scelta da nascondere.


In [37]:
# --- Passo 30c. Valutazione con il fine turno corretto ----------------------------
!python evaluate_task.py --model {MODEL} --task {TASK} --split-dir {SPLIT} \
    --adapter {ADAPTER} --split test --out /kaggle/working/eval

GPU cc 7.5 -> float32  (Gemma senza bf16 nativo: in fp16 il forward va in NaN e le uscite escono vuote, quindi fp32)
Loading weights: 100%|█| 883/883 [00:07<00:00, 114.85it/s, Materializing param=m
  controllo numerico: logit finiti, primo token = 236745 't'
=== gemma-3-4b-it T1 ft | 267 esempi di test ===
Fine turno:
  chiusura del target renderizzato: id=106 repr='<end_of_turn>'
  tok.eos_token = '<eos>' (id 1)
  fine turno da usare = [1, 106, 212] (['<eos>', '<end_of_turn>', '</s>'])
  ! [212] NON erano in generation_config: senza il fix generate() arrivava sempre a max_new=64
Decoding: greedy

Frazione al tetto: 0.000   ok
Contrastiva (nessuna generazione, solo forward)...
{
  "run": "gemma-3-4b-it__T1__ft",
  "decoding": "greedy",
  "repo_id": "google/gemma-3-4b-it",
  "task": "T1",
  "tag": "ft",
  "architettura": "CausalLM",
  "n_test": 267,
  "F_riferimento": {
    "chrf++": 74.78,
    "bleu": 54.16
  },
  "F_bootstrap_chrf": {
    "ci95_basso": 72.52,
    "ci95_alto": 77.03
  

## Correzione: uscite di test tutte vuote (chrF++ 0.00)

Fra gli id di fine turno restituiti da `id_fine_turno()` c'e' il **107**, che non
e' un token speciale ma un semplice `'\n'`: il chat template mette un a capo
dopo `<end_of_turn>`, quindi e' l'ultimo token del target renderizzato ed e'
corretto che compaia nella lista.

Il problema era la guardia `min_new_tokens` nel decoder manuale: con `min_new=1`
la condizione `step < min_new - 1` e' falsa gia' al primo passo, quindi un `'\n'`
generato subito faceva uscire il loop restituendo stringa vuota. Su T1 e'
successo su **tutte e 267** le uscite: chrF++ 0.00, nessun errore, nessun avviso.

Due modifiche, in `evaluate_task.py` e in `common.py`:

1. l'EOS resta vietato finche' non e' stato generato almeno un carattere
   non-spazio. Fermarsi su `'\n'` dopo del contenuto e' giusto, prima no;
2. `genera()` conta le uscite vuote e lo dice a schermo. Un'uscita assente non e'
   una traduzione sbagliata, e non deve piu' arrivare a valle come una barra a
   zero in un grafico senza che nessuno se ne accorga.

**Questo non garantisce che il punteggio salga.** Se l'adapter e' compromesso
(nel warmup di T1 c'erano `grad_norm: nan` e `learning_rate: 0`, cioe' step
scartati dal GradScaler di fp16), il modello puo' produrre comunque testo privo
di senso. La cella diagnostica qui sotto distingue i due casi: genera ignorando
gli eos e mostra i token grezzi.


In [38]:
# --- Diagnostica: cosa genera davvero il modello, senza criteri di arresto ----
# Da eseguire SOLO se una valutazione torna con uscite vuote. Genera 24 token
# ignorando gli eos: distingue "il decoding si ferma troppo presto" da "il
# modello e' rotto e non produce niente di sensato".
import glob, torch
from collections import Counter
from transformers import AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from common import (resolve_model, load_backbone, load_hf_token, load_split,
                    render_prompt)
from evaluate_task import _decode_gemma3_manuale, id_fine_turno

# BUG CORRETTO. Il glob per solo task pescava anche l'adapter dello smoke test
# (gemma-3-1b-it__T1) e lo assegnava ad ADAPTER, la variabile usata da tutte le
# celle di valutazione qui sotto: la valutazione successiva ha girato sul base
# 4b con un adapter 1b (nel log: 638 chiavi LoRA mancanti, cioe' un adapter
# inerte). Si filtra per modello, come al Passo 26, e si usa una variabile
# separata cosi' ADAPTER non viene toccato.
import common
_slug_diag = common.slug(common.resolve_model(MODEL))
cand = (sorted(glob.glob(f"/kaggle/working/runs/{_slug_diag}__{TASK}/adapter_final"))
        or sorted(glob.glob(f"/kaggle/working/runs/{_slug_diag}__{TASK}/checkpoint-*")))
assert cand, f"nessun adapter {_slug_diag}__{TASK} in /kaggle/working/runs"
ADAPTER_DIAG = cand[-1]
print("adapter (solo per questa diagnostica):", ADAPTER_DIAG)

repo_id = resolve_model(MODEL); token = load_hf_token()
tok = AutoTokenizer.from_pretrained(repo_id, token=token, use_fast=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"
dtype, _ = common.scegli_dtype(repo_id)   # fp32 su Gemma senza bf16
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
model, _ = load_backbone(repo_id, dtype, quantization_config=quant,
                         device_map={"": 0}, token=token, low_cpu_mem_usage=True,
                         attn_implementation="eager")
model = PeftModel.from_pretrained(model, ADAPTER_DIAG); model.eval()
model.config.use_cache = True

rows = load_split(SPLIT, TASK, "test")[:20]
eot = set(id_fine_turno(tok, rows, verbose=False))
primi = []
for k, r in enumerate(rows):
    enc = tok(render_prompt(tok, r["prompt"]), return_tensors="pt",
              add_special_tokens=False).to(model.device)
    w = enc["input_ids"].shape[1]
    with torch.no_grad():
        # eos_ids=[] -> non si ferma mai: si vede cosa produrrebbe davvero
        g = _decode_gemma3_manuale(model, enc, "greedy", 24, 1, 0.9, 1.0,
                                   [], 1.0, 0, tok=tok)
    nuovi = g[0][w:].tolist()
    primi.append(nuovi[0])
    if k < 5:
        print(f"\n[{k}] target: {r['target']!r}")
        print("    id    :", nuovi[:12])
        print("    token :", tok.convert_ids_to_tokens(nuovi[:12]))
        print("    testo :", repr(tok.decode(nuovi, skip_special_tokens=False)))

print("\nprimo token generato, su 20 esempi:")
for i, n in Counter(primi).most_common():
    print(f"  id={i:6d} {tok.convert_ids_to_tokens([i])[0]!r:20s} x{n}"
          f"{'   <-- e nella lista di fine turno' if i in eot else ''}")
print("\nLettura: se i token dopo il primo sono napoletano sensato, era solo il\n"
      "decoding (gia' corretto). Se sono rumore o ripetizioni, l'adapter e'\n"
      "compromesso e va rifatto il training.")


adapter (solo per questa diagnostica): /kaggle/working/runs/gemma-3-4b-it__T1/adapter_final


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


[0] target: 'tutto a posto, grazie, sì sì'
    id    : [236745, 26611, 496, 46718, 236764, 63251, 236764, 111178, 111178, 106, 107, 106]
    token : ['t', 'utto', '▁a', '▁posto', ',', '▁grazie', ',', '▁sì', '▁sì', '<end_of_turn>', '\n', '<end_of_turn>']
    testo : 'tutto a posto, grazie, sì sì<end_of_turn>\n<end_of_turn>\n<end_of_turn>\n<end_of_turn>\n<end_of_turn>\n<end_of_turn>\n<end_of_turn>\n<end_of_turn>'

[1] target: 'tutto bbuono, grazie, sì sì, tutto bbuonissimo'
    id    : [236745, 26611, 518, 6489, 7503, 236764, 63251, 236764, 111178, 111178, 236764, 29018]
    token : ['t', 'utto', '▁b', 'bu', 'ono', ',', '▁grazie', ',', '▁sì', '▁sì', ',', '▁tutto']
    testo : 'tutto bbuono, grazie, sì sì, tutto bbuonissimo<end_of_turn>\n<end_of_turn>\n<end_of_turn>\n<end_of_turn>\n'

[2] target: "ll'unica cosa, 'o vino nun simmo riuscite a 'o fernì, 'o tenite ancora 'o tappo"
    id    : [859, 236789, 119388, 26617, 236764, 756, 236748, 62066, 18217, 1400, 1289, 3382]
    token : ['ll',

## Correzione 2 - la causa a monte: un a capo fra gli id di fine turno

La guardia nel decoder manuale (sopra) impedisce all'uscita di *restare* vuota,
ma il problema nasce prima, in `id_fine_turno()`: fra gli eos finisce
**l'ultimo token del target renderizzato**, che su Gemma-3 e' il **107**, cioe'
`'\n'`. Il chat template scrive `'<end_of_turn>' + a capo`, quindi il token che
chiude davvero il turno e' il **106** e il 107 e' solo la spaziatura che lo
segue.

Verifica, senza fidarsi dei commenti: in `tokenizer.json` di questo modello
106 = `<end_of_turn>` e 107 = `'\n'`; nel `chat_template.jinja` la riga e'
`{{ '<end_of_turn>\n' }}`. Con il 107 fra gli eos, un a capo emesso come primo
token chiude il turno: uscita vuota, chrF++ 0.00, nessun errore. Su T1 e'
successo su **267 item su 267**.

Le celle qui sotto, nell'ordine:

1. spostano la cartella `eval/` prodotta prima della correzione. Serve:
   `metriche_finali.py` globba **tutti** i `*.preds.jsonl` di `--preds-dir`, e
   quelli vecchi (vuoti) rientrerebbero nelle medie insieme ai nuovi;
2. rilanciano la valutazione su test con la lista di eos corretta;
3. **verificano** il risultato e si fermano con un errore se le uscite sono
   ancora vuote o se il 107 e' ancora fra gli eos. Un grafico a 0.00 non deve
   piu' poter uscire da questo notebook senza che nessuno se ne accorga.

Il training **non** va rifatto: gli adapter sono validi (chrF 77.35 in teacher
forcing, contrastiva 1.00 su 5 candidati). Cambia solo il decoding.

## Correzione 3 — la causa vera: overflow fp16 nel forward

Le uscite vuote non erano un problema di fine turno. Nei quattro
`*.metrics.json` prodotti finora c'e' `"logprob_riferimento_norm": NaN`, e la
metrica contrastiva **non genera niente**: fa solo forward. Il forward era gia'
rotto prima di arrivare al decoding.

Cosa succede, in ordine:

1. la GPU e' una T4 (cc 7.5), quindi `dtype = torch.float16`;
2. Gemma-3 ha attivazioni grandi: il flusso residuale in fp16 supera 65504,
   l'inf entra nelle RMSNorm e i logit diventano NaN;
3. `argmax` su un tensore di NaN restituisce l'indice **0**, che su Gemma e'
   `<pad>`. La diagnostica sopra lo mostra: primo token generato `<pad>` su 20
   esempi su 20;
4. `<pad>` non e' fra gli eos, quindi il loop non si ferma; con
   `skip_special_tokens=True` i 64 pad decodificano nella stringa vuota. chrF++
   0.00 su 267 item, nessuna eccezione.

**Perche' durante il training funzionava.** In `run()` il modello passa per
`prepare_model_for_kbit_training`, che porta a fp32 i parametri non quantizzati.
Con gli embedding in fp32 le hidden state sono fp32 e il residuo si accumula in
fp32; bitsandbytes casta l'ingresso a fp16 solo per il matmul 4-bit. Per questo
la generazione di monitoraggio stampava napoletano corretto sugli stessi prompt
e con lo stesso adapter, mentre `evaluate_task.py`, che carica il modello nudo,
restituiva 267 stringhe vuote.

### Cosa e' cambiato

| dove | modifica |
|---|---|
| `common.py` | `scegli_dtype()`, la regola del notebook T3: su GPU senza bf16 nativo Gemma gira in **fp32**, non in fp16, anche come `bnb_4bit_compute_dtype`. La usano tutti gli script di inferenza (`evaluate_task.py`, `prova_esempi.py`, `rerank_t3.py`, `decodifica_contestuale.py`, `diagnosi_cecita.py`) e `evaluate_task.py` espone `--dtype auto/bf16/fp16/fp32` |
| `common.py` | `stabilizza_fp16()`, chiamata da `load_backbone()` quando si carica in fp16 e 4-bit: stesso upcast che fa `prepare_model_for_kbit_training` in training. Seconda linea di difesa per i modelli che in fp16 girano davvero (Llama, Minerva) |
| `common.py`, `evaluate_task.py` | i decoder si fermano con un errore esplicito se i logit non sono finiti, invece di emettere `<pad>` in silenzio |
| `evaluate_task.py` | `<pad>` e `<bos>` non sono piu' generabili; `controllo_numerico()` fa un forward su un prompt vero prima di generare 267 volte |
| cella di diagnostica | non sovrascrive piu' `ADAPTER` con l'adapter dello smoke test: la valutazione precedente ha girato sul base 4b con un adapter 1b (638 chiavi LoRA mancanti, adapter inerte) |
| cella di verifica | fallisce anche su `logprob NaN`, che era l'unico segnale gia' presente e non guardato |
| `common.py`, `pretrain_dialect.py` | **anche il training** segue `scegli_dtype`: su Gemma senza bf16 si addestra in fp32, e `TrainingArguments` mette `fp16=False`/`bf16=False`, quindi niente autocast e niente GradScaler. `controlla_finito()` fa un forward di controllo prima di ogni run, con `--dtype` per forzare la precisione |

### Perche' fp32 e non l'upcast dei soli tensori non quantizzati

Perche' e' la configurazione gia' dimostrata sulla stessa GPU e sullo stesso
modello: il notebook T3 gira su T4 con `precisione float32`, `logit finiti:
True` e `"vuote": 0` in tutti gli arm. L'upcast dei soli parametri non
quantizzati replica la numerica del *training* (residuo in fp32, matmul 4-bit
ancora in fp16) ed e' probabilmente sufficiente, ma "probabilmente" non e' un
criterio: in valutazione sono 267 item e il raddoppio del tempo non conta
niente. Il training resta in fp16 com'era: e' gia' finito, era gia' stabile
grazie a `prepare_model_for_kbit_training`, e rifarlo cambierebbe gli adapter
da confrontare.

### Da rieseguire, in quest'ordine

`common.py` (Passo 6) → `evaluate_task.py` (Passo 15) → le tre celle qui sotto.
Il training **non** va rifatto: gli adapter sono validi, cambia solo come viene
caricato il modello in valutazione.


In [39]:
# --- Correzione 2a. Metti da parte la valutazione prodotta prima della patch --
# metriche_finali.py globba TUTTI i *.preds.jsonl di --preds-dir: se restano
# quelli con le uscite vuote, rientrano nel CSV finale e nelle medie insieme ai
# nuovi. Si spostano, non si cancellano: servono per il confronto prima/dopo.
import glob, os, shutil, time

VECCHI = "/kaggle/working/eval_prima_della_patch_" + time.strftime("%H%M%S")
da_spostare = glob.glob("/kaggle/working/eval/*")
if da_spostare:
    os.makedirs(VECCHI, exist_ok=True)
    for p in da_spostare:
        shutil.move(p, os.path.join(VECCHI, os.path.basename(p)))
    print(f"spostati {len(da_spostare)} file in {VECCHI}")
else:
    print("cartella eval/ gia' vuota: niente da spostare")

spostati 2 file in /kaggle/working/eval_prima_della_patch_052555


In [40]:
# --- Correzione 2b. Valutazione su test con il forward stabilizzato ----------
# Nell'output cercare, nell'ordine:
#   "stabilizzazione numerica: N tensori ... da fp16 a fp32"
#   "controllo numerico: logit finiti, primo token = ..."
# Se la prima riga manca, la cella di common.py non e' stata rieseguita.
import os, common
_slug_eval = common.slug(common.resolve_model(MODEL))
ADAPTER = f"/kaggle/working/runs/{_slug_eval}__{TASK}/adapter_final"
assert os.path.isdir(ADAPTER), f"adapter non trovato: {ADAPTER}"
print("adapter:", ADAPTER)

!python evaluate_task.py --model {MODEL} --task {TASK} --split-dir {SPLIT} \
    --adapter {ADAPTER} --split test --out /kaggle/working/eval

adapter: /kaggle/working/runs/gemma-3-4b-it__T1/adapter_final
GPU cc 7.5 -> float32  (Gemma senza bf16 nativo: in fp16 il forward va in NaN e le uscite escono vuote, quindi fp32)
Loading weights: 100%|█| 883/883 [00:07<00:00, 116.06it/s, Materializing param=m
  controllo numerico: logit finiti, primo token = 236745 't'
=== gemma-3-4b-it T1 ft | 267 esempi di test ===
Fine turno:
  chiusura del target renderizzato: id=106 repr='<end_of_turn>'
  tok.eos_token = '<eos>' (id 1)
  fine turno da usare = [1, 106, 212] (['<eos>', '<end_of_turn>', '</s>'])
  ! [212] NON erano in generation_config: senza il fix generate() arrivava sempre a max_new=64
Decoding: greedy

Frazione al tetto: 0.000   ok
Contrastiva (nessuna generazione, solo forward)...
{
  "run": "gemma-3-4b-it__T1__ft",
  "decoding": "greedy",
  "repo_id": "google/gemma-3-4b-it",
  "task": "T1",
  "tag": "ft",
  "architettura": "CausalLM",
  "n_test": 267,
  "F_riferimento": {
    "chrf++": 74.78,
    "bleu": 54.16
  },
  "F_bootstr

In [41]:
# --- Correzione 2c. Verifica: un 0.00 non deve piu' arrivare ai grafici -------
# Fallisce di proposito invece di lasciar passare metriche non interpretabili.
import glob, json

percorsi = sorted(glob.glob("/kaggle/working/eval/*.metrics.json"))
assert percorsi, "nessun *.metrics.json: la valutazione non e' arrivata in fondo"
with open(percorsi[-1], encoding="utf-8") as f:
    m = json.load(f)

chrf = m.get("F_riferimento", {}).get("chrf++")
dist = m.get("C_distribuzionali", {})
lung = dist.get("lunghezza_media")
lung_um = dist.get("_umano", {}).get("lunghezza_media")
base = m.get("F_baseline", {}).get("copia_italiano")
eos = m.get("gen_kw", {}).get("eos_token_id") or []
lp = m.get("B_contrastiva", {}).get("logprob_riferimento_norm")

print("file                :", percorsi[-1])
print("eos usati           :", eos)
print(f"chrF++ sistema      : {chrf}")
print(f"baseline copia ita  : {base}")
print(f"lunghezza media gen : {lung} parole   (riferimenti umani: {lung_um})")
print(f"logprob contrastiva : {lp}")

problemi = []
# Un NaN qui viene dal forward, non dal decoding: la contrastiva non genera
# niente. Era il sintomo che distingueva l'overflow fp16 dal problema di eos.
if lp is None or lp != lp:
    problemi.append("logprob_riferimento_norm NaN: il forward produce logit non "
                    "finiti (overflow fp16), quindi nessuna metrica di questo "
                    "file e' interpretabile")
if not lung:
    problemi.append("le uscite sono ancora vuote (lunghezza media 0)")
if not chrf:
    problemi.append("chrF++ nullo")
if any(t in eos for t in (107, 108)):
    problemi.append("un token di a capo (107/108) e' ancora fra gli eos: "
                    "la cella di common.py/evaluate_task.py non e' stata rieseguita")
assert not problemi, "DA NON PORTARE NEI GRAFICI -> " + "; ".join(problemi)

print("\nok: uscite non vuote, le metriche finali sono interpretabili.")
if base is not None and chrf is not None and chrf <= base:
    print(f"! chrF++ ({chrf:.2f}) non batte la copia dell'italiano ({base:.2f}). "
          "Questo si', e' un risultato da riportare: non e' un bug di decoding.")

file                : /kaggle/working/eval/gemma-3-4b-it__T1__ft.metrics.json
eos usati           : [1, 106, 212]
chrF++ sistema      : 74.78
baseline copia ita  : 35.41
lunghezza media gen : 7.09 parole   (riferimenti umani: 7.07)
logprob contrastiva : -0.972

ok: uscite non vuote, le metriche finali sono interpretabili.


In [42]:
# --- Passo 30g. CSV unico + metriche ----------------------------------------------
!python metriche_finali.py --preds-dir /kaggle/working/eval \
    --lessico {LEX_JSON} \
    --csv /kaggle/working/predizioni_test.csv \
    --out /kaggle/working/metriche_finali.json

import pandas as pd
pred = pd.read_csv("/kaggle/working/predizioni_test.csv")
print(f"{len(pred)} generazioni")
display(pred.groupby(["task", "tag", "decoding"])[["chrf_item", "bertscore_f1"]]
        .agg(["mean", "count"]).round(3))

Trovati 1 file di predizioni:
  gemma-3-4b-it__T1__ft.preds.jsonl                      267 item

BERTScore con bert-base-multilingual-cased su 801 coppie (sistema + tarature)...
Loading weights: 100%|█| 199/199 [00:00<00:00, 1415.77it/s, Materializing param=
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

Scri

chrf_item       bertscore_f1      
                       mean count         mean count
task tag decoding                                   
T1   ft  greedy      76.907   267        0.922   267

## Passo 32 — il modello è cieco al contesto? E il prompt può renderlo vedente?


### Passo 32a — i due contatori che hai già nei log

Prima di qualunque ipotesi: se il contesto **non arriva** al modello per un motivo
meccanico, ogni ora spesa su prompt, DPO o reranking è buttata. Due numeri in ogni
`summary.json`:

- **`troncati_train`** — prompt tagliati a `max_seq_len`. Se è alto, parte del
  contesto non entra mai nel modello: cieco alla lettera, e nessun intervento a
  valle recupera informazione che non è mai arrivata.
- **`prefix_mismatch_train`** — confine prompt/target sfasato. Deve essere **0**:
  altrimenti la loss è mascherata sul punto sbagliato e stai addestrando su un
  target che non è quello che credi.

Lo stesso script fa anche l'ispezione statica dei prompt: quante parole di
contesto ci sono davvero, quanti turni hanno l'etichetta del parlante, e — il
controllo che decide il 32c — **quante istruzioni distinte** esistono nel train.
Zero GPU.


In [43]:
%%writefile /kaggle/working/training/diagnosi_cecita.py
#!/usr/bin/env python3
"""
diagnosi_cecita.py — il modello e' cieco al contesto? Tre controlli, due gratis.

"Sembra cieco" e' un'impressione, e questa e' l'unica cosa di tutto il progetto
che non va trattata come tale: se il contesto non arriva al modello per un
motivo meccanico, ogni ora spesa su prompt, DPO o reranking e' buttata.

I tre controlli, in ordine di costo:

[1] CONTATORI DI TRAINING (zero GPU, legge i summary.json)
    troncati_train        prompt tagliati a max_seq_len. Se e' alto, parte del
                          contesto NON ARRIVA MAI al modello: cieco alla
                          lettera, e nessun intervento a valle serve.
    prefix_mismatch_train confine prompt/target sfasato: la loss e'
                          mascherata sul punto sbagliato. Deve essere 0.

[2] ABLAZIONE SUL CONTESTO (~20 min di GPU)
    Genera lo stesso item due volte: una col contesto reale, una col contesto
    preso da un ALTRO punto della conversazione (sostituisci_contesto di
    contesto_metrica.py), poi misura il chrF++ FRA LE DUE USCITE.

        chrF alto (>60)  le due uscite sono quasi identiche: il modello ignora
                         il contesto. E' la misura diretta della cecita', ed e'
                         il numero da riportare.
        chrF basso       il contesto entra nella generazione. Il problema non e'
                         che sia cieco, e' che ci vede male.

    Perche' e' meglio della contrastiva: la contrastiva misura il PUNTEGGIO che
    il modello assegna a stringhe date, questa misura la GENERAZIONE - cioe'
    esattamente il comportamento che stai osservando quando dici che sembra
    cieco. Un modello puo' avere contrastiva 0.33 (qualche preferenza) e
    generare comunque la stessa cosa a prescindere dal contesto.

[3] ISPEZIONE DEI PROMPT (zero GPU)
    Stampa prompt reali per layout e controlla quattro cose concrete:
      - i turni di contesto hanno l'etichetta del parlante?
      - l'istruzione dice A QUALE parlante rispondere, con lo stesso
        identificativo usato nel contesto?
      - quante parole di contesto ci sono davvero? 3 turni da ~7 parole sono
        ~21 parole di parlato frammentario: bastano a stabilire l'argomento?
      - il separatore distingue visibilmente contesto e istruzione?
    Piu' un controllo automatico: l'istruzione e' COSTANTE su tutti gli esempi?
    Se lo e', ha informazione mutua zero col target, il modello non puo' usarla
    per abbassare la loss e impara a ignorarla. Riformularla meglio non serve:
    va resa variabile (vedi arricchisci_split.py).

Uso
---
    # solo i controlli gratis
    python diagnosi_cecita.py --split-dir /kaggle/working/split --solo-statico

    # tutto, incluso il [2]
    python diagnosi_cecita.py --model minerva --split-dir /kaggle/working/split \\
        --adapter /kaggle/working/runs/<slug>__T3/adapter_final --task T3
"""

import argparse
import glob
import json
import os
import random
import statistics
from collections import Counter
from pathlib import Path

from common import load_split, render_prompt, resolve_model
from contesto_metrica import estrai_blocco_contesto, sostituisci_contesto
from evaluate_task import MAX_NEW, id_fine_turno, norm


# --------------------------------------------------------------------------- #
# [1] contatori di training
# --------------------------------------------------------------------------- #

def contatori(runs_dir):
    righe = []
    for p in sorted(glob.glob(os.path.join(runs_dir, "*", "summary.json"))):
        s = json.load(open(p, encoding="utf-8"))
        d = s.get("dataset", {})
        n_tr = d.get("train") or 1
        righe.append({
            "run": os.path.basename(os.path.dirname(p)),
            "layout": s.get("task", {}).get("layout"),
            "max_seq_len": s.get("hyperparams", {}).get("max_seq_len"),
            "train": d.get("train"),
            "troncati_train": d.get("troncati_train"),
            "frazione_troncati": round((d.get("troncati_train") or 0) / n_tr, 3),
            "prefix_mismatch_train": d.get("prefix_mismatch_train"),
        })
    return righe


def leggi_contatori(righe):
    problemi = []
    for r in righe:
        if (r["frazione_troncati"] or 0) > 0.02:
            problemi.append(
                f"{r['run']}: {r['frazione_troncati']:.1%} dei prompt di train "
                f"troncati a max_seq_len={r['max_seq_len']}. Parte del contesto "
                f"non arriva al modello. Alza --max-seq-len o accorcia la "
                f"finestra: nessun intervento a valle recupera informazione "
                f"che non e' mai entrata.")
        if r["prefix_mismatch_train"]:
            problemi.append(
                f"{r['run']}: prefix_mismatch_train={r['prefix_mismatch_train']} "
                f"(deve essere 0). Il confine prompt/target e' sfasato: la loss "
                f"e' mascherata sul punto sbagliato e stai addestrando su un "
                f"target che non e' quello che credi.")
    return problemi


# --------------------------------------------------------------------------- #
# [3] ispezione dei prompt (statico)
# --------------------------------------------------------------------------- #

def analizza_prompt(rows, layout):
    """Cosa c'e' e cosa manca nei prompt, in numeri."""
    con_ctx, turni_n, parole_ctx = 0, [], []
    etichettati, non_etichettati = 0, 0
    istruzioni = Counter()

    for r in rows:
        trovato = estrai_blocco_contesto(r["prompt"])
        if trovato is None:
            continue
        con_ctx += 1
        i0, i1, turni = trovato
        turni_n.append(len(turni))
        parole_ctx.append(sum(len(t.split()) for t in turni))
        # etichetta del parlante: "X:" all'inizio della riga
        for t in turni:
            testa = t.strip().split(":", 1)
            if len(testa) == 2 and 0 < len(testa[0]) <= 12:
                etichettati += 1
            else:
                non_etichettati += 1
        # l'istruzione: tutto cio' che segue il blocco, separatori esclusi
        righe = r["prompt"].split("\n")
        istr = "\n".join(x for x in righe[i1:] if x.strip()
                         and not set(x.strip()) <= set("-=*_"))
        istruzioni[istr] += 1

    n = len(rows)
    tot_t = etichettati + non_etichettati
    piu_comune, freq = (istruzioni.most_common(1)[0] if istruzioni else ("", 0))
    return {
        "layout": layout,
        "n": n,
        "con_contesto": con_ctx,
        "senza_contesto": n - con_ctx,
        "turni_medi": round(statistics.mean(turni_n), 2) if turni_n else 0,
        "parole_contesto_medie": round(statistics.mean(parole_ctx), 1) if parole_ctx else 0,
        "parole_contesto_mediane": statistics.median(parole_ctx) if parole_ctx else 0,
        "turni_etichettati": round(etichettati / tot_t, 3) if tot_t else 0,
        "istruzioni_distinte": len(istruzioni),
        "frazione_istruzione_piu_comune": round(freq / con_ctx, 3) if con_ctx else 0,
        "istruzione_piu_comune": piu_comune,
    }


def leggi_prompt(a_):
    """Le quattro domande, con la risposta che i numeri danno."""
    p = []
    if a_["istruzioni_distinte"] <= 1 or a_["frazione_istruzione_piu_comune"] > 0.98:
        p.append(
            f"{a_['layout']}: l'istruzione e' COSTANTE su tutti gli esempi "
            f"({a_['istruzioni_distinte']} varianti distinte). Una feature "
            f"costante ha informazione mutua zero col target: il modello non "
            f"puo' usarla per abbassare la loss, quindi impara a ignorarla e "
            f"la legge come un delimitatore, non come un'istruzione. "
            f"Riformularla meglio NON serve: va resa variabile.")
    if a_["turni_etichettati"] < 0.9:
        p.append(
            f"{a_['layout']}: solo il {a_['turni_etichettati']:.0%} dei turni di "
            f"contesto ha l'etichetta del parlante. Senza, il modello non puo' "
            f"sapere che i turni si alternano ne' a chi tocca rispondere. "
            f"Il CSV ha la colonna `speaker`: l'informazione c'e' e non e' usata.")
    if a_["parole_contesto_medie"] < 30:
        p.append(
            f"{a_['layout']}: {a_['parole_contesto_medie']} parole di contesto in "
            f"media ({a_['turni_medi']} turni). E' parlato frammentario: puo' non "
            f"bastare a stabilire l'argomento. Prova ad allargare la finestra "
            f"(igiene_split.py --finestre) e vedi se la contrastiva si muove: se "
            f"non si muove, il contesto non e' il collo di bottiglia e lo sai.")
    if a_["senza_contesto"]:
        p.append(
            f"{a_['layout']}: {a_['senza_contesto']}/{a_['n']} item senza blocco "
            f"di contesto estraibile. Su quelli il delta di contesto e il CFG "
            f"non sono definiti e vengono esclusi dalle misure.")
    return p


# --------------------------------------------------------------------------- #
# [2] ablazione: contesto vero contro contesto falso
# --------------------------------------------------------------------------- #

def coppie_contesto(rows, seed=0):
    """(prompt_vero, prompt_falso, target). Il contesto falso viene da un ALTRO
    item, non da turni sintetici: cosi' i due prompt sono entrambi plausibili e
    ben formati, e la differenza fra le uscite misura la pertinenza e non la
    stranezza del testo."""
    rng = random.Random(seed)
    con = [(i, estrai_blocco_contesto(r["prompt"])) for i, r in enumerate(rows)]
    con = [(i, t) for i, t in con if t is not None]
    fuori = []
    for i, _ in con:
        j = i
        for _ in range(20):
            j = rng.choice(con)[0]
            if j != i:
                break
        alt = estrai_blocco_contesto(rows[j]["prompt"])
        falso = sostituisci_contesto(rows[i]["prompt"], alt[2])
        if falso is None or falso == rows[i]["prompt"]:
            continue
        fuori.append((rows[i]["prompt"], falso, rows[i]["target"]))
    return fuori


def ablazione(model, tok, torch, coppie, max_new, eot, batch=8, limite=64,
              temperature=0.0):
    """chrF++ fra uscita-con-contesto-vero e uscita-con-contesto-falso.

    temperature=0 (greedy) di proposito: col campionamento due uscite
    differiscono anche a parita' di prompt, e il numero misurerebbe la
    stocasticita' del decoder invece della sensibilita' al contesto.
    """
    import sacrebleu
    from evaluate_task import pulisci_generato

    sub = coppie[:limite]
    prev = tok.padding_side
    tok.padding_side = "left"

    # BUG CORRETTO. Qui c'era una generate() batched con left-padding: su
    # Gemma-3 (Gemma3ForConditionalGeneration) crasha dentro transformers
    # ("Tensors must have same number of dimensions"), quindi l'ablazione non
    # produceva nessun numero. Si passa da common.genera_una, che su Gemma-3 usa
    # il loop di decoding manuale (batch 1, greedy) e sugli altri modelli
    # continua a usare generate(). L'ablazione e' greedy per costruzione, quindi
    # il loop manuale calcola esattamente la stessa cosa.
    from common import genera_una

    def genera(testi):
        out = []
        for i, x in enumerate(testi, 1):
            txt = genera_una(model, tok, render_prompt(tok, x), max_new, eot)
            out.append(pulisci_generato(txt))
            print(f"    {i}/{len(testi)}", end="\r")
        return out

    try:
        veri = genera([c[0] for c in sub])
        print()
        falsi = genera([c[1] for c in sub])
        print()
    finally:
        tok.padding_side = prev

    def chrf(h, r):
        return sacrebleu.sentence_chrf(h, [r], word_order=2).score

    per_item = [chrf(v, f) if (v.strip() and f.strip()) else None
                for v, f in zip(veri, falsi)]
    validi = [x for x in per_item if x is not None]
    identici = sum(1 for v, f in zip(veri, falsi) if norm(v) == norm(f))
    refs = [c[2] for c in sub]
    chrf_vero = round(sacrebleu.corpus_chrf(veri, [refs], word_order=2).score, 2)
    chrf_falso = round(sacrebleu.corpus_chrf(falsi, [refs], word_order=2).score, 2)

    return {
        "n": len(validi),
        "chrf_tra_le_due_uscite": round(statistics.mean(validi), 2) if validi else None,
        "frazione_uscite_identiche": round(identici / len(sub), 3) if sub else None,
        "chrf_contro_riferimento_contesto_vero": chrf_vero,
        "chrf_contro_riferimento_contesto_falso": chrf_falso,
        "delta_chrf_vero_meno_falso": round(chrf_vero - chrf_falso, 2),
        "_lettura": "chrf_tra_le_due_uscite ALTO (>60) o frazione_identiche alta "
                    "= il modello ignora il contesto: cecita' misurata. "
                    "delta_chrf_vero_meno_falso <= 0 = il contesto reale non "
                    "aiuta nemmeno contro il riferimento, che e' la forma piu' "
                    "netta dello stesso risultato.",
        "esempi": [{"contesto_vero": v, "contesto_falso": f, "riferimento": r}
                   for v, f, r in list(zip(veri, falsi, refs))[:10]],
    }


# --------------------------------------------------------------------------- #

def main():
    ap = argparse.ArgumentParser(
        description="Il modello e' cieco al contesto? Tre controlli.",
        formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--split-dir", default="/kaggle/working/split")
    ap.add_argument("--runs-dir", default="/kaggle/working/runs")
    ap.add_argument("--out", default="/kaggle/working/eval/diagnosi_cecita.json")
    ap.add_argument("--task", default="T3", nargs="+", choices=["T1", "T2", "T3"])
    ap.add_argument("--split", default="dev",
                    help="dev per default: la diagnosi non deve consumare il test")
    ap.add_argument("--solo-statico", action="store_true",
                    help="solo [1] e [3]: nessuna GPU")
    ap.add_argument("--model", default=None)
    ap.add_argument("--adapter", default=None)
    ap.add_argument("--limite", type=int, default=64)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--hf-token", default=None)
    a = ap.parse_args()

    tasks = a.task if isinstance(a.task, list) else [a.task]
    rapporto = {"split_dir": a.split_dir, "split": a.split}

    print("=" * 74)
    print("[1] CONTATORI DI TRAINING  (troncamento e masking)")
    print("=" * 74)
    righe = contatori(a.runs_dir)
    if not righe:
        print(f"  nessun summary.json in {a.runs_dir}: salto")
    else:
        for r in righe:
            print(f"  {r['run']:<38} {r['layout']}  troncati "
                  f"{r['troncati_train']}/{r['train']} "
                  f"({r['frazione_troncati']:.1%})  mismatch "
                  f"{r['prefix_mismatch_train']}")
    problemi = leggi_contatori(righe)
    rapporto["contatori"] = {"righe": righe, "problemi": problemi,
                             "verificato": bool(righe)}
    if not righe:
        print("\n  ? NON VERIFICATO: senza summary.json non si puo' escludere il "
              "troncamento.\n     Questo controllo resta aperto: rilancialo dopo "
              "il primo run di training.")
    elif problemi:
        print("\n  ! PROBLEMI MECCANICI (da risolvere PRIMA di tutto il resto):")
        for p in problemi:
            print(f"    - {p}")
    else:
        print("\n  ok: nessun troncamento significativo, masking allineato.")
        print("     Il contesto ARRIVA al modello: se sembra cieco, non e' per "
              "questo.")

    print("\n" + "=" * 74)
    print("[3] ISPEZIONE DEI PROMPT")
    print("=" * 74)
    rapporto["prompt"] = {}
    for t in tasks:
        rows = load_split(a.split_dir, t, "train")
        an = analizza_prompt(rows, t)
        rapporto["prompt"][t] = an
        print(f"\n  {t}: {an['con_contesto']}/{an['n']} con contesto | "
              f"{an['turni_medi']} turni | {an['parole_contesto_medie']} parole | "
              f"turni etichettati {an['turni_etichettati']:.0%} | "
              f"istruzioni distinte {an['istruzioni_distinte']}")
        print(f"  --- un prompt reale di {t} " + "-" * 40)
        for riga in rows[0]["prompt"].split("\n"):
            print(f"  | {riga}")
        print(f"  | TARGET: {rows[0]['target']}")
        oss = leggi_prompt(an)
        rapporto["prompt"][t]["osservazioni"] = oss
        for o in oss:
            print(f"    - {o}")

    if a.solo_statico:
        Path(a.out).parent.mkdir(parents=True, exist_ok=True)
        Path(a.out).write_text(json.dumps(rapporto, ensure_ascii=False, indent=2),
                               encoding="utf-8")
        print(f"\nSalvato {a.out}  (ablazione [2] non eseguita: --solo-statico)")
        return

    if not (a.model and a.adapter):
        print("\n[2] salto l'ablazione: servono --model e --adapter")
        return

    print("\n" + "=" * 74)
    print("[2] ABLAZIONE SUL CONTESTO  (contesto vero contro contesto falso)")
    print("=" * 74)
    import torch
    from transformers import AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel
    from common import load_backbone, load_hf_token, scegli_dtype

    repo = resolve_model(a.model)
    token = load_hf_token(a.hf_token)
    tok = AutoTokenizer.from_pretrained(repo, token=token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dt, _ = scegli_dtype(repo)            # fp32 su Gemma senza bf16
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_use_double_quant=True,
                               bnb_4bit_compute_dtype=dt)
    model, arch = load_backbone(repo, dt, quantization_config=quant,
                                device_map={"": 0}, token=token,
                                low_cpu_mem_usage=True,
                                attn_implementation="eager")
    model = PeftModel.from_pretrained(model, a.adapter)
    model.eval()
    model.config.use_cache = True

    rapporto["ablazione"] = {}
    for t in tasks:
        rows = load_split(a.split_dir, t, a.split)
        coppie = coppie_contesto(rows, a.seed)
        if not coppie:
            print(f"  {t}: nessuna coppia costruibile (contesto non estraibile)")
            continue
        eot = id_fine_turno(tok, rows, verbose=False)
        print(f"\n  {t}: {len(coppie)} coppie, ne uso {min(a.limite, len(coppie))}")
        res = ablazione(model, tok, torch, coppie, MAX_NEW[t], eot,
                        batch=a.batch, limite=a.limite)
        rapporto["ablazione"][t] = res
        print(f"    chrF fra le due uscite      {res['chrf_tra_le_due_uscite']}")
        print(f"    uscite identiche            "
              f"{res['frazione_uscite_identiche']:.1%}")
        print(f"    chrF vs riferimento  vero   "
              f"{res['chrf_contro_riferimento_contesto_vero']}")
        print(f"                         falso  "
              f"{res['chrf_contro_riferimento_contesto_falso']}")
        print(f"    delta (vero - falso)        "
              f"{res['delta_chrf_vero_meno_falso']}")
        c = res["chrf_tra_le_due_uscite"] or 0
        d = res["delta_chrf_vero_meno_falso"]
        if c > 60 or (res["frazione_uscite_identiche"] or 0) > 0.3:
            print(f"    -> CIECO: cambiare il contesto non cambia l'uscita.")
        elif d <= 0:
            print(f"    -> CIECO in forma netta: il contesto reale non aiuta "
                  f"nemmeno contro il riferimento.")
        else:
            print(f"    -> il contesto ENTRA nella generazione. Il problema non "
                  f"e' la cecita': e' che ci vede male. Prompt e DPO sono la "
                  f"strada, non il troncamento.")
        print("\n    tre esempi (stesso item, contesto diverso):")
        for e in res["esempi"][:3]:
            print(f"      RIF   {e['riferimento']}")
            print(f"      vero  {e['contesto_vero']}")
            print(f"      falso {e['contesto_falso']}")

    Path(a.out).parent.mkdir(parents=True, exist_ok=True)
    Path(a.out).write_text(json.dumps(rapporto, ensure_ascii=False, indent=2),
                           encoding="utf-8")
    print(f"\nSalvato {a.out}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/diagnosi_cecita.py


In [44]:
# --- Passo 32a. Controlli statici: nessuna GPU ------------------------------------
!python diagnosi_cecita.py --split-dir {SPLIT} --runs-dir /kaggle/working/runs \
    --task {TASK} --solo-statico \
    --out /kaggle/working/eval/diagnosi_cecita_statica.json

[1] CONTATORI DI TRAINING  (troncamento e masking)
  gemma-3-1b-it__T1                      T1  troncati 0/48 (0.0%)  mismatch 0
  gemma-3-4b-it__A2                      A2  troncati 0/1742 (0.0%)  mismatch 0
  gemma-3-4b-it__T1                      T1  troncati 0/1134 (0.0%)  mismatch 0

  ok: nessun troncamento significativo, masking allineato.
     Il contesto ARRIVA al modello: se sembra cieco, non e' per questo.

[3] ISPEZIONE DEI PROMPT

  T1: 1133/1134 con contesto | 2.99 turni | 21.5 parole | turni etichettati 100% | istruzioni distinte 1120
  --- un prompt reale di T1 ----------------------------------------
  | Conversazione finora:
  | A: emh
  | ---
  | Traduci in napoletano: tu non hai mai fatto queste foto quando sei andata in barca
  | TARGET: tu nun l'hê maje fatte sti ffoto quanno si' ghiuta 'ncopp'â varca
    - T1: 21.5 parole di contesto in media (2.99 turni). E' parlato frammentario: puo' non bastare a stabilire l'argomento. Prova ad allargare la finestra (igiene_sp

### Passo 32b — ablazione sul contesto: la misura diretta della cecità

Genera lo stesso item **due volte**: una col contesto reale, una col contesto
preso da un altro punto della conversazione (`sostituisci_contesto`, che hai già),
poi misura il chrF++ **fra le due uscite**.

- **chrF alto (>60)**, o frazione di uscite identiche alta → cambiare il contesto
  non cambia l'output. **Cecità misurata**, ed è il numero da riportare.
- **`delta_chrf_vero_meno_falso` ≤ 0** → il contesto reale non aiuta nemmeno
  contro il riferimento: è la forma più netta dello stesso risultato.
- **chrF basso e delta positivo** → il contesto *entra* nella generazione. Il
  problema non è la cecità, è che ci vede male — e allora prompt e DPO sono la
  strada giusta.

Perché è più informativa della contrastiva: la contrastiva misura il **punteggio**
che il modello assegna a stringhe date, questa misura la **generazione** — cioè
esattamente il comportamento che stai osservando. Un modello può avere contrastiva
0,33 e generare comunque la stessa cosa a prescindere dal contesto.

Il decoding è **greedy di proposito**: col campionamento due uscite differiscono
anche a parità di prompt, e il numero misurerebbe la stocasticità del decoder
invece della sensibilità al contesto.


In [45]:
# --- Passo 32b. Ablazione contesto vero / contesto falso --------------------------
# Anche T1 ha il contesto conversazionale nel prompt: se l'output non cambia fra
# contesto vero e contesto di un altro punto, il contesto e' decorativo.
!python diagnosi_cecita.py --model {MODEL} --split-dir {SPLIT} \
    --adapter {ADAPTER} --task {TASK} --split dev --limite 64 \
    --out /kaggle/working/eval/diagnosi_cecita.json

[1] CONTATORI DI TRAINING  (troncamento e masking)
  gemma-3-1b-it__T1                      T1  troncati 0/48 (0.0%)  mismatch 0
  gemma-3-4b-it__A2                      A2  troncati 0/1742 (0.0%)  mismatch 0
  gemma-3-4b-it__T1                      T1  troncati 0/1134 (0.0%)  mismatch 0

  ok: nessun troncamento significativo, masking allineato.
     Il contesto ARRIVA al modello: se sembra cieco, non e' per questo.

[3] ISPEZIONE DEI PROMPT

  T1: 1133/1134 con contesto | 2.99 turni | 21.5 parole | turni etichettati 100% | istruzioni distinte 1120
  --- un prompt reale di T1 ----------------------------------------
  | Conversazione finora:
  | A: emh
  | ---
  | Traduci in napoletano: tu non hai mai fatto queste foto quando sei andata in barca
  | TARGET: tu nun l'hê maje fatte sti ffoto quanno si' ghiuta 'ncopp'â varca
    - T1: 21.5 parole di contesto in media (2.99 turni). E' parlato frammentario: puo' non bastare a stabilire l'argomento. Prova ad allargare la finestra (igiene_sp

## Opzionale — valutare su T1 l'adapter multi-task

Il run multi-task (Passo 32d) si addestra nel notebook **T3**: un adapter solo
per i tre task, su prompt arricchiti. La valutazione pero' e' per task, quindi
la parte che riguarda T1 sta qui.

Serve una cosa sola: che lo split arricchito sia **byte-identico** a quello visto
in training. Le due celle che lo ricostruiscono sono deterministiche a parita' di
`SEED` e di flag, quindi non serve trasportare la cartella: basta non cambiare
nessuna delle due righe di comando rispetto al notebook T3.

Le prime celle sono `%%writefile`: scrivono `igiene_split.py` e
`arricchisci_split.py` senza eseguire niente. Se non hai ancora l'adapter MT,
salta tutta questa sezione.

In [46]:
%%writefile /kaggle/working/training/igiene_split.py
#!/usr/bin/env python3
"""
igiene_split.py — riscrive lo split di T3 (e T2) senza aggiungere dati.

Tre patologie del corpus che l'addestramento attuale trasforma in
comportamento appreso, e che si correggono ricombinando i dati che hai:

1. I RISCONTRI COME TARGET.
   nel corpus:  <=1 parola 20.9%   |  mh    95 volte
                <=2 parole 33.7%   |  si'   71
                <=3 parole 43.8%   |  mhmh  58
                                   |  no    32
   Addestrare su questi insegna che una replica senza contenuto e' corretta, e
   il modello media fra due distribuzioni molto diverse: "decidi se serve un
   riscontro" e "produci un contenuto". Due rimedi possibili:
     --tipo-turno   li tiene, ma sotto un marcatore nel prompt, cosi' il modello
                    impara QUANDO un riscontro e' giusto invece di mediare
     --filtra-riscontri  li toglie dal train (non da dev/test: la valutazione
                    resta sulla distribuzione reale)

2. I DUPLICATI. 495 target su 2568 sono duplicati esatti. "mh" pesa 95 volte
   quanto un turno contenutistico. `--sottocampiona-duplicati` tiene al massimo
   ceil(sqrt(freq)) copie per target: e' il peso 1/sqrt(freq) ottenuto per
   sottocampionamento, senza toccare il Trainer.

3. UNA SOLA FINESTRA DI CONTESTO. Ogni item ha 3 turni di contesto, sempre.
   `--finestre 1,2,3,4` genera un item per ogni ampiezza dagli stessi turni:
   ~3x le istanze, zero dati nuovi, e insegna robustezza all'ampiezza del
   contesto - utile anche in inferenza, dove non e' garantito che ce ne siano 3.

Sul marcatore di tipo turno
---------------------------
Bucket derivato dalla lunghezza del target:

    riscontro  <= 2 parole      breve  3-6      medio  7-12      lungo  13+

In TRAIN il bucket viene dal target reale. In DEV e TEST il target non si puo'
guardare, quindi il bucket viene CAMPIONATO dalla distribuzione empirica del
train con seme fisso. Questo e' il punto: la valutazione resta onesta (il
modello non vede la lunghezza del riferimento) e la distribuzione delle
lunghezze generate diventa compatibile con l'umano per costruzione, che e'
quello che fa crollare `controllo_solo_lunghezza` - 0.776 su 0.80 di accuracy
avversariale nel run misurato, cioe' il 97% del potere discriminante.

E' controllable generation standard (Kikuchi et al. 2016; Fan et al. 2018).

Sicurezza
---------
Scrive in una cartella NUOVA: lo split originale non viene toccato, e i due
arm sono confrontabili perche' esistono entrambi. Prima di scrivere verifica che
`estrai_blocco_contesto` e `rimuovi_contesto` funzionino ancora sui prompt
modificati: se il marcatore rompe il parsing, contesto_metrica.py e
decodifica_contestuale.py darebbero numeri plausibili e sbagliati, quindi lo
script si ferma invece di procedere.

Uso
---
    python igiene_split.py --split-dir /kaggle/working/split \\
        --out-dir /kaggle/working/split_igiene \\
        --tipo-turno --sottocampiona-duplicati --finestre 1,2,3,4

    # variante minimale: solo il marcatore
    python igiene_split.py --split-dir ... --out-dir ... --tipo-turno
"""

import argparse
import json
import math
import random
import shutil
import statistics
from collections import Counter
from pathlib import Path

from common import LAYOUT_DIRS, load_split
from contesto_metrica import (_e_separatore, estrai_blocco_contesto,
                              sostituisci_contesto)
from decodifica_contestuale import rimuovi_contesto

MARCATORE = "Tipo di turno: {b}."


def bucket(n_parole):
    if n_parole <= 2:
        return "riscontro"
    if n_parole <= 6:
        return "breve"
    if n_parole <= 12:
        return "medio"
    return "lungo"


def inserisci_marcatore(prompt, b):
    """Il marcatore va DOPO il separatore che chiude il contesto.

    Non prima: estrai_blocco_contesto delimita il blocco fra l'intestazione e il
    primo separatore (riga vuota o trattini), quindi una riga inserita a indice
    i1 finisce DENTRO i turni di contesto - verrebbe letta come un turno, e
    sostituisci_contesto la cancellerebbe. Il delta di contesto e il CFG
    misurerebbero la cosa sbagliata senza dare errore.

    Non all'inizio: sposterebbe l'indice dell'intestazione. Non alla fine: deve
    stare adiacente all'istruzione, dove il modello lo legge prima di generare.
    """
    trovato = estrai_blocco_contesto(prompt)
    if trovato is None:
        return None
    _, i1, _ = trovato
    righe = prompt.split("\n")
    k = i1
    while k < len(righe) and _e_separatore(righe[k]):
        k += 1                       # oltre il separatore, prima dell'istruzione
    righe.insert(k, MARCATORE.format(b=b))
    return "\n".join(righe)


def finestra(prompt, n):
    """Lo stesso item con solo gli ultimi n turni di contesto."""
    trovato = estrai_blocco_contesto(prompt)
    if trovato is None:
        return None
    _, _, turni = trovato
    if n >= len(turni):
        return prompt if n == len(turni) else None
    return sostituisci_contesto(prompt, turni[-n:])


def sottocampiona(rows, seed):
    """Tiene al massimo ceil(sqrt(freq)) item per ogni target distinto."""
    rng = random.Random(seed)
    per_target = {}
    for r in rows:
        per_target.setdefault(r["target"].strip().lower(), []).append(r)
    fuori, tolti = [], 0
    for _, gruppo in per_target.items():
        tetto = math.ceil(math.sqrt(len(gruppo)))
        if len(gruppo) <= tetto:
            fuori.extend(gruppo)
        else:
            fuori.extend(rng.sample(gruppo, tetto))
            tolti += len(gruppo) - tetto
    rng.shuffle(fuori)
    return fuori, tolti


def statistiche(rows, etichetta):
    L = [len(r["target"].split()) for r in rows]
    c = Counter(bucket(x) for x in L)
    return {"nome": etichetta, "n": len(rows),
            "lunghezza_media": round(statistics.mean(L), 2) if L else 0,
            "lunghezza_mediana": statistics.median(L) if L else 0,
            "bucket": {k: round(v / len(rows), 3) for k, v in c.items()} if rows else {}}


def main():
    ap = argparse.ArgumentParser(
        description=__doc__.split("\n")[1],
        formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--split-dir", default="/kaggle/working/split")
    ap.add_argument("--out-dir", default="/kaggle/working/split_igiene")
    ap.add_argument("--task", default="T3", choices=["T2", "T3"], nargs="+")
    ap.add_argument("--tipo-turno", action="store_true")
    ap.add_argument("--filtra-riscontri", action="store_true",
                    help="toglie i target <=2 parole dal TRAIN (dev/test intatti). "
                         "Alternativa a --tipo-turno, non complementare")
    ap.add_argument("--sottocampiona-duplicati", action="store_true")
    ap.add_argument("--finestre", default=None,
                    help="es. 1,2,3,4 — un item per ogni ampiezza di contesto")
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()

    tasks = a.task if isinstance(a.task, list) else [a.task]
    if a.tipo_turno and a.filtra_riscontri:
        print("! --tipo-turno e --filtra-riscontri fanno la stessa cosa in due "
              "modi opposti. Con entrambi attivi il marcatore 'riscontro' non "
              "compare mai in train ma verra' campionato in test: incoerente.")
        return

    out = Path(a.out_dir)
    src = Path(a.split_dir)
    rapporto = {"origine": str(src), "opzioni": vars(a), "task": {}}
    finestre = ([int(x) for x in a.finestre.split(",")] if a.finestre else None)
    rng_bucket = random.Random(a.seed)

    for t in tasks:
        d = out / LAYOUT_DIRS[t]
        d.mkdir(parents=True, exist_ok=True)
        rows_train = load_split(str(src), t, "train")
        dist_bucket = Counter(bucket(len(r["target"].split()))
                              for r in rows_train)
        chiavi = sorted(dist_bucket)
        pesi = [dist_bucket[k] for k in chiavi]
        info_t = {"distribuzione_bucket_train":
                  {k: round(dist_bucket[k] / len(rows_train), 3) for k in chiavi}}

        for nome in ("train", "dev", "test"):
            rows = load_split(str(src), t, nome)
            prima = statistiche(rows, f"{t}/{nome} prima")
            n_ctx_persi = 0

            # 1. finestre variabili: solo sul train
            if finestre and nome == "train":
                espanse = []
                for r in rows:
                    for n in finestre:
                        p = finestra(r["prompt"], n)
                        if p is None:
                            continue
                        q = dict(r)
                        q["prompt"] = p
                        q["finestra_contesto"] = n
                        espanse.append(q)
                if espanse:
                    rows = espanse
                else:
                    n_ctx_persi = 1

            # 2. filtro riscontri: solo sul train
            if a.filtra_riscontri and nome == "train":
                rows = [r for r in rows if len(r["target"].split()) > 2]

            # 3. sottocampionamento dei duplicati: solo sul train
            tolti = 0
            if a.sottocampiona_duplicati and nome == "train":
                rows, tolti = sottocampiona(rows, a.seed)

            # 4. marcatore di tipo turno
            senza_marcatore = 0
            if a.tipo_turno:
                nuove = []
                for r in rows:
                    if nome == "train":
                        b = bucket(len(r["target"].split()))
                    else:
                        # dev/test: il target non si guarda. Bucket campionato
                        # dalla distribuzione del TRAIN, seme fisso.
                        b = rng_bucket.choices(chiavi, weights=pesi, k=1)[0]
                    p = inserisci_marcatore(r["prompt"], b)
                    if p is None:
                        senza_marcatore += 1
                        nuove.append(r)
                        continue
                    q = dict(r)
                    q["prompt"] = p
                    q["tipo_turno"] = b
                    nuove.append(q)
                rows = nuove

           
            import re as _re
            _marca = _re.compile(r"^Tipo di turno: .*\.$")
            def _senza(p):
                return "\n".join(x for x in p.split("\n") if not _marca.match(x))
            campione = rows[:min(32, len(rows))]
            rotti_ctx = sum(1 for r in campione
                            if estrai_blocco_contesto(r["prompt"]) is None
                            and estrai_blocco_contesto(_senza(r["prompt"])) is not None)
            rotti_rm = sum(1 for r in campione
                           if rimuovi_contesto(r["prompt"]) is None
                           and rimuovi_contesto(_senza(r["prompt"])) is not None)
            if rotti_ctx or rotti_rm:
                print(f"ERRORE su {t}/{nome}: il marcatore rompe il parsing "
                      f"({rotti_ctx}/{len(campione)} estrai_blocco_contesto, "
                      f"{rotti_rm}/{len(campione)} rimuovi_contesto).")
                print("  contesto_metrica.py e decodifica_contestuale.py "
                      "darebbero numeri sbagliati. Niente scritto.")
                return

            json.dump(rows, open(d / f"{nome}.json", "w", encoding="utf-8"),
                      ensure_ascii=False, indent=1)
            info_t[nome] = {"prima": prima,
                            "dopo": statistiche(rows, f"{t}/{nome} dopo"),
                            "duplicati_tolti": tolti,
                            "prompt_senza_marcatore": senza_marcatore,
                            "contesto_non_estraibile": n_ctx_persi}
            print(f"  {t}/{nome}: {prima['n']} -> {len(rows)}"
                  + (f"  (-{tolti} duplicati)" if tolti else "")
                  + (f"  ! {senza_marcatore} senza marcatore"
                     if senza_marcatore else ""))
        rapporto["task"][t] = info_t

    # A2 e gli altri layout vanno copiati intatti, altrimenti lo split nuovo e'
    # incompleto e i run che li usano fallirebbero con un errore oscuro.
    for k, sub in LAYOUT_DIRS.items():
        if k in tasks:
            continue
        s = src / sub
        if s.exists():
            shutil.copytree(s, out / sub, dirs_exist_ok=True)
            print(f"  {k}: copiato intatto")

    (out / "igiene.json").write_text(
        json.dumps(rapporto, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nScritto {out}")
    print(f"Rapporto: {out}/igiene.json")
    print(f"Usalo con --split-dir {out} negli script di finetune E in "
          f"evaluate_task.py: i prompt devono essere gli stessi da entrambe "
          f"le parti.")


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/igiene_split.py


In [47]:
%%writefile /kaggle/working/training/arricchisci_split.py
#!/usr/bin/env python3
"""
arricchisci_split.py — aggiunge INFORMAZIONE ai prompt, e costruisce il mix
multi-task. Nessun dato nuovo.

Il punto di partenza
--------------------
`diagnosi_cecita.py` mostra che nel training di T3 l'istruzione e' la STESSA
stringa in tutti i 748 esempi. Una feature costante ha informazione mutua zero
col target: il modello non puo' usarla per abbassare la loss, quindi impara a
ignorarla. Dopo l'SFT l'istruzione non e' piu' letta come un'istruzione, e'
diventata un DELIMITATORE - un pattern che dice "qui inizia il pezzo dove devo
produrre testo tipo-parlato-trascritto".

Da cui la conseguenza che va detta chiaramente: riscrivere meglio l'istruzione
NON serve, perche' il modello non la sta leggendo. Prima va resa informativa.

Le quattro modifiche (--variazioni, --parlanti, --ancora, e igiene_split
--finestre per la quarta) aggiungono informazione che ora NON c'e'. Non sono
riformulazioni.

1. --variazioni  L'istruzione viene campionata fra N parafrasi. Smette di essere
                 costante, quindi smette di essere ignorabile: il modello deve
                 leggere il prompt per capire cosa produrre. In DEV e TEST si usa
                 SEMPRE la parafrasi canonica (indice 0), non una campionata: il
                 training vede varieta', la valutazione vede una forma fissa,
                 cosi' due arm restano confrontabili e nessun run e' fortunato.

2. --parlanti    Etichetta del parlante sui turni di contesto + istruzione che
                 nomina il destinatario con lo STESSO identificativo. Senza,
                 il modello non puo' sapere che i turni si alternano ne' a chi
                 tocca. Il CSV ha la colonna `speaker`: l'informazione esiste e
                 non e' usata. Se i turni sono gia' etichettati nello split,
                 l'opzione aggiunge solo il riferimento nell'istruzione.

3. --ancora      Una riga con l'argomento del segmento, ricavata via TF-IDF dalle
                 parole di contenuto piu' distintive dei turni attorno. Attacca
                 direttamente il "non sa di cosa si sta parlando". NON e' dato
                 nuovo: e' informazione derivata dai turni che hai, resa
                 esplicita invece di lasciata implicita in ~21 parole di parlato
                 frammentario.

4. --multitask   Scrive un layout "MT" con T1+T2+T3 MESCOLATI, per addestrare UN
                 SOLO adapter invece di tre.

Perche' il multi-task e' la modifica strutturale
------------------------------------------------
Ora addestri tre adapter separati. Ognuno vede una sola istruzione, sempre
identica -> costante -> ignorata. Mescolando i tre task, l'istruzione diventa
l'UNICA cosa che li distingue: il modello DEVE leggerla, per necessita' e non
per esortazione.

Due effetti che si sommano:

    istanze viste dall'adapter che poi usi su T3
        ora            748
        multi-task     1134 + 683 + 748 = 2565

E il secondo, che conta di piu': T1 fa da ANCORA SEMANTICA. La traduzione e'
uno-a-uno e obbliga a una mappatura che preserva il significato - non puoi
tradurre bene ignorando cosa dice la frase. Addestrando insieme, quel vincolo
tiene il modello agganciato al contenuto anche su T3, dove la loss da sola non
lo richiede. E' l'opposto di quello che succede ora, dove l'SFT su T3 isolato
spinge verso la superficie del parlato.

Richiede una riga in common.py: LAYOUT_DIRS["MT"] = "multitask". La cella del
notebook la applica.

Uso
---
    python arricchisci_split.py --split-dir /kaggle/working/split \\
        --out-dir /kaggle/working/split_arr \\
        --variazioni --parlanti --ancora --multitask

    # compone con igiene_split.py: prima igiene, poi arricchimento
    python igiene_split.py --split-dir SPLIT --out-dir /kaggle/working/split_ig \\
        --task T3 T2 --tipo-turno --sottocampiona-duplicati --finestre 1,2,3,4,5
    python arricchisci_split.py --split-dir /kaggle/working/split_ig \\
        --out-dir /kaggle/working/split_arr --variazioni --parlanti --ancora \\
        --multitask

ATTENZIONE: i prompt devono essere BYTE-IDENTICI fra training e inferenza.
Qualunque cartella usi per addestrare, usa la stessa in evaluate_task.py,
rerank_t3.py, finetune_t3_dpo.py e pipeline_due_stadi.py.
"""

import argparse
import json
import random
import re
import shutil
import statistics
from collections import Counter
from pathlib import Path

from common import LAYOUT_DIRS, load_split
from contesto_metrica import _e_separatore, estrai_blocco_contesto

# Parafrasi dell'istruzione, per layout. L'indice 0 e' la CANONICA: e' quella
# usata in dev/test, quindi deve restare la formulazione piu' neutra.
# Le altre non sono sinonimi decorativi: variano il modo in cui il compito e'
# descritto (imperativo, ruolo, obiettivo), cosi' il modello impara a mappare
# INTENTI diversi sullo stesso comportamento invece di riconoscere una stringa.
VARIAZIONI = {
    "T1": [
        "Traduci in napoletano la frase italiana.",
        "Rendi in napoletano quello che segue, restando fedele al significato.",
        "Qual e' la versione napoletana di questa frase italiana?",
        "Riscrivi la frase in napoletano senza cambiare cio' che dice.",
        "Volgi in napoletano, mantenendo il senso e il registro parlato.",
        "Da italiano a napoletano: traduci.",
        "Come si dice in napoletano?",
        "Trasponi in napoletano il contenuto della frase italiana.",
    ],
    "T2": [
        "Completa il turno.",
        "Continua e concludi il turno iniziato, restando coerente col discorso.",
        "Come finisce questo turno?",
        "Porta a termine la frase del parlante in napoletano.",
        "Il turno e' interrotto: completalo in modo coerente col contesto.",
        "Prosegui il turno fino alla fine.",
        "Scrivi la parte che manca del turno.",
        "Termina l'enunciato cominciato sopra.",
    ],
    "T3": [
        "Rispondi come il parlante successivo.",
        "Cosa dice adesso l'altro parlante? Rispondi nel merito di quello che e' "
        "stato detto.",
        "Tocca all'altro parlante: scrivi il suo turno, pertinente al discorso.",
        "Prosegui la conversazione con il turno successivo, in napoletano.",
        "Sei il parlante successivo: replica a quello che hai appena sentito.",
        "Scrivi la replica che viene ora, coerente con l'argomento della "
        "conversazione.",
        "Che cosa risponde l'interlocutore?",
        "Continua il dialogo: il prossimo turno tocca a te.",
    ],
}

# Parole troppo frequenti nel parlato per essere distintive di un argomento.
# Non e' una stoplist italiana generica: e' tarata sui riscontri e sui
# segnalidiscorsivi che dominano questo corpus (mh 95, si' 71, mhmh 58, no 32).
STOP = set("""
e ma pe' po' ca che chi cu nun no si' se mh mhmh eh ah oh uh ehm mm cioe' pero'
allora quindi insomma tipo ecco vabbuo' okay ok gia' pure comme quanno quando
dove chesta chesto chella chello chisto chillo stu sta 'o 'a 'e 'nu 'na nu na
me te ce ve se ne ci vi mi ti lo la li le gli il un una uno dei delle del della
a ai al alla da dai dal dalla di in con su per tra fra non piu' meno molto poco
tutto tutta tutti tutte cosa cose fatto fatta essere stato stata avere aveva
sono era erano ho hai ha abbiamo avete hanno faccio fa fai famo facimmo
grazie prego scusa senti sai vedi guarda dimmi niente nulla qualcosa
""".split())

MIN_LEN_PAROLA = 4


def normalizza(s):
    return re.sub(r"[^\w'\s]", " ", (s or "").lower())


def parole_contenuto(testo):
    return [w for w in normalizza(testo).split()
            if len(w) >= MIN_LEN_PAROLA and w not in STOP and not w.isdigit()]


def costruisci_ancore(rows, k=4, finestra=8):
    """Per ogni item: le k parole piu' distintive del suo intorno.

    TF-IDF fatto a mano invece che con sklearn: il documento e' l'intorno di un
    item, il corpus sono tutti gli intorni. Cosi' l'ancora non contiene le parole
    frequenti in TUTTA la conversazione (che non distinguono un segmento
    dall'altro) ma quelle caratteristiche di QUESTO punto.

    L'intorno include il contesto e i turni vicini, MAI il target: altrimenti
    l'ancora conterrebbe le parole della risposta e sarebbe leakage - il modello
    imparerebbe a copiarle e i numeri sarebbero gonfiati senza che nulla di
    reale sia migliorato.
    """
    import math
    intorni = []
    for i, r in enumerate(rows):
        trovato = estrai_blocco_contesto(r["prompt"])
        turni = trovato[2] if trovato else []
        vicini = []
        for j in range(max(0, i - finestra // 2), min(len(rows), i + finestra // 2)):
            t = estrai_blocco_contesto(rows[j]["prompt"])
            if t:
                vicini += t[2]
        intorni.append(parole_contenuto(" ".join(turni + vicini)))

    df = Counter()
    for doc in intorni:
        df.update(set(doc))
    n_doc = max(1, len(intorni))

    ancore = []
    for doc in intorni:
        tf = Counter(doc)
        punteggi = {w: c * math.log(n_doc / (1 + df[w])) for w, c in tf.items()}
        top = [w for w, _ in sorted(punteggi.items(), key=lambda x: -x[1])[:k]]
        ancore.append(top)
    return ancore


def scomponi(prompt):
    """(intestazione+turni, separatori, istruzione) come liste di righe."""
    trovato = estrai_blocco_contesto(prompt)
    if trovato is None:
        return None
    i0, i1, _ = trovato
    righe = prompt.split("\n")
    k = i1
    while k < len(righe) and _e_separatore(righe[k]):
        k += 1
    return righe[:i1], righe[i1:k], righe[k:]


def etichette_presenti(turni):
    n = 0
    for t in turni:
        p = t.strip().split(":", 1)
        if len(p) == 2 and 0 < len(p[0]) <= 12:
            n += 1
    return n / len(turni) if turni else 0.0


def etichetta_turni(turni, nomi=("A", "B")):
    """Etichetta alternata. Non e' la vera identita' dei parlanti - quella sta
    nella colonna `speaker` del CSV e andrebbe propagata in split_dataset.py -
    ma rende esplicita l'ALTERNANZA, che e' l'informazione di cui il modello ha
    bisogno per sapere a chi tocca. Riportalo come approssimazione."""
    return [f"{nomi[i % len(nomi)]}: {t.strip()}" for i, t in enumerate(turni)]


def prossimo_parlante(turni, nomi=("A", "B")):
    return nomi[len(turni) % len(nomi)]


def riscrivi(rows, layout, split, a, rng, ancore=None):
    fuori = []
    n_var, n_lab, n_anc, n_salti = 0, 0, 0, 0
    for i, r in enumerate(rows):
        sc = scomponi(r["prompt"])
        if sc is None:
            n_salti += 1
            fuori.append(dict(r))
            continue
        testa, sep, istr = sc
        trovato = estrai_blocco_contesto(r["prompt"])
        turni = trovato[2]
        intest = testa[:len(testa) - len(turni)]
        q = dict(r)

        # 1. etichette dei parlanti
        if a.parlanti:
            if etichette_presenti(turni) < 0.9:
                turni = etichetta_turni(turni)
                n_lab += 1
            q["parlante_target"] = prossimo_parlante(turni)

        # 2. istruzione: variata (train) o canonica (dev/test)
        if a.variazioni:
            varianti = VARIAZIONI.get(layout) or []
            if varianti:
                testo = varianti[0] if split != "train" else rng.choice(varianti)
                if split == "train":
                    n_var += 1
                if a.parlanti and layout == "T3":
                    testo += f" Rispondi a {turni[-1].split(':')[0].strip()}."
                # L'istruzione va ULTIMA, adiacente al punto di generazione:
                # le altre righe (ancora, tipo di turno di igiene_split) sono
                # contesto, non comando. Le righe preesistenti che sono la
                # vecchia istruzione vengono rimosse, non duplicate.
                resto = [x for x in istr if x.strip()
                         and not _somiglia_istruzione(x, varianti)]
                istr = resto + [testo]
                q["variante_istruzione"] = testo

        # 3. ancora tematica: prima di tutto il resto dell'istruzione
        if a.ancora and ancore and ancore[i]:
            istr = [f"Si sta parlando di: {', '.join(ancore[i])}."] + istr
            q["ancora"] = ancore[i]
            n_anc += 1

        q["prompt"] = "\n".join(intest + turni + sep + istr)
        q["layout"] = layout
        fuori.append(q)
    return fuori, {"istruzioni_variate": n_var, "turni_etichettati": n_lab,
                   "ancore_aggiunte": n_anc, "prompt_non_scomponibili": n_salti}


def _somiglia_istruzione(riga, varianti):
    """Riconosce la vecchia istruzione per non lasciarne due nel prompt.

    Confronto sulle prime parole di contenuto: le formulazioni originali
    ("Rispondi come il parlante successivo.") e le parafrasi condividono il
    verbo iniziale, e un match esatto non basterebbe.
    """
    r = normalizza(riga).split()
    if not r:
        return False
    teste = {normalizza(v).split()[0] for v in varianti if v.strip()}
    return r[0] in teste


def statistiche(rows, etichetta):
    n_istr = Counter()
    for r in rows:
        sc = scomponi(r["prompt"])
        if sc:
            n_istr["\n".join(sc[2])] += 1
    L = [len(r["prompt"].split()) for r in rows]
    return {"nome": etichetta, "n": len(rows),
            "istruzioni_distinte": len(n_istr),
            "parole_prompt_medie": round(statistics.mean(L), 1) if L else 0}


def main():
    ap = argparse.ArgumentParser(
        description="Aggiunge informazione ai prompt + mix multi-task",
        formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--split-dir", default="/kaggle/working/split")
    ap.add_argument("--out-dir", default="/kaggle/working/split_arr")
    ap.add_argument("--task", default=["T1", "T2", "T3"], nargs="+",
                    choices=["T1", "T2", "T3"])
    ap.add_argument("--variazioni", action="store_true")
    ap.add_argument("--parlanti", action="store_true")
    ap.add_argument("--ancora", action="store_true")
    ap.add_argument("--ancora-k", type=int, default=4)
    ap.add_argument("--multitask", action="store_true",
                    help="scrive anche il layout MT con i task mescolati")
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()

    tasks = a.task if isinstance(a.task, list) else [a.task]
    if not (a.variazioni or a.parlanti or a.ancora or a.multitask):
        print("Nessuna modifica richiesta. Usa almeno una fra --variazioni, "
              "--parlanti, --ancora, --multitask.")
        return

    src, out = Path(a.split_dir), Path(a.out_dir)
    rng = random.Random(a.seed)
    rapporto = {"origine": str(src), "opzioni": vars(a), "task": {}}
    mix = {"train": [], "dev": [], "test": []}

    for t in tasks:
        d = out / LAYOUT_DIRS[t]
        d.mkdir(parents=True, exist_ok=True)
        rapporto["task"][t] = {}
        for split in ("train", "dev", "test"):
            rows = load_split(str(src), t, split)
            ancore = (costruisci_ancore(rows, a.ancora_k) if a.ancora else None)
            nuove, info = riscrivi(rows, t, split, a, rng, ancore)
            json.dump(nuove, open(d / f"{split}.json", "w", encoding="utf-8"),
                      ensure_ascii=False, indent=1)
            rapporto["task"][t][split] = {
                "prima": statistiche(rows, f"{t}/{split} prima"),
                "dopo": statistiche(nuove, f"{t}/{split} dopo"), **info}
            print(f"  {t}/{split}: {len(rows)} item | istruzioni distinte "
                  f"{rapporto['task'][t][split]['prima']['istruzioni_distinte']}"
                  f" -> {rapporto['task'][t][split]['dopo']['istruzioni_distinte']}"
                  + (f" | {info['ancore_aggiunte']} ancore" if a.ancora else "")
                  + (f" | ! {info['prompt_non_scomponibili']} non scomponibili"
                     if info["prompt_non_scomponibili"] else ""))
            if a.multitask:
                for r in nuove:
                    q = dict(r)
                    q["task_origine"] = t
                    q["layout"] = "MT"      # load_split filtra su questo campo
                    mix[split].append(q)

    if a.multitask:
        dm = out / "multitask"
        dm.mkdir(parents=True, exist_ok=True)
        for split in ("train", "dev", "test"):
            rng.shuffle(mix[split])
            json.dump(mix[split], open(dm / f"{split}.json", "w", encoding="utf-8"),
                      ensure_ascii=False, indent=1)
        comp = {s: dict(Counter(r["task_origine"] for r in mix[s]))
                for s in mix}
        rapporto["multitask"] = {"n": {s: len(v) for s, v in mix.items()},
                                 "composizione": comp}
        print(f"\n  MT: train {len(mix['train'])} | dev {len(mix['dev'])} | "
              f"test {len(mix['test'])}")
        print(f"      composizione train: {comp['train']}")
        print("      ricorda la riga in common.py: "
              'LAYOUT_DIRS["MT"] = "multitask"')

    # gli altri layout (A2) vanno copiati intatti
    for k, sub in LAYOUT_DIRS.items():
        if k in tasks or k == "MT":
            continue
        s = src / sub
        if s.exists():
            shutil.copytree(s, out / sub, dirs_exist_ok=True)
            print(f"  {k}: copiato intatto")

    (out / "arricchimento.json").write_text(
        json.dumps(rapporto, ensure_ascii=False, indent=2), encoding="utf-8")

    # un prompt di esempio per far vedere il risultato
    esempio = load_split(str(out), tasks[-1], "train")[0]
    print(f"\n  --- un prompt di {tasks[-1]}/train dopo l'arricchimento ---")
    for riga in esempio["prompt"].split("\n"):
        print(f"  | {riga}")
    print(f"  | TARGET: {esempio['target']}")

    print(f"\nScritto {out}")
    print(f"Usa --split-dir {out} in TUTTI gli script, training e valutazione: "
          f"i prompt devono essere byte-identici da entrambe le parti.")


if __name__ == "__main__":
    main()


Writing /kaggle/working/training/arricchisci_split.py


In [48]:
# --- Ricostruisci lo split arricchito (deve combaciare con quello del notebook T3)
# Deterministico a parita' di SEED e di flag: NON modificare i due comandi.
# Nomi di cartella diversi da quelli del Passo 31c: i percorsi non entrano nei
# prompt, quindi non devono combaciare con quelli del notebook T3 — devono
# combaciare i FLAG e il SEED, che sono gli stessi qui sotto.
SPLIT_IG_MT  = "/kaggle/working/split_igiene_mt"
SPLIT_ARR = "/kaggle/working/split_arr"

!python igiene_split.py --split-dir {SPLIT} --out-dir {SPLIT_IG_MT} \
    --task T3 T2 --tipo-turno --sottocampiona-duplicati --finestre 1,2,3,4,5 \
    --seed {SEED}

!python arricchisci_split.py --split-dir {SPLIT_IG_MT} --out-dir {SPLIT_ARR} \
    --task T1 T2 T3 --variazioni --parlanti --ancora --multitask --seed {SEED}

  T3/train: 748 -> 1482  (-756 duplicati)
  T3/dev: 187 -> 187
  T3/test: 192 -> 192
  T2/train: 683 -> 1363  (-681 duplicati)
  T2/dev: 142 -> 142  ! 1 senza marcatore
  T2/test: 147 -> 147  ! 2 senza marcatore
  T1: copiato intatto
  A2: copiato intatto

Scritto /kaggle/working/split_igiene_mt
Rapporto: /kaggle/working/split_igiene_mt/igiene.json
Usalo con --split-dir /kaggle/working/split_igiene_mt negli script di finetune E in evaluate_task.py: i prompt devono essere gli stessi da entrambe le parti.
  T1/train: 1134 item | istruzioni distinte 1120 -> 1121 | 1133 ancore | ! 1 non scomponibili
  T1/dev: 270 item | istruzioni distinte 266 -> 244 | 268 ancore | ! 2 non scomponibili
  T1/test: 267 item | istruzioni distinte 263 -> 236 | 265 ancore | ! 2 non scomponibili
  T2/train: 1363 item | istruzioni distinte 681 -> 1357 | 1363 ancore
  T2/dev: 142 item | istruzioni distinte 141 -> 134 | 141 ancore | ! 1 non scomponibili
  T2/test: 147 item | istruzioni distinte 145 -> 143 | 145 anc

In [49]:
# --- Valuta l'adapter MT su T1 ------------------------------------------------
# ATTENZIONE: --split-dir SPLIT_ARR anche qui. I prompt devono essere
# byte-identici fra training e inferenza.
MT_ADAPTER = ""   # percorso dell'adapter *__MT prodotto dal notebook T3

if not MT_ADAPTER:
    print("MT_ADAPTER vuoto: sezione saltata.")
else:
    !python evaluate_task.py --model {MODEL} --task {TASK} --split-dir {SPLIT_ARR} \
        --adapter {MT_ADAPTER} --split test --out /kaggle/working/eval

MT_ADAPTER vuoto: sezione saltata.


## Salvare gli adapter

`/kaggle/working` viene azzerato alla fine della sessione. *Save Version > Save &
Run All* persiste l'output: da lì gli adapter sono recuperabili e riutilizzabili
come Dataset di input per la valutazione finale.

Per riprendere un run interrotto: allega quell'output come Dataset e rilancia con
`--resume-dir /kaggle/input/<nome-output>`. Lo script copia i `checkpoint-*` in
working e riparte con ottimizzatore, scheduler e `log_history` ripristinati.

## Se Gemma-3 dà problemi

`gemma-3-4b-it` è multimodale (`Gemma3ForConditionalGeneration`). `common.py`
prova `AutoModelForCausalLM` e ripiega su `AutoModelForImageTextToText`;
`summary.json` registra quale in `architettura_caricata`, e all'avvio viene
stampato quanti moduli del vision tower sono stati esclusi dagli adapter.

Se compare `loss=nan`: `--lr 5e-5`, poi `--max-seq-len 384`, poi
`--model gemma-tiny` (gemma-3-1b-it) dichiarando la sostituzione.

Nota sulle taglie: 7B (Llama-2) / 7B (Minerva) / 4B (Gemma-3). L'asimmetria va
dichiarata: un punteggio più basso di Gemma non è di per sé un limite del modello.

## Le metriche di questo notebook non sono i risultati

Tutto ciò che viene stampato durante il training è calcolato in **teacher
forcing** (argmax dei logit) e serve per il monitoraggio e l'early stopping.

I numeri da riportare vengono da una valutazione separata su `test.json` con
`generate()` reale e decoding deterministico. E per T2/T3, dove il riferimento è
uno fra molti validi, il chrF va accompagnato da metriche che non presuppongono
una risposta unica: discriminazione avversariale umano-vs-macchina (controllo
umano-vs-umano = 0,52 ± 0,04), ranking contrastivo con distrattori dal corpus
parallelo, dialettalità `P(nap)` dal classificatore ita/nap, e confronto
distribuzionale (distinct-n, rep-n, KS sulle lunghezze) contro i valori umani:
distinct-2 = 0,758, rep-3 = 0,016, lunghezza mediana 6.

## Riepilogo del disegno sperimentale

Cosa hai in mano quando i 15 run sono finiti, per ciascuno dei tre modelli
(Minerva-7B = italiano, Llama-2-7b = multilingua, Gemma-3-4b = piu' piccolo):

| run | stadio | riusato da | nel confronto? |
|---|---|---|---|
| 1 | A — continued pretraining | T1, T2, T3 | no (perplexity di monitoraggio) |
| 2 | A2 — iniezione lessicale | T1, T2, T3 | no (controllo che abbia preso) |
| 3-5 | B — T1, T2, T3 | — | **si** |

Piu' i tre run Llama vecchi come arm "solo stadio B", che danno l'ablation sul
curriculum senza costare GPU.

Se vuoi anche l'ablation sulla **pesatura** (e non solo sul curriculum), il modo
economico e' un quarto run su un solo modello: `finetune_t1_traduzione.py` con
`{INIT2}` ma **senza** `--lessico`. Isola la pesatura tenendo fisso il
curriculum, e costa 1 run invece di 3.

## Cosa riportare, e cosa no

Tutto ciò che viene stampato durante il training è in **teacher forcing** e
serve al monitoraggio e all'early stopping. I numeri del paper vengono da
`evaluate_task.py` su `test.json` con `generate()` reale.

Per la domanda "ha imparato il dialetto?" il chrF da solo non basta: va
accompagnato dal **recall dialettale** e dal **tasso di italianismi**, che hanno
fondo scala (copia-italiano) e tetto (riferimento umano) noti dal passo 22. Un
modello che alza il chrF lasciando il tasso di italianismi vicino a 1,00 ha
imparato il compito e non la varieta'.

## Nota sui dati

`dataset_finale.csv` ha `regione = emilia-romagna` e `languages = italian` su
tutte le 2.568 righe, e come varieta' target solo il napoletano. Il confronto
cross-varieta', se serve, deve arrivare dai dati degli altri gruppi: da qui non
si ricava.
## Arm di T3 dopo il Passo 30

Tutti sullo stesso adapter: nessuno di questi richiede di riaddestrare.

| arm | cosa cambia | costo |
|---|---|---|
| `T3__ft__nucleus` (v1) | — | già fatto, **non confrontabile** (fine turno rotto) |
| `T3__ft__nucleus` (v2) | fine turno + rep. penalty | ~10 min |
| `T3__ft__typical` | locally typical, T 0,7 | ~10 min |
| `T3__ft__typical_bon12_mbr` | best-of-12, consenso chrF | ~10 min |
| `T3__ft__typical_bon12_composito` | best-of-12, d_ctx + d_nap + d_len | ~15 min |
| `T3__ft__typical_bon12_ibrido` | i due combinati | incluso sopra |
| `T3__2stadi` | composizione via adapter T1 | già previsto al Passo 29 |
| `T3__cfg_gamma*` | `decodifica_contestuale.py`, sweep su γ | ~15 min per γ |

L'ultimo non ha ancora una cella: `decodifica_contestuale.py` è scritto e mai
usato, e la contrastiva a 0,33 contro 0,20 casuale dice che c'è segnale di
contesto da amplificare. Sweep di γ ∈ {0,25; 0,5; 1,0; 1,5} sul dev, selezione
sulla contrastiva, e riporta la **curva**: γ alto alza la pertinenza ma peggiora
la fluenza, il punto migliore da solo non è un risultato.

## Il tetto vero, da scrivere nei limiti

`dataset_finale.csv` ha **2 conversazioni**, 1.102 e 1.466 turni. `regione =
emilia-romagna`, `macro_regione = nord`, `languages = italian`: la fonte è
parlato italiano settentrionale tradotto in napoletano (2.053 righe `golden` +
515 `gemma4`). Quindi la pragmatica dei riferimenti di T3 è pragmatica di parlato
italiano del nord, non di conversazione napoletana nativa: T3 misura "produrre il
turno successivo di una conversazione italiana resa in napoletano", che non è la
stessa cosa.

Con 2 conversazioni **lo split non può essere per conversazione**. Verifica che
`split_dataset.py` divida per blocchi contigui: con uno split random a livello di
turno i 3 turni di contesto di un item di test sono target di training, e va
dichiarato.

Nessuno degli arm sopra sposta questo tetto. Le due strade che lo spostano:

1. **Silver data via T1.** T1 fa 70,80 chrF++ e batte il dizionario oracolo
   (68,46): usalo per tradurre un corpus di dialogo italiano molto più grande e
   addestra T3 su quello. Distillazione standard, porta T3 da ~750 a 10-50k
   istanze, e diventa un arm di ablation pulito (gold contro gold+silver).
2. **Token di controllo del tipo di turno.** Nel prompt di T3, un marcatore
   derivato dalla lunghezza del target: `riscontro` (≤2 parole, 33,7% del
   corpus), `breve` (≤6), `medio` (≤12), `lungo`. In inferenza si campiona il
   bucket dalla distribuzione di **train**. Rende la distribuzione delle
   lunghezze compatibile con l'umano per costruzione e fa smettere il modello di
   produrre 17 parole dove l'umano ha detto `mh`; soprattutto fa crollare
   `controllo_solo_lunghezza`, che è ciò che finalmente rende leggibile
   `accuracy_umano_vs_macchina` come misura di qualità dialettale invece che di
   lunghezza. Controllable generation standard (Kikuchi et al. 2016; Fan et al.
   2018). Richiede di rigenerare lo split e un run T3 di ~1-1,5 h.


## Coordinamento fra i tre notebook

Cosa produce questo notebook e chi lo usa:

| artefatto | percorso | serve a |
|---|---|---|
| adapter stadio A2 | `/kaggle/working/runs/*__A2/adapter_final` | `ADAPTER_A2_ESTERNO` in T2 e T3 |
| adapter T1 | `/kaggle/working/runs/*__T1/adapter_final` | `ADAPTER_T1_ESTERNO` in T2 e T3 (pipeline a due stadi) |
| lessico allineato | `/kaggle/working/lessico_train.json` | si rigenera in pochi secondi, non serve trasportarlo |

`/kaggle/working` viene azzerato a fine sessione: *Save Version > Save & Run All*
e' cio' che rende questi percorsi riutilizzabili come Dataset di input negli
altri notebook. Con la quota di 30 h GPU/settimana e il tetto di 12 h per
sessione, una sessione per notebook e' la divisione naturale.

Per la tabella comparativa finale fra i tre task: allega gli output dei tre
notebook a una sessione nuova, copia i `.preds.jsonl` in un'unica cartella e
lancia `metriche_finali.py` una volta sola su quella.